In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Inital Split

In [ ]:
import pickle

with open("siamese isic_split.pkl", "rb") as f:
    data = pickle.load(f)

(x1, x2), y = data
print("✅ Number of pairs:", len(y))
print("Shape of x1:", len(x1), "Shape of x2:", len(x2))


In [ ]:
# ✅ Number of pairs: 140
# Shape of x1: 140 Shape of x2: 140

Only has : Number of pairs: 140
Shape of x1: 140 Shape of x2: 140 will return high training accuracy because of small number of pairs i think the network isnt able to find pairs for postive since negative pair are in abundance

Splitting data

Better Criteria for splitting data

In [ ]:

import pickle

with open("/content/drive/MyDrive/siam2/siamese_isic_split5000.pkl", "rb") as f:
    (x1, x2), y = pickle.load(f)

print("✅ Loaded successfully!")
print("Pairs:", len(y))
print("x1 shape:", x1.shape)
print("x2 shape:", x2.shape)
print("Example labels:", y[:10])

In [ ]:
# ✅ Loaded successfully!
# Pairs: 5000
# x1 shape: (5000, 105, 105, 3)
# x2 shape: (5000, 105, 105, 3)
# Example labels: [1 1 1 1 0 1 0 1 1 1]

Training and validation data is mixed together causing very high accuracy

To run training

In [ ]:
!python3 train_siam.py


In [ ]:
# !ls -lh /content/drive/MyDrive/siam2


In [ ]:
# total 91M
# -rw------- 1 root root 2.6K Oct 30 05:46 dataset_siam.py
# -rw------- 1 root root 2.0M Oct 30 05:57 Groundtruth.csv
# -rw------- 1 root root 1.2K Oct 30 05:46 model_siam.py
# -rw------- 1 root root 1.1K Oct 30 05:46 plot_classification.py
# -rw------- 1 root root 1.6K Oct 30 05:46 plot_perform.py
# -rw------- 1 root root  736 Oct 30 05:46 predict_siam.py
# drwx------ 2 root root 4.0K Oct 30 06:00 __pycache__
# -rw------- 1 root root    6 Oct 30 05:46 README.md
# -rw------- 1 root root  45M Oct 30 05:57 siamese_isic.pkl
# -rw------- 1 root root    0 Oct 30 06:07 siamese_isic_split.pkl
# -rw------- 1 root root  45M Oct 30 05:57 train_pairs.pkl
# -rw------- 1 root root 3.3K Oct 30 06:04 train_siam.py

In [ ]:
# !mv "siamese isic_split.pkl" siamese_isic_split.pkl


In [ ]:
# !ls -lh siamese_isic_split.pkl


In [ ]:
# -rw------- 1 root root 36M Oct 30 06:11 siamese_isic_split.pkl

In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import os
from model_siam import SiameseNetwork

# --- paths ---
img_dir = "/content/drive/MyDrive/siam2/test"
model_path = "/content/drive/MyDrive/siam2/siamese_best.pth"

# --- model + preprocessing ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SiameseNetwork().to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

def compute_similarity(img1_path, img2_path):
    img1 = transform(Image.open(img1_path).convert("RGB")).unsqueeze(0).to(device)
    img2 = transform(Image.open(img2_path).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        score, _, _ = model(img1, img2)
        return torch.sigmoid(score).item()

# --- compare reference image to all others ---
img_names = sorted(os.listdir(img_dir))
ref = img_names[0]
ref_path = os.path.join(img_dir, ref)

results = []
for other in img_names[1:]:
    sim = compute_similarity(ref_path, os.path.join(img_dir, other))
    results.append((ref, other, sim))
    print(f"{ref} vs {other}: {sim:.4f}")

# Optional: save to CSV
import pandas as pd
df = pd.DataFrame(results, columns=["Image1", "Image2", "Similarity"])
df.to_csv("/content/drive/MyDrive/siam2/similarity_results_reference.csv", index=False)
print("✅ Saved to similarity_results_reference.csv")


In [ ]:
# Streaming output truncated to the last 5000 lines.
# ISIC_1510820.jpg vs ISIC_2423944.jpg: 0.7713
# ISIC_1510820.jpg vs ISIC_2425720.jpg: 0.8551
# ISIC_1510820.jpg vs ISIC_2426423.jpg: 0.7071
# ISIC_1510820.jpg vs ISIC_2427838.jpg: 0.5122
# ISIC_1510820.jpg vs ISIC_2428361.jpg: 0.2864
# ISIC_1510820.jpg vs ISIC_2428652.jpg: 0.8231
# ISIC_1510820.jpg vs ISIC_2428691.jpg: 0.8025
# ISIC_1510820.jpg vs ISIC_2430330.jpg: 0.6310
# ISIC_1510820.jpg vs ISIC_2431958.jpg: 0.4240
ISIC_1510820.jpg vs ISIC_2433578.jpg: 0.1855
ISIC_1510820.jpg vs ISIC_2434912.jpg: 0.7922
ISIC_1510820.jpg vs ISIC_2435805.jpg: 0.7450
ISIC_1510820.jpg vs ISIC_2437222.jpg: 0.7499
ISIC_1510820.jpg vs ISIC_2438069.jpg: 0.6194
ISIC_1510820.jpg vs ISIC_2438481.jpg: 0.4285
ISIC_1510820.jpg vs ISIC_2439227.jpg: 0.5573
ISIC_1510820.jpg vs ISIC_2447694.jpg: 0.5569
ISIC_1510820.jpg vs ISIC_2447802.jpg: 0.8185
ISIC_1510820.jpg vs ISIC_2453607.jpg: 0.7824
ISIC_1510820.jpg vs ISIC_2454187.jpg: 0.6428
ISIC_1510820.jpg vs ISIC_2455435.jpg: 0.8338
ISIC_1510820.jpg vs ISIC_2457108.jpg: 0.7734
ISIC_1510820.jpg vs ISIC_2457967.jpg: 0.5830
ISIC_1510820.jpg vs ISIC_2458215.jpg: 0.8107
ISIC_1510820.jpg vs ISIC_2460791.jpg: 0.4282
ISIC_1510820.jpg vs ISIC_2461537.jpg: 0.7295
ISIC_1510820.jpg vs ISIC_2461994.jpg: 0.6901
ISIC_1510820.jpg vs ISIC_2462762.jpg: 0.6420
ISIC_1510820.jpg vs ISIC_2463217.jpg: 0.6738
ISIC_1510820.jpg vs ISIC_2463262.jpg: 0.4221
ISIC_1510820.jpg vs ISIC_2464369.jpg: 0.6253
ISIC_1510820.jpg vs ISIC_2466179.jpg: 0.8388
ISIC_1510820.jpg vs ISIC_2466641.jpg: 0.7779
ISIC_1510820.jpg vs ISIC_2467705.jpg: 0.8762
ISIC_1510820.jpg vs ISIC_2467742.jpg: 0.8358
ISIC_1510820.jpg vs ISIC_2467985.jpg: 0.5886
ISIC_1510820.jpg vs ISIC_2468035.jpg: 0.6582
ISIC_1510820.jpg vs ISIC_2469321.jpg: 0.4881
ISIC_1510820.jpg vs ISIC_2472358.jpg: 0.7036
ISIC_1510820.jpg vs ISIC_2474708.jpg: 0.2608
ISIC_1510820.jpg vs ISIC_2476534.jpg: 0.4583
ISIC_1510820.jpg vs ISIC_2476859.jpg: 0.8219
ISIC_1510820.jpg vs ISIC_2477987.jpg: 0.8096
ISIC_1510820.jpg vs ISIC_2478371.jpg: 0.6682
ISIC_1510820.jpg vs ISIC_2478446.jpg: 0.6104
ISIC_1510820.jpg vs ISIC_2479848.jpg: 0.2803
ISIC_1510820.jpg vs ISIC_2481946.jpg: 0.3716
ISIC_1510820.jpg vs ISIC_2481958.jpg: 0.8625
ISIC_1510820.jpg vs ISIC_2483982.jpg: 0.7516
ISIC_1510820.jpg vs ISIC_2484115.jpg: 0.2755
ISIC_1510820.jpg vs ISIC_2484178.jpg: 0.4909
ISIC_1510820.jpg vs ISIC_2484203.jpg: 0.8024
ISIC_1510820.jpg vs ISIC_2485658.jpg: 0.3928
ISIC_1510820.jpg vs ISIC_2485868.jpg: 0.4765
ISIC_1510820.jpg vs ISIC_2486784.jpg: 0.4578
ISIC_1510820.jpg vs ISIC_2489071.jpg: 0.7358
ISIC_1510820.jpg vs ISIC_2490163.jpg: 0.7983
ISIC_1510820.jpg vs ISIC_2490327.jpg: 0.8606
ISIC_1510820.jpg vs ISIC_2491883.jpg: 0.5825
ISIC_1510820.jpg vs ISIC_2493881.jpg: 0.4041
ISIC_1510820.jpg vs ISIC_2495255.jpg: 0.8155
ISIC_1510820.jpg vs ISIC_2495802.jpg: 0.4667
ISIC_1510820.jpg vs ISIC_2497011.jpg: 0.6434
ISIC_1510820.jpg vs ISIC_2497422.jpg: 0.7168
ISIC_1510820.jpg vs ISIC_2498534.jpg: 0.8088
ISIC_1510820.jpg vs ISIC_2498620.jpg: 0.4347
ISIC_1510820.jpg vs ISIC_2501699.jpg: 0.5677
ISIC_1510820.jpg vs ISIC_2504419.jpg: 0.6254
ISIC_1510820.jpg vs ISIC_2505832.jpg: 0.8852
ISIC_1510820.jpg vs ISIC_2510291.jpg: 0.7592
ISIC_1510820.jpg vs ISIC_2512303.jpg: 0.1075
ISIC_1510820.jpg vs ISIC_2512550.jpg: 0.7982
ISIC_1510820.jpg vs ISIC_2513195.jpg: 0.7318
ISIC_1510820.jpg vs ISIC_2515419.jpg: 0.6302
ISIC_1510820.jpg vs ISIC_2516818.jpg: 0.8110
ISIC_1510820.jpg vs ISIC_2516903.jpg: 0.7770
ISIC_1510820.jpg vs ISIC_2518102.jpg: 0.5956
ISIC_1510820.jpg vs ISIC_2519003.jpg: 0.4401
ISIC_1510820.jpg vs ISIC_2521951.jpg: 0.3126
ISIC_1510820.jpg vs ISIC_2524304.jpg: 0.6129
ISIC_1510820.jpg vs ISIC_2525897.jpg: 0.2444
ISIC_1510820.jpg vs ISIC_2528358.jpg: 0.6793
ISIC_1510820.jpg vs ISIC_2529965.jpg: 0.6943
ISIC_1510820.jpg vs ISIC_2530380.jpg: 0.3145
ISIC_1510820.jpg vs ISIC_2530646.jpg: 0.7444
ISIC_1510820.jpg vs ISIC_2530832.jpg: 0.6486
ISIC_1510820.jpg vs ISIC_2531776.jpg: 0.7261
ISIC_1510820.jpg vs ISIC_2535069.jpg: 0.6743
ISIC_1510820.jpg vs ISIC_2535128.jpg: 0.6745
ISIC_1510820.jpg vs ISIC_2538799.jpg: 0.6628
ISIC_1510820.jpg vs ISIC_2542440.jpg: 0.3122
ISIC_1510820.jpg vs ISIC_2543386.jpg: 0.1496
ISIC_1510820.jpg vs ISIC_2544347.jpg: 0.4031
ISIC_1510820.jpg vs ISIC_2545971.jpg: 0.4669
ISIC_1510820.jpg vs ISIC_2548780.jpg: 0.7908
ISIC_1510820.jpg vs ISIC_2550073.jpg: 0.8437
ISIC_1510820.jpg vs ISIC_2550217.jpg: 0.7784
ISIC_1510820.jpg vs ISIC_2550632.jpg: 0.9116
ISIC_1510820.jpg vs ISIC_2550946.jpg: 0.6568
ISIC_1510820.jpg vs ISIC_2550997.jpg: 0.1705
ISIC_1510820.jpg vs ISIC_2551086.jpg: 0.8666
ISIC_1510820.jpg vs ISIC_2551089.jpg: 0.6623
ISIC_1510820.jpg vs ISIC_2551111.jpg: 0.4664
ISIC_1510820.jpg vs ISIC_2552351.jpg: 0.7394
ISIC_1510820.jpg vs ISIC_2552770.jpg: 0.7757
ISIC_1510820.jpg vs ISIC_2559237.jpg: 0.8468
ISIC_1510820.jpg vs ISIC_2560699.jpg: 0.2717
ISIC_1510820.jpg vs ISIC_2561895.jpg: 0.8036
ISIC_1510820.jpg vs ISIC_2563545.jpg: 0.7914
ISIC_1510820.jpg vs ISIC_2563680.jpg: 0.5304
ISIC_1510820.jpg vs ISIC_2566474.jpg: 0.7706
ISIC_1510820.jpg vs ISIC_2566780.jpg: 0.7992
ISIC_1510820.jpg vs ISIC_2567217.jpg: 0.7892
ISIC_1510820.jpg vs ISIC_2567231.jpg: 0.7688
ISIC_1510820.jpg vs ISIC_2568399.jpg: 0.5733
ISIC_1510820.jpg vs ISIC_2568903.jpg: 0.0608
ISIC_1510820.jpg vs ISIC_2569931.jpg: 0.6191
ISIC_1510820.jpg vs ISIC_2570815.jpg: 0.7556
ISIC_1510820.jpg vs ISIC_2571600.jpg: 0.3591
# ISIC_1510820.jpg vs ISIC_2572301.jpg: 0.1744
# ISIC_1510820.jpg vs ISIC_2573270.jpg: 0.6848
# ISIC_1510820.jpg vs ISIC_2574347.jpg: 0.8034
# ISIC_1510820.jpg vs ISIC_2574854.jpg: 0.8163
# ISIC_1510820.jpg vs ISIC_2575740.jpg: 0.6234
# ISIC_1510820.jpg vs ISIC_2576499.jpg: 0.7516
# ISIC_1510820.jpg vs ISIC_2577765.jpg: 0.8676
# ISIC_1510820.jpg vs ISIC_2578191.jpg: 0.7821
# ISIC_1510820.jpg vs ISIC_2578562.jpg: 0.8266
# ISIC_1510820.jpg vs ISIC_2582691.jpg: 0.3511
# ISIC_1510820.jpg vs ISIC_2586009.jpg: 0.3689
# ISIC_1510820.jpg vs ISIC_2586249.jpg: 0.8150
# ISIC_1510820.jpg vs ISIC_2588457.jpg: 0.8365
# ISIC_1510820.jpg vs ISIC_2588690.jpg: 0.4406
# ISIC_1510820.jpg vs ISIC_2588932.jpg: 0.3104
# ISIC_1510820.jpg vs ISIC_2589886.jpg: 0.7530
# ISIC_1510820.jpg vs ISIC_2591810.jpg: 0.2969
# ISIC_1510820.jpg vs ISIC_2594538.jpg: 0.6084
# ISIC_1510820.jpg vs ISIC_2594596.jpg: 0.3651
# ISIC_1510820.jpg vs ISIC_2595762.jpg: 0.7467
# ISIC_1510820.jpg vs ISIC_2599282.jpg: 0.9070
# ISIC_1510820.jpg vs ISIC_2600298.jpg: 0.6178
# ISIC_1510820.jpg vs ISIC_2600649.jpg: 0.6171
# ISIC_1510820.jpg vs ISIC_2600849.jpg: 0.8622
# ISIC_1510820.jpg vs ISIC_2601658.jpg: 0.4532
# ISIC_1510820.jpg vs ISIC_2601827.jpg: 0.8102
# ISIC_1510820.jpg vs ISIC_2603428.jpg: 0.7093
# ISIC_1510820.jpg vs ISIC_2605953.jpg: 0.7783
# ISIC_1510820.jpg vs ISIC_2606454.jpg: 0.5778
# ISIC_1510820.jpg vs ISIC_2607622.jpg: 0.5184
# ISIC_1510820.jpg vs ISIC_2607761.jpg: 0.8137
# ISIC_1510820.jpg vs ISIC_2609978.jpg: 0.5415
# ISIC_1510820.jpg vs ISIC_2610536.jpg: 0.3390
# ISIC_1510820.jpg vs ISIC_2610835.jpg: 0.7649
# ISIC_1510820.jpg vs ISIC_2611114.jpg: 0.8418
# ISIC_1510820.jpg vs ISIC_2611496.jpg: 0.8299
# ISIC_1510820.jpg vs ISIC_2613083.jpg: 0.6570
# ISIC_1510820.jpg vs ISIC_2613282.jpg: 0.8133
# ISIC_1510820.jpg vs ISIC_2613780.jpg: 0.5582
# ISIC_1510820.jpg vs ISIC_2613789.jpg: 0.7303
# ISIC_1510820.jpg vs ISIC_2613930.jpg: 0.5903
# ISIC_1510820.jpg vs ISIC_2618851.jpg: 0.7503
# ISIC_1510820.jpg vs ISIC_2621151.jpg: 0.6379
# ISIC_1510820.jpg vs ISIC_2625601.jpg: 0.6625
# ISIC_1510820.jpg vs ISIC_2626522.jpg: 0.7238
# ISIC_1510820.jpg vs ISIC_2630389.jpg: 0.5368
# ISIC_1510820.jpg vs ISIC_2631078.jpg: 0.4168
# ISIC_1510820.jpg vs ISIC_2632644.jpg: 0.6981
# ISIC_1510820.jpg vs ISIC_2634029.jpg: 0.9133
# ISIC_1510820.jpg vs ISIC_2635420.jpg: 0.6682
# ISIC_1510820.jpg vs ISIC_2635909.jpg: 0.2759
# ISIC_1510820.jpg vs ISIC_2636640.jpg: 0.7428
# ISIC_1510820.jpg vs ISIC_2640777.jpg: 0.7499
# ISIC_1510820.jpg vs ISIC_2642228.jpg: 0.8648
# ISIC_1510820.jpg vs ISIC_2646237.jpg: 0.8565
# ISIC_1510820.jpg vs ISIC_2647881.jpg: 0.3354
# ISIC_1510820.jpg vs ISIC_2650588.jpg: 0.1974
# ISIC_1510820.jpg vs ISIC_2651134.jpg: 0.8029
# ISIC_1510820.jpg vs ISIC_2653979.jpg: 0.6862
# ISIC_1510820.jpg vs ISIC_2656229.jpg: 0.7138
# ISIC_1510820.jpg vs ISIC_2661607.jpg: 0.2205
# ISIC_1510820.jpg vs ISIC_2664224.jpg: 0.5846
# ISIC_1510820.jpg vs ISIC_2665667.jpg: 0.4200
# ISIC_1510820.jpg vs ISIC_2667536.jpg: 0.3886
# ISIC_1510820.jpg vs ISIC_2670221.jpg: 0.6404
# ISIC_1510820.jpg vs ISIC_2672734.jpg: 0.6059
# ISIC_1510820.jpg vs ISIC_2672880.jpg: 0.8479
# ISIC_1510820.jpg vs ISIC_2674741.jpg: 0.7826
# ISIC_1510820.jpg vs ISIC_2675336.jpg: 0.7987
# ISIC_1510820.jpg vs ISIC_2680409.jpg: 0.7140
# ISIC_1510820.jpg vs ISIC_2683058.jpg: 0.4182
# ISIC_1510820.jpg vs ISIC_2684650.jpg: 0.5744
# ISIC_1510820.jpg vs ISIC_2690375.jpg: 0.4435
# ISIC_1510820.jpg vs ISIC_2691534.jpg: 0.5455
# ISIC_1510820.jpg vs ISIC_2693087.jpg: 0.7033
# ISIC_1510820.jpg vs ISIC_2695905.jpg: 0.5968
# ISIC_1510820.jpg vs ISIC_2698346.jpg: 0.8598
# ISIC_1510820.jpg vs ISIC_2698764.jpg: 0.7578
# ISIC_1510820.jpg vs ISIC_2701399.jpg: 0.4267
# ISIC_1510820.jpg vs ISIC_2702465.jpg: 0.6795
# ISIC_1510820.jpg vs ISIC_2705044.jpg: 0.4156
# ISIC_1510820.jpg vs ISIC_2709532.jpg: 0.3887
# ISIC_1510820.jpg vs ISIC_2757879.jpg: 0.7028
# ISIC_1510820.jpg vs ISIC_2758441.jpg: 0.8557
# ISIC_1510820.jpg vs ISIC_2759341.jpg: 0.8040
# ISIC_1510820.jpg vs ISIC_2761222.jpg: 0.5484
# ISIC_1510820.jpg vs ISIC_2764446.jpg: 0.4311
# ISIC_1510820.jpg vs ISIC_2766032.jpg: 0.6349
# ISIC_1510820.jpg vs ISIC_2766590.jpg: 0.1796
# ISIC_1510820.jpg vs ISIC_2768407.jpg: 0.8776
# ISIC_1510820.jpg vs ISIC_2768840.jpg: 0.6639
# ISIC_1510820.jpg vs ISIC_2769811.jpg: 0.7551
# ISIC_1510820.jpg vs ISIC_2771030.jpg: 0.3479
# ISIC_1510820.jpg vs ISIC_2775262.jpg: 0.3485
# ISIC_1510820.jpg vs ISIC_2775569.jpg: 0.7352
# ISIC_1510820.jpg vs ISIC_2780480.jpg: 0.7177
# ISIC_1510820.jpg vs ISIC_2781714.jpg: 0.6538
# ISIC_1510820.jpg vs ISIC_2783061.jpg: 0.6475
# ISIC_1510820.jpg vs ISIC_2783585.jpg: 0.5142
# ISIC_1510820.jpg vs ISIC_2785976.jpg: 0.8196
# ISIC_1510820.jpg vs ISIC_2786319.jpg: 0.6206
# ISIC_1510820.jpg vs ISIC_2788245.jpg: 0.3395
# ISIC_1510820.jpg vs ISIC_2789714.jpg: 0.7770
# ISIC_1510820.jpg vs ISIC_2791655.jpg: 0.7930
# ISIC_1510820.jpg vs ISIC_2793133.jpg: 0.7436
# ISIC_1510820.jpg vs ISIC_2793703.jpg: 0.6733
# ISIC_1510820.jpg vs ISIC_2797372.jpg: 0.4019
# ISIC_1510820.jpg vs ISIC_2799021.jpg: 0.7049
# ISIC_1510820.jpg vs ISIC_2799268.jpg: 0.8686
# ISIC_1510820.jpg vs ISIC_2801139.jpg: 0.2094
# ISIC_1510820.jpg vs ISIC_2802301.jpg: 0.8433
# ISIC_1510820.jpg vs ISIC_2802346.jpg: 0.9063
# ISIC_1510820.jpg vs ISIC_2802408.jpg: 0.7834
# ISIC_1510820.jpg vs ISIC_2803506.jpg: 0.6616
# ISIC_1510820.jpg vs ISIC_2804825.jpg: 0.3677
# ISIC_1510820.jpg vs ISIC_2806461.jpg: 0.7978
# ISIC_1510820.jpg vs ISIC_2807040.jpg: 0.7285
# ISIC_1510820.jpg vs ISIC_2807921.jpg: 0.8079
# ISIC_1510820.jpg vs ISIC_2808393.jpg: 0.7479
# ISIC_1510820.jpg vs ISIC_2809430.jpg: 0.8758
# ISIC_1510820.jpg vs ISIC_2811093.jpg: 0.8364
# ISIC_1510820.jpg vs ISIC_2811919.jpg: 0.3983
# ISIC_1510820.jpg vs ISIC_2812284.jpg: 0.5643
# ISIC_1510820.jpg vs ISIC_2814272.jpg: 0.8750
# ISIC_1510820.jpg vs ISIC_2817197.jpg: 0.6923
# ISIC_1510820.jpg vs ISIC_2818153.jpg: 0.7807
# ISIC_1510820.jpg vs ISIC_2822174.jpg: 0.1347
# ISIC_1510820.jpg vs ISIC_2822889.jpg: 0.7305
# ISIC_1510820.jpg vs ISIC_2823266.jpg: 0.6929
# ISIC_1510820.jpg vs ISIC_2823299.jpg: 0.6844
# ISIC_1510820.jpg vs ISIC_2824061.jpg: 0.7880
# ISIC_1510820.jpg vs ISIC_2824922.jpg: 0.3666
# ISIC_1510820.jpg vs ISIC_2827942.jpg: 0.3087
# ISIC_1510820.jpg vs ISIC_2828156.jpg: 0.1879
# ISIC_1510820.jpg vs ISIC_2828811.jpg: 0.7740
# ISIC_1510820.jpg vs ISIC_2829040.jpg: 0.1648
# ISIC_1510820.jpg vs ISIC_2829089.jpg: 0.7960
# ISIC_1510820.jpg vs ISIC_2834420.jpg: 0.7848
# ISIC_1510820.jpg vs ISIC_2835116.jpg: 0.8428
# ISIC_1510820.jpg vs ISIC_2837857.jpg: 0.6789
# ISIC_1510820.jpg vs ISIC_2838169.jpg: 0.7894
# ISIC_1510820.jpg vs ISIC_2838180.jpg: 0.8720
# ISIC_1510820.jpg vs ISIC_2839742.jpg: 0.3268
# ISIC_1510820.jpg vs ISIC_2840019.jpg: 0.2785
# ISIC_1510820.jpg vs ISIC_2841788.jpg: 0.1101
# ISIC_1510820.jpg vs ISIC_2845014.jpg: 0.5105
# ISIC_1510820.jpg vs ISIC_2845028.jpg: 0.7011
# ISIC_1510820.jpg vs ISIC_2853193.jpg: 0.5526
# ISIC_1510820.jpg vs ISIC_2855421.jpg: 0.8180
# ISIC_1510820.jpg vs ISIC_2855923.jpg: 0.8468
# ISIC_1510820.jpg vs ISIC_2858057.jpg: 0.2380
# ISIC_1510820.jpg vs ISIC_2861073.jpg: 0.7593
# ISIC_1510820.jpg vs ISIC_2861615.jpg: 0.6046
# ISIC_1510820.jpg vs ISIC_2864571.jpg: 0.7138
# ISIC_1510820.jpg vs ISIC_2865092.jpg: 0.6905
# ISIC_1510820.jpg vs ISIC_2869298.jpg: 0.5841
# ISIC_1510820.jpg vs ISIC_2869500.jpg: 0.7971
# ISIC_1510820.jpg vs ISIC_2871866.jpg: 0.7732
# ISIC_1510820.jpg vs ISIC_2879103.jpg: 0.7756
# ISIC_1510820.jpg vs ISIC_2879319.jpg: 0.7197
# ISIC_1510820.jpg vs ISIC_2880938.jpg: 0.1927
# ISIC_1510820.jpg vs ISIC_2882183.jpg: 0.4941
# ISIC_1510820.jpg vs ISIC_2882715.jpg: 0.8055
# ISIC_1510820.jpg vs ISIC_2882963.jpg: 0.5282
# ISIC_1510820.jpg vs ISIC_2883463.jpg: 0.1587
# ISIC_1510820.jpg vs ISIC_2883756.jpg: 0.5711
# ISIC_1510820.jpg vs ISIC_2884058.jpg: 0.3095
# ISIC_1510820.jpg vs ISIC_2885348.jpg: 0.8596
# ISIC_1510820.jpg vs ISIC_2885577.jpg: 0.6710
# ISIC_1510820.jpg vs ISIC_2885718.jpg: 0.1417
# ISIC_1510820.jpg vs ISIC_2885741.jpg: 0.8806
# ISIC_1510820.jpg vs ISIC_2887299.jpg: 0.2129
# ISIC_1510820.jpg vs ISIC_2889328.jpg: 0.8315
# ISIC_1510820.jpg vs ISIC_2890008.jpg: 0.7628
# ISIC_1510820.jpg vs ISIC_2893969.jpg: 0.8156
# ISIC_1510820.jpg vs ISIC_2894619.jpg: 0.8700
# ISIC_1510820.jpg vs ISIC_2897386.jpg: 0.8559
# ISIC_1510820.jpg vs ISIC_2899485.jpg: 0.4939
# ISIC_1510820.jpg vs ISIC_2900065.jpg: 0.7964
# ISIC_1510820.jpg vs ISIC_2900150.jpg: 0.1234
# ISIC_1510820.jpg vs ISIC_2901036.jpg: 0.0806
# ISIC_1510820.jpg vs ISIC_2906353.jpg: 0.6346
# ISIC_1510820.jpg vs ISIC_2906357.jpg: 0.8976
# ISIC_1510820.jpg vs ISIC_2906649.jpg: 0.4505
# ISIC_1510820.jpg vs ISIC_2907414.jpg: 0.3814
# ISIC_1510820.jpg vs ISIC_2908210.jpg: 0.4142
# ISIC_1510820.jpg vs ISIC_2909250.jpg: 0.8347
# ISIC_1510820.jpg vs ISIC_2909747.jpg: 0.8543
# ISIC_1510820.jpg vs ISIC_2910374.jpg: 0.4497
# ISIC_1510820.jpg vs ISIC_2911033.jpg: 0.7597
# ISIC_1510820.jpg vs ISIC_2911378.jpg: 0.4915
# ISIC_1510820.jpg vs ISIC_2912359.jpg: 0.7781
# ISIC_1510820.jpg vs ISIC_2914164.jpg: 0.5222
# ISIC_1510820.jpg vs ISIC_2914431.jpg: 0.5545
# ISIC_1510820.jpg vs ISIC_2914650.jpg: 0.5482
# ISIC_1510820.jpg vs ISIC_2919248.jpg: 0.8862
# ISIC_1510820.jpg vs ISIC_2922450.jpg: 0.2868
# ISIC_1510820.jpg vs ISIC_2922876.jpg: 0.6133
# ISIC_1510820.jpg vs ISIC_2927268.jpg: 0.8120
# ISIC_1510820.jpg vs ISIC_2928166.jpg: 0.3722
# ISIC_1510820.jpg vs ISIC_2928329.jpg: 0.8740
# ISIC_1510820.jpg vs ISIC_2929442.jpg: 0.8574
# ISIC_1510820.jpg vs ISIC_2930982.jpg: 0.2614
# ISIC_1510820.jpg vs ISIC_2931955.jpg: 0.2774
# ISIC_1510820.jpg vs ISIC_2933054.jpg: 0.7978
# ISIC_1510820.jpg vs ISIC_2939559.jpg: 0.7183
# ISIC_1510820.jpg vs ISIC_2939632.jpg: 0.3900
# ISIC_1510820.jpg vs ISIC_2940341.jpg: 0.7624
# ISIC_1510820.jpg vs ISIC_2941506.jpg: 0.6542
# ISIC_1510820.jpg vs ISIC_2944785.jpg: 0.8357
# ISIC_1510820.jpg vs ISIC_2945698.jpg: 0.5779
# ISIC_1510820.jpg vs ISIC_2945726.jpg: 0.8663
# ISIC_1510820.jpg vs ISIC_2946358.jpg: 0.5259
# ISIC_1510820.jpg vs ISIC_2947373.jpg: 0.8987
# ISIC_1510820.jpg vs ISIC_2948729.jpg: 0.5362
# ISIC_1510820.jpg vs ISIC_2950913.jpg: 0.3426
# ISIC_1510820.jpg vs ISIC_2952110.jpg: 0.3908
# ISIC_1510820.jpg vs ISIC_2953931.jpg: 0.6942
# ISIC_1510820.jpg vs ISIC_2954422.jpg: 0.4749
# ISIC_1510820.jpg vs ISIC_2954737.jpg: 0.6744
# ISIC_1510820.jpg vs ISIC_2955552.jpg: 0.2472
# ISIC_1510820.jpg vs ISIC_2958724.jpg: 0.3696
# ISIC_1510820.jpg vs ISIC_2961790.jpg: 0.3883
# ISIC_1510820.jpg vs ISIC_2961837.jpg: 0.7378
# ISIC_1510820.jpg vs ISIC_2962245.jpg: 0.7950
# ISIC_1510820.jpg vs ISIC_2964421.jpg: 0.6080
# ISIC_1510820.jpg vs ISIC_2966319.jpg: 0.7967
# ISIC_1510820.jpg vs ISIC_2966600.jpg: 0.3752
# ISIC_1510820.jpg vs ISIC_2968104.jpg: 0.6190
# ISIC_1510820.jpg vs ISIC_2970783.jpg: 0.7716
# ISIC_1510820.jpg vs ISIC_2973598.jpg: 0.8078
# ISIC_1510820.jpg vs ISIC_2973910.jpg: 0.8618
# ISIC_1510820.jpg vs ISIC_2974420.jpg: 0.7626
# ISIC_1510820.jpg vs ISIC_2976697.jpg: 0.2420
# ISIC_1510820.jpg vs ISIC_2976936.jpg: 0.5931
# ISIC_1510820.jpg vs ISIC_2979177.jpg: 0.5546
# ISIC_1510820.jpg vs ISIC_2982140.jpg: 0.6797
# ISIC_1510820.jpg vs ISIC_2987555.jpg: 0.3339
# ISIC_1510820.jpg vs ISIC_2988031.jpg: 0.8117
# ISIC_1510820.jpg vs ISIC_2989863.jpg: 0.4665
# ISIC_1510820.jpg vs ISIC_2990790.jpg: 0.4661
# ISIC_1510820.jpg vs ISIC_2990982.jpg: 0.6284
# ISIC_1510820.jpg vs ISIC_2994593.jpg: 0.8985
# ISIC_1510820.jpg vs ISIC_2995868.jpg: 0.5446
# ISIC_1510820.jpg vs ISIC_2996554.jpg: 0.3594
# ISIC_1510820.jpg vs ISIC_2997653.jpg: 0.7879
# ISIC_1510820.jpg vs ISIC_2999870.jpg: 0.7678
# ISIC_1510820.jpg vs ISIC_3003131.jpg: 0.3687
# ISIC_1510820.jpg vs ISIC_3004536.jpg: 0.7160
# ISIC_1510820.jpg vs ISIC_3006001.jpg: 0.8314
# ISIC_1510820.jpg vs ISIC_3006353.jpg: 0.8416
# ISIC_1510820.jpg vs ISIC_3006985.jpg: 0.6636
# ISIC_1510820.jpg vs ISIC_3009035.jpg: 0.2581
# ISIC_1510820.jpg vs ISIC_3010492.jpg: 0.7462
# ISIC_1510820.jpg vs ISIC_3016106.jpg: 0.5030
# ISIC_1510820.jpg vs ISIC_3016243.jpg: 0.3799
# ISIC_1510820.jpg vs ISIC_3016409.jpg: 0.4153
# ISIC_1510820.jpg vs ISIC_3017624.jpg: 0.8358
# ISIC_1510820.jpg vs ISIC_3017944.jpg: 0.5264
# ISIC_1510820.jpg vs ISIC_3018548.jpg: 0.7970
# ISIC_1510820.jpg vs ISIC_3018869.jpg: 0.8958
# ISIC_1510820.jpg vs ISIC_3019231.jpg: 0.5865
# ISIC_1510820.jpg vs ISIC_3019849.jpg: 0.8303
# ISIC_1510820.jpg vs ISIC_3021712.jpg: 0.3905
# ISIC_1510820.jpg vs ISIC_3024297.jpg: 0.3895
# ISIC_1510820.jpg vs ISIC_3025179.jpg: 0.8670
# ISIC_1510820.jpg vs ISIC_3025776.jpg: 0.6526
# ISIC_1510820.jpg vs ISIC_3026301.jpg: 0.5541
# ISIC_1510820.jpg vs ISIC_3029757.jpg: 0.8992
# ISIC_1510820.jpg vs ISIC_3031742.jpg: 0.3304
# ISIC_1510820.jpg vs ISIC_3034496.jpg: 0.4566
# ISIC_1510820.jpg vs ISIC_3035013.jpg: 0.8271
# ISIC_1510820.jpg vs ISIC_3035671.jpg: 0.8259
# ISIC_1510820.jpg vs ISIC_3035934.jpg: 0.5063
# ISIC_1510820.jpg vs ISIC_3036806.jpg: 0.6538
# ISIC_1510820.jpg vs ISIC_3038801.jpg: 0.8921
# ISIC_1510820.jpg vs ISIC_3040215.jpg: 0.5786
# ISIC_1510820.jpg vs ISIC_3046579.jpg: 0.8227
# ISIC_1510820.jpg vs ISIC_3048391.jpg: 0.6464
# ISIC_1510820.jpg vs ISIC_3048601.jpg: 0.8816
# ISIC_1510820.jpg vs ISIC_3050241.jpg: 0.2765
# ISIC_1510820.jpg vs ISIC_3050591.jpg: 0.6137
# ISIC_1510820.jpg vs ISIC_3050726.jpg: 0.5070
# ISIC_1510820.jpg vs ISIC_3051087.jpg: 0.7050
# ISIC_1510820.jpg vs ISIC_3051428.jpg: 0.7786
# ISIC_1510820.jpg vs ISIC_3052742.jpg: 0.7940
# ISIC_1510820.jpg vs ISIC_3053615.jpg: 0.7021
# ISIC_1510820.jpg vs ISIC_3054208.jpg: 0.7987
# ISIC_1510820.jpg vs ISIC_3056153.jpg: 0.4049
# ISIC_1510820.jpg vs ISIC_3056816.jpg: 0.6436
# ISIC_1510820.jpg vs ISIC_3057622.jpg: 0.7538
# ISIC_1510820.jpg vs ISIC_3059224.jpg: 0.6789
# ISIC_1510820.jpg vs ISIC_3059916.jpg: 0.4556
# ISIC_1510820.jpg vs ISIC_3060292.jpg: 0.8838
# ISIC_1510820.jpg vs ISIC_3061715.jpg: 0.5577
# ISIC_1510820.jpg vs ISIC_3061920.jpg: 0.6866
# ISIC_1510820.jpg vs ISIC_3064913.jpg: 0.6867
# ISIC_1510820.jpg vs ISIC_3065173.jpg: 0.5790
# ISIC_1510820.jpg vs ISIC_3069602.jpg: 0.4933
# ISIC_1510820.jpg vs ISIC_3073305.jpg: 0.6026
# ISIC_1510820.jpg vs ISIC_3073626.jpg: 0.6818
# ISIC_1510820.jpg vs ISIC_3073822.jpg: 0.3142
# ISIC_1510820.jpg vs ISIC_3075041.jpg: 0.7849
# ISIC_1510820.jpg vs ISIC_3077337.jpg: 0.5429
# ISIC_1510820.jpg vs ISIC_3077737.jpg: 0.7164
# ISIC_1510820.jpg vs ISIC_3078390.jpg: 0.2365
# ISIC_1510820.jpg vs ISIC_3078669.jpg: 0.6681
# ISIC_1510820.jpg vs ISIC_3078876.jpg: 0.4266
# ISIC_1510820.jpg vs ISIC_3080264.jpg: 0.8513
# ISIC_1510820.jpg vs ISIC_3080291.jpg: 0.8504
# ISIC_1510820.jpg vs ISIC_3080994.jpg: 0.5496
# ISIC_1510820.jpg vs ISIC_3081722.jpg: 0.3227
# ISIC_1510820.jpg vs ISIC_3082434.jpg: 0.8559
# ISIC_1510820.jpg vs ISIC_3082829.jpg: 0.7258
# ISIC_1510820.jpg vs ISIC_3084021.jpg: 0.5668
# ISIC_1510820.jpg vs ISIC_3085039.jpg: 0.5354
# ISIC_1510820.jpg vs ISIC_3086004.jpg: 0.8015
# ISIC_1510820.jpg vs ISIC_3086392.jpg: 0.5060
# ISIC_1510820.jpg vs ISIC_3086566.jpg: 0.8752
# ISIC_1510820.jpg vs ISIC_3086993.jpg: 0.8535
# ISIC_1510820.jpg vs ISIC_3088239.jpg: 0.8032
# ISIC_1510820.jpg vs ISIC_3090818.jpg: 0.7863
# ISIC_1510820.jpg vs ISIC_3091500.jpg: 0.2167
# ISIC_1510820.jpg vs ISIC_3091594.jpg: 0.8276
# ISIC_1510820.jpg vs ISIC_3091975.jpg: 0.7085
# ISIC_1510820.jpg vs ISIC_3092993.jpg: 0.4400
# ISIC_1510820.jpg vs ISIC_3097494.jpg: 0.3929
# ISIC_1510820.jpg vs ISIC_3100039.jpg: 0.8247
# ISIC_1510820.jpg vs ISIC_3100379.jpg: 0.6398
# ISIC_1510820.jpg vs ISIC_3100513.jpg: 0.8816
# ISIC_1510820.jpg vs ISIC_3101113.jpg: 0.7959
# ISIC_1510820.jpg vs ISIC_3101558.jpg: 0.7156
# ISIC_1510820.jpg vs ISIC_3105094.jpg: 0.3501
# ISIC_1510820.jpg vs ISIC_3105463.jpg: 0.3651
# ISIC_1510820.jpg vs ISIC_3105863.jpg: 0.8395
# ISIC_1510820.jpg vs ISIC_3109975.jpg: 0.7007
# ISIC_1510820.jpg vs ISIC_3111332.jpg: 0.5484
# ISIC_1510820.jpg vs ISIC_3112707.jpg: 0.5260
# ISIC_1510820.jpg vs ISIC_3113447.jpg: 0.7718
# ISIC_1510820.jpg vs ISIC_3114484.jpg: 0.7310
# ISIC_1510820.jpg vs ISIC_3115114.jpg: 0.5894
# ISIC_1510820.jpg vs ISIC_3115397.jpg: 0.1363
# ISIC_1510820.jpg vs ISIC_3115713.jpg: 0.6245
# ISIC_1510820.jpg vs ISIC_3116793.jpg: 0.2005
# ISIC_1510820.jpg vs ISIC_3117522.jpg: 0.7507
# ISIC_1510820.jpg vs ISIC_3118476.jpg: 0.3102
# ISIC_1510820.jpg vs ISIC_3125596.jpg: 0.8448
# ISIC_1510820.jpg vs ISIC_3130286.jpg: 0.3674
# ISIC_1510820.jpg vs ISIC_3130826.jpg: 0.1648
# ISIC_1510820.jpg vs ISIC_3133510.jpg: 0.7629
# ISIC_1510820.jpg vs ISIC_3136416.jpg: 0.8132
# ISIC_1510820.jpg vs ISIC_3136517.jpg: 0.7534
# ISIC_1510820.jpg vs ISIC_3137228.jpg: 0.8749
# ISIC_1510820.jpg vs ISIC_3140781.jpg: 0.8309
# ISIC_1510820.jpg vs ISIC_3140894.jpg: 0.6780
# ISIC_1510820.jpg vs ISIC_3143811.jpg: 0.5914
# ISIC_1510820.jpg vs ISIC_3144013.jpg: 0.5312
# ISIC_1510820.jpg vs ISIC_3147141.jpg: 0.6129
# ISIC_1510820.jpg vs ISIC_3149300.jpg: 0.4418
# ISIC_1510820.jpg vs ISIC_3150364.jpg: 0.7371
# ISIC_1510820.jpg vs ISIC_3150831.jpg: 0.8577
# ISIC_1510820.jpg vs ISIC_3158528.jpg: 0.6712
# ISIC_1510820.jpg vs ISIC_3159265.jpg: 0.1847
# ISIC_1510820.jpg vs ISIC_3160366.jpg: 0.7877
# ISIC_1510820.jpg vs ISIC_3161587.jpg: 0.8640
# ISIC_1510820.jpg vs ISIC_3162378.jpg: 0.6353
# ISIC_1510820.jpg vs ISIC_3162606.jpg: 0.6384
# ISIC_1510820.jpg vs ISIC_3163483.jpg: 0.7796
# ISIC_1510820.jpg vs ISIC_3164325.jpg: 0.7769
# ISIC_1510820.jpg vs ISIC_3166213.jpg: 0.5965
# ISIC_1510820.jpg vs ISIC_3166913.jpg: 0.7482
# ISIC_1510820.jpg vs ISIC_3169364.jpg: 0.8228
# ISIC_1510820.jpg vs ISIC_3169421.jpg: 0.5804
# ISIC_1510820.jpg vs ISIC_3169710.jpg: 0.6522
# ISIC_1510820.jpg vs ISIC_3169779.jpg: 0.6240
# ISIC_1510820.jpg vs ISIC_3170107.jpg: 0.5762
# ISIC_1510820.jpg vs ISIC_3170352.jpg: 0.5735
# ISIC_1510820.jpg vs ISIC_3171000.jpg: 0.4144
# ISIC_1510820.jpg vs ISIC_3172378.jpg: 0.1880
# ISIC_1510820.jpg vs ISIC_3174508.jpg: 0.8202
# ISIC_1510820.jpg vs ISIC_3174720.jpg: 0.7497
# ISIC_1510820.jpg vs ISIC_3174785.jpg: 0.6699
# ISIC_1510820.jpg vs ISIC_3177069.jpg: 0.3283
# ISIC_1510820.jpg vs ISIC_3177698.jpg: 0.7605
# ISIC_1510820.jpg vs ISIC_3178424.jpg: 0.7799
# ISIC_1510820.jpg vs ISIC_3178543.jpg: 0.6210
# ISIC_1510820.jpg vs ISIC_3182083.jpg: 0.5360
# ISIC_1510820.jpg vs ISIC_3182255.jpg: 0.8391
# ISIC_1510820.jpg vs ISIC_3182801.jpg: 0.1614
# ISIC_1510820.jpg vs ISIC_3187460.jpg: 0.6245
# ISIC_1510820.jpg vs ISIC_3187793.jpg: 0.8255
# ISIC_1510820.jpg vs ISIC_3188729.jpg: 0.8989
# ISIC_1510820.jpg vs ISIC_3191573.jpg: 0.6001
# ISIC_1510820.jpg vs ISIC_3191607.jpg: 0.8023
# ISIC_1510820.jpg vs ISIC_3192101.jpg: 0.5640
# ISIC_1510820.jpg vs ISIC_3193920.jpg: 0.8474
# ISIC_1510820.jpg vs ISIC_3196004.jpg: 0.6198
# ISIC_1510820.jpg vs ISIC_3197088.jpg: 0.4868
# ISIC_1510820.jpg vs ISIC_3197746.jpg: 0.8614
# ISIC_1510820.jpg vs ISIC_3197993.jpg: 0.4277
# ISIC_1510820.jpg vs ISIC_3199911.jpg: 0.3740
# ISIC_1510820.jpg vs ISIC_3201901.jpg: 0.7063
# ISIC_1510820.jpg vs ISIC_3202345.jpg: 0.7790
# ISIC_1510820.jpg vs ISIC_3203927.jpg: 0.8942
# ISIC_1510820.jpg vs ISIC_3204260.jpg: 0.8520
# ISIC_1510820.jpg vs ISIC_3205782.jpg: 0.8324
# ISIC_1510820.jpg vs ISIC_3206928.jpg: 0.7475
# ISIC_1510820.jpg vs ISIC_3207493.jpg: 0.5388
# ISIC_1510820.jpg vs ISIC_3208994.jpg: 0.4986
# ISIC_1510820.jpg vs ISIC_3209894.jpg: 0.6100
# ISIC_1510820.jpg vs ISIC_3210295.jpg: 0.8206
# ISIC_1510820.jpg vs ISIC_3211391.jpg: 0.6880
# ISIC_1510820.jpg vs ISIC_3212230.jpg: 0.4627
# ISIC_1510820.jpg vs ISIC_3214327.jpg: 0.7964
# ISIC_1510820.jpg vs ISIC_3216128.jpg: 0.2141
# ISIC_1510820.jpg vs ISIC_3217925.jpg: 0.4119
# ISIC_1510820.jpg vs ISIC_3221148.jpg: 0.2099
# ISIC_1510820.jpg vs ISIC_3223721.jpg: 0.8282
# ISIC_1510820.jpg vs ISIC_3224219.jpg: 0.4028
# ISIC_1510820.jpg vs ISIC_3224330.jpg: 0.2817
# ISIC_1510820.jpg vs ISIC_3224580.jpg: 0.5812
# ISIC_1510820.jpg vs ISIC_3226112.jpg: 0.5228
# ISIC_1510820.jpg vs ISIC_3226250.jpg: 0.8256
# ISIC_1510820.jpg vs ISIC_3227329.jpg: 0.8382
# ISIC_1510820.jpg vs ISIC_3228270.jpg: 0.8760
# ISIC_1510820.jpg vs ISIC_3230995.jpg: 0.1575
# ISIC_1510820.jpg vs ISIC_3231416.jpg: 0.8084
# ISIC_1510820.jpg vs ISIC_3231502.jpg: 0.0951
# ISIC_1510820.jpg vs ISIC_3232262.jpg: 0.7636
# ISIC_1510820.jpg vs ISIC_3233449.jpg: 0.5848
# ISIC_1510820.jpg vs ISIC_3233743.jpg: 0.6622
# ISIC_1510820.jpg vs ISIC_3235082.jpg: 0.1092
# ISIC_1510820.jpg vs ISIC_3235363.jpg: 0.8357
# ISIC_1510820.jpg vs ISIC_3240195.jpg: 0.3326
# ISIC_1510820.jpg vs ISIC_3242339.jpg: 0.6755
# ISIC_1510820.jpg vs ISIC_3242898.jpg: 0.6918
# ISIC_1510820.jpg vs ISIC_3243897.jpg: 0.8872
# ISIC_1510820.jpg vs ISIC_3245209.jpg: 0.7349
# ISIC_1510820.jpg vs ISIC_3248430.jpg: 0.2940
# ISIC_1510820.jpg vs ISIC_3249230.jpg: 0.8586
# ISIC_1510820.jpg vs ISIC_3249625.jpg: 0.2337
# ISIC_1510820.jpg vs ISIC_3250051.jpg: 0.8153
# ISIC_1510820.jpg vs ISIC_3250526.jpg: 0.6102
# ISIC_1510820.jpg vs ISIC_3252554.jpg: 0.5819
# ISIC_1510820.jpg vs ISIC_3254014.jpg: 0.8556
# ISIC_1510820.jpg vs ISIC_3254862.jpg: 0.7443
# ISIC_1510820.jpg vs ISIC_3258116.jpg: 0.4600
# ISIC_1510820.jpg vs ISIC_3260623.jpg: 0.1757
# ISIC_1510820.jpg vs ISIC_3263214.jpg: 0.7531
# ISIC_1510820.jpg vs ISIC_3263776.jpg: 0.6527
# ISIC_1510820.jpg vs ISIC_3264572.jpg: 0.8061
# ISIC_1510820.jpg vs ISIC_3265261.jpg: 0.5041
# ISIC_1510820.jpg vs ISIC_3265514.jpg: 0.3153
# ISIC_1510820.jpg vs ISIC_3266762.jpg: 0.5464
# ISIC_1510820.jpg vs ISIC_3266844.jpg: 0.8745
# ISIC_1510820.jpg vs ISIC_3269036.jpg: 0.8566
# ISIC_1510820.jpg vs ISIC_3269752.jpg: 0.7241
# ISIC_1510820.jpg vs ISIC_3270093.jpg: 0.6698
# ISIC_1510820.jpg vs ISIC_3270302.jpg: 0.6218
# ISIC_1510820.jpg vs ISIC_3272340.jpg: 0.7820
# ISIC_1510820.jpg vs ISIC_3273639.jpg: 0.8740
# ISIC_1510820.jpg vs ISIC_3274488.jpg: 0.4755
# ISIC_1510820.jpg vs ISIC_3280235.jpg: 0.8729
# ISIC_1510820.jpg vs ISIC_3280384.jpg: 0.6083
# ISIC_1510820.jpg vs ISIC_3280956.jpg: 0.5148
# ISIC_1510820.jpg vs ISIC_3281881.jpg: 0.8554
# ISIC_1510820.jpg vs ISIC_3283688.jpg: 0.1509
# ISIC_1510820.jpg vs ISIC_3284664.jpg: 0.4177
# ISIC_1510820.jpg vs ISIC_3288360.jpg: 0.7938
# ISIC_1510820.jpg vs ISIC_3292354.jpg: 0.2662
# ISIC_1510820.jpg vs ISIC_3292979.jpg: 0.5792
# ISIC_1510820.jpg vs ISIC_3293577.jpg: 0.3063
# ISIC_1510820.jpg vs ISIC_3294265.jpg: 0.6250
# ISIC_1510820.jpg vs ISIC_3295754.jpg: 0.7631
# ISIC_1510820.jpg vs ISIC_3296102.jpg: 0.7084
# ISIC_1510820.jpg vs ISIC_3299574.jpg: 0.6826
# ISIC_1510820.jpg vs ISIC_3301470.jpg: 0.1170
# ISIC_1510820.jpg vs ISIC_3301584.jpg: 0.2270
# ISIC_1510820.jpg vs ISIC_3301814.jpg: 0.3734
# ISIC_1510820.jpg vs ISIC_3303838.jpg: 0.7969
# ISIC_1510820.jpg vs ISIC_3307624.jpg: 0.7368
# ISIC_1510820.jpg vs ISIC_3308101.jpg: 0.7521
# ISIC_1510820.jpg vs ISIC_3308539.jpg: 0.7072
# ISIC_1510820.jpg vs ISIC_3309618.jpg: 0.5250
# ISIC_1510820.jpg vs ISIC_3310253.jpg: 0.2643
# ISIC_1510820.jpg vs ISIC_3317890.jpg: 0.3522
# ISIC_1510820.jpg vs ISIC_3318037.jpg: 0.8793
# ISIC_1510820.jpg vs ISIC_3319446.jpg: 0.8910
# ISIC_1510820.jpg vs ISIC_3319746.jpg: 0.4548
# ISIC_1510820.jpg vs ISIC_3326069.jpg: 0.5613
# ISIC_1510820.jpg vs ISIC_3327858.jpg: 0.9241
# ISIC_1510820.jpg vs ISIC_3327932.jpg: 0.3841
# ISIC_1510820.jpg vs ISIC_3330108.jpg: 0.1930
# ISIC_1510820.jpg vs ISIC_3330580.jpg: 0.4541
# ISIC_1510820.jpg vs ISIC_3330773.jpg: 0.3715
# ISIC_1510820.jpg vs ISIC_3331297.jpg: 0.8377
# ISIC_1510820.jpg vs ISIC_3332378.jpg: 0.4004
# ISIC_1510820.jpg vs ISIC_3332687.jpg: 0.8277
# ISIC_1510820.jpg vs ISIC_3334793.jpg: 0.4237
# ISIC_1510820.jpg vs ISIC_3335184.jpg: 0.5569
# ISIC_1510820.jpg vs ISIC_3335904.jpg: 0.6771
# ISIC_1510820.jpg vs ISIC_3337583.jpg: 0.5188
# ISIC_1510820.jpg vs ISIC_3343138.jpg: 0.8103
# ISIC_1510820.jpg vs ISIC_3343372.jpg: 0.1367
# ISIC_1510820.jpg vs ISIC_3345936.jpg: 0.8083
# ISIC_1510820.jpg vs ISIC_3348364.jpg: 0.8025
# ISIC_1510820.jpg vs ISIC_3349193.jpg: 0.8314
# ISIC_1510820.jpg vs ISIC_3350064.jpg: 0.6347
# ISIC_1510820.jpg vs ISIC_3353247.jpg: 0.5784
# ISIC_1510820.jpg vs ISIC_3364143.jpg: 0.3466
# ISIC_1510820.jpg vs ISIC_3373667.jpg: 0.7146
# ISIC_1510820.jpg vs ISIC_3373821.jpg: 0.8951
# ISIC_1510820.jpg vs ISIC_3374412.jpg: 0.6135
# ISIC_1510820.jpg vs ISIC_3379766.jpg: 0.7849
# ISIC_1510820.jpg vs ISIC_3380844.jpg: 0.7256
# ISIC_1510820.jpg vs ISIC_3382817.jpg: 0.6218
# ISIC_1510820.jpg vs ISIC_3383938.jpg: 0.7456
# ISIC_1510820.jpg vs ISIC_3392622.jpg: 0.9151
# ISIC_1510820.jpg vs ISIC_3393209.jpg: 0.6176
# ISIC_1510820.jpg vs ISIC_3393442.jpg: 0.2266
# ISIC_1510820.jpg vs ISIC_3394090.jpg: 0.7055
# ISIC_1510820.jpg vs ISIC_3397549.jpg: 0.8079
# ISIC_1510820.jpg vs ISIC_3404515.jpg: 0.1941
# ISIC_1510820.jpg vs ISIC_3405704.jpg: 0.7991
# ISIC_1510820.jpg vs ISIC_3406505.jpg: 0.6835
# ISIC_1510820.jpg vs ISIC_3406688.jpg: 0.3862
# ISIC_1510820.jpg vs ISIC_3409294.jpg: 0.5815
# ISIC_1510820.jpg vs ISIC_3412626.jpg: 0.6411
# ISIC_1510820.jpg vs ISIC_3537350.jpg: 0.5840
# ISIC_1510820.jpg vs ISIC_3537735.jpg: 0.7922
# ISIC_1510820.jpg vs ISIC_3539380.jpg: 0.4037
# ISIC_1510820.jpg vs ISIC_3548128.jpg: 0.9125
# ISIC_1510820.jpg vs ISIC_3549355.jpg: 0.2876
# ISIC_1510820.jpg vs ISIC_3549689.jpg: 0.8121
# ISIC_1510820.jpg vs ISIC_3550074.jpg: 0.6615
# ISIC_1510820.jpg vs ISIC_3550301.jpg: 0.9327
# ISIC_1510820.jpg vs ISIC_3550427.jpg: 0.5102
# ISIC_1510820.jpg vs ISIC_3554298.jpg: 0.8314
# ISIC_1510820.jpg vs ISIC_3555756.jpg: 0.6438
# ISIC_1510820.jpg vs ISIC_3556753.jpg: 0.8325
# ISIC_1510820.jpg vs ISIC_3557281.jpg: 0.7884
# ISIC_1510820.jpg vs ISIC_3557534.jpg: 0.6073
# ISIC_1510820.jpg vs ISIC_3560198.jpg: 0.3793
# ISIC_1510820.jpg vs ISIC_3561053.jpg: 0.2765
# ISIC_1510820.jpg vs ISIC_3562440.jpg: 0.4810
# ISIC_1510820.jpg vs ISIC_3563115.jpg: 0.6827
# ISIC_1510820.jpg vs ISIC_3566617.jpg: 0.7928
# ISIC_1510820.jpg vs ISIC_3570453.jpg: 0.6870
# ISIC_1510820.jpg vs ISIC_3570575.jpg: 0.6490
# ISIC_1510820.jpg vs ISIC_3571742.jpg: 0.8035
# ISIC_1510820.jpg vs ISIC_3571995.jpg: 0.2105
# ISIC_1510820.jpg vs ISIC_3572957.jpg: 0.3456
# ISIC_1510820.jpg vs ISIC_3573529.jpg: 0.1154
# ISIC_1510820.jpg vs ISIC_3574848.jpg: 0.8284
# ISIC_1510820.jpg vs ISIC_3575125.jpg: 0.6496
# ISIC_1510820.jpg vs ISIC_3575423.jpg: 0.4702
# ISIC_1510820.jpg vs ISIC_3575990.jpg: 0.5607
# ISIC_1510820.jpg vs ISIC_3576876.jpg: 0.7605
# ISIC_1510820.jpg vs ISIC_3577035.jpg: 0.7289
# ISIC_1510820.jpg vs ISIC_3577605.jpg: 0.7783
# ISIC_1510820.jpg vs ISIC_3577723.jpg: 0.6730
# ISIC_1510820.jpg vs ISIC_3580555.jpg: 0.5526
# ISIC_1510820.jpg vs ISIC_3581674.jpg: 0.1692
# ISIC_1510820.jpg vs ISIC_3582682.jpg: 0.6524
# ISIC_1510820.jpg vs ISIC_3582780.jpg: 0.3853
# ISIC_1510820.jpg vs ISIC_3584949.jpg: 0.2318
# ISIC_1510820.jpg vs ISIC_3586356.jpg: 0.6247
# ISIC_1510820.jpg vs ISIC_3588706.jpg: 0.7729
# ISIC_1510820.jpg vs ISIC_3589090.jpg: 0.8407
# ISIC_1510820.jpg vs ISIC_3590140.jpg: 0.6290
# ISIC_1510820.jpg vs ISIC_3592077.jpg: 0.6442
# ISIC_1510820.jpg vs ISIC_3592668.jpg: 0.7063
# ISIC_1510820.jpg vs ISIC_3592904.jpg: 0.7741
# ISIC_1510820.jpg vs ISIC_3594380.jpg: 0.8360
# ISIC_1510820.jpg vs ISIC_3596358.jpg: 0.3362
# ISIC_1510820.jpg vs ISIC_3597076.jpg: 0.5608
# ISIC_1510820.jpg vs ISIC_3597077.jpg: 0.8370
# ISIC_1510820.jpg vs ISIC_3597421.jpg: 0.8990
# ISIC_1510820.jpg vs ISIC_3597902.jpg: 0.4007
# ISIC_1510820.jpg vs ISIC_3598655.jpg: 0.8427
# ISIC_1510820.jpg vs ISIC_3599510.jpg: 0.2519
# ISIC_1510820.jpg vs ISIC_3600277.jpg: 0.5061
# ISIC_1510820.jpg vs ISIC_3600426.jpg: 0.4254
# ISIC_1510820.jpg vs ISIC_3607787.jpg: 0.2948
# ISIC_1510820.jpg vs ISIC_3608296.jpg: 0.2567
# ISIC_1510820.jpg vs ISIC_3608624.jpg: 0.7640
# ISIC_1510820.jpg vs ISIC_3609057.jpg: 0.5833
# ISIC_1510820.jpg vs ISIC_3610466.jpg: 0.1907
# ISIC_1510820.jpg vs ISIC_3612582.jpg: 0.7592
# ISIC_1510820.jpg vs ISIC_3613018.jpg: 0.4613
# ISIC_1510820.jpg vs ISIC_3614842.jpg: 0.4756
# ISIC_1510820.jpg vs ISIC_3616064.jpg: 0.7176
# ISIC_1510820.jpg vs ISIC_3616494.jpg: 0.7943
# ISIC_1510820.jpg vs ISIC_3616814.jpg: 0.5169
# ISIC_1510820.jpg vs ISIC_3616871.jpg: 0.4610
# ISIC_1510820.jpg vs ISIC_3617966.jpg: 0.6439
# ISIC_1510820.jpg vs ISIC_3619413.jpg: 0.6993
# ISIC_1510820.jpg vs ISIC_3620608.jpg: 0.8073
# ISIC_1510820.jpg vs ISIC_3622629.jpg: 0.9078
# ISIC_1510820.jpg vs ISIC_3623502.jpg: 0.8495
# ISIC_1510820.jpg vs ISIC_3624539.jpg: 0.5996
# ISIC_1510820.jpg vs ISIC_3625918.jpg: 0.8419
# ISIC_1510820.jpg vs ISIC_3626651.jpg: 0.6116
# ISIC_1510820.jpg vs ISIC_3627405.jpg: 0.7935
# ISIC_1510820.jpg vs ISIC_3628995.jpg: 0.6691
# ISIC_1510820.jpg vs ISIC_3630213.jpg: 0.3380
# ISIC_1510820.jpg vs ISIC_3631216.jpg: 0.2006
# ISIC_1510820.jpg vs ISIC_3632526.jpg: 0.3763
# ISIC_1510820.jpg vs ISIC_3633517.jpg: 0.8621
# ISIC_1510820.jpg vs ISIC_3634305.jpg: 0.8397
# ISIC_1510820.jpg vs ISIC_3636661.jpg: 0.6464
# ISIC_1510820.jpg vs ISIC_3637224.jpg: 0.7071
# ISIC_1510820.jpg vs ISIC_3639434.jpg: 0.6573
# ISIC_1510820.jpg vs ISIC_3641962.jpg: 0.1302
# ISIC_1510820.jpg vs ISIC_3642019.jpg: 0.8367
# ISIC_1510820.jpg vs ISIC_3642221.jpg: 0.6611
# ISIC_1510820.jpg vs ISIC_3646060.jpg: 0.8097
# ISIC_1510820.jpg vs ISIC_3648148.jpg: 0.4892
# ISIC_1510820.jpg vs ISIC_3649426.jpg: 0.6913
# ISIC_1510820.jpg vs ISIC_3653199.jpg: 0.4183
# ISIC_1510820.jpg vs ISIC_3653457.jpg: 0.5004
# ISIC_1510820.jpg vs ISIC_3656476.jpg: 0.6975
# ISIC_1510820.jpg vs ISIC_3663431.jpg: 0.6461
# ISIC_1510820.jpg vs ISIC_3664196.jpg: 0.7219
# ISIC_1510820.jpg vs ISIC_3664322.jpg: 0.5541
# ISIC_1510820.jpg vs ISIC_3665246.jpg: 0.6783
# ISIC_1510820.jpg vs ISIC_3666418.jpg: 0.7211
# ISIC_1510820.jpg vs ISIC_3666465.jpg: 0.7901
# ISIC_1510820.jpg vs ISIC_3666967.jpg: 0.1086
# ISIC_1510820.jpg vs ISIC_3668633.jpg: 0.8844
# ISIC_1510820.jpg vs ISIC_3669370.jpg: 0.4679
# ISIC_1510820.jpg vs ISIC_3669944.jpg: 0.4333
# ISIC_1510820.jpg vs ISIC_3670111.jpg: 0.8190
# ISIC_1510820.jpg vs ISIC_3671995.jpg: 0.6202
# ISIC_1510820.jpg vs ISIC_3673079.jpg: 0.3242
# ISIC_1510820.jpg vs ISIC_3673788.jpg: 0.5434
# ISIC_1510820.jpg vs ISIC_3674608.jpg: 0.8086
# ISIC_1510820.jpg vs ISIC_3674889.jpg: 0.7584
# ISIC_1510820.jpg vs ISIC_3675678.jpg: 0.8782
# ISIC_1510820.jpg vs ISIC_3677875.jpg: 0.7438
# ISIC_1510820.jpg vs ISIC_3678349.jpg: 0.8288
# ISIC_1510820.jpg vs ISIC_3680848.jpg: 0.6483
# ISIC_1510820.jpg vs ISIC_3681319.jpg: 0.8156
# ISIC_1510820.jpg vs ISIC_3681605.jpg: 0.2530
# ISIC_1510820.jpg vs ISIC_3683317.jpg: 0.8537
# ISIC_1510820.jpg vs ISIC_3684862.jpg: 0.3449
# ISIC_1510820.jpg vs ISIC_3685534.jpg: 0.5422
# ISIC_1510820.jpg vs ISIC_3685585.jpg: 0.6984
# ISIC_1510820.jpg vs ISIC_3689081.jpg: 0.7442
# ISIC_1510820.jpg vs ISIC_3689093.jpg: 0.8036
# ISIC_1510820.jpg vs ISIC_3689148.jpg: 0.8345
# ISIC_1510820.jpg vs ISIC_3689236.jpg: 0.4383
# ISIC_1510820.jpg vs ISIC_3689290.jpg: 0.1406
# ISIC_1510820.jpg vs ISIC_3689531.jpg: 0.8491
# ISIC_1510820.jpg vs ISIC_3689836.jpg: 0.8569
# ISIC_1510820.jpg vs ISIC_3690426.jpg: 0.6856
# ISIC_1510820.jpg vs ISIC_3691906.jpg: 0.3836
# ISIC_1510820.jpg vs ISIC_3692505.jpg: 0.8097
# ISIC_1510820.jpg vs ISIC_3693659.jpg: 0.5699
# ISIC_1510820.jpg vs ISIC_3695954.jpg: 0.7753
# ISIC_1510820.jpg vs ISIC_3697615.jpg: 0.5495
# ISIC_1510820.jpg vs ISIC_3699368.jpg: 0.7497
# ISIC_1510820.jpg vs ISIC_3699566.jpg: 0.7395
# ISIC_1510820.jpg vs ISIC_3701760.jpg: 0.6838
# ISIC_1510820.jpg vs ISIC_3701864.jpg: 0.6488
# ISIC_1510820.jpg vs ISIC_3701985.jpg: 0.6655
# ISIC_1510820.jpg vs ISIC_3702535.jpg: 0.7780
# ISIC_1510820.jpg vs ISIC_3702871.jpg: 0.7738
# ISIC_1510820.jpg vs ISIC_3703228.jpg: 0.4663
# ISIC_1510820.jpg vs ISIC_3703571.jpg: 0.7648
# ISIC_1510820.jpg vs ISIC_3705025.jpg: 0.4084
# ISIC_1510820.jpg vs ISIC_3705644.jpg: 0.1759
# ISIC_1510820.jpg vs ISIC_3707677.jpg: 0.8809
# ISIC_1510820.jpg vs ISIC_3707681.jpg: 0.7922
# ISIC_1510820.jpg vs ISIC_3709384.jpg: 0.8953
# ISIC_1510820.jpg vs ISIC_3710118.jpg: 0.7964
# ISIC_1510820.jpg vs ISIC_3710566.jpg: 0.6503
# ISIC_1510820.jpg vs ISIC_3711604.jpg: 0.3915
# ISIC_1510820.jpg vs ISIC_3712623.jpg: 0.7981
# ISIC_1510820.jpg vs ISIC_3713521.jpg: 0.5641
# ISIC_1510820.jpg vs ISIC_3713800.jpg: 0.5118
# ISIC_1510820.jpg vs ISIC_3715808.jpg: 0.5354
# ISIC_1510820.jpg vs ISIC_3715917.jpg: 0.6711
# ISIC_1510820.jpg vs ISIC_3716403.jpg: 0.6933
# ISIC_1510820.jpg vs ISIC_3717405.jpg: 0.7683
# ISIC_1510820.jpg vs ISIC_3721415.jpg: 0.6747
# ISIC_1510820.jpg vs ISIC_3722094.jpg: 0.3529
# ISIC_1510820.jpg vs ISIC_3722127.jpg: 0.5646
# ISIC_1510820.jpg vs ISIC_3722660.jpg: 0.3238
# ISIC_1510820.jpg vs ISIC_3723953.jpg: 0.3744
# ISIC_1510820.jpg vs ISIC_3724889.jpg: 0.6662
# ISIC_1510820.jpg vs ISIC_3725123.jpg: 0.5631
# ISIC_1510820.jpg vs ISIC_3727502.jpg: 0.4980
# ISIC_1510820.jpg vs ISIC_3730304.jpg: 0.3837
# ISIC_1510820.jpg vs ISIC_3734151.jpg: 0.4799
# ISIC_1510820.jpg vs ISIC_3735686.jpg: 0.7775
# ISIC_1510820.jpg vs ISIC_3735739.jpg: 0.4455
# ISIC_1510820.jpg vs ISIC_3735944.jpg: 0.6642
# ISIC_1510820.jpg vs ISIC_3736099.jpg: 0.2946
# ISIC_1510820.jpg vs ISIC_3737914.jpg: 0.7297
# ISIC_1510820.jpg vs ISIC_3738610.jpg: 0.7991
# ISIC_1510820.jpg vs ISIC_3740111.jpg: 0.5265
# ISIC_1510820.jpg vs ISIC_3740887.jpg: 0.7985
# ISIC_1510820.jpg vs ISIC_3742754.jpg: 0.6861
# ISIC_1510820.jpg vs ISIC_3744551.jpg: 0.7612
# ISIC_1510820.jpg vs ISIC_3744571.jpg: 0.8451
# ISIC_1510820.jpg vs ISIC_3744712.jpg: 0.6671
# ISIC_1510820.jpg vs ISIC_3746770.jpg: 0.6642
# ISIC_1510820.jpg vs ISIC_3751401.jpg: 0.9159
# ISIC_1510820.jpg vs ISIC_3751463.jpg: 0.2417
# ISIC_1510820.jpg vs ISIC_3751922.jpg: 0.5733
# ISIC_1510820.jpg vs ISIC_3752717.jpg: 0.8857
# ISIC_1510820.jpg vs ISIC_3752905.jpg: 0.4944
# ISIC_1510820.jpg vs ISIC_3753916.jpg: 0.7248
# ISIC_1510820.jpg vs ISIC_3757475.jpg: 0.3146
# ISIC_1510820.jpg vs ISIC_3759005.jpg: 0.4383
# ISIC_1510820.jpg vs ISIC_3760758.jpg: 0.5453
# ISIC_1510820.jpg vs ISIC_3761263.jpg: 0.1892
# ISIC_1510820.jpg vs ISIC_3761540.jpg: 0.7628
# ISIC_1510820.jpg vs ISIC_3764100.jpg: 0.3738
# ISIC_1510820.jpg vs ISIC_3764369.jpg: 0.8183
# ISIC_1510820.jpg vs ISIC_3765694.jpg: 0.6106
# ISIC_1510820.jpg vs ISIC_3765884.jpg: 0.5870
# ISIC_1510820.jpg vs ISIC_3766707.jpg: 0.6291
# ISIC_1510820.jpg vs ISIC_3768327.jpg: 0.8009
# ISIC_1510820.jpg vs ISIC_3771998.jpg: 0.7172
# ISIC_1510820.jpg vs ISIC_3772083.jpg: 0.7177
# ISIC_1510820.jpg vs ISIC_3773083.jpg: 0.8039
# ISIC_1510820.jpg vs ISIC_3778164.jpg: 0.5952
# ISIC_1510820.jpg vs ISIC_3778773.jpg: 0.6119
# ISIC_1510820.jpg vs ISIC_3781034.jpg: 0.4826
# ISIC_1510820.jpg vs ISIC_3784911.jpg: 0.2885
# ISIC_1510820.jpg vs ISIC_3784989.jpg: 0.8193
# ISIC_1510820.jpg vs ISIC_3790496.jpg: 0.4608
# ISIC_1510820.jpg vs ISIC_3793804.jpg: 0.5807
# ISIC_1510820.jpg vs ISIC_3794213.jpg: 0.3446
# ISIC_1510820.jpg vs ISIC_3795144.jpg: 0.5780
# ISIC_1510820.jpg vs ISIC_3798140.jpg: 0.4210
# ISIC_1510820.jpg vs ISIC_3801213.jpg: 0.4494
# ISIC_1510820.jpg vs ISIC_3802684.jpg: 0.4323
# ISIC_1510820.jpg vs ISIC_3804041.jpg: 0.7453
# ISIC_1510820.jpg vs ISIC_3804339.jpg: 0.7410
# ISIC_1510820.jpg vs ISIC_3805982.jpg: 0.7465
# ISIC_1510820.jpg vs ISIC_3807024.jpg: 0.7444
# ISIC_1510820.jpg vs ISIC_3807083.jpg: 0.7214
# ISIC_1510820.jpg vs ISIC_3809089.jpg: 0.7282
# ISIC_1510820.jpg vs ISIC_3809167.jpg: 0.6378
# ISIC_1510820.jpg vs ISIC_3809896.jpg: 0.1572
# ISIC_1510820.jpg vs ISIC_3809996.jpg: 0.8527
# ISIC_1510820.jpg vs ISIC_3810006.jpg: 0.6559
# ISIC_1510820.jpg vs ISIC_3810856.jpg: 0.7551
# ISIC_1510820.jpg vs ISIC_3811472.jpg: 0.8259
# ISIC_1510820.jpg vs ISIC_3812398.jpg: 0.7422
# ISIC_1510820.jpg vs ISIC_3812758.jpg: 0.5848
# ISIC_1510820.jpg vs ISIC_3812886.jpg: 0.7234
# ISIC_1510820.jpg vs ISIC_3813860.jpg: 0.4313
# ISIC_1510820.jpg vs ISIC_3814018.jpg: 0.9143
# ISIC_1510820.jpg vs ISIC_3815181.jpg: 0.1964
# ISIC_1510820.jpg vs ISIC_3815385.jpg: 0.7476
# ISIC_1510820.jpg vs ISIC_3817642.jpg: 0.5184
# ISIC_1510820.jpg vs ISIC_3817681.jpg: 0.8675
# ISIC_1510820.jpg vs ISIC_3817768.jpg: 0.5508
# ISIC_1510820.jpg vs ISIC_3818797.jpg: 0.7690
# ISIC_1510820.jpg vs ISIC_3819363.jpg: 0.3307
# ISIC_1510820.jpg vs ISIC_3819475.jpg: 0.6519
# ISIC_1510820.jpg vs ISIC_3819795.jpg: 0.4520
# ISIC_1510820.jpg vs ISIC_3820411.jpg: 0.3492
# ISIC_1510820.jpg vs ISIC_3821169.jpg: 0.6582
# ISIC_1510820.jpg vs ISIC_3821340.jpg: 0.1307
# ISIC_1510820.jpg vs ISIC_3822511.jpg: 0.5673
# ISIC_1510820.jpg vs ISIC_3822853.jpg: 0.7154
# ISIC_1510820.jpg vs ISIC_3823679.jpg: 0.8751
# ISIC_1510820.jpg vs ISIC_3824624.jpg: 0.7794
# ISIC_1510820.jpg vs ISIC_3825943.jpg: 0.7976
# ISIC_1510820.jpg vs ISIC_3826140.jpg: 0.7603
# ISIC_1510820.jpg vs ISIC_3828515.jpg: 0.6138
# ISIC_1510820.jpg vs ISIC_3829545.jpg: 0.5362
# ISIC_1510820.jpg vs ISIC_3829596.jpg: 0.7809
# ISIC_1510820.jpg vs ISIC_3829724.jpg: 0.5735
# ISIC_1510820.jpg vs ISIC_3832497.jpg: 0.8900
# ISIC_1510820.jpg vs ISIC_3833945.jpg: 0.8573
# ISIC_1510820.jpg vs ISIC_3834329.jpg: 0.5058
# ISIC_1510820.jpg vs ISIC_3834484.jpg: 0.6370
# ISIC_1510820.jpg vs ISIC_3834937.jpg: 0.4362
# ISIC_1510820.jpg vs ISIC_3835548.jpg: 0.8485
# ISIC_1510820.jpg vs ISIC_3837036.jpg: 0.7873
# ISIC_1510820.jpg vs ISIC_3837251.jpg: 0.7000
# ISIC_1510820.jpg vs ISIC_3840152.jpg: 0.7320
# ISIC_1510820.jpg vs ISIC_3840945.jpg: 0.5676
# ISIC_1510820.jpg vs ISIC_3841665.jpg: 0.8310
# ISIC_1510820.jpg vs ISIC_3843316.jpg: 0.6765
# ISIC_1510820.jpg vs ISIC_3845109.jpg: 0.7592
# ISIC_1510820.jpg vs ISIC_3849267.jpg: 0.7101
# ISIC_1510820.jpg vs ISIC_3849492.jpg: 0.7546
# ISIC_1510820.jpg vs ISIC_3852835.jpg: 0.8297
# ISIC_1510820.jpg vs ISIC_3853277.jpg: 0.8038
# ISIC_1510820.jpg vs ISIC_3855425.jpg: 0.7620
# ISIC_1510820.jpg vs ISIC_3855503.jpg: 0.6350
# ISIC_1510820.jpg vs ISIC_3858353.jpg: 0.5497
# ISIC_1510820.jpg vs ISIC_3861014.jpg: 0.5496
# ISIC_1510820.jpg vs ISIC_3861559.jpg: 0.2054
# ISIC_1510820.jpg vs ISIC_3864247.jpg: 0.4609
# ISIC_1510820.jpg vs ISIC_3864836.jpg: 0.5741
# ISIC_1510820.jpg vs ISIC_3867821.jpg: 0.7297
# ISIC_1510820.jpg vs ISIC_3869964.jpg: 0.1816
# ISIC_1510820.jpg vs ISIC_3870547.jpg: 0.7609
# ISIC_1510820.jpg vs ISIC_3870921.jpg: 0.3268
# ISIC_1510820.jpg vs ISIC_3871000.jpg: 0.8494
# ISIC_1510820.jpg vs ISIC_3872067.jpg: 0.6712
# ISIC_1510820.jpg vs ISIC_3872083.jpg: 0.2456
# ISIC_1510820.jpg vs ISIC_3872285.jpg: 0.5715
# ISIC_1510820.jpg vs ISIC_3874776.jpg: 0.3736
# ISIC_1510820.jpg vs ISIC_3875284.jpg: 0.8664
# ISIC_1510820.jpg vs ISIC_3875740.jpg: 0.6484
# ISIC_1510820.jpg vs ISIC_3876748.jpg: 0.7692
# ISIC_1510820.jpg vs ISIC_3878964.jpg: 0.7359
# ISIC_1510820.jpg vs ISIC_3879706.jpg: 0.6633
# ISIC_1510820.jpg vs ISIC_3879850.jpg: 0.9032
# ISIC_1510820.jpg vs ISIC_3880022.jpg: 0.7050
# ISIC_1510820.jpg vs ISIC_3881407.jpg: 0.6590
# ISIC_1510820.jpg vs ISIC_3881920.jpg: 0.8146
# ISIC_1510820.jpg vs ISIC_3882145.jpg: 0.7762
# ISIC_1510820.jpg vs ISIC_3883117.jpg: 0.4539
# ISIC_1510820.jpg vs ISIC_3883863.jpg: 0.8439
# ISIC_1510820.jpg vs ISIC_3884398.jpg: 0.9144
# ISIC_1510820.jpg vs ISIC_3884657.jpg: 0.6170
# ISIC_1510820.jpg vs ISIC_3885191.jpg: 0.2955
# ISIC_1510820.jpg vs ISIC_3887326.jpg: 0.6309
# ISIC_1510820.jpg vs ISIC_3887998.jpg: 0.2728
# ISIC_1510820.jpg vs ISIC_3890626.jpg: 0.2363
# ISIC_1510820.jpg vs ISIC_3891628.jpg: 0.7175
# ISIC_1510820.jpg vs ISIC_3891712.jpg: 0.6628
# ISIC_1510820.jpg vs ISIC_3893736.jpg: 0.8742
# ISIC_1510820.jpg vs ISIC_3895426.jpg: 0.1575
# ISIC_1510820.jpg vs ISIC_3895545.jpg: 0.8117
# ISIC_1510820.jpg vs ISIC_3898500.jpg: 0.6385
# ISIC_1510820.jpg vs ISIC_3899765.jpg: 0.4567
# ISIC_1510820.jpg vs ISIC_3901048.jpg: 0.5057
# ISIC_1510820.jpg vs ISIC_3901209.jpg: 0.2885
# ISIC_1510820.jpg vs ISIC_3903912.jpg: 0.8548
# ISIC_1510820.jpg vs ISIC_3907519.jpg: 0.6621
# ISIC_1510820.jpg vs ISIC_3908046.jpg: 0.2282
# ISIC_1510820.jpg vs ISIC_3909787.jpg: 0.8479
# ISIC_1510820.jpg vs ISIC_3911173.jpg: 0.8342
# ISIC_1510820.jpg vs ISIC_3912898.jpg: 0.5411
# ISIC_1510820.jpg vs ISIC_3913043.jpg: 0.5113
# ISIC_1510820.jpg vs ISIC_3914490.jpg: 0.7947
# ISIC_1510820.jpg vs ISIC_3918613.jpg: 0.2722
# ISIC_1510820.jpg vs ISIC_3918617.jpg: 0.6113
# ISIC_1510820.jpg vs ISIC_3918811.jpg: 0.7146
# ISIC_1510820.jpg vs ISIC_3920311.jpg: 0.6337
# ISIC_1510820.jpg vs ISIC_3921222.jpg: 0.8335
# ISIC_1510820.jpg vs ISIC_3922363.jpg: 0.5825
# ISIC_1510820.jpg vs ISIC_3923249.jpg: 0.6800
# ISIC_1510820.jpg vs ISIC_3923398.jpg: 0.5955
# ISIC_1510820.jpg vs ISIC_3924003.jpg: 0.6522
# ISIC_1510820.jpg vs ISIC_3924221.jpg: 0.6675
# ISIC_1510820.jpg vs ISIC_3926962.jpg: 0.3134
# ISIC_1510820.jpg vs ISIC_3927334.jpg: 0.3622
# ISIC_1510820.jpg vs ISIC_3928059.jpg: 0.8187
# ISIC_1510820.jpg vs ISIC_3928185.jpg: 0.7934
# ISIC_1510820.jpg vs ISIC_3928327.jpg: 0.1765
# ISIC_1510820.jpg vs ISIC_3930196.jpg: 0.7652
# ISIC_1510820.jpg vs ISIC_3931939.jpg: 0.6378
# ISIC_1510820.jpg vs ISIC_3934137.jpg: 0.9120
# ISIC_1510820.jpg vs ISIC_3937563.jpg: 0.0822
# ISIC_1510820.jpg vs ISIC_3937767.jpg: 0.8135
# ISIC_1510820.jpg vs ISIC_3939975.jpg: 0.5202
# ISIC_1510820.jpg vs ISIC_3940305.jpg: 0.5878
# ISIC_1510820.jpg vs ISIC_3943687.jpg: 0.2592
# ISIC_1510820.jpg vs ISIC_3944323.jpg: 0.7148
# ISIC_1510820.jpg vs ISIC_3945700.jpg: 0.4226
# ISIC_1510820.jpg vs ISIC_3947097.jpg: 0.8011
# ISIC_1510820.jpg vs ISIC_3947454.jpg: 0.4495
# ISIC_1510820.jpg vs ISIC_3949543.jpg: 0.4526
# ISIC_1510820.jpg vs ISIC_3951614.jpg: 0.6361
# ISIC_1510820.jpg vs ISIC_3952076.jpg: 0.8840
# ISIC_1510820.jpg vs ISIC_3955981.jpg: 0.4711
# ISIC_1510820.jpg vs ISIC_3956322.jpg: 0.8938
# ISIC_1510820.jpg vs ISIC_3957302.jpg: 0.6534
# ISIC_1510820.jpg vs ISIC_3958018.jpg: 0.8265
# ISIC_1510820.jpg vs ISIC_3958469.jpg: 0.7500
# ISIC_1510820.jpg vs ISIC_3958513.jpg: 0.6919
# ISIC_1510820.jpg vs ISIC_3959866.jpg: 0.3265
# ISIC_1510820.jpg vs ISIC_3963047.jpg: 0.3552
# ISIC_1510820.jpg vs ISIC_3963758.jpg: 0.9154
# ISIC_1510820.jpg vs ISIC_3965027.jpg: 0.6198
# ISIC_1510820.jpg vs ISIC_3965044.jpg: 0.3595
# ISIC_1510820.jpg vs ISIC_3966780.jpg: 0.7367
# ISIC_1510820.jpg vs ISIC_3968987.jpg: 0.4852
# ISIC_1510820.jpg vs ISIC_3969038.jpg: 0.4905
# ISIC_1510820.jpg vs ISIC_3970274.jpg: 0.7784
# ISIC_1510820.jpg vs ISIC_3970277.jpg: 0.4090
# ISIC_1510820.jpg vs ISIC_3971465.jpg: 0.3691
# ISIC_1510820.jpg vs ISIC_3973166.jpg: 0.5569
# ISIC_1510820.jpg vs ISIC_3974325.jpg: 0.7582
# ISIC_1510820.jpg vs ISIC_3974947.jpg: 0.2780
# ISIC_1510820.jpg vs ISIC_3985101.jpg: 0.8728
# ISIC_1510820.jpg vs ISIC_3987387.jpg: 0.8343
# ISIC_1510820.jpg vs ISIC_3988372.jpg: 0.6146
# ISIC_1510820.jpg vs ISIC_3989004.jpg: 0.8721
# ISIC_1510820.jpg vs ISIC_3989338.jpg: 0.6054
# ISIC_1510820.jpg vs ISIC_3990959.jpg: 0.8388
# ISIC_1510820.jpg vs ISIC_3991293.jpg: 0.5503
# ISIC_1510820.jpg vs ISIC_3991801.jpg: 0.6999
# ISIC_1510820.jpg vs ISIC_3994900.jpg: 0.4647
# ISIC_1510820.jpg vs ISIC_3995640.jpg: 0.6887
# ISIC_1510820.jpg vs ISIC_3997576.jpg: 0.4497
# ISIC_1510820.jpg vs ISIC_4002083.jpg: 0.7882
# ISIC_1510820.jpg vs ISIC_4003979.jpg: 0.6885
# ISIC_1510820.jpg vs ISIC_4004059.jpg: 0.8461
# ISIC_1510820.jpg vs ISIC_4004858.jpg: 0.5996
# ISIC_1510820.jpg vs ISIC_4005292.jpg: 0.4186
# ISIC_1510820.jpg vs ISIC_4006453.jpg: 0.6474
# ISIC_1510820.jpg vs ISIC_4016138.jpg: 0.8944
# ISIC_1510820.jpg vs ISIC_4017245.jpg: 0.6838
# ISIC_1510820.jpg vs ISIC_4017926.jpg: 0.6676
# ISIC_1510820.jpg vs ISIC_4018651.jpg: 0.5081
# ISIC_1510820.jpg vs ISIC_4019458.jpg: 0.4875
# ISIC_1510820.jpg vs ISIC_4019832.jpg: 0.4615
# ISIC_1510820.jpg vs ISIC_4021183.jpg: 0.4867
# ISIC_1510820.jpg vs ISIC_4022414.jpg: 0.7773
# ISIC_1510820.jpg vs ISIC_4119147.jpg: 0.4623
# ISIC_1510820.jpg vs ISIC_4123597.jpg: 0.7094
# ISIC_1510820.jpg vs ISIC_4134016.jpg: 0.4955
# ISIC_1510820.jpg vs ISIC_4143173.jpg: 0.5614
# ISIC_1510820.jpg vs ISIC_4156931.jpg: 0.2796
# ISIC_1510820.jpg vs ISIC_4160484.jpg: 0.5002
# ISIC_1510820.jpg vs ISIC_4161439.jpg: 0.6825
# ISIC_1510820.jpg vs ISIC_4163501.jpg: 0.7463
# ISIC_1510820.jpg vs ISIC_4167596.jpg: 0.8635
# ISIC_1510820.jpg vs ISIC_4171132.jpg: 0.5791
# ISIC_1510820.jpg vs ISIC_4173173.jpg: 0.5683
# ISIC_1510820.jpg vs ISIC_4174959.jpg: 0.8700
# ISIC_1510820.jpg vs ISIC_4179187.jpg: 0.5715
# ISIC_1510820.jpg vs ISIC_4179495.jpg: 0.5528
# ISIC_1510820.jpg vs ISIC_4181203.jpg: 0.6243
# ISIC_1510820.jpg vs ISIC_4181217.jpg: 0.7888
# ISIC_1510820.jpg vs ISIC_4181903.jpg: 0.6876
# ISIC_1510820.jpg vs ISIC_4182656.jpg: 0.7183
# ISIC_1510820.jpg vs ISIC_4186549.jpg: 0.7753
# ISIC_1510820.jpg vs ISIC_4186863.jpg: 0.5166
# ISIC_1510820.jpg vs ISIC_4187088.jpg: 0.6718
# ISIC_1510820.jpg vs ISIC_4187200.jpg: 0.4779
# ISIC_1510820.jpg vs ISIC_4187880.jpg: 0.2662
# ISIC_1510820.jpg vs ISIC_4188214.jpg: 0.7216
# ISIC_1510820.jpg vs ISIC_4188332.jpg: 0.8340
# ISIC_1510820.jpg vs ISIC_4191055.jpg: 0.2498
# ISIC_1510820.jpg vs ISIC_4192645.jpg: 0.6863
# ISIC_1510820.jpg vs ISIC_4194430.jpg: 0.2530
# ISIC_1510820.jpg vs ISIC_4194768.jpg: 0.5374
# ISIC_1510820.jpg vs ISIC_4195092.jpg: 0.3341
# ISIC_1510820.jpg vs ISIC_4197980.jpg: 0.8792
# ISIC_1510820.jpg vs ISIC_4198983.jpg: 0.4744
# ISIC_1510820.jpg vs ISIC_4199184.jpg: 0.8739
# ISIC_1510820.jpg vs ISIC_4199799.jpg: 0.8546
# ISIC_1510820.jpg vs ISIC_4201584.jpg: 0.3737
# ISIC_1510820.jpg vs ISIC_4203773.jpg: 0.4149
# ISIC_1510820.jpg vs ISIC_4204053.jpg: 0.7156
# ISIC_1510820.jpg vs ISIC_4204071.jpg: 0.7840
# ISIC_1510820.jpg vs ISIC_4205309.jpg: 0.5429
# ISIC_1510820.jpg vs ISIC_4206108.jpg: 0.7756
# ISIC_1510820.jpg vs ISIC_4206434.jpg: 0.6073
# ISIC_1510820.jpg vs ISIC_4206778.jpg: 0.8785
# ISIC_1510820.jpg vs ISIC_4209184.jpg: 0.6101
# ISIC_1510820.jpg vs ISIC_4210823.jpg: 0.7905
# ISIC_1510820.jpg vs ISIC_4211013.jpg: 0.8477
# ISIC_1510820.jpg vs ISIC_4211964.jpg: 0.8677
# ISIC_1510820.jpg vs ISIC_4214570.jpg: 0.4650
# ISIC_1510820.jpg vs ISIC_4216580.jpg: 0.4858
# ISIC_1510820.jpg vs ISIC_4218911.jpg: 0.4051
# ISIC_1510820.jpg vs ISIC_4219443.jpg: 0.2431
# ISIC_1510820.jpg vs ISIC_4222477.jpg: 0.4639
# ISIC_1510820.jpg vs ISIC_4223005.jpg: 0.5562
# ISIC_1510820.jpg vs ISIC_4225544.jpg: 0.2761
# ISIC_1510820.jpg vs ISIC_4226938.jpg: 0.7238
# ISIC_1510820.jpg vs ISIC_4233105.jpg: 0.7625
# ISIC_1510820.jpg vs ISIC_4233329.jpg: 0.7788
# ISIC_1510820.jpg vs ISIC_4235180.jpg: 0.4292
# ISIC_1510820.jpg vs ISIC_4235197.jpg: 0.1810
# ISIC_1510820.jpg vs ISIC_4236192.jpg: 0.4433
# ISIC_1510820.jpg vs ISIC_4236332.jpg: 0.4492
# ISIC_1510820.jpg vs ISIC_4240414.jpg: 0.8393
# ISIC_1510820.jpg vs ISIC_4241007.jpg: 0.7395
# ISIC_1510820.jpg vs ISIC_4242171.jpg: 0.5453
# ISIC_1510820.jpg vs ISIC_4242517.jpg: 0.5408
# ISIC_1510820.jpg vs ISIC_4247203.jpg: 0.5437
# ISIC_1510820.jpg vs ISIC_4247641.jpg: 0.8048
# ISIC_1510820.jpg vs ISIC_4248295.jpg: 0.3909
# ISIC_1510820.jpg vs ISIC_4249430.jpg: 0.7945
# ISIC_1510820.jpg vs ISIC_4251600.jpg: 0.1609
# ISIC_1510820.jpg vs ISIC_4252331.jpg: 0.4018
# ISIC_1510820.jpg vs ISIC_4253256.jpg: 0.3065
# ISIC_1510820.jpg vs ISIC_4255739.jpg: 0.5182
# ISIC_1510820.jpg vs ISIC_4256374.jpg: 0.6010
# ISIC_1510820.jpg vs ISIC_4257030.jpg: 0.8578
# ISIC_1510820.jpg vs ISIC_4258047.jpg: 0.2883
# ISIC_1510820.jpg vs ISIC_4258959.jpg: 0.3060
# ISIC_1510820.jpg vs ISIC_4259013.jpg: 0.3851
# ISIC_1510820.jpg vs ISIC_4261613.jpg: 0.4319
# ISIC_1510820.jpg vs ISIC_4263498.jpg: 0.7765
# ISIC_1510820.jpg vs ISIC_4266201.jpg: 0.6997
# ISIC_1510820.jpg vs ISIC_4267507.jpg: 0.3868
# ISIC_1510820.jpg vs ISIC_4267994.jpg: 0.3522
# ISIC_1510820.jpg vs ISIC_4270186.jpg: 0.8340
# ISIC_1510820.jpg vs ISIC_4271057.jpg: 0.2973
# ISIC_1510820.jpg vs ISIC_4271848.jpg: 0.7804
# ISIC_1510820.jpg vs ISIC_4273322.jpg: 0.8239
# ISIC_1510820.jpg vs ISIC_4274756.jpg: 0.9140
# ISIC_1510820.jpg vs ISIC_4278988.jpg: 0.2273
# ISIC_1510820.jpg vs ISIC_4279681.jpg: 0.5220
# ISIC_1510820.jpg vs ISIC_4281444.jpg: 0.6932
# ISIC_1510820.jpg vs ISIC_4281475.jpg: 0.6020
# ISIC_1510820.jpg vs ISIC_4282399.jpg: 0.4868
# ISIC_1510820.jpg vs ISIC_4285809.jpg: 0.8532
# ISIC_1510820.jpg vs ISIC_4286823.jpg: 0.6286
# ISIC_1510820.jpg vs ISIC_4286970.jpg: 0.1420
# ISIC_1510820.jpg vs ISIC_4288417.jpg: 0.8769
# ISIC_1510820.jpg vs ISIC_4289007.jpg: 0.8562
# ISIC_1510820.jpg vs ISIC_4290148.jpg: 0.5884
# ISIC_1510820.jpg vs ISIC_4290765.jpg: 0.1631
# ISIC_1510820.jpg vs ISIC_4291126.jpg: 0.7118
# ISIC_1510820.jpg vs ISIC_4293728.jpg: 0.5100
# ISIC_1510820.jpg vs ISIC_4294619.jpg: 0.2746
# ISIC_1510820.jpg vs ISIC_4296526.jpg: 0.0982
# ISIC_1510820.jpg vs ISIC_4296557.jpg: 0.6808
# ISIC_1510820.jpg vs ISIC_4296605.jpg: 0.7551
# ISIC_1510820.jpg vs ISIC_4300625.jpg: 0.9000
# ISIC_1510820.jpg vs ISIC_4300846.jpg: 0.2934
# ISIC_1510820.jpg vs ISIC_4301916.jpg: 0.2551
# ISIC_1510820.jpg vs ISIC_4302347.jpg: 0.8374
# ISIC_1510820.jpg vs ISIC_4302545.jpg: 0.3726
# ISIC_1510820.jpg vs ISIC_4303326.jpg: 0.7977
# ISIC_1510820.jpg vs ISIC_4305063.jpg: 0.7275
# ISIC_1510820.jpg vs ISIC_4305480.jpg: 0.4268
# ISIC_1510820.jpg vs ISIC_4305950.jpg: 0.8084
# ISIC_1510820.jpg vs ISIC_4305990.jpg: 0.6870
# ISIC_1510820.jpg vs ISIC_4308121.jpg: 0.6363
# ISIC_1510820.jpg vs ISIC_4309705.jpg: 0.8227
# ISIC_1510820.jpg vs ISIC_4309998.jpg: 0.5364
# ISIC_1510820.jpg vs ISIC_4312836.jpg: 0.9040
# ISIC_1510820.jpg vs ISIC_4313110.jpg: 0.8036
# ISIC_1510820.jpg vs ISIC_4313524.jpg: 0.8069
# ISIC_1510820.jpg vs ISIC_4314656.jpg: 0.7604
# ISIC_1510820.jpg vs ISIC_4316187.jpg: 0.5991
# ISIC_1510820.jpg vs ISIC_4316707.jpg: 0.7177
# ISIC_1510820.jpg vs ISIC_4317606.jpg: 0.7543
# ISIC_1510820.jpg vs ISIC_4319839.jpg: 0.6031
# ISIC_1510820.jpg vs ISIC_4320065.jpg: 0.5780
# ISIC_1510820.jpg vs ISIC_4323634.jpg: 0.7326
# ISIC_1510820.jpg vs ISIC_4323977.jpg: 0.7384
# ISIC_1510820.jpg vs ISIC_4327951.jpg: 0.8063
# ISIC_1510820.jpg vs ISIC_4327964.jpg: 0.4114
# ISIC_1510820.jpg vs ISIC_4328195.jpg: 0.7928
# ISIC_1510820.jpg vs ISIC_4330485.jpg: 0.8495
# ISIC_1510820.jpg vs ISIC_4331890.jpg: 0.5474
# ISIC_1510820.jpg vs ISIC_4332761.jpg: 0.8015
# ISIC_1510820.jpg vs ISIC_4333529.jpg: 0.3607
# ISIC_1510820.jpg vs ISIC_4333983.jpg: 0.6720
# ISIC_1510820.jpg vs ISIC_4336438.jpg: 0.6372
# ISIC_1510820.jpg vs ISIC_4339577.jpg: 0.2645
# ISIC_1510820.jpg vs ISIC_4341449.jpg: 0.3954
# ISIC_1510820.jpg vs ISIC_4341708.jpg: 0.2331
# ISIC_1510820.jpg vs ISIC_4342558.jpg: 0.7985
# ISIC_1510820.jpg vs ISIC_4342571.jpg: 0.7743
# ISIC_1510820.jpg vs ISIC_4342824.jpg: 0.5930
# ISIC_1510820.jpg vs ISIC_4342825.jpg: 0.4154
# ISIC_1510820.jpg vs ISIC_4343422.jpg: 0.6458
# ISIC_1510820.jpg vs ISIC_4346003.jpg: 0.7526
# ISIC_1510820.jpg vs ISIC_4347871.jpg: 0.7512
# ISIC_1510820.jpg vs ISIC_4348477.jpg: 0.5229
# ISIC_1510820.jpg vs ISIC_4351185.jpg: 0.4442
# ISIC_1510820.jpg vs ISIC_4351557.jpg: 0.4387
# ISIC_1510820.jpg vs ISIC_4352355.jpg: 0.7299
# ISIC_1510820.jpg vs ISIC_4355988.jpg: 0.8837
# ISIC_1510820.jpg vs ISIC_4356606.jpg: 0.8246
# ISIC_1510820.jpg vs ISIC_4356936.jpg: 0.8320
# ISIC_1510820.jpg vs ISIC_4358060.jpg: 0.4422
# ISIC_1510820.jpg vs ISIC_4359352.jpg: 0.1560
# ISIC_1510820.jpg vs ISIC_4360093.jpg: 0.5927
# ISIC_1510820.jpg vs ISIC_4360109.jpg: 0.6668
# ISIC_1510820.jpg vs ISIC_4361794.jpg: 0.7108
# ISIC_1510820.jpg vs ISIC_4365909.jpg: 0.5236
# ISIC_1510820.jpg vs ISIC_4367244.jpg: 0.8412
# ISIC_1510820.jpg vs ISIC_4369210.jpg: 0.7142
# ISIC_1510820.jpg vs ISIC_4370063.jpg: 0.7364
# ISIC_1510820.jpg vs ISIC_4371027.jpg: 0.7758
# ISIC_1510820.jpg vs ISIC_4372186.jpg: 0.1978
# ISIC_1510820.jpg vs ISIC_4373405.jpg: 0.5902
# ISIC_1510820.jpg vs ISIC_4373849.jpg: 0.6829
# ISIC_1510820.jpg vs ISIC_4374956.jpg: 0.3898
# ISIC_1510820.jpg vs ISIC_4375150.jpg: 0.8498
# ISIC_1510820.jpg vs ISIC_4376572.jpg: 0.6411
# ISIC_1510820.jpg vs ISIC_4377671.jpg: 0.8523
# ISIC_1510820.jpg vs ISIC_4381076.jpg: 0.4661
# ISIC_1510820.jpg vs ISIC_4382016.jpg: 0.8315
# ISIC_1510820.jpg vs ISIC_4382778.jpg: 0.6580
# ISIC_1510820.jpg vs ISIC_4383639.jpg: 0.7834
# ISIC_1510820.jpg vs ISIC_4384149.jpg: 0.8340
# ISIC_1510820.jpg vs ISIC_4384631.jpg: 0.7418
# ISIC_1510820.jpg vs ISIC_4387462.jpg: 0.8331
# ISIC_1510820.jpg vs ISIC_4387742.jpg: 0.4530
# ISIC_1510820.jpg vs ISIC_4389169.jpg: 0.6147
# ISIC_1510820.jpg vs ISIC_4389340.jpg: 0.4203
# ISIC_1510820.jpg vs ISIC_4390316.jpg: 0.6051
# ISIC_1510820.jpg vs ISIC_4391116.jpg: 0.8599
# ISIC_1510820.jpg vs ISIC_4392497.jpg: 0.8312
# ISIC_1510820.jpg vs ISIC_4394582.jpg: 0.7207
# ISIC_1510820.jpg vs ISIC_4395233.jpg: 0.7245
# ISIC_1510820.jpg vs ISIC_4397375.jpg: 0.5206
# ISIC_1510820.jpg vs ISIC_4397698.jpg: 0.8488
# ISIC_1510820.jpg vs ISIC_4398622.jpg: 0.7130
# ISIC_1510820.jpg vs ISIC_4399176.jpg: 0.5409
# ISIC_1510820.jpg vs ISIC_4399932.jpg: 0.1773
# ISIC_1510820.jpg vs ISIC_4400063.jpg: 0.8882
# ISIC_1510820.jpg vs ISIC_4400253.jpg: 0.6056
# ISIC_1510820.jpg vs ISIC_4402945.jpg: 0.3893
# ISIC_1510820.jpg vs ISIC_4403324.jpg: 0.8023
# ISIC_1510820.jpg vs ISIC_4403513.jpg: 0.8201
# ISIC_1510820.jpg vs ISIC_4405508.jpg: 0.5039
# ISIC_1510820.jpg vs ISIC_4405616.jpg: 0.8519
# ISIC_1510820.jpg vs ISIC_4408816.jpg: 0.5898
# ISIC_1510820.jpg vs ISIC_4410698.jpg: 0.8231
# ISIC_1510820.jpg vs ISIC_4411659.jpg: 0.7553
# ISIC_1510820.jpg vs ISIC_4411770.jpg: 0.8164
# ISIC_1510820.jpg vs ISIC_4413303.jpg: 0.8531
# ISIC_1510820.jpg vs ISIC_4415527.jpg: 0.4265
# ISIC_1510820.jpg vs ISIC_4415966.jpg: 0.4836
# ISIC_1510820.jpg vs ISIC_4416068.jpg: 0.3250
# ISIC_1510820.jpg vs ISIC_4416545.jpg: 0.3153
# ISIC_1510820.jpg vs ISIC_4416593.jpg: 0.7544
# ISIC_1510820.jpg vs ISIC_4416877.jpg: 0.7988
# ISIC_1510820.jpg vs ISIC_4418073.jpg: 0.6337
# ISIC_1510820.jpg vs ISIC_4420890.jpg: 0.1618
# ISIC_1510820.jpg vs ISIC_4420896.jpg: 0.5457
# ISIC_1510820.jpg vs ISIC_4421057.jpg: 0.7284
# ISIC_1510820.jpg vs ISIC_4422783.jpg: 0.8804
# ISIC_1510820.jpg vs ISIC_4422883.jpg: 0.8028
# ISIC_1510820.jpg vs ISIC_4423021.jpg: 0.2654
# ISIC_1510820.jpg vs ISIC_4423247.jpg: 0.2806
# ISIC_1510820.jpg vs ISIC_4424749.jpg: 0.7049
# ISIC_1510820.jpg vs ISIC_4424842.jpg: 0.2149
# ISIC_1510820.jpg vs ISIC_4426517.jpg: 0.8378
# ISIC_1510820.jpg vs ISIC_4427247.jpg: 0.2747
# ISIC_1510820.jpg vs ISIC_4428229.jpg: 0.9073
# ISIC_1510820.jpg vs ISIC_4430138.jpg: 0.4556
# ISIC_1510820.jpg vs ISIC_4430541.jpg: 0.6596
# ISIC_1510820.jpg vs ISIC_4431794.jpg: 0.2233
# ISIC_1510820.jpg vs ISIC_4431911.jpg: 0.7872
# ISIC_1510820.jpg vs ISIC_4432275.jpg: 0.5414
# ISIC_1510820.jpg vs ISIC_4433924.jpg: 0.6905
# ISIC_1510820.jpg vs ISIC_4435605.jpg: 0.3834
# ISIC_1510820.jpg vs ISIC_4438148.jpg: 0.5915
# ISIC_1510820.jpg vs ISIC_4438918.jpg: 0.3077
# ISIC_1510820.jpg vs ISIC_4442628.jpg: 0.8884
# ISIC_1510820.jpg vs ISIC_4442884.jpg: 0.8643
# ISIC_1510820.jpg vs ISIC_4443268.jpg: 0.6300
# ISIC_1510820.jpg vs ISIC_4443863.jpg: 0.6317
# ISIC_1510820.jpg vs ISIC_4445424.jpg: 0.7862
# ISIC_1510820.jpg vs ISIC_4447049.jpg: 0.4259
# ISIC_1510820.jpg vs ISIC_4448639.jpg: 0.3774
# ISIC_1510820.jpg vs ISIC_4454742.jpg: 0.6589
# ISIC_1510820.jpg vs ISIC_4455746.jpg: 0.7743
# ISIC_1510820.jpg vs ISIC_4457555.jpg: 0.4711
# ISIC_1510820.jpg vs ISIC_4458831.jpg: 0.7523
# ISIC_1510820.jpg vs ISIC_4459057.jpg: 0.7527
# ISIC_1510820.jpg vs ISIC_4459725.jpg: 0.8787
# ISIC_1510820.jpg vs ISIC_4461030.jpg: 0.7290
# ISIC_1510820.jpg vs ISIC_4461248.jpg: 0.4902
# ISIC_1510820.jpg vs ISIC_4461960.jpg: 0.2418
# ISIC_1510820.jpg vs ISIC_4462816.jpg: 0.8339
# ISIC_1510820.jpg vs ISIC_4463349.jpg: 0.3422
# ISIC_1510820.jpg vs ISIC_4464420.jpg: 0.8282
# ISIC_1510820.jpg vs ISIC_4464801.jpg: 0.7074
# ISIC_1510820.jpg vs ISIC_4466015.jpg: 0.6587
# ISIC_1510820.jpg vs ISIC_4467372.jpg: 0.8116
# ISIC_1510820.jpg vs ISIC_4467804.jpg: 0.3783
# ISIC_1510820.jpg vs ISIC_4468350.jpg: 0.4422
# ISIC_1510820.jpg vs ISIC_4468975.jpg: 0.5803
# ISIC_1510820.jpg vs ISIC_4469799.jpg: 0.7402
# ISIC_1510820.jpg vs ISIC_4470148.jpg: 0.4737
# ISIC_1510820.jpg vs ISIC_4470155.jpg: 0.4259
# ISIC_1510820.jpg vs ISIC_4471395.jpg: 0.4841
# ISIC_1510820.jpg vs ISIC_4471947.jpg: 0.7590
# ISIC_1510820.jpg vs ISIC_4472255.jpg: 0.8932
# ISIC_1510820.jpg vs ISIC_4473665.jpg: 0.7969
# ISIC_1510820.jpg vs ISIC_4475717.jpg: 0.7839
# ISIC_1510820.jpg vs ISIC_4477796.jpg: 0.7561
# ISIC_1510820.jpg vs ISIC_4478000.jpg: 0.2738
# ISIC_1510820.jpg vs ISIC_4480540.jpg: 0.3470
# ISIC_1510820.jpg vs ISIC_4482611.jpg: 0.7465
# ISIC_1510820.jpg vs ISIC_4483006.jpg: 0.5154
# ISIC_1510820.jpg vs ISIC_4486116.jpg: 0.2706
# ISIC_1510820.jpg vs ISIC_4486493.jpg: 0.3441
# ISIC_1510820.jpg vs ISIC_4486657.jpg: 0.8218
# ISIC_1510820.jpg vs ISIC_4489148.jpg: 0.4669
# ISIC_1510820.jpg vs ISIC_4490095.jpg: 0.4017
# ISIC_1510820.jpg vs ISIC_4490633.jpg: 0.7886
# ISIC_1510820.jpg vs ISIC_4490969.jpg: 0.2534
# ISIC_1510820.jpg vs ISIC_4491724.jpg: 0.6920
# ISIC_1510820.jpg vs ISIC_4494730.jpg: 0.7260
# ISIC_1510820.jpg vs ISIC_4494895.jpg: 0.4058
# ISIC_1510820.jpg vs ISIC_4495576.jpg: 0.4186
# ISIC_1510820.jpg vs ISIC_4495728.jpg: 0.4428
# ISIC_1510820.jpg vs ISIC_4497053.jpg: 0.8266
# ISIC_1510820.jpg vs ISIC_4498705.jpg: 0.9093
# ISIC_1510820.jpg vs ISIC_4501640.jpg: 0.7353
# ISIC_1510820.jpg vs ISIC_4502826.jpg: 0.4762
# ISIC_1510820.jpg vs ISIC_4503813.jpg: 0.6729
# ISIC_1510820.jpg vs ISIC_4504054.jpg: 0.2682
# ISIC_1510820.jpg vs ISIC_4504273.jpg: 0.7085
# ISIC_1510820.jpg vs ISIC_4505694.jpg: 0.5398
# ISIC_1510820.jpg vs ISIC_4505831.jpg: 0.5123
# ISIC_1510820.jpg vs ISIC_4507305.jpg: 0.8735
# ISIC_1510820.jpg vs ISIC_4510884.jpg: 0.4931
# ISIC_1510820.jpg vs ISIC_4513283.jpg: 0.3084
# ISIC_1510820.jpg vs ISIC_4513838.jpg: 0.7557
# ISIC_1510820.jpg vs ISIC_4516431.jpg: 0.9198
# ISIC_1510820.jpg vs ISIC_4520822.jpg: 0.8751
# ISIC_1510820.jpg vs ISIC_4524070.jpg: 0.8364
# ISIC_1510820.jpg vs ISIC_4526190.jpg: 0.8884
# ISIC_1510820.jpg vs ISIC_4526571.jpg: 0.8825
# ISIC_1510820.jpg vs ISIC_4527263.jpg: 0.4147
# ISIC_1510820.jpg vs ISIC_4529482.jpg: 0.7401
# ISIC_1510820.jpg vs ISIC_4529721.jpg: 0.4240
# ISIC_1510820.jpg vs ISIC_4532093.jpg: 0.7293
# ISIC_1510820.jpg vs ISIC_4533109.jpg: 0.8111
# ISIC_1510820.jpg vs ISIC_4533497.jpg: 0.8191
# ISIC_1510820.jpg vs ISIC_4535403.jpg: 0.8713
# ISIC_1510820.jpg vs ISIC_4537621.jpg: 0.7266
# ISIC_1510820.jpg vs ISIC_4537689.jpg: 0.6097
# ISIC_1510820.jpg vs ISIC_4537734.jpg: 0.8850
# ISIC_1510820.jpg vs ISIC_4537949.jpg: 0.8719
# ISIC_1510820.jpg vs ISIC_4538214.jpg: 0.8570
# ISIC_1510820.jpg vs ISIC_4538371.jpg: 0.2016
# ISIC_1510820.jpg vs ISIC_4539865.jpg: 0.7552
# ISIC_1510820.jpg vs ISIC_4543297.jpg: 0.5470
# ISIC_1510820.jpg vs ISIC_4544545.jpg: 0.8072
# ISIC_1510820.jpg vs ISIC_4546060.jpg: 0.8701
# ISIC_1510820.jpg vs ISIC_4546591.jpg: 0.6676
# ISIC_1510820.jpg vs ISIC_4550711.jpg: 0.7603
# ISIC_1510820.jpg vs ISIC_4550876.jpg: 0.2617
# ISIC_1510820.jpg vs ISIC_4553607.jpg: 0.7991
# ISIC_1510820.jpg vs ISIC_4555302.jpg: 0.7370
# ISIC_1510820.jpg vs ISIC_4557263.jpg: 0.1669
# ISIC_1510820.jpg vs ISIC_4559201.jpg: 0.8734
# ISIC_1510820.jpg vs ISIC_4560021.jpg: 0.1271
# ISIC_1510820.jpg vs ISIC_4560577.jpg: 0.7609
# ISIC_1510820.jpg vs ISIC_4561222.jpg: 0.3932
# ISIC_1510820.jpg vs ISIC_4561897.jpg: 0.5705
# ISIC_1510820.jpg vs ISIC_4563869.jpg: 0.7502
# ISIC_1510820.jpg vs ISIC_4564430.jpg: 0.7520
# ISIC_1510820.jpg vs ISIC_4566744.jpg: 0.6866
# ISIC_1510820.jpg vs ISIC_4567243.jpg: 0.6998
# ISIC_1510820.jpg vs ISIC_4569441.jpg: 0.8477
# ISIC_1510820.jpg vs ISIC_4573692.jpg: 0.7967
# ISIC_1510820.jpg vs ISIC_4573723.jpg: 0.1447
# ISIC_1510820.jpg vs ISIC_4574906.jpg: 0.7673
# ISIC_1510820.jpg vs ISIC_4575527.jpg: 0.5805
# ISIC_1510820.jpg vs ISIC_4576656.jpg: 0.5924
# ISIC_1510820.jpg vs ISIC_4578458.jpg: 0.7286
# ISIC_1510820.jpg vs ISIC_4578961.jpg: 0.8374
# ISIC_1510820.jpg vs ISIC_4580050.jpg: 0.3531
# ISIC_1510820.jpg vs ISIC_4580366.jpg: 0.2618
# ISIC_1510820.jpg vs ISIC_4583119.jpg: 0.6930
# ISIC_1510820.jpg vs ISIC_4583349.jpg: 0.3964
# ISIC_1510820.jpg vs ISIC_4583712.jpg: 0.9046
# ISIC_1510820.jpg vs ISIC_4584784.jpg: 0.6247
# ISIC_1510820.jpg vs ISIC_4586633.jpg: 0.7515
# ISIC_1510820.jpg vs ISIC_4587316.jpg: 0.5772
# ISIC_1510820.jpg vs ISIC_4587366.jpg: 0.8014
# ISIC_1510820.jpg vs ISIC_4589576.jpg: 0.4861
# ISIC_1510820.jpg vs ISIC_4590186.jpg: 0.6444
# ISIC_1510820.jpg vs ISIC_4590442.jpg: 0.3435
# ISIC_1510820.jpg vs ISIC_4593553.jpg: 0.6756
# ISIC_1510820.jpg vs ISIC_4593605.jpg: 0.7869
# ISIC_1510820.jpg vs ISIC_4594387.jpg: 0.6961
# ISIC_1510820.jpg vs ISIC_4597293.jpg: 0.5632
# ISIC_1510820.jpg vs ISIC_4599325.jpg: 0.8739
# ISIC_1510820.jpg vs ISIC_4600378.jpg: 0.8036
# ISIC_1510820.jpg vs ISIC_4600893.jpg: 0.2649
# ISIC_1510820.jpg vs ISIC_4605739.jpg: 0.6258
# ISIC_1510820.jpg vs ISIC_4608543.jpg: 0.8246
# ISIC_1510820.jpg vs ISIC_4610167.jpg: 0.7139
# ISIC_1510820.jpg vs ISIC_4610381.jpg: 0.4029
# ISIC_1510820.jpg vs ISIC_4610774.jpg: 0.8374
# ISIC_1510820.jpg vs ISIC_4612428.jpg: 0.7529
# ISIC_1510820.jpg vs ISIC_4615677.jpg: 0.8849
# ISIC_1510820.jpg vs ISIC_4616381.jpg: 0.6514
# ISIC_1510820.jpg vs ISIC_4616408.jpg: 0.6112
# ISIC_1510820.jpg vs ISIC_4618929.jpg: 0.3683
# ISIC_1510820.jpg vs ISIC_4619326.jpg: 0.5111
# ISIC_1510820.jpg vs ISIC_4621064.jpg: 0.7424
# ISIC_1510820.jpg vs ISIC_4621918.jpg: 0.6286
# ISIC_1510820.jpg vs ISIC_4622981.jpg: 0.7610
# ISIC_1510820.jpg vs ISIC_4625309.jpg: 0.4671
# ISIC_1510820.jpg vs ISIC_4625562.jpg: 0.8446
# ISIC_1510820.jpg vs ISIC_4626600.jpg: 0.8464
# ISIC_1510820.jpg vs ISIC_4626672.jpg: 0.7959
# ISIC_1510820.jpg vs ISIC_4627066.jpg: 0.6514
# ISIC_1510820.jpg vs ISIC_4629020.jpg: 0.8096
# ISIC_1510820.jpg vs ISIC_4629709.jpg: 0.8030
# ISIC_1510820.jpg vs ISIC_4630161.jpg: 0.7295
# ISIC_1510820.jpg vs ISIC_4630423.jpg: 0.3597
# ISIC_1510820.jpg vs ISIC_4631399.jpg: 0.7439
# ISIC_1510820.jpg vs ISIC_4632131.jpg: 0.5784
# ISIC_1510820.jpg vs ISIC_4633043.jpg: 0.6305
# ISIC_1510820.jpg vs ISIC_4634217.jpg: 0.3576
# ISIC_1510820.jpg vs ISIC_4634556.jpg: 0.3348
# ISIC_1510820.jpg vs ISIC_4634975.jpg: 0.6324
# ISIC_1510820.jpg vs ISIC_4638233.jpg: 0.6739
# ISIC_1510820.jpg vs ISIC_4638451.jpg: 0.4424
# ISIC_1510820.jpg vs ISIC_4639151.jpg: 0.8458
# ISIC_1510820.jpg vs ISIC_4639746.jpg: 0.3747
# ISIC_1510820.jpg vs ISIC_4640206.jpg: 0.1717
# ISIC_1510820.jpg vs ISIC_4643058.jpg: 0.2235
# ISIC_1510820.jpg vs ISIC_4644307.jpg: 0.7519
# ISIC_1510820.jpg vs ISIC_4646907.jpg: 0.2492
# ISIC_1510820.jpg vs ISIC_4649621.jpg: 0.2516
# ISIC_1510820.jpg vs ISIC_4651671.jpg: 0.8280
# ISIC_1510820.jpg vs ISIC_4656170.jpg: 0.6663
# ISIC_1510820.jpg vs ISIC_4657579.jpg: 0.5912
# ISIC_1510820.jpg vs ISIC_4790433.jpg: 0.7083
# ISIC_1510820.jpg vs ISIC_4791409.jpg: 0.4702
# ISIC_1510820.jpg vs ISIC_4795502.jpg: 0.5579
# ISIC_1510820.jpg vs ISIC_4797031.jpg: 0.8830
# ISIC_1510820.jpg vs ISIC_4797111.jpg: 0.3233
# ISIC_1510820.jpg vs ISIC_4798486.jpg: 0.6535
# ISIC_1510820.jpg vs ISIC_4799228.jpg: 0.7859
# ISIC_1510820.jpg vs ISIC_4799458.jpg: 0.7690
# ISIC_1510820.jpg vs ISIC_4801332.jpg: 0.2731
# ISIC_1510820.jpg vs ISIC_4805956.jpg: 0.4999
# ISIC_1510820.jpg vs ISIC_4806625.jpg: 0.9010
# ISIC_1510820.jpg vs ISIC_4807050.jpg: 0.4998
# ISIC_1510820.jpg vs ISIC_4807815.jpg: 0.2356
# ISIC_1510820.jpg vs ISIC_4808532.jpg: 0.5768
# ISIC_1510820.jpg vs ISIC_4808848.jpg: 0.9054
# ISIC_1510820.jpg vs ISIC_4809071.jpg: 0.4924
# ISIC_1510820.jpg vs ISIC_4810227.jpg: 0.1258
# ISIC_1510820.jpg vs ISIC_4811580.jpg: 0.3500
# ISIC_1510820.jpg vs ISIC_4811778.jpg: 0.7383
# ISIC_1510820.jpg vs ISIC_4812485.jpg: 0.6022
# ISIC_1510820.jpg vs ISIC_4814164.jpg: 0.2129
# ISIC_1510820.jpg vs ISIC_4814785.jpg: 0.3994
# ISIC_1510820.jpg vs ISIC_4815388.jpg: 0.5995
# ISIC_1510820.jpg vs ISIC_4817313.jpg: 0.5346
# ISIC_1510820.jpg vs ISIC_4817940.jpg: 0.5424
# ISIC_1510820.jpg vs ISIC_4819603.jpg: 0.8349
# ISIC_1510820.jpg vs ISIC_4820261.jpg: 0.8619
# ISIC_1510820.jpg vs ISIC_4821916.jpg: 0.4316
# ISIC_1510820.jpg vs ISIC_4825339.jpg: 0.1863
# ISIC_1510820.jpg vs ISIC_4825933.jpg: 0.3653
# ISIC_1510820.jpg vs ISIC_4827036.jpg: 0.8486
# ISIC_1510820.jpg vs ISIC_4827286.jpg: 0.7959
# ISIC_1510820.jpg vs ISIC_4828981.jpg: 0.6952
# ISIC_1510820.jpg vs ISIC_4829446.jpg: 0.1977
# ISIC_1510820.jpg vs ISIC_4829472.jpg: 0.5449
# ISIC_1510820.jpg vs ISIC_4831237.jpg: 0.6245
# ISIC_1510820.jpg vs ISIC_4831295.jpg: 0.4214
# ISIC_1510820.jpg vs ISIC_4832339.jpg: 0.7878
# ISIC_1510820.jpg vs ISIC_4832498.jpg: 0.8457
# ISIC_1510820.jpg vs ISIC_4836753.jpg: 0.6377
# ISIC_1510820.jpg vs ISIC_4837220.jpg: 0.8252
# ISIC_1510820.jpg vs ISIC_4837951.jpg: 0.6585
# ISIC_1510820.jpg vs ISIC_4840667.jpg: 0.7868
# ISIC_1510820.jpg vs ISIC_4841584.jpg: 0.4800
# ISIC_1510820.jpg vs ISIC_4841875.jpg: 0.6768
# ISIC_1510820.jpg vs ISIC_4842020.jpg: 0.4365
# ISIC_1510820.jpg vs ISIC_4842371.jpg: 0.7631
# ISIC_1510820.jpg vs ISIC_4842490.jpg: 0.7434
# ISIC_1510820.jpg vs ISIC_4844211.jpg: 0.4241
# ISIC_1510820.jpg vs ISIC_4845699.jpg: 0.7376
# ISIC_1510820.jpg vs ISIC_4850670.jpg: 0.7886
# ISIC_1510820.jpg vs ISIC_4852647.jpg: 0.1849
# ISIC_1510820.jpg vs ISIC_4853617.jpg: 0.2317
# ISIC_1510820.jpg vs ISIC_4854164.jpg: 0.6394
# ISIC_1510820.jpg vs ISIC_4854177.jpg: 0.6615
# ISIC_1510820.jpg vs ISIC_4855263.jpg: 0.5773
# ISIC_1510820.jpg vs ISIC_4858118.jpg: 0.7186
# ISIC_1510820.jpg vs ISIC_4860646.jpg: 0.7112
# ISIC_1510820.jpg vs ISIC_4864264.jpg: 0.7681
# ISIC_1510820.jpg vs ISIC_4864699.jpg: 0.8329
# ISIC_1510820.jpg vs ISIC_4870115.jpg: 0.7675
# ISIC_1510820.jpg vs ISIC_4870183.jpg: 0.6256
# ISIC_1510820.jpg vs ISIC_4870714.jpg: 0.8154
# ISIC_1510820.jpg vs ISIC_4871413.jpg: 0.7518
# ISIC_1510820.jpg vs ISIC_4872245.jpg: 0.7266
# ISIC_1510820.jpg vs ISIC_4872399.jpg: 0.7714
# ISIC_1510820.jpg vs ISIC_4875186.jpg: 0.8766
# ISIC_1510820.jpg vs ISIC_4876529.jpg: 0.4036
# ISIC_1510820.jpg vs ISIC_4876871.jpg: 0.5398
# ISIC_1510820.jpg vs ISIC_4878152.jpg: 0.9220
# ISIC_1510820.jpg vs ISIC_4878732.jpg: 0.8519
# ISIC_1510820.jpg vs ISIC_4880962.jpg: 0.5922
# ISIC_1510820.jpg vs ISIC_4881393.jpg: 0.6559
# ISIC_1510820.jpg vs ISIC_4885871.jpg: 0.7377
# ISIC_1510820.jpg vs ISIC_4887134.jpg: 0.8612
# ISIC_1510820.jpg vs ISIC_4887484.jpg: 0.8447
# ISIC_1510820.jpg vs ISIC_4890586.jpg: 0.3980
# ISIC_1510820.jpg vs ISIC_4892281.jpg: 0.3442
# ISIC_1510820.jpg vs ISIC_4892438.jpg: 0.6939
# ISIC_1510820.jpg vs ISIC_4893770.jpg: 0.8641
# ISIC_1510820.jpg vs ISIC_4895887.jpg: 0.8746
# ISIC_1510820.jpg vs ISIC_4899688.jpg: 0.2319
# ISIC_1510820.jpg vs ISIC_4900231.jpg: 0.8568
# ISIC_1510820.jpg vs ISIC_4901230.jpg: 0.7900
# ISIC_1510820.jpg vs ISIC_4901478.jpg: 0.7791
# ISIC_1510820.jpg vs ISIC_4903290.jpg: 0.3822
# ISIC_1510820.jpg vs ISIC_4905375.jpg: 0.4457
# ISIC_1510820.jpg vs ISIC_4906070.jpg: 0.4930
# ISIC_1510820.jpg vs ISIC_4906097.jpg: 0.5114
# ISIC_1510820.jpg vs ISIC_4906230.jpg: 0.3813
# ISIC_1510820.jpg vs ISIC_4907312.jpg: 0.4521
# ISIC_1510820.jpg vs ISIC_4910741.jpg: 0.0912
# ISIC_1510820.jpg vs ISIC_4911748.jpg: 0.7827
# ISIC_1510820.jpg vs ISIC_4913954.jpg: 0.4080
# ISIC_1510820.jpg vs ISIC_4913985.jpg: 0.5345
# ISIC_1510820.jpg vs ISIC_4916489.jpg: 0.8350
# ISIC_1510820.jpg vs ISIC_4917144.jpg: 0.7538
# ISIC_1510820.jpg vs ISIC_4918167.jpg: 0.5918
# ISIC_1510820.jpg vs ISIC_4918621.jpg: 0.6698
# ISIC_1510820.jpg vs ISIC_4920450.jpg: 0.4286
# ISIC_1510820.jpg vs ISIC_4920466.jpg: 0.4535
# ISIC_1510820.jpg vs ISIC_4921590.jpg: 0.6048
# ISIC_1510820.jpg vs ISIC_4921639.jpg: 0.8485
# ISIC_1510820.jpg vs ISIC_4921916.jpg: 0.6051
# ISIC_1510820.jpg vs ISIC_4922339.jpg: 0.1385
# ISIC_1510820.jpg vs ISIC_4923214.jpg: 0.6772
# ISIC_1510820.jpg vs ISIC_4924167.jpg: 0.7718
# ISIC_1510820.jpg vs ISIC_4924889.jpg: 0.8291
# ISIC_1510820.jpg vs ISIC_4925317.jpg: 0.8563
# ISIC_1510820.jpg vs ISIC_4926849.jpg: 0.6675
# ISIC_1510820.jpg vs ISIC_4927902.jpg: 0.7000
# ISIC_1510820.jpg vs ISIC_4928736.jpg: 0.6092
# ISIC_1510820.jpg vs ISIC_4928785.jpg: 0.7250
# ISIC_1510820.jpg vs ISIC_4930702.jpg: 0.9188
# ISIC_1510820.jpg vs ISIC_4933952.jpg: 0.8275
# ISIC_1510820.jpg vs ISIC_4934388.jpg: 0.6472
# ISIC_1510820.jpg vs ISIC_4934693.jpg: 0.5866
# ISIC_1510820.jpg vs ISIC_4935297.jpg: 0.8068
# ISIC_1510820.jpg vs ISIC_4936935.jpg: 0.6865
# ISIC_1510820.jpg vs ISIC_4938202.jpg: 0.5288
# ISIC_1510820.jpg vs ISIC_4938348.jpg: 0.7968
# ISIC_1510820.jpg vs ISIC_4939110.jpg: 0.3428
# ISIC_1510820.jpg vs ISIC_4939299.jpg: 0.7651
# ISIC_1510820.jpg vs ISIC_4940049.jpg: 0.5495
# ISIC_1510820.jpg vs ISIC_4940163.jpg: 0.3819
# ISIC_1510820.jpg vs ISIC_4944703.jpg: 0.8977
# ISIC_1510820.jpg vs ISIC_4946396.jpg: 0.3312
# ISIC_1510820.jpg vs ISIC_4947097.jpg: 0.6046
# ISIC_1510820.jpg vs ISIC_4949979.jpg: 0.6442
# ISIC_1510820.jpg vs ISIC_4954090.jpg: 0.5264
# ISIC_1510820.jpg vs ISIC_4955533.jpg: 0.3411
# ISIC_1510820.jpg vs ISIC_4956153.jpg: 0.8598
# ISIC_1510820.jpg vs ISIC_4957213.jpg: 0.8076
# ISIC_1510820.jpg vs ISIC_4959196.jpg: 0.8701
# ISIC_1510820.jpg vs ISIC_4959359.jpg: 0.6840
# ISIC_1510820.jpg vs ISIC_4961146.jpg: 0.1266
# ISIC_1510820.jpg vs ISIC_4963175.jpg: 0.5426
# ISIC_1510820.jpg vs ISIC_4964804.jpg: 0.7376
# ISIC_1510820.jpg vs ISIC_4964889.jpg: 0.7395
# ISIC_1510820.jpg vs ISIC_4965292.jpg: 0.1460
# ISIC_1510820.jpg vs ISIC_4965436.jpg: 0.8200
# ISIC_1510820.jpg vs ISIC_4967356.jpg: 0.5437
# ISIC_1510820.jpg vs ISIC_4968241.jpg: 0.5471
# ISIC_1510820.jpg vs ISIC_4968384.jpg: 0.5898
# ISIC_1510820.jpg vs ISIC_4969524.jpg: 0.7943
# ISIC_1510820.jpg vs ISIC_4971479.jpg: 0.4975
# ISIC_1510820.jpg vs ISIC_4973223.jpg: 0.6905
# ISIC_1510820.jpg vs ISIC_4974922.jpg: 0.6322
# ISIC_1510820.jpg vs ISIC_4975524.jpg: 0.1157
# ISIC_1510820.jpg vs ISIC_4979450.jpg: 0.5436
# ISIC_1510820.jpg vs ISIC_4980704.jpg: 0.7170
# ISIC_1510820.jpg vs ISIC_4982046.jpg: 0.8236
# ISIC_1510820.jpg vs ISIC_4982137.jpg: 0.8136
# ISIC_1510820.jpg vs ISIC_4985114.jpg: 0.5833
# ISIC_1510820.jpg vs ISIC_4986320.jpg: 0.4398
# ISIC_1510820.jpg vs ISIC_4990450.jpg: 0.8306
# ISIC_1510820.jpg vs ISIC_4991055.jpg: 0.7113
# ISIC_1510820.jpg vs ISIC_4991081.jpg: 0.8926
# ISIC_1510820.jpg vs ISIC_4991866.jpg: 0.8483
# ISIC_1510820.jpg vs ISIC_4994074.jpg: 0.3122
# ISIC_1510820.jpg vs ISIC_4998162.jpg: 0.8196
# ISIC_1510820.jpg vs ISIC_4998467.jpg: 0.9034
# ISIC_1510820.jpg vs ISIC_5000253.jpg: 0.6639
# ISIC_1510820.jpg vs ISIC_5003318.jpg: 0.4637
# ISIC_1510820.jpg vs ISIC_5003396.jpg: 0.7856
# ISIC_1510820.jpg vs ISIC_5003515.jpg: 0.2893
# ISIC_1510820.jpg vs ISIC_5003617.jpg: 0.8003
# ISIC_1510820.jpg vs ISIC_5005015.jpg: 0.2136
# ISIC_1510820.jpg vs ISIC_5005138.jpg: 0.4582
# ISIC_1510820.jpg vs ISIC_5005933.jpg: 0.4884
# ISIC_1510820.jpg vs ISIC_5006243.jpg: 0.3786
# ISIC_1510820.jpg vs ISIC_5006561.jpg: 0.9140
# ISIC_1510820.jpg vs ISIC_5006966.jpg: 0.4684
# ISIC_1510820.jpg vs ISIC_5007320.jpg: 0.4512
# ISIC_1510820.jpg vs ISIC_5007420.jpg: 0.2285
# ISIC_1510820.jpg vs ISIC_5008110.jpg: 0.4887
# ISIC_1510820.jpg vs ISIC_5008595.jpg: 0.6975
# ISIC_1510820.jpg vs ISIC_5009524.jpg: 0.8023
# ISIC_1510820.jpg vs ISIC_5011656.jpg: 0.7592
# ISIC_1510820.jpg vs ISIC_5011671.jpg: 0.5913
# ISIC_1510820.jpg vs ISIC_5013845.jpg: 0.3659
# ISIC_1510820.jpg vs ISIC_5014918.jpg: 0.8982
# ISIC_1510820.jpg vs ISIC_5014922.jpg: 0.4459
# ISIC_1510820.jpg vs ISIC_5015604.jpg: 0.8036
# ISIC_1510820.jpg vs ISIC_5016755.jpg: 0.5943
# ISIC_1510820.jpg vs ISIC_5017172.jpg: 0.9032
# ISIC_1510820.jpg vs ISIC_5017576.jpg: 0.7433
# ISIC_1510820.jpg vs ISIC_5018995.jpg: 0.7552
# ISIC_1510820.jpg vs ISIC_5019438.jpg: 0.7688
# ISIC_1510820.jpg vs ISIC_5021072.jpg: 0.8356
# ISIC_1510820.jpg vs ISIC_5021202.jpg: 0.3661
# ISIC_1510820.jpg vs ISIC_5022165.jpg: 0.5832
# ISIC_1510820.jpg vs ISIC_5024441.jpg: 0.5339
# ISIC_1510820.jpg vs ISIC_5024697.jpg: 0.7609
# ISIC_1510820.jpg vs ISIC_5025033.jpg: 0.5526
# ISIC_1510820.jpg vs ISIC_5026157.jpg: 0.7465
# ISIC_1510820.jpg vs ISIC_5027183.jpg: 0.2779
# ISIC_1510820.jpg vs ISIC_5029232.jpg: 0.8693
# ISIC_1510820.jpg vs ISIC_5031847.jpg: 0.8376
# ISIC_1510820.jpg vs ISIC_5032734.jpg: 0.8435
# ISIC_1510820.jpg vs ISIC_5033399.jpg: 0.7518
# ISIC_1510820.jpg vs ISIC_5033500.jpg: 0.7675
# ISIC_1510820.jpg vs ISIC_5033622.jpg: 0.8402
# ISIC_1510820.jpg vs ISIC_5034479.jpg: 0.5071
# ISIC_1510820.jpg vs ISIC_5034532.jpg: 0.6873
# ISIC_1510820.jpg vs ISIC_5036511.jpg: 0.3373
# ISIC_1510820.jpg vs ISIC_5039173.jpg: 0.8711
# ISIC_1510820.jpg vs ISIC_5043590.jpg: 0.4375
# ISIC_1510820.jpg vs ISIC_5045049.jpg: 0.8577
# ISIC_1510820.jpg vs ISIC_5046595.jpg: 0.4534
# ISIC_1510820.jpg vs ISIC_5046748.jpg: 0.6687
# ISIC_1510820.jpg vs ISIC_5048708.jpg: 0.3151
# ISIC_1510820.jpg vs ISIC_5049288.jpg: 0.5993
# ISIC_1510820.jpg vs ISIC_5050453.jpg: 0.8449
# ISIC_1510820.jpg vs ISIC_5050580.jpg: 0.6068
# ISIC_1510820.jpg vs ISIC_5052489.jpg: 0.8694
# ISIC_1510820.jpg vs ISIC_5052908.jpg: 0.4900
# ISIC_1510820.jpg vs ISIC_5053329.jpg: 0.8323
# ISIC_1510820.jpg vs ISIC_5053923.jpg: 0.3505
# ISIC_1510820.jpg vs ISIC_5056243.jpg: 0.1445
# ISIC_1510820.jpg vs ISIC_5060482.jpg: 0.7040
# ISIC_1510820.jpg vs ISIC_5061591.jpg: 0.1553
# ISIC_1510820.jpg vs ISIC_5061941.jpg: 0.1478
# ISIC_1510820.jpg vs ISIC_5064788.jpg: 0.6753
# ISIC_1510820.jpg vs ISIC_5066711.jpg: 0.7861
# ISIC_1510820.jpg vs ISIC_5067615.jpg: 0.8185
# ISIC_1510820.jpg vs ISIC_5069265.jpg: 0.7825
# ISIC_1510820.jpg vs ISIC_5069634.jpg: 0.8502
# ISIC_1510820.jpg vs ISIC_5072815.jpg: 0.8755
# ISIC_1510820.jpg vs ISIC_5073520.jpg: 0.8365
# ISIC_1510820.jpg vs ISIC_5081250.jpg: 0.8326
# ISIC_1510820.jpg vs ISIC_5081326.jpg: 0.8924
# ISIC_1510820.jpg vs ISIC_5082251.jpg: 0.6524
# ISIC_1510820.jpg vs ISIC_5082319.jpg: 0.2299
# ISIC_1510820.jpg vs ISIC_5082705.jpg: 0.5393
# ISIC_1510820.jpg vs ISIC_5084762.jpg: 0.7944
# ISIC_1510820.jpg vs ISIC_5085659.jpg: 0.4834
# ISIC_1510820.jpg vs ISIC_5085823.jpg: 0.6049
# ISIC_1510820.jpg vs ISIC_5087344.jpg: 0.3316
# ISIC_1510820.jpg vs ISIC_5091263.jpg: 0.5839
# ISIC_1510820.jpg vs ISIC_5092653.jpg: 0.7877
# ISIC_1510820.jpg vs ISIC_5093855.jpg: 0.8579
# ISIC_1510820.jpg vs ISIC_5093941.jpg: 0.4108
# ISIC_1510820.jpg vs ISIC_5094022.jpg: 0.6782
# ISIC_1510820.jpg vs ISIC_5095231.jpg: 0.3873
# ISIC_1510820.jpg vs ISIC_5097563.jpg: 0.8338
# ISIC_1510820.jpg vs ISIC_5097951.jpg: 0.7209
# ISIC_1510820.jpg vs ISIC_5098309.jpg: 0.6695
# ISIC_1510820.jpg vs ISIC_5098512.jpg: 0.5607
# ISIC_1510820.jpg vs ISIC_5098519.jpg: 0.5400
# ISIC_1510820.jpg vs ISIC_5098625.jpg: 0.8758
# ISIC_1510820.jpg vs ISIC_5098669.jpg: 0.1950
# ISIC_1510820.jpg vs ISIC_5099850.jpg: 0.7941
# ISIC_1510820.jpg vs ISIC_5100308.jpg: 0.2446
# ISIC_1510820.jpg vs ISIC_5106300.jpg: 0.6204
# ISIC_1510820.jpg vs ISIC_5107234.jpg: 0.5505
# ISIC_1510820.jpg vs ISIC_5107830.jpg: 0.7048
# ISIC_1510820.jpg vs ISIC_5108974.jpg: 0.8202
# ISIC_1510820.jpg vs ISIC_5110226.jpg: 0.4862
# ISIC_1510820.jpg vs ISIC_5112073.jpg: 0.7729
# ISIC_1510820.jpg vs ISIC_5112206.jpg: 0.8444
# ISIC_1510820.jpg vs ISIC_5114735.jpg: 0.4912
# ISIC_1510820.jpg vs ISIC_5116301.jpg: 0.8083
# ISIC_1510820.jpg vs ISIC_5116490.jpg: 0.3820
# ISIC_1510820.jpg vs ISIC_5117203.jpg: 0.1892
# ISIC_1510820.jpg vs ISIC_5119528.jpg: 0.4241
# ISIC_1510820.jpg vs ISIC_5120471.jpg: 0.5762
# ISIC_1510820.jpg vs ISIC_5121946.jpg: 0.4743
# ISIC_1510820.jpg vs ISIC_5125245.jpg: 0.7922
# ISIC_1510820.jpg vs ISIC_5128929.jpg: 0.7187
# ISIC_1510820.jpg vs ISIC_5129659.jpg: 0.1950
# ISIC_1510820.jpg vs ISIC_5132018.jpg: 0.7791
# ISIC_1510820.jpg vs ISIC_5132556.jpg: 0.3269
# ISIC_1510820.jpg vs ISIC_5134044.jpg: 0.3788
# ISIC_1510820.jpg vs ISIC_5135803.jpg: 0.6369
# ISIC_1510820.jpg vs ISIC_5136208.jpg: 0.8084
# ISIC_1510820.jpg vs ISIC_5136406.jpg: 0.3854
# ISIC_1510820.jpg vs ISIC_5137296.jpg: 0.1679
# ISIC_1510820.jpg vs ISIC_5137329.jpg: 0.7029
# ISIC_1510820.jpg vs ISIC_5139650.jpg: 0.7409
# ISIC_1510820.jpg vs ISIC_5140224.jpg: 0.9037
# ISIC_1510820.jpg vs ISIC_5141003.jpg: 0.3029
# ISIC_1510820.jpg vs ISIC_5143609.jpg: 0.8263
# ISIC_1510820.jpg vs ISIC_5143719.jpg: 0.7999
# ISIC_1510820.jpg vs ISIC_5143980.jpg: 0.6528
# ISIC_1510820.jpg vs ISIC_5144800.jpg: 0.7146
# ISIC_1510820.jpg vs ISIC_5145268.jpg: 0.5302
# ISIC_1510820.jpg vs ISIC_5148020.jpg: 0.8907
# ISIC_1510820.jpg vs ISIC_5149540.jpg: 0.6187
# ISIC_1510820.jpg vs ISIC_5151708.jpg: 0.7825
# ISIC_1510820.jpg vs ISIC_5152615.jpg: 0.6948
# ISIC_1510820.jpg vs ISIC_5154538.jpg: 0.6326
# ISIC_1510820.jpg vs ISIC_5154690.jpg: 0.6438
# ISIC_1510820.jpg vs ISIC_5155477.jpg: 0.7441
# ISIC_1510820.jpg vs ISIC_5157067.jpg: 0.7606
# ISIC_1510820.jpg vs ISIC_5158115.jpg: 0.9203
# ISIC_1510820.jpg vs ISIC_5159813.jpg: 0.7472
# ISIC_1510820.jpg vs ISIC_5165301.jpg: 0.6024
# ISIC_1510820.jpg vs ISIC_5167100.jpg: 0.4877
# ISIC_1510820.jpg vs ISIC_5167130.jpg: 0.7892
# ISIC_1510820.jpg vs ISIC_5168048.jpg: 0.6528
# ISIC_1510820.jpg vs ISIC_5172440.jpg: 0.2633
# ISIC_1510820.jpg vs ISIC_5172576.jpg: 0.5653
# ISIC_1510820.jpg vs ISIC_5172724.jpg: 0.5932
# ISIC_1510820.jpg vs ISIC_5172770.jpg: 0.8559
# ISIC_1510820.jpg vs ISIC_5173928.jpg: 0.8107
# ISIC_1510820.jpg vs ISIC_5173944.jpg: 0.7101
# ISIC_1510820.jpg vs ISIC_5175967.jpg: 0.6511
# ISIC_1510820.jpg vs ISIC_5176953.jpg: 0.7936
# ISIC_1510820.jpg vs ISIC_5176973.jpg: 0.5606
# ISIC_1510820.jpg vs ISIC_5180609.jpg: 0.6092
# ISIC_1510820.jpg vs ISIC_5183234.jpg: 0.5182
# ISIC_1510820.jpg vs ISIC_5183542.jpg: 0.8670
# ISIC_1510820.jpg vs ISIC_5184339.jpg: 0.6685
# ISIC_1510820.jpg vs ISIC_5186148.jpg: 0.7703
# ISIC_1510820.jpg vs ISIC_5187012.jpg: 0.1285
# ISIC_1510820.jpg vs ISIC_5187844.jpg: 0.8691
# ISIC_1510820.jpg vs ISIC_5189120.jpg: 0.2151
# ISIC_1510820.jpg vs ISIC_5191060.jpg: 0.5261
# ISIC_1510820.jpg vs ISIC_5192096.jpg: 0.7910
# ISIC_1510820.jpg vs ISIC_5194092.jpg: 0.4018
# ISIC_1510820.jpg vs ISIC_5195456.jpg: 0.6922
# ISIC_1510820.jpg vs ISIC_5196119.jpg: 0.7055
# ISIC_1510820.jpg vs ISIC_5196410.jpg: 0.6484
# ISIC_1510820.jpg vs ISIC_5196606.jpg: 0.7913
# ISIC_1510820.jpg vs ISIC_5197483.jpg: 0.8611
# ISIC_1510820.jpg vs ISIC_5198138.jpg: 0.8436
# ISIC_1510820.jpg vs ISIC_5198297.jpg: 0.8054
# ISIC_1510820.jpg vs ISIC_5200854.jpg: 0.7620
# ISIC_1510820.jpg vs ISIC_5202792.jpg: 0.7972
# ISIC_1510820.jpg vs ISIC_5202819.jpg: 0.7906
# ISIC_1510820.jpg vs ISIC_5203650.jpg: 0.3168
# ISIC_1510820.jpg vs ISIC_5209191.jpg: 0.6953
# ISIC_1510820.jpg vs ISIC_5210778.jpg: 0.8303
# ISIC_1510820.jpg vs ISIC_5211515.jpg: 0.8329
# ISIC_1510820.jpg vs ISIC_5213317.jpg: 0.6323
# ISIC_1510820.jpg vs ISIC_5215433.jpg: 0.2245
# ISIC_1510820.jpg vs ISIC_5216021.jpg: 0.8351
# ISIC_1510820.jpg vs ISIC_5217919.jpg: 0.2903
# ISIC_1510820.jpg vs ISIC_5219445.jpg: 0.4359
# ISIC_1510820.jpg vs ISIC_5221781.jpg: 0.6381
# ISIC_1510820.jpg vs ISIC_5221810.jpg: 0.6994
# ISIC_1510820.jpg vs ISIC_5224820.jpg: 0.7316
# ISIC_1510820.jpg vs ISIC_5224893.jpg: 0.7507
# ISIC_1510820.jpg vs ISIC_5224960.jpg: 0.2024
# ISIC_1510820.jpg vs ISIC_5226679.jpg: 0.3772
# ISIC_1510820.jpg vs ISIC_5228119.jpg: 0.7651
# ISIC_1510820.jpg vs ISIC_5229373.jpg: 0.6657
# ISIC_1510820.jpg vs ISIC_5230011.jpg: 0.4350
# ISIC_1510820.jpg vs ISIC_5230205.jpg: 0.8338
# ISIC_1510820.jpg vs ISIC_5231939.jpg: 0.6502
# ISIC_1510820.jpg vs ISIC_5233054.jpg: 0.7563
# ISIC_1510820.jpg vs ISIC_5236540.jpg: 0.7423
# ISIC_1510820.jpg vs ISIC_5237186.jpg: 0.7015
# ISIC_1510820.jpg vs ISIC_5239624.jpg: 0.8351
# ISIC_1510820.jpg vs ISIC_5242261.jpg: 0.8412
# ISIC_1510820.jpg vs ISIC_5244932.jpg: 0.8279
# ISIC_1510820.jpg vs ISIC_5246727.jpg: 0.8426
# ISIC_1510820.jpg vs ISIC_5247005.jpg: 0.2706
# ISIC_1510820.jpg vs ISIC_5249675.jpg: 0.1578
# ISIC_1510820.jpg vs ISIC_5251954.jpg: 0.2757
# ISIC_1510820.jpg vs ISIC_5253450.jpg: 0.7845
# ISIC_1510820.jpg vs ISIC_5253465.jpg: 0.6049
# ISIC_1510820.jpg vs ISIC_5253795.jpg: 0.8495
# ISIC_1510820.jpg vs ISIC_5254577.jpg: 0.8197
# ISIC_1510820.jpg vs ISIC_5256047.jpg: 0.8480
# ISIC_1510820.jpg vs ISIC_5258301.jpg: 0.7861
# ISIC_1510820.jpg vs ISIC_5260081.jpg: 0.8183
# ISIC_1510820.jpg vs ISIC_5260274.jpg: 0.5992
# ISIC_1510820.jpg vs ISIC_5260558.jpg: 0.2136
# ISIC_1510820.jpg vs ISIC_5264406.jpg: 0.5876
# ISIC_1510820.jpg vs ISIC_5264509.jpg: 0.8276
# ISIC_1510820.jpg vs ISIC_5265185.jpg: 0.5782
# ISIC_1510820.jpg vs ISIC_5267225.jpg: 0.6142
# ISIC_1510820.jpg vs ISIC_5269066.jpg: 0.3919
# ISIC_1510820.jpg vs ISIC_5269993.jpg: 0.9157
# ISIC_1510820.jpg vs ISIC_5271610.jpg: 0.5727
# ISIC_1510820.jpg vs ISIC_5274319.jpg: 0.6417
# ISIC_1510820.jpg vs ISIC_5275484.jpg: 0.5886
# ISIC_1510820.jpg vs ISIC_5276317.jpg: 0.8851
# ISIC_1510820.jpg vs ISIC_5278233.jpg: 0.6805
# ISIC_1510820.jpg vs ISIC_5278466.jpg: 0.7676
# ISIC_1510820.jpg vs ISIC_5280324.jpg: 0.6522
# ISIC_1510820.jpg vs ISIC_5281393.jpg: 0.8308
# ISIC_1510820.jpg vs ISIC_5283427.jpg: 0.4955
# ISIC_1510820.jpg vs ISIC_5284795.jpg: 0.4782
# ISIC_1510820.jpg vs ISIC_5285042.jpg: 0.8364
# ISIC_1510820.jpg vs ISIC_5285074.jpg: 0.8196
# ISIC_1510820.jpg vs ISIC_5285323.jpg: 0.6439
# ISIC_1510820.jpg vs ISIC_5287531.jpg: 0.6294
# ISIC_1510820.jpg vs ISIC_5287696.jpg: 0.5043
# ISIC_1510820.jpg vs ISIC_5291409.jpg: 0.6779
# ISIC_1510820.jpg vs ISIC_5292407.jpg: 0.7653
# ISIC_1510820.jpg vs ISIC_5294230.jpg: 0.3204
# ISIC_1510820.jpg vs ISIC_5294387.jpg: 0.5063
# ISIC_1510820.jpg vs ISIC_5294678.jpg: 0.6994
# ISIC_1510820.jpg vs ISIC_5306862.jpg: 0.7838
# ISIC_1510820.jpg vs ISIC_5307788.jpg: 0.8061
# ISIC_1510820.jpg vs ISIC_5309403.jpg: 0.7076
# ISIC_1510820.jpg vs ISIC_5309864.jpg: 0.5053
# ISIC_1510820.jpg vs ISIC_5311634.jpg: 0.7758
# ISIC_1510820.jpg vs ISIC_5319047.jpg: 0.6039
# ISIC_1510820.jpg vs ISIC_5320315.jpg: 0.4927
# ISIC_1510820.jpg vs ISIC_5330007.jpg: 0.4328
# ISIC_1510820.jpg vs ISIC_5386993.jpg: 0.0782
# ISIC_1510820.jpg vs ISIC_5387264.jpg: 0.4150
# ISIC_1510820.jpg vs ISIC_5388371.jpg: 0.1111
# ISIC_1510820.jpg vs ISIC_5390118.jpg: 0.6237
# ISIC_1510820.jpg vs ISIC_5390308.jpg: 0.6019
# ISIC_1510820.jpg vs ISIC_5391024.jpg: 0.6177
# ISIC_1510820.jpg vs ISIC_5393971.jpg: 0.5521
# ISIC_1510820.jpg vs ISIC_5399201.jpg: 0.7129
# ISIC_1510820.jpg vs ISIC_5399722.jpg: 0.9033
# ISIC_1510820.jpg vs ISIC_5399850.jpg: 0.3513
# ISIC_1510820.jpg vs ISIC_5400056.jpg: 0.4245
# ISIC_1510820.jpg vs ISIC_5401032.jpg: 0.8943
# ISIC_1510820.jpg vs ISIC_5401768.jpg: 0.4010
# ISIC_1510820.jpg vs ISIC_5403261.jpg: 0.7322
# ISIC_1510820.jpg vs ISIC_5403285.jpg: 0.7035
# ISIC_1510820.jpg vs ISIC_5403880.jpg: 0.5165
# ISIC_1510820.jpg vs ISIC_5406945.jpg: 0.5555
# ISIC_1510820.jpg vs ISIC_5407909.jpg: 0.3944
# ISIC_1510820.jpg vs ISIC_5408357.jpg: 0.6311
# ISIC_1510820.jpg vs ISIC_5410137.jpg: 0.7293
# ISIC_1510820.jpg vs ISIC_5410641.jpg: 0.6721
# ISIC_1510820.jpg vs ISIC_5410816.jpg: 0.4503
# ISIC_1510820.jpg vs ISIC_5411152.jpg: 0.2210
# ISIC_1510820.jpg vs ISIC_5412416.jpg: 0.6355
# ISIC_1510820.jpg vs ISIC_5412438.jpg: 0.8869
# ISIC_1510820.jpg vs ISIC_5415096.jpg: 0.9063
# ISIC_1510820.jpg vs ISIC_5415822.jpg: 0.7817
# ISIC_1510820.jpg vs ISIC_5416338.jpg: 0.7503
# ISIC_1510820.jpg vs ISIC_5418007.jpg: 0.8100
# ISIC_1510820.jpg vs ISIC_5418351.jpg: 0.2517
# ISIC_1510820.jpg vs ISIC_5418992.jpg: 0.7096
# ISIC_1510820.jpg vs ISIC_5419254.jpg: 0.8181
# ISIC_1510820.jpg vs ISIC_5419426.jpg: 0.2921
# ISIC_1510820.jpg vs ISIC_5419657.jpg: 0.1260
# ISIC_1510820.jpg vs ISIC_5419668.jpg: 0.7444
# ISIC_1510820.jpg vs ISIC_5421651.jpg: 0.6844
# ISIC_1510820.jpg vs ISIC_5422892.jpg: 0.6218
# ISIC_1510820.jpg vs ISIC_5423251.jpg: 0.2316
# ISIC_1510820.jpg vs ISIC_5423554.jpg: 0.6862
# ISIC_1510820.jpg vs ISIC_5423741.jpg: 0.2762
# ISIC_1510820.jpg vs ISIC_5424954.jpg: 0.5943
# ISIC_1510820.jpg vs ISIC_5426249.jpg: 0.3045
# ISIC_1510820.jpg vs ISIC_5428000.jpg: 0.8716
# ISIC_1510820.jpg vs ISIC_5429359.jpg: 0.8894
# ISIC_1510820.jpg vs ISIC_5430817.jpg: 0.4008
# ISIC_1510820.jpg vs ISIC_5433808.jpg: 0.4968
# ISIC_1510820.jpg vs ISIC_5433830.jpg: 0.2828
# ISIC_1510820.jpg vs ISIC_5434740.jpg: 0.7409
# ISIC_1510820.jpg vs ISIC_5435156.jpg: 0.3895
# ISIC_1510820.jpg vs ISIC_5437769.jpg: 0.6290
# ISIC_1510820.jpg vs ISIC_5438598.jpg: 0.5660
# ISIC_1510820.jpg vs ISIC_5438774.jpg: 0.9110
# ISIC_1510820.jpg vs ISIC_5440349.jpg: 0.4044
# ISIC_1510820.jpg vs ISIC_5442471.jpg: 0.5750
# ISIC_1510820.jpg vs ISIC_5443694.jpg: 0.8114
# ISIC_1510820.jpg vs ISIC_5443967.jpg: 0.6635
# ISIC_1510820.jpg vs ISIC_5446126.jpg: 0.8040
# ISIC_1510820.jpg vs ISIC_5446229.jpg: 0.7307
# ISIC_1510820.jpg vs ISIC_5447745.jpg: 0.6097
# ISIC_1510820.jpg vs ISIC_5453550.jpg: 0.5869
# ISIC_1510820.jpg vs ISIC_5453965.jpg: 0.6186
# ISIC_1510820.jpg vs ISIC_5454664.jpg: 0.7527
# ISIC_1510820.jpg vs ISIC_5458475.jpg: 0.8261
# ISIC_1510820.jpg vs ISIC_5462474.jpg: 0.8751
# ISIC_1510820.jpg vs ISIC_5466110.jpg: 0.6043
# ISIC_1510820.jpg vs ISIC_5467877.jpg: 0.6141
# ISIC_1510820.jpg vs ISIC_5470845.jpg: 0.4529
# ISIC_1510820.jpg vs ISIC_5471585.jpg: 0.8080
# ISIC_1510820.jpg vs ISIC_5472565.jpg: 0.7619
# ISIC_1510820.jpg vs ISIC_5473977.jpg: 0.6259
# ISIC_1510820.jpg vs ISIC_5475276.jpg: 0.8027
# ISIC_1510820.jpg vs ISIC_5476552.jpg: 0.5499
# ISIC_1510820.jpg vs ISIC_5483744.jpg: 0.8062
# ISIC_1510820.jpg vs ISIC_5484280.jpg: 0.6062
# ISIC_1510820.jpg vs ISIC_5484843.jpg: 0.1882
# ISIC_1510820.jpg vs ISIC_5485368.jpg: 0.6440
# ISIC_1510820.jpg vs ISIC_5486638.jpg: 0.6280
# ISIC_1510820.jpg vs ISIC_5488695.jpg: 0.7390
# ISIC_1510820.jpg vs ISIC_5488923.jpg: 0.5805
# ISIC_1510820.jpg vs ISIC_5490573.jpg: 0.5230
# ISIC_1510820.jpg vs ISIC_5490874.jpg: 0.7814
# ISIC_1510820.jpg vs ISIC_5495612.jpg: 0.5568
# ISIC_1510820.jpg vs ISIC_5496072.jpg: 0.6612
# ISIC_1510820.jpg vs ISIC_5496889.jpg: 0.8123
# ISIC_1510820.jpg vs ISIC_5497040.jpg: 0.3637
# ISIC_1510820.jpg vs ISIC_5497586.jpg: 0.2650
# ISIC_1510820.jpg vs ISIC_5499841.jpg: 0.6748
# ISIC_1510820.jpg vs ISIC_5500783.jpg: 0.7061
# ISIC_1510820.jpg vs ISIC_5501466.jpg: 0.3394
# ISIC_1510820.jpg vs ISIC_5502735.jpg: 0.7885
# ISIC_1510820.jpg vs ISIC_5502759.jpg: 0.7835
# ISIC_1510820.jpg vs ISIC_5503017.jpg: 0.7093
# ISIC_1510820.jpg vs ISIC_5503450.jpg: 0.6882
# ISIC_1510820.jpg vs ISIC_5505130.jpg: 0.8002
# ISIC_1510820.jpg vs ISIC_5505677.jpg: 0.5176
# ISIC_1510820.jpg vs ISIC_5507883.jpg: 0.3423
# ISIC_1510820.jpg vs ISIC_5508489.jpg: 0.4869
# ISIC_1510820.jpg vs ISIC_5511356.jpg: 0.5664
# ISIC_1510820.jpg vs ISIC_5515689.jpg: 0.9146
# ISIC_1510820.jpg vs ISIC_5515761.jpg: 0.7549
# ISIC_1510820.jpg vs ISIC_5516080.jpg: 0.7401
# ISIC_1510820.jpg vs ISIC_5519109.jpg: 0.8649
# ISIC_1510820.jpg vs ISIC_5519153.jpg: 0.4253
# ISIC_1510820.jpg vs ISIC_5520837.jpg: 0.7700
# ISIC_1510820.jpg vs ISIC_5520987.jpg: 0.7402
# ISIC_1510820.jpg vs ISIC_5527106.jpg: 0.6206
# ISIC_1510820.jpg vs ISIC_5530515.jpg: 0.8656
# ISIC_1510820.jpg vs ISIC_5531040.jpg: 0.7405
# ISIC_1510820.jpg vs ISIC_5531729.jpg: 0.6306
# ISIC_1510820.jpg vs ISIC_5533324.jpg: 0.8383
# ISIC_1510820.jpg vs ISIC_5533685.jpg: 0.7919
# ISIC_1510820.jpg vs ISIC_5533724.jpg: 0.6815
# ISIC_1510820.jpg vs ISIC_5533867.jpg: 0.0978
# ISIC_1510820.jpg vs ISIC_5535061.jpg: 0.2376
# ISIC_1510820.jpg vs ISIC_5535308.jpg: 0.7727
# ISIC_1510820.jpg vs ISIC_5537005.jpg: 0.7554
# ISIC_1510820.jpg vs ISIC_5537240.jpg: 0.6460
# ISIC_1510820.jpg vs ISIC_5539423.jpg: 0.1997
# ISIC_1510820.jpg vs ISIC_5539445.jpg: 0.5732
# ISIC_1510820.jpg vs ISIC_5539665.jpg: 0.8352
# ISIC_1510820.jpg vs ISIC_5539687.jpg: 0.6626
# ISIC_1510820.jpg vs ISIC_5540957.jpg: 0.7689
# ISIC_1510820.jpg vs ISIC_5541162.jpg: 0.2558
# ISIC_1510820.jpg vs ISIC_5541445.jpg: 0.0608
# ISIC_1510820.jpg vs ISIC_5541526.jpg: 0.7086
# ISIC_1510820.jpg vs ISIC_5541720.jpg: 0.7108
# ISIC_1510820.jpg vs ISIC_5541989.jpg: 0.7742
# ISIC_1510820.jpg vs ISIC_5542248.jpg: 0.8547
# ISIC_1510820.jpg vs ISIC_5543962.jpg: 0.6824
# ISIC_1510820.jpg vs ISIC_5544312.jpg: 0.2945
# ISIC_1510820.jpg vs ISIC_5545935.jpg: 0.5587
# ISIC_1510820.jpg vs ISIC_5548310.jpg: 0.7950
# ISIC_1510820.jpg vs ISIC_5549462.jpg: 0.2486
# ISIC_1510820.jpg vs ISIC_5550366.jpg: 0.7028
# ISIC_1510820.jpg vs ISIC_5554930.jpg: 0.7380
# ISIC_1510820.jpg vs ISIC_5555122.jpg: 0.7871
# ISIC_1510820.jpg vs ISIC_5559277.jpg: 0.8702
# ISIC_1510820.jpg vs ISIC_5559931.jpg: 0.8536
# ISIC_1510820.jpg vs ISIC_5560509.jpg: 0.5959
# ISIC_1510820.jpg vs ISIC_5560779.jpg: 0.7430
# ISIC_1510820.jpg vs ISIC_5561701.jpg: 0.8481
# ISIC_1510820.jpg vs ISIC_5563510.jpg: 0.8449
# ISIC_1510820.jpg vs ISIC_5564375.jpg: 0.7881
# ISIC_1510820.jpg vs ISIC_5564959.jpg: 0.1663
# ISIC_1510820.jpg vs ISIC_5565342.jpg: 0.6584
# ISIC_1510820.jpg vs ISIC_5565859.jpg: 0.8070
# ISIC_1510820.jpg vs ISIC_5566502.jpg: 0.6599
# ISIC_1510820.jpg vs ISIC_5567033.jpg: 0.4695
# ISIC_1510820.jpg vs ISIC_5567187.jpg: 0.8592
# ISIC_1510820.jpg vs ISIC_5568321.jpg: 0.7554
# ISIC_1510820.jpg vs ISIC_5568735.jpg: 0.7692
# ISIC_1510820.jpg vs ISIC_5569800.jpg: 0.3463
# ISIC_1510820.jpg vs ISIC_5571037.jpg: 0.1624
# ISIC_1510820.jpg vs ISIC_5575241.jpg: 0.4777
# ISIC_1510820.jpg vs ISIC_5576230.jpg: 0.5528
# ISIC_1510820.jpg vs ISIC_5576438.jpg: 0.2325
# ISIC_1510820.jpg vs ISIC_5576756.jpg: 0.8007
# ISIC_1510820.jpg vs ISIC_5576762.jpg: 0.7124
# ISIC_1510820.jpg vs ISIC_5577319.jpg: 0.7937
# ISIC_1510820.jpg vs ISIC_5578839.jpg: 0.8576
# ISIC_1510820.jpg vs ISIC_5580945.jpg: 0.5706
# ISIC_1510820.jpg vs ISIC_5581560.jpg: 0.6901
# ISIC_1510820.jpg vs ISIC_5582992.jpg: 0.8759
# ISIC_1510820.jpg vs ISIC_5583376.jpg: 0.7944
# ISIC_1510820.jpg vs ISIC_5583936.jpg: 0.6864
# ISIC_1510820.jpg vs ISIC_5585090.jpg: 0.7841
# ISIC_1510820.jpg vs ISIC_5586123.jpg: 0.4242
# ISIC_1510820.jpg vs ISIC_5590761.jpg: 0.4489
# ISIC_1510820.jpg vs ISIC_5592614.jpg: 0.6222
# ISIC_1510820.jpg vs ISIC_5597578.jpg: 0.8308
# ISIC_1510820.jpg vs ISIC_5598610.jpg: 0.2030
# ISIC_1510820.jpg vs ISIC_5599751.jpg: 0.2850
# ISIC_1510820.jpg vs ISIC_5600375.jpg: 0.7315
# ISIC_1510820.jpg vs ISIC_5600551.jpg: 0.8326
# ISIC_1510820.jpg vs ISIC_5600700.jpg: 0.5323
# ISIC_1510820.jpg vs ISIC_5602249.jpg: 0.4674
# ISIC_1510820.jpg vs ISIC_5602856.jpg: 0.4599
# ISIC_1510820.jpg vs ISIC_5603057.jpg: 0.7844
# ISIC_1510820.jpg vs ISIC_5605645.jpg: 0.3193
# ISIC_1510820.jpg vs ISIC_5606526.jpg: 0.7969
# ISIC_1510820.jpg vs ISIC_5607261.jpg: 0.8467
# ISIC_1510820.jpg vs ISIC_5612963.jpg: 0.7241
# ISIC_1510820.jpg vs ISIC_5613408.jpg: 0.8073
# ISIC_1510820.jpg vs ISIC_5618182.jpg: 0.2197
# ISIC_1510820.jpg vs ISIC_5619486.jpg: 0.4254
# ISIC_1510820.jpg vs ISIC_5620287.jpg: 0.8309
# ISIC_1510820.jpg vs ISIC_5622428.jpg: 0.3335
# ISIC_1510820.jpg vs ISIC_5623119.jpg: 0.8207
# ISIC_1510820.jpg vs ISIC_5624387.jpg: 0.7192
# ISIC_1510820.jpg vs ISIC_5624407.jpg: 0.8239
# ISIC_1510820.jpg vs ISIC_5624772.jpg: 0.8652
# ISIC_1510820.jpg vs ISIC_5626138.jpg: 0.5706
# ISIC_1510820.jpg vs ISIC_5626995.jpg: 0.5927
# ISIC_1510820.jpg vs ISIC_5627557.jpg: 0.5969
# ISIC_1510820.jpg vs ISIC_5628761.jpg: 0.8770
# ISIC_1510820.jpg vs ISIC_5629414.jpg: 0.7808
# ISIC_1510820.jpg vs ISIC_5630715.jpg: 0.2973
# ISIC_1510820.jpg vs ISIC_5631254.jpg: 0.8107
# ISIC_1510820.jpg vs ISIC_5631587.jpg: 0.8700
# ISIC_1510820.jpg vs ISIC_5631948.jpg: 0.8273
# ISIC_1510820.jpg vs ISIC_5635429.jpg: 0.7662
# ISIC_1510820.jpg vs ISIC_5635478.jpg: 0.8179
# ISIC_1510820.jpg vs ISIC_5637379.jpg: 0.3753
# ISIC_1510820.jpg vs ISIC_5637617.jpg: 0.1548
# ISIC_1510820.jpg vs ISIC_5638807.jpg: 0.5598
# ISIC_1510820.jpg vs ISIC_5643279.jpg: 0.6682
# ISIC_1510820.jpg vs ISIC_5655022.jpg: 0.7695
# ISIC_1510820.jpg vs ISIC_5659637.jpg: 0.2156
# ISIC_1510820.jpg vs ISIC_5662562.jpg: 0.4622
# ISIC_1510820.jpg vs ISIC_5664024.jpg: 0.6519
# ISIC_1510820.jpg vs ISIC_5664060.jpg: 0.7799
# ISIC_1510820.jpg vs ISIC_5664143.jpg: 0.7534
# ISIC_1510820.jpg vs ISIC_5664695.jpg: 0.3870
# ISIC_1510820.jpg vs ISIC_5665733.jpg: 0.8199
# ISIC_1510820.jpg vs ISIC_5666134.jpg: 0.6667
# ISIC_1510820.jpg vs ISIC_5669245.jpg: 0.6620
# ISIC_1510820.jpg vs ISIC_5670543.jpg: 0.5485
# ISIC_1510820.jpg vs ISIC_5670844.jpg: 0.7431
# ISIC_1510820.jpg vs ISIC_5672457.jpg: 0.5362
# ISIC_1510820.jpg vs ISIC_5673944.jpg: 0.8778
# ISIC_1510820.jpg vs ISIC_5674930.jpg: 0.8328
# ISIC_1510820.jpg vs ISIC_5675376.jpg: 0.5410
# ISIC_1510820.jpg vs ISIC_5678849.jpg: 0.2019
# ISIC_1510820.jpg vs ISIC_5679644.jpg: 0.7721
# ISIC_1510820.jpg vs ISIC_5679779.jpg: 0.2095
# ISIC_1510820.jpg vs ISIC_5679940.jpg: 0.6249
# ISIC_1510820.jpg vs ISIC_5681289.jpg: 0.6387
# ISIC_1510820.jpg vs ISIC_5681301.jpg: 0.7660
# ISIC_1510820.jpg vs ISIC_5681752.jpg: 0.1692
# ISIC_1510820.jpg vs ISIC_5682842.jpg: 0.7698
# ISIC_1510820.jpg vs ISIC_5683180.jpg: 0.6253
# ISIC_1510820.jpg vs ISIC_5689896.jpg: 0.2171
# ISIC_1510820.jpg vs ISIC_5690889.jpg: 0.6405
# ISIC_1510820.jpg vs ISIC_5690955.jpg: 0.7380
# ISIC_1510820.jpg vs ISIC_5692444.jpg: 0.8464
# ISIC_1510820.jpg vs ISIC_5693635.jpg: 0.8198
# ISIC_1510820.jpg vs ISIC_5694968.jpg: 0.6117
# ISIC_1510820.jpg vs ISIC_5697382.jpg: 0.6894
# ISIC_1510820.jpg vs ISIC_5698863.jpg: 0.1121
# ISIC_1510820.jpg vs ISIC_5700242.jpg: 0.4950
# ISIC_1510820.jpg vs ISIC_5700419.jpg: 0.7158
# ISIC_1510820.jpg vs ISIC_5701922.jpg: 0.4097
# ISIC_1510820.jpg vs ISIC_5702257.jpg: 0.6452
# ISIC_1510820.jpg vs ISIC_5702422.jpg: 0.7435
# ISIC_1510820.jpg vs ISIC_5703542.jpg: 0.6876
# ISIC_1510820.jpg vs ISIC_5703633.jpg: 0.8304
# ISIC_1510820.jpg vs ISIC_5705356.jpg: 0.7524
# ISIC_1510820.jpg vs ISIC_5706387.jpg: 0.8639
# ISIC_1510820.jpg vs ISIC_5707071.jpg: 0.8463
# ISIC_1510820.jpg vs ISIC_5708420.jpg: 0.5258
# ISIC_1510820.jpg vs ISIC_5709287.jpg: 0.6399
# ISIC_1510820.jpg vs ISIC_5711802.jpg: 0.8226
# ISIC_1510820.jpg vs ISIC_5712569.jpg: 0.5249
# ISIC_1510820.jpg vs ISIC_5712915.jpg: 0.7721
# ISIC_1510820.jpg vs ISIC_5712969.jpg: 0.6585
# ISIC_1510820.jpg vs ISIC_5713011.jpg: 0.4936
# ISIC_1510820.jpg vs ISIC_5713093.jpg: 0.7750
# ISIC_1510820.jpg vs ISIC_5713173.jpg: 0.8164
# ISIC_1510820.jpg vs ISIC_5714065.jpg: 0.7338
# ISIC_1510820.jpg vs ISIC_5714211.jpg: 0.8739
# ISIC_1510820.jpg vs ISIC_5714278.jpg: 0.8127
# ISIC_1510820.jpg vs ISIC_5714720.jpg: 0.6643
# ISIC_1510820.jpg vs ISIC_5714747.jpg: 0.8748
# ISIC_1510820.jpg vs ISIC_5714992.jpg: 0.4190
# ISIC_1510820.jpg vs ISIC_5718241.jpg: 0.2232
# ISIC_1510820.jpg vs ISIC_5718520.jpg: 0.7890
# ISIC_1510820.jpg vs ISIC_5718931.jpg: 0.4663
# ISIC_1510820.jpg vs ISIC_5719373.jpg: 0.8501
# ISIC_1510820.jpg vs ISIC_5719436.jpg: 0.2092
# ISIC_1510820.jpg vs ISIC_5720944.jpg: 0.5074
# ISIC_1510820.jpg vs ISIC_5721530.jpg: 0.8707
# ISIC_1510820.jpg vs ISIC_5725417.jpg: 0.7567
# ISIC_1510820.jpg vs ISIC_5725671.jpg: 0.3915
# ISIC_1510820.jpg vs ISIC_5730606.jpg: 0.6590
# ISIC_1510820.jpg vs ISIC_5733991.jpg: 0.7950
# ISIC_1510820.jpg vs ISIC_5736865.jpg: 0.6040
# ISIC_1510820.jpg vs ISIC_5737211.jpg: 0.7532
# ISIC_1510820.jpg vs ISIC_5737322.jpg: 0.8710
# ISIC_1510820.jpg vs ISIC_5738160.jpg: 0.4085
# ISIC_1510820.jpg vs ISIC_5739375.jpg: 0.6741
# ISIC_1510820.jpg vs ISIC_5740353.jpg: 0.3998
# ISIC_1510820.jpg vs ISIC_5741024.jpg: 0.8895
# ISIC_1510820.jpg vs ISIC_5742807.jpg: 0.5549
# ISIC_1510820.jpg vs ISIC_5749591.jpg: 0.7536
# ISIC_1510820.jpg vs ISIC_5749893.jpg: 0.2194
# ISIC_1510820.jpg vs ISIC_5750800.jpg: 0.7186
# ISIC_1510820.jpg vs ISIC_5752353.jpg: 0.7787
# ISIC_1510820.jpg vs ISIC_5758834.jpg: 0.5933
# ISIC_1510820.jpg vs ISIC_5761233.jpg: 0.4592
# ISIC_1510820.jpg vs ISIC_5763460.jpg: 0.5989
# ISIC_1510820.jpg vs ISIC_5763939.jpg: 0.5619
# ISIC_1510820.jpg vs ISIC_5764132.jpg: 0.5429
# ISIC_1510820.jpg vs ISIC_5764228.jpg: 0.8343
# ISIC_1510820.jpg vs ISIC_5764234.jpg: 0.4047
# ISIC_1510820.jpg vs ISIC_5767868.jpg: 0.8555
# ISIC_1510820.jpg vs ISIC_5769412.jpg: 0.6254
# ISIC_1510820.jpg vs ISIC_5774467.jpg: 0.7921
# ISIC_1510820.jpg vs ISIC_5774484.jpg: 0.3472
# ISIC_1510820.jpg vs ISIC_5775179.jpg: 0.7029
# ISIC_1510820.jpg vs ISIC_5775317.jpg: 0.3232
# ISIC_1510820.jpg vs ISIC_5776649.jpg: 0.8280
# ISIC_1510820.jpg vs ISIC_5777319.jpg: 0.8666
# ISIC_1510820.jpg vs ISIC_5782456.jpg: 0.6405
# ISIC_1510820.jpg vs ISIC_5784853.jpg: 0.4278
# ISIC_1510820.jpg vs ISIC_5787155.jpg: 0.8857
# ISIC_1510820.jpg vs ISIC_5787158.jpg: 0.4261
# ISIC_1510820.jpg vs ISIC_5787501.jpg: 0.1626
# ISIC_1510820.jpg vs ISIC_5787648.jpg: 0.7380
# ISIC_1510820.jpg vs ISIC_5788972.jpg: 0.4488
# ISIC_1510820.jpg vs ISIC_5789959.jpg: 0.5953
# ISIC_1510820.jpg vs ISIC_5790931.jpg: 0.5252
# ISIC_1510820.jpg vs ISIC_5791555.jpg: 0.6524
# ISIC_1510820.jpg vs ISIC_5793340.jpg: 0.3203
# ISIC_1510820.jpg vs ISIC_5794395.jpg: 0.7871
# ISIC_1510820.jpg vs ISIC_5795173.jpg: 0.8642
# ISIC_1510820.jpg vs ISIC_5795436.jpg: 0.4694
# ISIC_1510820.jpg vs ISIC_5797446.jpg: 0.2767
# ISIC_1510820.jpg vs ISIC_5797747.jpg: 0.3396
# ISIC_1510820.jpg vs ISIC_5797949.jpg: 0.6920
# ISIC_1510820.jpg vs ISIC_5798657.jpg: 0.2487
# ISIC_1510820.jpg vs ISIC_5798789.jpg: 0.8674
# ISIC_1510820.jpg vs ISIC_5800043.jpg: 0.6663
# ISIC_1510820.jpg vs ISIC_5800224.jpg: 0.8194
# ISIC_1510820.jpg vs ISIC_5800765.jpg: 0.6507
# ISIC_1510820.jpg vs ISIC_5801108.jpg: 0.2109
# ISIC_1510820.jpg vs ISIC_5801193.jpg: 0.6491
# ISIC_1510820.jpg vs ISIC_5802864.jpg: 0.8706
# ISIC_1510820.jpg vs ISIC_5802893.jpg: 0.2925
# ISIC_1510820.jpg vs ISIC_5803768.jpg: 0.7696
# ISIC_1510820.jpg vs ISIC_5804623.jpg: 0.8364
# ISIC_1510820.jpg vs ISIC_5807012.jpg: 0.8846
# ISIC_1510820.jpg vs ISIC_5807899.jpg: 0.2508
# ISIC_1510820.jpg vs ISIC_5808864.jpg: 0.7878
# ISIC_1510820.jpg vs ISIC_5815193.jpg: 0.8581
# ISIC_1510820.jpg vs ISIC_5815409.jpg: 0.1025
# ISIC_1510820.jpg vs ISIC_5817062.jpg: 0.7391
# ISIC_1510820.jpg vs ISIC_5817984.jpg: 0.5206
# ISIC_1510820.jpg vs ISIC_5818519.jpg: 0.7899
# ISIC_1510820.jpg vs ISIC_5819533.jpg: 0.1852
# ISIC_1510820.jpg vs ISIC_5819597.jpg: 0.7076
# ISIC_1510820.jpg vs ISIC_5820450.jpg: 0.7488
# ISIC_1510820.jpg vs ISIC_5821781.jpg: 0.8269
# ISIC_1510820.jpg vs ISIC_5822125.jpg: 0.5483
# ISIC_1510820.jpg vs ISIC_5823667.jpg: 0.6618
# ISIC_1510820.jpg vs ISIC_5827723.jpg: 0.4708
# ISIC_1510820.jpg vs ISIC_5827759.jpg: 0.8961
# ISIC_1510820.jpg vs ISIC_5828210.jpg: 0.5766
# ISIC_1510820.jpg vs ISIC_5828980.jpg: 0.7621
# ISIC_1510820.jpg vs ISIC_5831411.jpg: 0.5907
# ISIC_1510820.jpg vs ISIC_5832048.jpg: 0.7212
# ISIC_1510820.jpg vs ISIC_5832642.jpg: 0.7317
# ISIC_1510820.jpg vs ISIC_5832709.jpg: 0.6100
# ISIC_1510820.jpg vs ISIC_5842884.jpg: 0.1589
# ISIC_1510820.jpg vs ISIC_5844164.jpg: 0.8462
# ISIC_1510820.jpg vs ISIC_5844458.jpg: 0.6283
# ISIC_1510820.jpg vs ISIC_5844609.jpg: 0.3111
# ISIC_1510820.jpg vs ISIC_5845864.jpg: 0.7497
# ISIC_1510820.jpg vs ISIC_5847839.jpg: 0.7243
# ISIC_1510820.jpg vs ISIC_5847979.jpg: 0.8451
# ISIC_1510820.jpg vs ISIC_5850513.jpg: 0.4152
# ISIC_1510820.jpg vs ISIC_5852300.jpg: 0.7689
# ISIC_1510820.jpg vs ISIC_5852478.jpg: 0.5276
# ISIC_1510820.jpg vs ISIC_5854410.jpg: 0.6713
# ISIC_1510820.jpg vs ISIC_5856636.jpg: 0.6648
# ISIC_1510820.jpg vs ISIC_5856841.jpg: 0.7299
# ISIC_1510820.jpg vs ISIC_5856952.jpg: 0.8687
# ISIC_1510820.jpg vs ISIC_5857128.jpg: 0.5332
# ISIC_1510820.jpg vs ISIC_5857931.jpg: 0.5364
# ISIC_1510820.jpg vs ISIC_5858452.jpg: 0.8016
# ISIC_1510820.jpg vs ISIC_5858575.jpg: 0.8841
# ISIC_1510820.jpg vs ISIC_5864666.jpg: 0.4048
# ISIC_1510820.jpg vs ISIC_5865496.jpg: 0.5437
# ISIC_1510820.jpg vs ISIC_5866313.jpg: 0.6598
# ISIC_1510820.jpg vs ISIC_5866452.jpg: 0.7296
# ISIC_1510820.jpg vs ISIC_5867064.jpg: 0.7104
# ISIC_1510820.jpg vs ISIC_5867760.jpg: 0.5972
# ISIC_1510820.jpg vs ISIC_5869109.jpg: 0.7351
# ISIC_1510820.jpg vs ISIC_5872519.jpg: 0.4865
# ISIC_1510820.jpg vs ISIC_5873723.jpg: 0.3913
# ISIC_1510820.jpg vs ISIC_5874097.jpg: 0.4427
# ISIC_1510820.jpg vs ISIC_5875752.jpg: 0.7495
# ISIC_1510820.jpg vs ISIC_5876026.jpg: 0.8202
# ISIC_1510820.jpg vs ISIC_5876203.jpg: 0.1866
# ISIC_1510820.jpg vs ISIC_5877881.jpg: 0.6169
# ISIC_1510820.jpg vs ISIC_5878196.jpg: 0.8865
# ISIC_1510820.jpg vs ISIC_5879037.jpg: 0.4659
# ISIC_1510820.jpg vs ISIC_5879313.jpg: 0.2300
# ISIC_1510820.jpg vs ISIC_5882237.jpg: 0.7652
# ISIC_1510820.jpg vs ISIC_5883000.jpg: 0.2212
# ISIC_1510820.jpg vs ISIC_5883709.jpg: 0.6397
# ISIC_1510820.jpg vs ISIC_5885225.jpg: 0.8572
# ISIC_1510820.jpg vs ISIC_5887793.jpg: 0.8487
# ISIC_1510820.jpg vs ISIC_5888756.jpg: 0.2837
# ISIC_1510820.jpg vs ISIC_5889171.jpg: 0.1807
# ISIC_1510820.jpg vs ISIC_5889229.jpg: 0.8094
# ISIC_1510820.jpg vs ISIC_5889822.jpg: 0.7491
# ISIC_1510820.jpg vs ISIC_5890128.jpg: 0.3362
# ISIC_1510820.jpg vs ISIC_5890372.jpg: 0.7626
# ISIC_1510820.jpg vs ISIC_5891752.jpg: 0.8212
# ISIC_1510820.jpg vs ISIC_5892469.jpg: 0.6516
# ISIC_1510820.jpg vs ISIC_5894347.jpg: 0.6443
# ISIC_1510820.jpg vs ISIC_5897093.jpg: 0.7903
# ISIC_1510820.jpg vs ISIC_5898235.jpg: 0.4071
# ISIC_1510820.jpg vs ISIC_5901955.jpg: 0.1275
# ISIC_1510820.jpg vs ISIC_5901962.jpg: 0.8363
# ISIC_1510820.jpg vs ISIC_5902290.jpg: 0.8371
# ISIC_1510820.jpg vs ISIC_5904715.jpg: 0.6361
# ISIC_1510820.jpg vs ISIC_5906485.jpg: 0.5614
# ISIC_1510820.jpg vs ISIC_5907145.jpg: 0.4764
# ISIC_1510820.jpg vs ISIC_5909829.jpg: 0.7647
# ISIC_1510820.jpg vs ISIC_5910903.jpg: 0.6431
# ISIC_1510820.jpg vs ISIC_5911171.jpg: 0.6723
# ISIC_1510820.jpg vs ISIC_5913285.jpg: 0.6344
# ISIC_1510820.jpg vs ISIC_5915563.jpg: 0.6798
# ISIC_1510820.jpg vs ISIC_5916147.jpg: 0.3297
# ISIC_1510820.jpg vs ISIC_5916689.jpg: 0.5911
# ISIC_1510820.jpg vs ISIC_5916999.jpg: 0.7956
# ISIC_1510820.jpg vs ISIC_5917729.jpg: 0.3705
# ISIC_1510820.jpg vs ISIC_5917737.jpg: 0.4827
# ISIC_1510820.jpg vs ISIC_5919372.jpg: 0.7604
# ISIC_1510820.jpg vs ISIC_5921042.jpg: 0.5696
# ISIC_1510820.jpg vs ISIC_5921996.jpg: 0.1268
# ISIC_1510820.jpg vs ISIC_5922835.jpg: 0.6184
# ISIC_1510820.jpg vs ISIC_5923107.jpg: 0.5812
# ISIC_1510820.jpg vs ISIC_5926184.jpg: 0.3362
# ISIC_1510820.jpg vs ISIC_5926577.jpg: 0.7170
# ISIC_1510820.jpg vs ISIC_5928225.jpg: 0.6953
# ISIC_1510820.jpg vs ISIC_5928882.jpg: 0.6539
# ISIC_1510820.jpg vs ISIC_5930869.jpg: 0.2169
# ISIC_1510820.jpg vs ISIC_5930965.jpg: 0.5103
# ISIC_1510820.jpg vs ISIC_5931154.jpg: 0.1257
# ISIC_1510820.jpg vs ISIC_5931285.jpg: 0.9007
# ISIC_1510820.jpg vs ISIC_5932219.jpg: 0.6601
# ISIC_1510820.jpg vs ISIC_5933157.jpg: 0.7580
# ISIC_1510820.jpg vs ISIC_5934718.jpg: 0.1115
# ISIC_1510820.jpg vs ISIC_5935067.jpg: 0.8822
# ISIC_1510820.jpg vs ISIC_5936480.jpg: 0.6015
# ISIC_1510820.jpg vs ISIC_5936603.jpg: 0.2081
# ISIC_1510820.jpg vs ISIC_5936614.jpg: 0.7406
# ISIC_1510820.jpg vs ISIC_5938720.jpg: 0.7203
# ISIC_1510820.jpg vs ISIC_5938978.jpg: 0.6204
# ISIC_1510820.jpg vs ISIC_5940742.jpg: 0.8410
# ISIC_1510820.jpg vs ISIC_5940904.jpg: 0.3966
# ISIC_1510820.jpg vs ISIC_5942063.jpg: 0.6870
# ISIC_1510820.jpg vs ISIC_5942663.jpg: 0.7272
# ISIC_1510820.jpg vs ISIC_6048978.jpg: 0.5378
# ISIC_1510820.jpg vs ISIC_6051029.jpg: 0.6108
# ISIC_1510820.jpg vs ISIC_6052137.jpg: 0.8541
# ISIC_1510820.jpg vs ISIC_6053337.jpg: 0.3751
# ISIC_1510820.jpg vs ISIC_6057427.jpg: 0.8613
# ISIC_1510820.jpg vs ISIC_6060411.jpg: 0.7783
# ISIC_1510820.jpg vs ISIC_6061137.jpg: 0.8126
# ISIC_1510820.jpg vs ISIC_6062422.jpg: 0.7828
# ISIC_1510820.jpg vs ISIC_6064159.jpg: 0.5590
# ISIC_1510820.jpg vs ISIC_6070288.jpg: 0.4246
# ISIC_1510820.jpg vs ISIC_6071805.jpg: 0.3345
# ISIC_1510820.jpg vs ISIC_6072532.jpg: 0.7962
# ISIC_1510820.jpg vs ISIC_6073734.jpg: 0.2942
# ISIC_1510820.jpg vs ISIC_6074397.jpg: 0.7050
# ISIC_1510820.jpg vs ISIC_6081327.jpg: 0.6988
# ISIC_1510820.jpg vs ISIC_6081653.jpg: 0.6470
# ISIC_1510820.jpg vs ISIC_6087077.jpg: 0.6191
# ISIC_1510820.jpg vs ISIC_6096565.jpg: 0.3518
# ISIC_1510820.jpg vs ISIC_6097857.jpg: 0.6334
# ISIC_1510820.jpg vs ISIC_6098830.jpg: 0.7183
# ISIC_1510820.jpg vs ISIC_6102229.jpg: 0.5245
# ISIC_1510820.jpg vs ISIC_6102805.jpg: 0.4080
# ISIC_1510820.jpg vs ISIC_6102938.jpg: 0.8261
# ISIC_1510820.jpg vs ISIC_6105717.jpg: 0.8644
# ISIC_1510820.jpg vs ISIC_6107317.jpg: 0.2250
# ISIC_1510820.jpg vs ISIC_6113741.jpg: 0.8363
# ISIC_1510820.jpg vs ISIC_6115225.jpg: 0.1960
# ISIC_1510820.jpg vs ISIC_6115711.jpg: 0.2554
# ISIC_1510820.jpg vs ISIC_6118637.jpg: 0.7748
# ISIC_1510820.jpg vs ISIC_6118712.jpg: 0.2498
# ISIC_1510820.jpg vs ISIC_6118768.jpg: 0.8112
# ISIC_1510820.jpg vs ISIC_6120143.jpg: 0.8232
# ISIC_1510820.jpg vs ISIC_6120158.jpg: 0.6881
# ISIC_1510820.jpg vs ISIC_6120545.jpg: 0.7548
# ISIC_1510820.jpg vs ISIC_6122007.jpg: 0.7293
# ISIC_1510820.jpg vs ISIC_6122407.jpg: 0.4087
# ISIC_1510820.jpg vs ISIC_6122897.jpg: 0.4079
# ISIC_1510820.jpg vs ISIC_6122906.jpg: 0.1421
# ISIC_1510820.jpg vs ISIC_6125633.jpg: 0.1621
# ISIC_1510820.jpg vs ISIC_6128464.jpg: 0.2945
# ISIC_1510820.jpg vs ISIC_6129073.jpg: 0.2396
# ISIC_1510820.jpg vs ISIC_6130016.jpg: 0.7163
# ISIC_1510820.jpg vs ISIC_6131888.jpg: 0.5116
# ISIC_1510820.jpg vs ISIC_6132443.jpg: 0.1678
# ISIC_1510820.jpg vs ISIC_6132521.jpg: 0.6465
# ISIC_1510820.jpg vs ISIC_6133252.jpg: 0.9053
# ISIC_1510820.jpg vs ISIC_6133571.jpg: 0.8303
# ISIC_1510820.jpg vs ISIC_6135759.jpg: 0.4240
# ISIC_1510820.jpg vs ISIC_6136227.jpg: 0.7882
# ISIC_1510820.jpg vs ISIC_6137841.jpg: 0.7844
# ISIC_1510820.jpg vs ISIC_6139839.jpg: 0.8373
# ISIC_1510820.jpg vs ISIC_6140475.jpg: 0.6637
# ISIC_1510820.jpg vs ISIC_6140637.jpg: 0.7612
# ISIC_1510820.jpg vs ISIC_6143045.jpg: 0.7649
# ISIC_1510820.jpg vs ISIC_6144806.jpg: 0.6632
# ISIC_1510820.jpg vs ISIC_6147174.jpg: 0.3937
# ISIC_1510820.jpg vs ISIC_6147889.jpg: 0.2187
# ISIC_1510820.jpg vs ISIC_6148454.jpg: 0.6325
# ISIC_1510820.jpg vs ISIC_6148710.jpg: 0.7555
# ISIC_1510820.jpg vs ISIC_6149394.jpg: 0.3923
# ISIC_1510820.jpg vs ISIC_6149823.jpg: 0.7711
# ISIC_1510820.jpg vs ISIC_6151236.jpg: 0.5607
# ISIC_1510820.jpg vs ISIC_6153943.jpg: 0.7758
# ISIC_1510820.jpg vs ISIC_6154102.jpg: 0.5031
# ISIC_1510820.jpg vs ISIC_6159592.jpg: 0.8305
# ISIC_1510820.jpg vs ISIC_6159600.jpg: 0.1444
# ISIC_1510820.jpg vs ISIC_6164205.jpg: 0.7334
# ISIC_1510820.jpg vs ISIC_6164865.jpg: 0.6867
# ISIC_1510820.jpg vs ISIC_6165518.jpg: 0.7012
# ISIC_1510820.jpg vs ISIC_6165920.jpg: 0.0845
# ISIC_1510820.jpg vs ISIC_6166514.jpg: 0.6713
# ISIC_1510820.jpg vs ISIC_6168190.jpg: 0.4453
# ISIC_1510820.jpg vs ISIC_6169239.jpg: 0.3838
# ISIC_1510820.jpg vs ISIC_6171146.jpg: 0.8389
# ISIC_1510820.jpg vs ISIC_6174589.jpg: 0.7218
# ISIC_1510820.jpg vs ISIC_6174690.jpg: 0.7347
# ISIC_1510820.jpg vs ISIC_6174724.jpg: 0.8489
# ISIC_1510820.jpg vs ISIC_6180307.jpg: 0.4585
# ISIC_1510820.jpg vs ISIC_6180603.jpg: 0.8274
# ISIC_1510820.jpg vs ISIC_6181315.jpg: 0.8106
# ISIC_1510820.jpg vs ISIC_6182346.jpg: 0.8124
# ISIC_1510820.jpg vs ISIC_6182913.jpg: 0.9555
# ISIC_1510820.jpg vs ISIC_6184049.jpg: 0.6912
# ISIC_1510820.jpg vs ISIC_6184616.jpg: 0.4671
# ISIC_1510820.jpg vs ISIC_6184815.jpg: 0.7354
# ISIC_1510820.jpg vs ISIC_6185319.jpg: 0.6446
# ISIC_1510820.jpg vs ISIC_6185606.jpg: 0.7789
# ISIC_1510820.jpg vs ISIC_6185801.jpg: 0.7687
# ISIC_1510820.jpg vs ISIC_6186432.jpg: 0.2953
# ISIC_1510820.jpg vs ISIC_6188565.jpg: 0.6464
# ISIC_1510820.jpg vs ISIC_6190621.jpg: 0.3750
# ISIC_1510820.jpg vs ISIC_6192132.jpg: 0.3831
# ISIC_1510820.jpg vs ISIC_6192205.jpg: 0.5368
# ISIC_1510820.jpg vs ISIC_6192610.jpg: 0.1409
# ISIC_1510820.jpg vs ISIC_6193215.jpg: 0.2673
# ISIC_1510820.jpg vs ISIC_6193472.jpg: 0.8603
# ISIC_1510820.jpg vs ISIC_6195527.jpg: 0.7076
# ISIC_1510820.jpg vs ISIC_6197041.jpg: 0.6757
# ISIC_1510820.jpg vs ISIC_6197896.jpg: 0.8066
# ISIC_1510820.jpg vs ISIC_6198726.jpg: 0.8386
# ISIC_1510820.jpg vs ISIC_6199822.jpg: 0.5134
# ISIC_1510820.jpg vs ISIC_6203295.jpg: 0.8487
# ISIC_1510820.jpg vs ISIC_6204442.jpg: 0.6810
# ISIC_1510820.jpg vs ISIC_6204751.jpg: 0.5449
# ISIC_1510820.jpg vs ISIC_6206241.jpg: 0.8122
# ISIC_1510820.jpg vs ISIC_6208151.jpg: 0.1025
# ISIC_1510820.jpg vs ISIC_6211950.jpg: 0.6817
# ISIC_1510820.jpg vs ISIC_6214753.jpg: 0.5277
# ISIC_1510820.jpg vs ISIC_6218246.jpg: 0.4389
# ISIC_1510820.jpg vs ISIC_6218648.jpg: 0.8291
# ISIC_1510820.jpg vs ISIC_6220493.jpg: 0.6366
# ISIC_1510820.jpg vs ISIC_6222213.jpg: 0.7373
# ISIC_1510820.jpg vs ISIC_6223199.jpg: 0.7417
# ISIC_1510820.jpg vs ISIC_6227605.jpg: 0.7104
# ISIC_1510820.jpg vs ISIC_6228369.jpg: 0.5353
# ISIC_1510820.jpg vs ISIC_6230135.jpg: 0.2556
# ISIC_1510820.jpg vs ISIC_6233969.jpg: 0.8469
# ISIC_1510820.jpg vs ISIC_6235516.jpg: 0.7469
# ISIC_1510820.jpg vs ISIC_6240439.jpg: 0.5461
# ISIC_1510820.jpg vs ISIC_6240886.jpg: 0.8530
# ISIC_1510820.jpg vs ISIC_6244226.jpg: 0.7164
# ISIC_1510820.jpg vs ISIC_6244894.jpg: 0.8966
# ISIC_1510820.jpg vs ISIC_6246998.jpg: 0.6150
# ISIC_1510820.jpg vs ISIC_6248112.jpg: 0.3845
# ISIC_1510820.jpg vs ISIC_6248504.jpg: 0.3066
# ISIC_1510820.jpg vs ISIC_6251679.jpg: 0.6841
# ISIC_1510820.jpg vs ISIC_6251700.jpg: 0.3262
# ISIC_1510820.jpg vs ISIC_6252644.jpg: 0.7378
# ISIC_1510820.jpg vs ISIC_6256102.jpg: 0.5880
# ISIC_1510820.jpg vs ISIC_6257335.jpg: 0.7684
# ISIC_1510820.jpg vs ISIC_6259642.jpg: 0.8102
# ISIC_1510820.jpg vs ISIC_6259682.jpg: 0.6561
# ISIC_1510820.jpg vs ISIC_6260052.jpg: 0.6027
# ISIC_1510820.jpg vs ISIC_6262744.jpg: 0.8689
# ISIC_1510820.jpg vs ISIC_6263151.jpg: 0.3343
# ISIC_1510820.jpg vs ISIC_6264872.jpg: 0.5665
# ISIC_1510820.jpg vs ISIC_6270723.jpg: 0.3797
# ISIC_1510820.jpg vs ISIC_6272060.jpg: 0.4482
# ISIC_1510820.jpg vs ISIC_6272071.jpg: 0.6639
# ISIC_1510820.jpg vs ISIC_6272206.jpg: 0.8241
# ISIC_1510820.jpg vs ISIC_6272978.jpg: 0.4629
# ISIC_1510820.jpg vs ISIC_6273296.jpg: 0.8423
# ISIC_1510820.jpg vs ISIC_6274125.jpg: 0.8420
# ISIC_1510820.jpg vs ISIC_6275700.jpg: 0.7673
# ISIC_1510820.jpg vs ISIC_6276358.jpg: 0.3193
# ISIC_1510820.jpg vs ISIC_6276543.jpg: 0.6675
# ISIC_1510820.jpg vs ISIC_6278817.jpg: 0.7762
# ISIC_1510820.jpg vs ISIC_6279071.jpg: 0.7380
# ISIC_1510820.jpg vs ISIC_6285105.jpg: 0.5711
# ISIC_1510820.jpg vs ISIC_6287859.jpg: 0.1110
# ISIC_1510820.jpg vs ISIC_6288405.jpg: 0.7692
# ISIC_1510820.jpg vs ISIC_6288517.jpg: 0.4345
# ISIC_1510820.jpg vs ISIC_6290714.jpg: 0.7185
# ISIC_1510820.jpg vs ISIC_6293576.jpg: 0.6830
# ISIC_1510820.jpg vs ISIC_6295508.jpg: 0.5067
# ISIC_1510820.jpg vs ISIC_6295805.jpg: 0.6365
# ISIC_1510820.jpg vs ISIC_6299694.jpg: 0.5912
# ISIC_1510820.jpg vs ISIC_6300866.jpg: 0.2889
# ISIC_1510820.jpg vs ISIC_6302625.jpg: 0.5726
# ISIC_1510820.jpg vs ISIC_6302745.jpg: 0.6511
# ISIC_1510820.jpg vs ISIC_6304786.jpg: 0.7351
# ISIC_1510820.jpg vs ISIC_6304831.jpg: 0.6294
# ISIC_1510820.jpg vs ISIC_6307097.jpg: 0.8296
# ISIC_1510820.jpg vs ISIC_6307284.jpg: 0.7772
# ISIC_1510820.jpg vs ISIC_6312849.jpg: 0.5697
# ISIC_1510820.jpg vs ISIC_6315939.jpg: 0.7533
# ISIC_1510820.jpg vs ISIC_6316211.jpg: 0.5627
# ISIC_1510820.jpg vs ISIC_6319908.jpg: 0.6864
# ISIC_1510820.jpg vs ISIC_6321296.jpg: 0.7661
# ISIC_1510820.jpg vs ISIC_6321957.jpg: 0.6854
# ISIC_1510820.jpg vs ISIC_6322638.jpg: 0.7540
# ISIC_1510820.jpg vs ISIC_6324134.jpg: 0.7663
# ISIC_1510820.jpg vs ISIC_6324738.jpg: 0.4983
# ISIC_1510820.jpg vs ISIC_6326045.jpg: 0.7353
# ISIC_1510820.jpg vs ISIC_6326439.jpg: 0.6793
# ISIC_1510820.jpg vs ISIC_6327696.jpg: 0.8634
# ISIC_1510820.jpg vs ISIC_6329267.jpg: 0.7837
# ISIC_1510820.jpg vs ISIC_6330302.jpg: 0.2920
# ISIC_1510820.jpg vs ISIC_6330336.jpg: 0.8358
# ISIC_1510820.jpg vs ISIC_6330632.jpg: 0.4291
# ISIC_1510820.jpg vs ISIC_6332124.jpg: 0.5560
# ISIC_1510820.jpg vs ISIC_6332746.jpg: 0.8528
# ISIC_1510820.jpg vs ISIC_6333868.jpg: 0.6929
# ISIC_1510820.jpg vs ISIC_6334718.jpg: 0.8843
# ISIC_1510820.jpg vs ISIC_6335117.jpg: 0.6820
# ISIC_1510820.jpg vs ISIC_6336463.jpg: 0.8467
# ISIC_1510820.jpg vs ISIC_6336553.jpg: 0.6314
# ISIC_1510820.jpg vs ISIC_6337591.jpg: 0.7887
# ISIC_1510820.jpg vs ISIC_6337981.jpg: 0.8342
# ISIC_1510820.jpg vs ISIC_6338356.jpg: 0.1472
# ISIC_1510820.jpg vs ISIC_6340119.jpg: 0.3564
# ISIC_1510820.jpg vs ISIC_6340846.jpg: 0.5716
# ISIC_1510820.jpg vs ISIC_6341261.jpg: 0.8056
# ISIC_1510820.jpg vs ISIC_6345695.jpg: 0.7917
# ISIC_1510820.jpg vs ISIC_6347478.jpg: 0.8229
# ISIC_1510820.jpg vs ISIC_6348542.jpg: 0.8802
# ISIC_1510820.jpg vs ISIC_6349053.jpg: 0.8311
# ISIC_1510820.jpg vs ISIC_6350227.jpg: 0.3829
# ISIC_1510820.jpg vs ISIC_6350420.jpg: 0.7152
# ISIC_1510820.jpg vs ISIC_6350718.jpg: 0.3659
# ISIC_1510820.jpg vs ISIC_6353090.jpg: 0.8764
# ISIC_1510820.jpg vs ISIC_6353402.jpg: 0.6244
# ISIC_1510820.jpg vs ISIC_6354781.jpg: 0.7757
# ISIC_1510820.jpg vs ISIC_6355799.jpg: 0.8291
# ISIC_1510820.jpg vs ISIC_6355985.jpg: 0.5609
# ISIC_1510820.jpg vs ISIC_6358503.jpg: 0.5920
# ISIC_1510820.jpg vs ISIC_6359785.jpg: 0.6531
# ISIC_1510820.jpg vs ISIC_6360613.jpg: 0.1911
# ISIC_1510820.jpg vs ISIC_6362191.jpg: 0.6393
# ISIC_1510820.jpg vs ISIC_6362598.jpg: 0.7084
# ISIC_1510820.jpg vs ISIC_6363298.jpg: 0.7023
# ISIC_1510820.jpg vs ISIC_6364521.jpg: 0.7990
# ISIC_1510820.jpg vs ISIC_6364739.jpg: 0.7292
# ISIC_1510820.jpg vs ISIC_6365507.jpg: 0.7699
# ISIC_1510820.jpg vs ISIC_6371918.jpg: 0.4301
# ISIC_1510820.jpg vs ISIC_6371939.jpg: 0.6125
# ISIC_1510820.jpg vs ISIC_6372732.jpg: 0.8956
# ISIC_1510820.jpg vs ISIC_6376631.jpg: 0.6338
# ISIC_1510820.jpg vs ISIC_6377740.jpg: 0.1055
# ISIC_1510820.jpg vs ISIC_6381819.jpg: 0.5547
# ISIC_1510820.jpg vs ISIC_6383313.jpg: 0.6188
# ISIC_1510820.jpg vs ISIC_6384675.jpg: 0.5110
# ISIC_1510820.jpg vs ISIC_6385118.jpg: 0.1303
# ISIC_1510820.jpg vs ISIC_6385839.jpg: 0.3574
# ISIC_1510820.jpg vs ISIC_6387900.jpg: 0.7153
# ISIC_1510820.jpg vs ISIC_6388741.jpg: 0.7533
# ISIC_1510820.jpg vs ISIC_6388895.jpg: 0.4658
# ISIC_1510820.jpg vs ISIC_6389588.jpg: 0.7122
# ISIC_1510820.jpg vs ISIC_6390348.jpg: 0.5928
# ISIC_1510820.jpg vs ISIC_6390419.jpg: 0.8055
# ISIC_1510820.jpg vs ISIC_6394052.jpg: 0.6978
# ISIC_1510820.jpg vs ISIC_6395203.jpg: 0.7940
# ISIC_1510820.jpg vs ISIC_6396303.jpg: 0.6750
# ISIC_1510820.jpg vs ISIC_6397215.jpg: 0.6339
# ISIC_1510820.jpg vs ISIC_6397471.jpg: 0.5762
# ISIC_1510820.jpg vs ISIC_6401781.jpg: 0.6076
# ISIC_1510820.jpg vs ISIC_6404736.jpg: 0.4148
# ISIC_1510820.jpg vs ISIC_6407209.jpg: 0.6239
# ISIC_1510820.jpg vs ISIC_6409218.jpg: 0.7733
# ISIC_1510820.jpg vs ISIC_6410742.jpg: 0.3066
# ISIC_1510820.jpg vs ISIC_6410833.jpg: 0.7380
# ISIC_1510820.jpg vs ISIC_6413760.jpg: 0.3435
# ISIC_1510820.jpg vs ISIC_6416091.jpg: 0.4118
# ISIC_1510820.jpg vs ISIC_6417661.jpg: 0.1624
# ISIC_1510820.jpg vs ISIC_6417770.jpg: 0.6280
# ISIC_1510820.jpg vs ISIC_6418859.jpg: 0.8622
# ISIC_1510820.jpg vs ISIC_6420072.jpg: 0.8248
# ISIC_1510820.jpg vs ISIC_6420618.jpg: 0.4205
# ISIC_1510820.jpg vs ISIC_6420980.jpg: 0.8078
# ISIC_1510820.jpg vs ISIC_6420986.jpg: 0.8822
# ISIC_1510820.jpg vs ISIC_6422928.jpg: 0.7192
# ISIC_1510820.jpg vs ISIC_6423812.jpg: 0.8878
# ISIC_1510820.jpg vs ISIC_6425676.jpg: 0.7627
# ISIC_1510820.jpg vs ISIC_6426806.jpg: 0.8085
# ISIC_1510820.jpg vs ISIC_6428265.jpg: 0.8022
# ISIC_1510820.jpg vs ISIC_6432856.jpg: 0.1783
# ISIC_1510820.jpg vs ISIC_6434719.jpg: 0.6727
# ISIC_1510820.jpg vs ISIC_6435735.jpg: 0.1992
# ISIC_1510820.jpg vs ISIC_6436323.jpg: 0.8076
# ISIC_1510820.jpg vs ISIC_6437816.jpg: 0.6220
# ISIC_1510820.jpg vs ISIC_6438534.jpg: 0.7485
# ISIC_1510820.jpg vs ISIC_6440128.jpg: 0.8089
# ISIC_1510820.jpg vs ISIC_6440823.jpg: 0.1790
# ISIC_1510820.jpg vs ISIC_6441462.jpg: 0.4638
# ISIC_1510820.jpg vs ISIC_6441653.jpg: 0.7619
# ISIC_1510820.jpg vs ISIC_6441975.jpg: 0.5595
# ISIC_1510820.jpg vs ISIC_6442196.jpg: 0.8300
# ISIC_1510820.jpg vs ISIC_6442624.jpg: 0.8572
# ISIC_1510820.jpg vs ISIC_6443328.jpg: 0.6332
# ISIC_1510820.jpg vs ISIC_6444637.jpg: 0.2239
# ISIC_1510820.jpg vs ISIC_6447876.jpg: 0.3620
# ISIC_1510820.jpg vs ISIC_6450938.jpg: 0.8278
# ISIC_1510820.jpg vs ISIC_6451105.jpg: 0.7502
# ISIC_1510820.jpg vs ISIC_6452363.jpg: 0.7247
# ISIC_1510820.jpg vs ISIC_6452610.jpg: 0.7166
# ISIC_1510820.jpg vs ISIC_6453432.jpg: 0.5541
# ISIC_1510820.jpg vs ISIC_6453867.jpg: 0.7515
# ISIC_1510820.jpg vs ISIC_6454808.jpg: 0.5418
# ISIC_1510820.jpg vs ISIC_6454901.jpg: 0.2937
# ISIC_1510820.jpg vs ISIC_6455851.jpg: 0.6356
# ISIC_1510820.jpg vs ISIC_6456359.jpg: 0.4019
# ISIC_1510820.jpg vs ISIC_6456904.jpg: 0.7021
# ISIC_1510820.jpg vs ISIC_6456968.jpg: 0.7068
# ISIC_1510820.jpg vs ISIC_6457041.jpg: 0.8807
# ISIC_1510820.jpg vs ISIC_6457527.jpg: 0.3803
# ISIC_1510820.jpg vs ISIC_6458793.jpg: 0.7651
# ISIC_1510820.jpg vs ISIC_6465527.jpg: 0.2084
# ISIC_1510820.jpg vs ISIC_6465712.jpg: 0.3342
# ISIC_1510820.jpg vs ISIC_6465748.jpg: 0.7635
# ISIC_1510820.jpg vs ISIC_6465955.jpg: 0.7293
# ISIC_1510820.jpg vs ISIC_6466968.jpg: 0.6043
# ISIC_1510820.jpg vs ISIC_6467041.jpg: 0.5197
# ISIC_1510820.jpg vs ISIC_6467817.jpg: 0.8096
# ISIC_1510820.jpg vs ISIC_6470007.jpg: 0.4004
# ISIC_1510820.jpg vs ISIC_6470574.jpg: 0.4197
# ISIC_1510820.jpg vs ISIC_6471549.jpg: 0.3630
# ISIC_1510820.jpg vs ISIC_6473784.jpg: 0.2974
# ISIC_1510820.jpg vs ISIC_6474260.jpg: 0.6727
# ISIC_1510820.jpg vs ISIC_6474483.jpg: 0.4195
# ISIC_1510820.jpg vs ISIC_6476591.jpg: 0.4002
# ISIC_1510820.jpg vs ISIC_6477494.jpg: 0.7476
# ISIC_1510820.jpg vs ISIC_6477955.jpg: 0.7516
# ISIC_1510820.jpg vs ISIC_6477960.jpg: 0.3067
# ISIC_1510820.jpg vs ISIC_6478369.jpg: 0.7530
# ISIC_1510820.jpg vs ISIC_6482254.jpg: 0.1861
# ISIC_1510820.jpg vs ISIC_6482346.jpg: 0.6922
# ISIC_1510820.jpg vs ISIC_6483735.jpg: 0.5360
# ISIC_1510820.jpg vs ISIC_6484204.jpg: 0.6555
# ISIC_1510820.jpg vs ISIC_6485104.jpg: 0.6434
# ISIC_1510820.jpg vs ISIC_6485174.jpg: 0.7706
# ISIC_1510820.jpg vs ISIC_6485524.jpg: 0.8382
# ISIC_1510820.jpg vs ISIC_6489270.jpg: 0.6294
# ISIC_1510820.jpg vs ISIC_6489436.jpg: 0.9193
# ISIC_1510820.jpg vs ISIC_6492133.jpg: 0.2419
# ISIC_1510820.jpg vs ISIC_6493546.jpg: 0.6672
# ISIC_1510820.jpg vs ISIC_6496183.jpg: 0.5956
# ISIC_1510820.jpg vs ISIC_6496831.jpg: 0.3634
# ISIC_1510820.jpg vs ISIC_6499111.jpg: 0.8522
# ISIC_1510820.jpg vs ISIC_6499677.jpg: 0.5634
# ISIC_1510820.jpg vs ISIC_6501156.jpg: 0.5570
# ISIC_1510820.jpg vs ISIC_6501761.jpg: 0.2868
# ISIC_1510820.jpg vs ISIC_6503955.jpg: 0.6606
# ISIC_1510820.jpg vs ISIC_6504056.jpg: 0.7957
# ISIC_1510820.jpg vs ISIC_6504083.jpg: 0.8154
# ISIC_1510820.jpg vs ISIC_6504643.jpg: 0.8345
# ISIC_1510820.jpg vs ISIC_6504764.jpg: 0.6912
# ISIC_1510820.jpg vs ISIC_6505572.jpg: 0.7199
# ISIC_1510820.jpg vs ISIC_6507238.jpg: 0.8651
# ISIC_1510820.jpg vs ISIC_6508409.jpg: 0.6827
# ISIC_1510820.jpg vs ISIC_6509669.jpg: 0.7710
# ISIC_1510820.jpg vs ISIC_6510708.jpg: 0.4423
# ISIC_1510820.jpg vs ISIC_6510730.jpg: 0.5758
# ISIC_1510820.jpg vs ISIC_6510897.jpg: 0.7165
# ISIC_1510820.jpg vs ISIC_6510933.jpg: 0.8327
# ISIC_1510820.jpg vs ISIC_6512437.jpg: 0.8353
# ISIC_1510820.jpg vs ISIC_6513708.jpg: 0.7836
# ISIC_1510820.jpg vs ISIC_6514347.jpg: 0.2675
# ISIC_1510820.jpg vs ISIC_6516058.jpg: 0.2959
# ISIC_1510820.jpg vs ISIC_6516973.jpg: 0.7795
# ISIC_1510820.jpg vs ISIC_6517556.jpg: 0.8240
# ISIC_1510820.jpg vs ISIC_6518088.jpg: 0.4055
# ISIC_1510820.jpg vs ISIC_6519116.jpg: 0.1510
# ISIC_1510820.jpg vs ISIC_6519195.jpg: 0.3050
# ISIC_1510820.jpg vs ISIC_6519660.jpg: 0.6389
# ISIC_1510820.jpg vs ISIC_6521534.jpg: 0.7436
# ISIC_1510820.jpg vs ISIC_6522232.jpg: 0.6087
# ISIC_1510820.jpg vs ISIC_6524335.jpg: 0.7759
# ISIC_1510820.jpg vs ISIC_6524570.jpg: 0.7339
# ISIC_1510820.jpg vs ISIC_6526343.jpg: 0.6294
# ISIC_1510820.jpg vs ISIC_6530984.jpg: 0.6274
# ISIC_1510820.jpg vs ISIC_6531810.jpg: 0.3185
# ISIC_1510820.jpg vs ISIC_6533587.jpg: 0.7860
# ISIC_1510820.jpg vs ISIC_6535708.jpg: 0.8470
# ISIC_1510820.jpg vs ISIC_6538442.jpg: 0.7250
# ISIC_1510820.jpg vs ISIC_6538759.jpg: 0.8016
# ISIC_1510820.jpg vs ISIC_6539905.jpg: 0.2331
# ISIC_1510820.jpg vs ISIC_6540771.jpg: 0.3097
# ISIC_1510820.jpg vs ISIC_6541434.jpg: 0.7122
# ISIC_1510820.jpg vs ISIC_6541945.jpg: 0.4195
# ISIC_1510820.jpg vs ISIC_6542061.jpg: 0.7308
# ISIC_1510820.jpg vs ISIC_6544818.jpg: 0.8154
# ISIC_1510820.jpg vs ISIC_6545625.jpg: 0.8325
# ISIC_1510820.jpg vs ISIC_6545813.jpg: 0.8360
# ISIC_1510820.jpg vs ISIC_6546383.jpg: 0.8955
# ISIC_1510820.jpg vs ISIC_6549364.jpg: 0.5185
# ISIC_1510820.jpg vs ISIC_6549522.jpg: 0.7475
# ISIC_1510820.jpg vs ISIC_6550440.jpg: 0.8833
# ISIC_1510820.jpg vs ISIC_6551637.jpg: 0.7085
# ISIC_1510820.jpg vs ISIC_6553836.jpg: 0.8693
# ISIC_1510820.jpg vs ISIC_6554420.jpg: 0.6015
# ISIC_1510820.jpg vs ISIC_6555901.jpg: 0.7372
# ISIC_1510820.jpg vs ISIC_6558015.jpg: 0.7666
# ISIC_1510820.jpg vs ISIC_6560711.jpg: 0.7716
# ISIC_1510820.jpg vs ISIC_6561134.jpg: 0.4000
# ISIC_1510820.jpg vs ISIC_6561559.jpg: 0.7160
# ISIC_1510820.jpg vs ISIC_6563141.jpg: 0.7491
# ISIC_1510820.jpg vs ISIC_6570595.jpg: 0.8871
# ISIC_1510820.jpg vs ISIC_6570791.jpg: 0.8053
# ISIC_1510820.jpg vs ISIC_6570876.jpg: 0.5537
# ISIC_1510820.jpg vs ISIC_6571223.jpg: 0.1453
# ISIC_1510820.jpg vs ISIC_6572392.jpg: 0.8920
# ISIC_1510820.jpg vs ISIC_6572503.jpg: 0.7366
# ISIC_1510820.jpg vs ISIC_6575327.jpg: 0.8169
# ISIC_1510820.jpg vs ISIC_6575489.jpg: 0.8091
# ISIC_1510820.jpg vs ISIC_6576250.jpg: 0.4225
# ISIC_1510820.jpg vs ISIC_6578095.jpg: 0.7624
# ISIC_1510820.jpg vs ISIC_6581064.jpg: 0.7383
# ISIC_1510820.jpg vs ISIC_6600704.jpg: 0.7934
# ISIC_1510820.jpg vs ISIC_6613137.jpg: 0.6827
# ISIC_1510820.jpg vs ISIC_6614545.jpg: 0.2048
# ISIC_1510820.jpg vs ISIC_6617622.jpg: 0.6860
# ISIC_1510820.jpg vs ISIC_6618544.jpg: 0.5618
# ISIC_1510820.jpg vs ISIC_6618864.jpg: 0.5719
# ISIC_1510820.jpg vs ISIC_6620053.jpg: 0.5132
# ISIC_1510820.jpg vs ISIC_6622777.jpg: 0.4714
# ISIC_1510820.jpg vs ISIC_6638136.jpg: 0.7511
# ISIC_1510820.jpg vs ISIC_6639597.jpg: 0.7911
# ISIC_1510820.jpg vs ISIC_6649139.jpg: 0.4745
# ISIC_1510820.jpg vs ISIC_6654360.jpg: 0.4336
# ISIC_1510820.jpg vs ISIC_6661454.jpg: 0.3474
# ISIC_1510820.jpg vs ISIC_6679395.jpg: 0.7476
# ISIC_1510820.jpg vs ISIC_6680545.jpg: 0.8455
# ISIC_1510820.jpg vs ISIC_6681100.jpg: 0.3787
# ISIC_1510820.jpg vs ISIC_6682047.jpg: 0.8444
# ISIC_1510820.jpg vs ISIC_6682323.jpg: 0.4553
# ISIC_1510820.jpg vs ISIC_6683074.jpg: 0.3054
# ISIC_1510820.jpg vs ISIC_6684390.jpg: 0.2108
# ISIC_1510820.jpg vs ISIC_6684791.jpg: 0.7896
# ISIC_1510820.jpg vs ISIC_6684827.jpg: 0.8235
# ISIC_1510820.jpg vs ISIC_6685364.jpg: 0.6459
# ISIC_1510820.jpg vs ISIC_6687916.jpg: 0.6216
# ISIC_1510820.jpg vs ISIC_6689584.jpg: 0.6787
# ISIC_1510820.jpg vs ISIC_6689820.jpg: 0.7622
# ISIC_1510820.jpg vs ISIC_6690051.jpg: 0.1823
# ISIC_1510820.jpg vs ISIC_6690278.jpg: 0.8947
# ISIC_1510820.jpg vs ISIC_6690331.jpg: 0.9034
# ISIC_1510820.jpg vs ISIC_6694996.jpg: 0.5341
# ISIC_1510820.jpg vs ISIC_6696474.jpg: 0.8267
# ISIC_1510820.jpg vs ISIC_6699429.jpg: 0.8189
# ISIC_1510820.jpg vs ISIC_6700104.jpg: 0.6889
# ISIC_1510820.jpg vs ISIC_6701908.jpg: 0.7992
# ISIC_1510820.jpg vs ISIC_6701974.jpg: 0.7973
# ISIC_1510820.jpg vs ISIC_6702499.jpg: 0.7074
# ISIC_1510820.jpg vs ISIC_6703533.jpg: 0.6194
# ISIC_1510820.jpg vs ISIC_6704967.jpg: 0.3159
# ISIC_1510820.jpg vs ISIC_6705213.jpg: 0.8096
# ISIC_1510820.jpg vs ISIC_6710398.jpg: 0.8629
# ISIC_1510820.jpg vs ISIC_6711260.jpg: 0.4035
# ISIC_1510820.jpg vs ISIC_6712374.jpg: 0.3804
# ISIC_1510820.jpg vs ISIC_6712576.jpg: 0.6171
# ISIC_1510820.jpg vs ISIC_6713125.jpg: 0.4312
# ISIC_1510820.jpg vs ISIC_6715217.jpg: 0.7801
# ISIC_1510820.jpg vs ISIC_6718057.jpg: 0.8525
# ISIC_1510820.jpg vs ISIC_6719897.jpg: 0.8570
# ISIC_1510820.jpg vs ISIC_6720144.jpg: 0.8639
# ISIC_1510820.jpg vs ISIC_6720892.jpg: 0.5973
# ISIC_1510820.jpg vs ISIC_6723980.jpg: 0.5947
# ISIC_1510820.jpg vs ISIC_6724105.jpg: 0.8820
# ISIC_1510820.jpg vs ISIC_6724237.jpg: 0.8226
# ISIC_1510820.jpg vs ISIC_6724343.jpg: 0.9210
# ISIC_1510820.jpg vs ISIC_6724629.jpg: 0.2319
# ISIC_1510820.jpg vs ISIC_6727968.jpg: 0.3792
# ISIC_1510820.jpg vs ISIC_6728356.jpg: 0.4720
# ISIC_1510820.jpg vs ISIC_6728492.jpg: 0.7247
# ISIC_1510820.jpg vs ISIC_6729337.jpg: 0.6973
# ISIC_1510820.jpg vs ISIC_6729820.jpg: 0.7037
# ISIC_1510820.jpg vs ISIC_6730696.jpg: 0.5174
# ISIC_1510820.jpg vs ISIC_6731469.jpg: 0.4533
# ISIC_1510820.jpg vs ISIC_6733544.jpg: 0.7037
# ISIC_1510820.jpg vs ISIC_6734840.jpg: 0.2653
# ISIC_1510820.jpg vs ISIC_6736963.jpg: 0.8208
# ISIC_1510820.jpg vs ISIC_6737042.jpg: 0.5405
# ISIC_1510820.jpg vs ISIC_6738385.jpg: 0.7242
# ISIC_1510820.jpg vs ISIC_6739853.jpg: 0.7564
# ISIC_1510820.jpg vs ISIC_6740312.jpg: 0.6452
# ISIC_1510820.jpg vs ISIC_6740393.jpg: 0.7920
# ISIC_1510820.jpg vs ISIC_6742788.jpg: 0.3854
# ISIC_1510820.jpg vs ISIC_6743437.jpg: 0.6256
# ISIC_1510820.jpg vs ISIC_6744358.jpg: 0.7120
# ISIC_1510820.jpg vs ISIC_6745056.jpg: 0.7750
# ISIC_1510820.jpg vs ISIC_6747997.jpg: 0.6139
# ISIC_1510820.jpg vs ISIC_6749430.jpg: 0.6073
# ISIC_1510820.jpg vs ISIC_6750286.jpg: 0.1681
# ISIC_1510820.jpg vs ISIC_6750614.jpg: 0.8298
# ISIC_1510820.jpg vs ISIC_6751174.jpg: 0.6313
# ISIC_1510820.jpg vs ISIC_6751717.jpg: 0.9001
# ISIC_1510820.jpg vs ISIC_6754458.jpg: 0.6331
# ISIC_1510820.jpg vs ISIC_6755755.jpg: 0.8998
# ISIC_1510820.jpg vs ISIC_6756392.jpg: 0.7242
# ISIC_1510820.jpg vs ISIC_6756448.jpg: 0.8069
# ISIC_1510820.jpg vs ISIC_6756925.jpg: 0.3833
# ISIC_1510820.jpg vs ISIC_6762136.jpg: 0.4400
# ISIC_1510820.jpg vs ISIC_6762834.jpg: 0.2532
# ISIC_1510820.jpg vs ISIC_6764328.jpg: 0.8938
# ISIC_1510820.jpg vs ISIC_6768098.jpg: 0.6111
# ISIC_1510820.jpg vs ISIC_6770701.jpg: 0.3697
# ISIC_1510820.jpg vs ISIC_6771747.jpg: 0.6832
# ISIC_1510820.jpg vs ISIC_6773252.jpg: 0.6227
# ISIC_1510820.jpg vs ISIC_6777221.jpg: 0.6863
# ISIC_1510820.jpg vs ISIC_6777761.jpg: 0.4136
# ISIC_1510820.jpg vs ISIC_6777956.jpg: 0.5196
# ISIC_1510820.jpg vs ISIC_6779189.jpg: 0.1113
# ISIC_1510820.jpg vs ISIC_6779885.jpg: 0.7558
# ISIC_1510820.jpg vs ISIC_6781832.jpg: 0.3168
# ISIC_1510820.jpg vs ISIC_6782432.jpg: 0.8604
# ISIC_1510820.jpg vs ISIC_6788736.jpg: 0.8016
# ISIC_1510820.jpg vs ISIC_6789406.jpg: 0.4203
# ISIC_1510820.jpg vs ISIC_6793148.jpg: 0.5387
# ISIC_1510820.jpg vs ISIC_6793377.jpg: 0.4678
# ISIC_1510820.jpg vs ISIC_6793888.jpg: 0.7037
# ISIC_1510820.jpg vs ISIC_6795412.jpg: 0.7957
# ISIC_1510820.jpg vs ISIC_6795855.jpg: 0.1557
# ISIC_1510820.jpg vs ISIC_6797452.jpg: 0.4673
# ISIC_1510820.jpg vs ISIC_6797664.jpg: 0.5652
# ISIC_1510820.jpg vs ISIC_6802138.jpg: 0.6755
# ISIC_1510820.jpg vs ISIC_6805686.jpg: 0.3121
# ISIC_1510820.jpg vs ISIC_6810659.jpg: 0.6194
# ISIC_1510820.jpg vs ISIC_6811237.jpg: 0.5221
# ISIC_1510820.jpg vs ISIC_6811416.jpg: 0.6171
# ISIC_1510820.jpg vs ISIC_6813530.jpg: 0.8827
# ISIC_1510820.jpg vs ISIC_6813763.jpg: 0.3109
# ISIC_1510820.jpg vs ISIC_6814727.jpg: 0.1042
# ISIC_1510820.jpg vs ISIC_6815865.jpg: 0.2194
# ISIC_1510820.jpg vs ISIC_6819277.jpg: 0.9018
# ISIC_1510820.jpg vs ISIC_6821845.jpg: 0.4969
# ISIC_1510820.jpg vs ISIC_6822272.jpg: 0.1464
# ISIC_1510820.jpg vs ISIC_6825910.jpg: 0.4774
# ISIC_1510820.jpg vs ISIC_6826117.jpg: 0.9041
# ISIC_1510820.jpg vs ISIC_6827668.jpg: 0.6790
# ISIC_1510820.jpg vs ISIC_6828354.jpg: 0.7093
# ISIC_1510820.jpg vs ISIC_6831983.jpg: 0.8728
# ISIC_1510820.jpg vs ISIC_6835703.jpg: 0.7046
# ISIC_1510820.jpg vs ISIC_6836798.jpg: 0.8938
# ISIC_1510820.jpg vs ISIC_6837581.jpg: 0.8025
# ISIC_1510820.jpg vs ISIC_6839259.jpg: 0.7410
# ISIC_1510820.jpg vs ISIC_6840007.jpg: 0.5825
# ISIC_1510820.jpg vs ISIC_6843228.jpg: 0.8719
# ISIC_1510820.jpg vs ISIC_6844611.jpg: 0.5331
# ISIC_1510820.jpg vs ISIC_6845523.jpg: 0.9022
# ISIC_1510820.jpg vs ISIC_6850984.jpg: 0.7296
# ISIC_1510820.jpg vs ISIC_6852747.jpg: 0.4773
# ISIC_1510820.jpg vs ISIC_6853184.jpg: 0.7222
# ISIC_1510820.jpg vs ISIC_6853620.jpg: 0.4499
# ISIC_1510820.jpg vs ISIC_6855256.jpg: 0.6448
# ISIC_1510820.jpg vs ISIC_6857184.jpg: 0.4646
# ISIC_1510820.jpg vs ISIC_6857576.jpg: 0.5756
# ISIC_1510820.jpg vs ISIC_6858056.jpg: 0.3130
# ISIC_1510820.jpg vs ISIC_6862118.jpg: 0.2261
# ISIC_1510820.jpg vs ISIC_6867311.jpg: 0.8660
# ISIC_1510820.jpg vs ISIC_6867950.jpg: 0.7544
# ISIC_1510820.jpg vs ISIC_6870521.jpg: 0.5759
# ISIC_1510820.jpg vs ISIC_6870887.jpg: 0.3328
# ISIC_1510820.jpg vs ISIC_6875383.jpg: 0.8837
# ISIC_1510820.jpg vs ISIC_6875932.jpg: 0.7452
# ISIC_1510820.jpg vs ISIC_6876145.jpg: 0.8369
# ISIC_1510820.jpg vs ISIC_6876493.jpg: 0.3526
# ISIC_1510820.jpg vs ISIC_6877005.jpg: 0.3831
# ISIC_1510820.jpg vs ISIC_6877629.jpg: 0.7837
# ISIC_1510820.jpg vs ISIC_6880676.jpg: 0.4387
# ISIC_1510820.jpg vs ISIC_6880871.jpg: 0.6100
# ISIC_1510820.jpg vs ISIC_6881441.jpg: 0.7051
# ISIC_1510820.jpg vs ISIC_6881924.jpg: 0.7276
# ISIC_1510820.jpg vs ISIC_6883060.jpg: 0.1768
# ISIC_1510820.jpg vs ISIC_6883104.jpg: 0.8308
# ISIC_1510820.jpg vs ISIC_6883430.jpg: 0.7557
# ISIC_1510820.jpg vs ISIC_6883524.jpg: 0.7567
# ISIC_1510820.jpg vs ISIC_6885218.jpg: 0.7842
# ISIC_1510820.jpg vs ISIC_6885654.jpg: 0.4619
# ISIC_1510820.jpg vs ISIC_6887266.jpg: 0.7853
# ISIC_1510820.jpg vs ISIC_6894377.jpg: 0.6658
# ISIC_1510820.jpg vs ISIC_6897661.jpg: 0.4701
# ISIC_1510820.jpg vs ISIC_6897793.jpg: 0.8050
# ISIC_1510820.jpg vs ISIC_6898013.jpg: 0.6531
# ISIC_1510820.jpg vs ISIC_6898290.jpg: 0.7182
# ISIC_1510820.jpg vs ISIC_6899150.jpg: 0.4668
# ISIC_1510820.jpg vs ISIC_6900078.jpg: 0.6420
# ISIC_1510820.jpg vs ISIC_6901140.jpg: 0.5713
# ISIC_1510820.jpg vs ISIC_6902302.jpg: 0.1555
# ISIC_1510820.jpg vs ISIC_6902548.jpg: 0.3776
# ISIC_1510820.jpg vs ISIC_6903486.jpg: 0.4287
# ISIC_1510820.jpg vs ISIC_6903770.jpg: 0.1976
# ISIC_1510820.jpg vs ISIC_6903816.jpg: 0.8345
# ISIC_1510820.jpg vs ISIC_6903971.jpg: 0.7858
# ISIC_1510820.jpg vs ISIC_6905379.jpg: 0.7967
# ISIC_1510820.jpg vs ISIC_6905962.jpg: 0.8061
# ISIC_1510820.jpg vs ISIC_6911199.jpg: 0.8707
# ISIC_1510820.jpg vs ISIC_6911963.jpg: 0.1216
# ISIC_1510820.jpg vs ISIC_6913320.jpg: 0.8093
# ISIC_1510820.jpg vs ISIC_6913600.jpg: 0.8766
# ISIC_1510820.jpg vs ISIC_6913983.jpg: 0.7756
# ISIC_1510820.jpg vs ISIC_6917374.jpg: 0.8885
# ISIC_1510820.jpg vs ISIC_6917597.jpg: 0.3631
# ISIC_1510820.jpg vs ISIC_6918208.jpg: 0.4952
# ISIC_1510820.jpg vs ISIC_6919598.jpg: 0.6537
# ISIC_1510820.jpg vs ISIC_6919934.jpg: 0.4735
# ISIC_1510820.jpg vs ISIC_6920450.jpg: 0.7870
# ISIC_1510820.jpg vs ISIC_6920596.jpg: 0.7733
# ISIC_1510820.jpg vs ISIC_6920609.jpg: 0.5355
# ISIC_1510820.jpg vs ISIC_6925576.jpg: 0.7256
# ISIC_1510820.jpg vs ISIC_6926294.jpg: 0.4245
# ISIC_1510820.jpg vs ISIC_6929794.jpg: 0.7741
# ISIC_1510820.jpg vs ISIC_6931246.jpg: 0.7379
# ISIC_1510820.jpg vs ISIC_6932354.jpg: 0.8139
# ISIC_1510820.jpg vs ISIC_6934272.jpg: 0.7916
# ISIC_1510820.jpg vs ISIC_6935489.jpg: 0.4309
# ISIC_1510820.jpg vs ISIC_6936286.jpg: 0.6401
# ISIC_1510820.jpg vs ISIC_6937025.jpg: 0.1607
# ISIC_1510820.jpg vs ISIC_6939698.jpg: 0.8000
# ISIC_1510820.jpg vs ISIC_6940742.jpg: 0.1917
# ISIC_1510820.jpg vs ISIC_6941854.jpg: 0.7644
# ISIC_1510820.jpg vs ISIC_6942800.jpg: 0.3445
# ISIC_1510820.jpg vs ISIC_6942995.jpg: 0.7714
# ISIC_1510820.jpg vs ISIC_6946895.jpg: 0.2746
# ISIC_1510820.jpg vs ISIC_6948933.jpg: 0.5843
# ISIC_1510820.jpg vs ISIC_6950754.jpg: 0.7839
# ISIC_1510820.jpg vs ISIC_6951279.jpg: 0.4105
# ISIC_1510820.jpg vs ISIC_6952190.jpg: 0.1580
# ISIC_1510820.jpg vs ISIC_6952589.jpg: 0.8301
# ISIC_1510820.jpg vs ISIC_6954370.jpg: 0.6982
# ISIC_1510820.jpg vs ISIC_6954625.jpg: 0.9101
# ISIC_1510820.jpg vs ISIC_6955630.jpg: 0.5723
# ISIC_1510820.jpg vs ISIC_6955637.jpg: 0.6400
# ISIC_1510820.jpg vs ISIC_6955838.jpg: 0.7828
# ISIC_1510820.jpg vs ISIC_6955962.jpg: 0.7002
# ISIC_1510820.jpg vs ISIC_6958771.jpg: 0.4986
# ISIC_1510820.jpg vs ISIC_6958842.jpg: 0.7110
# ISIC_1510820.jpg vs ISIC_6961265.jpg: 0.8244
# ISIC_1510820.jpg vs ISIC_6961583.jpg: 0.4674
# ISIC_1510820.jpg vs ISIC_6961770.jpg: 0.5338
# ISIC_1510820.jpg vs ISIC_6963944.jpg: 0.7036
# ISIC_1510820.jpg vs ISIC_6965271.jpg: 0.6764
# ISIC_1510820.jpg vs ISIC_6967784.jpg: 0.8177
# ISIC_1510820.jpg vs ISIC_6972434.jpg: 0.2608
# ISIC_1510820.jpg vs ISIC_6975236.jpg: 0.7567
# ISIC_1510820.jpg vs ISIC_6975486.jpg: 0.8331
# ISIC_1510820.jpg vs ISIC_6976452.jpg: 0.8349
# ISIC_1510820.jpg vs ISIC_6977473.jpg: 0.6309
# ISIC_1510820.jpg vs ISIC_6977916.jpg: 0.2987
# ISIC_1510820.jpg vs ISIC_6978556.jpg: 0.6235
# ISIC_1510820.jpg vs ISIC_6981145.jpg: 0.8003
# ISIC_1510820.jpg vs ISIC_6981516.jpg: 0.4126
# ISIC_1510820.jpg vs ISIC_6982312.jpg: 0.3927
# ISIC_1510820.jpg vs ISIC_6983099.jpg: 0.8819
# ISIC_1510820.jpg vs ISIC_6983482.jpg: 0.2469
# ISIC_1510820.jpg vs ISIC_6984644.jpg: 0.8305
# ISIC_1510820.jpg vs ISIC_6984658.jpg: 0.7400
# ISIC_1510820.jpg vs ISIC_6988163.jpg: 0.7642
# ISIC_1510820.jpg vs ISIC_6988262.jpg: 0.7099
# ISIC_1510820.jpg vs ISIC_6988497.jpg: 0.6318
# ISIC_1510820.jpg vs ISIC_6990099.jpg: 0.3442
# ISIC_1510820.jpg vs ISIC_6991088.jpg: 0.8614
# ISIC_1510820.jpg vs ISIC_6993184.jpg: 0.7817
# ISIC_1510820.jpg vs ISIC_6993349.jpg: 0.8813
# ISIC_1510820.jpg vs ISIC_6993350.jpg: 0.4746
# ISIC_1510820.jpg vs ISIC_7001013.jpg: 0.5455
# ISIC_1510820.jpg vs ISIC_7001504.jpg: 0.5124
# ISIC_1510820.jpg vs ISIC_7001659.jpg: 0.7428
# ISIC_1510820.jpg vs ISIC_7003908.jpg: 0.8015
# ISIC_1510820.jpg vs ISIC_7004578.jpg: 0.6793
# ISIC_1510820.jpg vs ISIC_7006251.jpg: 0.8742
# ISIC_1510820.jpg vs ISIC_7007065.jpg: 0.5669
# ISIC_1510820.jpg vs ISIC_7009636.jpg: 0.6007
# ISIC_1510820.jpg vs ISIC_7010755.jpg: 0.7326
# ISIC_1510820.jpg vs ISIC_7011312.jpg: 0.6854
# ISIC_1510820.jpg vs ISIC_7011852.jpg: 0.4081
# ISIC_1510820.jpg vs ISIC_7013717.jpg: 0.3335
# ISIC_1510820.jpg vs ISIC_7014635.jpg: 0.4595
# ISIC_1510820.jpg vs ISIC_7016381.jpg: 0.8969
# ISIC_1510820.jpg vs ISIC_7017459.jpg: 0.6194
# ISIC_1510820.jpg vs ISIC_7018756.jpg: 0.2356
# ISIC_1510820.jpg vs ISIC_7020654.jpg: 0.4830
# ISIC_1510820.jpg vs ISIC_7022512.jpg: 0.6946
# ISIC_1510820.jpg vs ISIC_7026533.jpg: 0.5237
# ISIC_1510820.jpg vs ISIC_7029435.jpg: 0.7344
# ISIC_1510820.jpg vs ISIC_7031413.jpg: 0.6945
# ISIC_1510820.jpg vs ISIC_7032767.jpg: 0.6173
# ISIC_1510820.jpg vs ISIC_7034056.jpg: 0.8798
# ISIC_1510820.jpg vs ISIC_7035208.jpg: 0.7757
# ISIC_1510820.jpg vs ISIC_7036830.jpg: 0.6964
# ISIC_1510820.jpg vs ISIC_7037702.jpg: 0.7371
# ISIC_1510820.jpg vs ISIC_7039072.jpg: 0.3532
# ISIC_1510820.jpg vs ISIC_7044408.jpg: 0.6245
# ISIC_1510820.jpg vs ISIC_7049364.jpg: 0.5923
# ISIC_1510820.jpg vs ISIC_7050025.jpg: 0.6598
# ISIC_1510820.jpg vs ISIC_7052464.jpg: 0.7614
# ISIC_1510820.jpg vs ISIC_7053272.jpg: 0.8769
# ISIC_1510820.jpg vs ISIC_7054725.jpg: 0.6441
# ISIC_1510820.jpg vs ISIC_7056434.jpg: 0.6738
# ISIC_1510820.jpg vs ISIC_7056912.jpg: 0.2367
# ISIC_1510820.jpg vs ISIC_7058873.jpg: 0.5477
# ISIC_1510820.jpg vs ISIC_7059105.jpg: 0.3312
# ISIC_1510820.jpg vs ISIC_7061139.jpg: 0.2914
# ISIC_1510820.jpg vs ISIC_7061494.jpg: 0.6516
# ISIC_1510820.jpg vs ISIC_7062201.jpg: 0.1753
# ISIC_1510820.jpg vs ISIC_7063177.jpg: 0.8706
# ISIC_1510820.jpg vs ISIC_7064337.jpg: 0.4988
# ISIC_1510820.jpg vs ISIC_7064511.jpg: 0.5386
# ISIC_1510820.jpg vs ISIC_7064592.jpg: 0.8249
# ISIC_1510820.jpg vs ISIC_7064704.jpg: 0.7129
# ISIC_1510820.jpg vs ISIC_7068535.jpg: 0.7476
# ISIC_1510820.jpg vs ISIC_7069639.jpg: 0.7041
# ISIC_1510820.jpg vs ISIC_7074909.jpg: 0.5496
# ISIC_1510820.jpg vs ISIC_7075768.jpg: 0.1896
# ISIC_1510820.jpg vs ISIC_7076194.jpg: 0.7573
# ISIC_1510820.jpg vs ISIC_7076983.jpg: 0.4087
# ISIC_1510820.jpg vs ISIC_7077024.jpg: 0.4882
# ISIC_1510820.jpg vs ISIC_7078308.jpg: 0.8651
# ISIC_1510820.jpg vs ISIC_7079491.jpg: 0.7588
# ISIC_1510820.jpg vs ISIC_7080192.jpg: 0.7906
# ISIC_1510820.jpg vs ISIC_7080469.jpg: 0.6368
# ISIC_1510820.jpg vs ISIC_7080556.jpg: 0.8931
# ISIC_1510820.jpg vs ISIC_7081220.jpg: 0.7598
# ISIC_1510820.jpg vs ISIC_7081223.jpg: 0.6127
# ISIC_1510820.jpg vs ISIC_7082609.jpg: 0.5117
# ISIC_1510820.jpg vs ISIC_7084116.jpg: 0.2288
# ISIC_1510820.jpg vs ISIC_7085260.jpg: 0.6599
# ISIC_1510820.jpg vs ISIC_7087406.jpg: 0.8193
# ISIC_1510820.jpg vs ISIC_7091133.jpg: 0.5711
# ISIC_1510820.jpg vs ISIC_7092566.jpg: 0.5724
# ISIC_1510820.jpg vs ISIC_7094353.jpg: 0.7591
# ISIC_1510820.jpg vs ISIC_7094448.jpg: 0.8679
# ISIC_1510820.jpg vs ISIC_7095065.jpg: 0.7527
# ISIC_1510820.jpg vs ISIC_7099778.jpg: 0.8177
# ISIC_1510820.jpg vs ISIC_7101448.jpg: 0.1898
# ISIC_1510820.jpg vs ISIC_7101450.jpg: 0.8029
# ISIC_1510820.jpg vs ISIC_7102106.jpg: 0.6573
# ISIC_1510820.jpg vs ISIC_7102421.jpg: 0.7712
# ISIC_1510820.jpg vs ISIC_7102450.jpg: 0.5802
# ISIC_1510820.jpg vs ISIC_7103029.jpg: 0.7692
# ISIC_1510820.jpg vs ISIC_7105259.jpg: 0.5638
# ISIC_1510820.jpg vs ISIC_7105294.jpg: 0.6550
# ISIC_1510820.jpg vs ISIC_7109036.jpg: 0.7853
# ISIC_1510820.jpg vs ISIC_7110004.jpg: 0.2109
# ISIC_1510820.jpg vs ISIC_7112065.jpg: 0.6028
# ISIC_1510820.jpg vs ISIC_7113087.jpg: 0.4561
# ISIC_1510820.jpg vs ISIC_7113110.jpg: 0.5264
# ISIC_1510820.jpg vs ISIC_7113256.jpg: 0.7807
# ISIC_1510820.jpg vs ISIC_7113418.jpg: 0.8655
# ISIC_1510820.jpg vs ISIC_7113586.jpg: 0.6300
# ISIC_1510820.jpg vs ISIC_7114023.jpg: 0.6848
# ISIC_1510820.jpg vs ISIC_7114727.jpg: 0.2737
# ISIC_1510820.jpg vs ISIC_7114912.jpg: 0.7119
# ISIC_1510820.jpg vs ISIC_7115225.jpg: 0.7815
# ISIC_1510820.jpg vs ISIC_7116294.jpg: 0.5922
# ISIC_1510820.jpg vs ISIC_7116684.jpg: 0.6587
# ISIC_1510820.jpg vs ISIC_7119932.jpg: 0.6841
# ISIC_1510820.jpg vs ISIC_7120406.jpg: 0.5138
# ISIC_1510820.jpg vs ISIC_7127226.jpg: 0.4673
# ISIC_1510820.jpg vs ISIC_7129969.jpg: 0.8373
# ISIC_1510820.jpg vs ISIC_7132534.jpg: 0.2530
# ISIC_1510820.jpg vs ISIC_7132780.jpg: 0.5116
# ISIC_1510820.jpg vs ISIC_7132896.jpg: 0.6739
# ISIC_1510820.jpg vs ISIC_7133530.jpg: 0.8134
# ISIC_1510820.jpg vs ISIC_7135160.jpg: 0.1572
# ISIC_1510820.jpg vs ISIC_7135304.jpg: 0.3046
# ISIC_1510820.jpg vs ISIC_7139089.jpg: 0.3744
# ISIC_1510820.jpg vs ISIC_7141412.jpg: 0.8924
# ISIC_1510820.jpg vs ISIC_7142335.jpg: 0.7559
# ISIC_1510820.jpg vs ISIC_7142471.jpg: 0.6883
# ISIC_1510820.jpg vs ISIC_7142776.jpg: 0.7113
# ISIC_1510820.jpg vs ISIC_7142991.jpg: 0.0975
# ISIC_1510820.jpg vs ISIC_7145784.jpg: 0.7075
# ISIC_1510820.jpg vs ISIC_7146714.jpg: 0.6739
# ISIC_1510820.jpg vs ISIC_7152033.jpg: 0.5807
# ISIC_1510820.jpg vs ISIC_7152851.jpg: 0.4520
# ISIC_1510820.jpg vs ISIC_7153281.jpg: 0.3837
# ISIC_1510820.jpg vs ISIC_7154902.jpg: 0.4628
# ISIC_1510820.jpg vs ISIC_7154992.jpg: 0.5581
# ISIC_1510820.jpg vs ISIC_7155184.jpg: 0.8388
# ISIC_1510820.jpg vs ISIC_7158189.jpg: 0.6894
# ISIC_1510820.jpg vs ISIC_7159168.jpg: 0.6852
# ISIC_1510820.jpg vs ISIC_7159776.jpg: 0.7064
# ISIC_1510820.jpg vs ISIC_7167759.jpg: 0.2575
# ISIC_1510820.jpg vs ISIC_7168654.jpg: 0.0747
# ISIC_1510820.jpg vs ISIC_7169582.jpg: 0.8127
# ISIC_1510820.jpg vs ISIC_7170016.jpg: 0.6127
# ISIC_1510820.jpg vs ISIC_7171310.jpg: 0.4787
# ISIC_1510820.jpg vs ISIC_7171951.jpg: 0.8353
# ISIC_1510820.jpg vs ISIC_7174145.jpg: 0.3543
# ISIC_1510820.jpg vs ISIC_7174811.jpg: 0.5587
# ISIC_1510820.jpg vs ISIC_7175329.jpg: 0.2610
# ISIC_1510820.jpg vs ISIC_7179756.jpg: 0.7472
# ISIC_1510820.jpg vs ISIC_7180429.jpg: 0.4682
# ISIC_1510820.jpg vs ISIC_7180740.jpg: 0.7751
# ISIC_1510820.jpg vs ISIC_7184527.jpg: 0.8191
# ISIC_1510820.jpg vs ISIC_7185224.jpg: 0.8703
# ISIC_1510820.jpg vs ISIC_7185618.jpg: 0.7871
# ISIC_1510820.jpg vs ISIC_7188491.jpg: 0.4783
# ISIC_1510820.jpg vs ISIC_7188780.jpg: 0.6512
# ISIC_1510820.jpg vs ISIC_7189518.jpg: 0.7931
# ISIC_1510820.jpg vs ISIC_7190153.jpg: 0.4230
# ISIC_1510820.jpg vs ISIC_7190671.jpg: 0.3574
# ISIC_1510820.jpg vs ISIC_7191641.jpg: 0.6916
# ISIC_1510820.jpg vs ISIC_7192464.jpg: 0.5917
# ISIC_1510820.jpg vs ISIC_7192471.jpg: 0.8302
# ISIC_1510820.jpg vs ISIC_7192748.jpg: 0.6925
# ISIC_1510820.jpg vs ISIC_7193199.jpg: 0.7968
# ISIC_1510820.jpg vs ISIC_7195248.jpg: 0.5940
# ISIC_1510820.jpg vs ISIC_7195286.jpg: 0.2707
# ISIC_1510820.jpg vs ISIC_7196620.jpg: 0.8115
# ISIC_1510820.jpg vs ISIC_7198276.jpg: 0.3811
# ISIC_1510820.jpg vs ISIC_7200910.jpg: 0.3511
# ISIC_1510820.jpg vs ISIC_7200911.jpg: 0.7206
# ISIC_1510820.jpg vs ISIC_7201379.jpg: 0.8101
# ISIC_1510820.jpg vs ISIC_7202197.jpg: 0.4331
# ISIC_1510820.jpg vs ISIC_7202841.jpg: 0.2933
# ISIC_1510820.jpg vs ISIC_7205346.jpg: 0.5457
# ISIC_1510820.jpg vs ISIC_7205470.jpg: 0.8447
# ISIC_1510820.jpg vs ISIC_7206270.jpg: 0.3011
# ISIC_1510820.jpg vs ISIC_7206758.jpg: 0.8895
# ISIC_1510820.jpg vs ISIC_7207612.jpg: 0.7869
# ISIC_1510820.jpg vs ISIC_7208438.jpg: 0.5475
# ISIC_1510820.jpg vs ISIC_7208452.jpg: 0.3301
# ISIC_1510820.jpg vs ISIC_7209713.jpg: 0.7041
# ISIC_1510820.jpg vs ISIC_7211673.jpg: 0.7677
# ISIC_1510820.jpg vs ISIC_7213439.jpg: 0.3809
# ISIC_1510820.jpg vs ISIC_7213459.jpg: 0.5401
# ISIC_1510820.jpg vs ISIC_7216563.jpg: 0.3918
# ISIC_1510820.jpg vs ISIC_7217280.jpg: 0.2122
# ISIC_1510820.jpg vs ISIC_7217766.jpg: 0.4739
# ISIC_1510820.jpg vs ISIC_7218038.jpg: 0.7205
# ISIC_1510820.jpg vs ISIC_7218942.jpg: 0.6297
# ISIC_1510820.jpg vs ISIC_7222316.jpg: 0.8408
# ISIC_1510820.jpg vs ISIC_7223807.jpg: 0.6630
# ISIC_1510820.jpg vs ISIC_7225062.jpg: 0.6414
# ISIC_1510820.jpg vs ISIC_7227997.jpg: 0.8642
# ISIC_1510820.jpg vs ISIC_7233451.jpg: 0.2066
# ISIC_1510820.jpg vs ISIC_7233969.jpg: 0.6725
# ISIC_1510820.jpg vs ISIC_7234089.jpg: 0.2754
# ISIC_1510820.jpg vs ISIC_7235152.jpg: 0.8773
# ISIC_1510820.jpg vs ISIC_7235719.jpg: 0.7169
# ISIC_1510820.jpg vs ISIC_7236296.jpg: 0.6834
# ISIC_1510820.jpg vs ISIC_7236531.jpg: 0.8164
# ISIC_1510820.jpg vs ISIC_7237885.jpg: 0.5758
# ISIC_1510820.jpg vs ISIC_7239806.jpg: 0.7815
# ISIC_1510820.jpg vs ISIC_7240170.jpg: 0.5528
# ISIC_1510820.jpg vs ISIC_7240964.jpg: 0.5459
# ISIC_1510820.jpg vs ISIC_7244482.jpg: 0.7788
# ISIC_1510820.jpg vs ISIC_7247219.jpg: 0.8052
# ISIC_1510820.jpg vs ISIC_7248498.jpg: 0.9120
# ISIC_1510820.jpg vs ISIC_7249001.jpg: 0.4489
# ISIC_1510820.jpg vs ISIC_7251830.jpg: 0.3734
# ISIC_1510820.jpg vs ISIC_7252720.jpg: 0.5520
# ISIC_1510820.jpg vs ISIC_7252876.jpg: 0.7832
# ISIC_1510820.jpg vs ISIC_7253599.jpg: 0.7124
# ISIC_1510820.jpg vs ISIC_7253799.jpg: 0.6627
# ISIC_1510820.jpg vs ISIC_7254179.jpg: 0.8406
# ISIC_1510820.jpg vs ISIC_7256487.jpg: 0.8461
# ISIC_1510820.jpg vs ISIC_7259222.jpg: 0.5777
# ISIC_1510820.jpg vs ISIC_7259800.jpg: 0.1995
# ISIC_1510820.jpg vs ISIC_7261608.jpg: 0.5855
# ISIC_1510820.jpg vs ISIC_7263763.jpg: 0.8282
# ISIC_1510820.jpg vs ISIC_7264521.jpg: 0.8152
# ISIC_1510820.jpg vs ISIC_7264689.jpg: 0.4958
# ISIC_1510820.jpg vs ISIC_7264722.jpg: 0.5220
# ISIC_1510820.jpg vs ISIC_7265782.jpg: 0.7875
# ISIC_1510820.jpg vs ISIC_7266688.jpg: 0.7456
# ISIC_1510820.jpg vs ISIC_7267148.jpg: 0.2868
# ISIC_1510820.jpg vs ISIC_7267800.jpg: 0.8555
# ISIC_1510820.jpg vs ISIC_7268291.jpg: 0.2625
# ISIC_1510820.jpg vs ISIC_7268637.jpg: 0.8345
# ISIC_1510820.jpg vs ISIC_7280569.jpg: 0.7698
# ISIC_1510820.jpg vs ISIC_7281152.jpg: 0.7640
# ISIC_1510820.jpg vs ISIC_7281652.jpg: 0.8289
# ISIC_1510820.jpg vs ISIC_7284217.jpg: 0.7447
# ISIC_1510820.jpg vs ISIC_7285430.jpg: 0.5150
# ISIC_1510820.jpg vs ISIC_7289947.jpg: 0.4079
# ISIC_1510820.jpg vs ISIC_7352675.jpg: 0.5642
# ISIC_1510820.jpg vs ISIC_7362188.jpg: 0.7770
# ISIC_1510820.jpg vs ISIC_7366427.jpg: 0.8565
# ISIC_1510820.jpg vs ISIC_7367079.jpg: 0.8946
# ISIC_1510820.jpg vs ISIC_7367641.jpg: 0.8210
# ISIC_1510820.jpg vs ISIC_7373084.jpg: 0.4984
# ISIC_1510820.jpg vs ISIC_7373877.jpg: 0.4519
# ISIC_1510820.jpg vs ISIC_7374271.jpg: 0.5547
# ISIC_1510820.jpg vs ISIC_7375926.jpg: 0.5277
# ISIC_1510820.jpg vs ISIC_7379934.jpg: 0.4715
# ISIC_1510820.jpg vs ISIC_7382419.jpg: 0.7228
# ISIC_1510820.jpg vs ISIC_7383713.jpg: 0.5024
# ISIC_1510820.jpg vs ISIC_7384016.jpg: 0.3327
# ISIC_1510820.jpg vs ISIC_7384826.jpg: 0.4597
# ISIC_1510820.jpg vs ISIC_7385553.jpg: 0.2402
# ISIC_1510820.jpg vs ISIC_7392354.jpg: 0.7475
# ISIC_1510820.jpg vs ISIC_7394850.jpg: 0.4657
# ISIC_1510820.jpg vs ISIC_7397275.jpg: 0.8924
# ISIC_1510820.jpg vs ISIC_7398705.jpg: 0.7926
# ISIC_1510820.jpg vs ISIC_7399356.jpg: 0.4577
# ISIC_1510820.jpg vs ISIC_7401344.jpg: 0.2791
# ISIC_1510820.jpg vs ISIC_7401900.jpg: 0.8545
# ISIC_1510820.jpg vs ISIC_7405314.jpg: 0.7365
# ISIC_1510820.jpg vs ISIC_7409499.jpg: 0.4992
# ISIC_1510820.jpg vs ISIC_7413080.jpg: 0.8561
# ISIC_1510820.jpg vs ISIC_7413622.jpg: 0.3884
# ISIC_1510820.jpg vs ISIC_7415965.jpg: 0.6901
# ISIC_1510820.jpg vs ISIC_7416462.jpg: 0.6671
# ISIC_1510820.jpg vs ISIC_7418191.jpg: 0.4009
# ISIC_1510820.jpg vs ISIC_7418611.jpg: 0.9037
# ISIC_1510820.jpg vs ISIC_7419326.jpg: 0.7558
# ISIC_1510820.jpg vs ISIC_7420003.jpg: 0.7595
# ISIC_1510820.jpg vs ISIC_7420156.jpg: 0.7608
# ISIC_1510820.jpg vs ISIC_7425333.jpg: 0.7317
# ISIC_1510820.jpg vs ISIC_7426756.jpg: 0.8720
# ISIC_1510820.jpg vs ISIC_7431726.jpg: 0.4080
# ISIC_1510820.jpg vs ISIC_7433838.jpg: 0.3893
# ISIC_1510820.jpg vs ISIC_7436486.jpg: 0.7778
# ISIC_1510820.jpg vs ISIC_7436513.jpg: 0.2969
# ISIC_1510820.jpg vs ISIC_7437829.jpg: 0.5039
# ISIC_1510820.jpg vs ISIC_7438856.jpg: 0.6186
# ISIC_1510820.jpg vs ISIC_7440908.jpg: 0.8689
# ISIC_1510820.jpg vs ISIC_7442694.jpg: 0.1240
# ISIC_1510820.jpg vs ISIC_7443279.jpg: 0.1968
# ISIC_1510820.jpg vs ISIC_7445432.jpg: 0.6883
# ISIC_1510820.jpg vs ISIC_7445535.jpg: 0.6437
# ISIC_1510820.jpg vs ISIC_7445753.jpg: 0.4477
# ISIC_1510820.jpg vs ISIC_7446208.jpg: 0.8398
# ISIC_1510820.jpg vs ISIC_7448043.jpg: 0.7412
# ISIC_1510820.jpg vs ISIC_7450400.jpg: 0.6925
# ISIC_1510820.jpg vs ISIC_7451544.jpg: 0.7693
# ISIC_1510820.jpg vs ISIC_7452111.jpg: 0.7017
# ISIC_1510820.jpg vs ISIC_7452361.jpg: 0.7574
# ISIC_1510820.jpg vs ISIC_7452557.jpg: 0.4851
# ISIC_1510820.jpg vs ISIC_7453678.jpg: 0.2964
# ISIC_1510820.jpg vs ISIC_7457700.jpg: 0.2676
# ISIC_1510820.jpg vs ISIC_7458062.jpg: 0.8292
# ISIC_1510820.jpg vs ISIC_7458366.jpg: 0.7562
# ISIC_1510820.jpg vs ISIC_7460098.jpg: 0.8047
# ISIC_1510820.jpg vs ISIC_7461873.jpg: 0.3811
# ISIC_1510820.jpg vs ISIC_7463689.jpg: 0.6620
# ISIC_1510820.jpg vs ISIC_7464332.jpg: 0.7212
# ISIC_1510820.jpg vs ISIC_7465205.jpg: 0.8456
# ISIC_1510820.jpg vs ISIC_7466378.jpg: 0.8802
# ISIC_1510820.jpg vs ISIC_7466714.jpg: 0.6504
# ISIC_1510820.jpg vs ISIC_7469476.jpg: 0.8184
# ISIC_1510820.jpg vs ISIC_7470317.jpg: 0.5753
# ISIC_1510820.jpg vs ISIC_7471956.jpg: 0.8871
# ISIC_1510820.jpg vs ISIC_7472363.jpg: 0.3498
# ISIC_1510820.jpg vs ISIC_7472702.jpg: 0.9145
# ISIC_1510820.jpg vs ISIC_7475630.jpg: 0.3488
# ISIC_1510820.jpg vs ISIC_7475644.jpg: 0.0604
# ISIC_1510820.jpg vs ISIC_7476014.jpg: 0.4139
# ISIC_1510820.jpg vs ISIC_7476093.jpg: 0.7445
# ISIC_1510820.jpg vs ISIC_7480165.jpg: 0.8847
# ISIC_1510820.jpg vs ISIC_7486739.jpg: 0.4356
# ISIC_1510820.jpg vs ISIC_7486911.jpg: 0.1404
# ISIC_1510820.jpg vs ISIC_7487858.jpg: 0.7775
# ISIC_1510820.jpg vs ISIC_7488189.jpg: 0.6782
# ISIC_1510820.jpg vs ISIC_7488192.jpg: 0.7863
# ISIC_1510820.jpg vs ISIC_7488921.jpg: 0.8300
# ISIC_1510820.jpg vs ISIC_7489203.jpg: 0.5269
# ISIC_1510820.jpg vs ISIC_7490240.jpg: 0.4597
# ISIC_1510820.jpg vs ISIC_7492542.jpg: 0.4563
# ISIC_1510820.jpg vs ISIC_7495312.jpg: 0.3848
# ISIC_1510820.jpg vs ISIC_7496557.jpg: 0.5366
# ISIC_1510820.jpg vs ISIC_7497192.jpg: 0.8953
# ISIC_1510820.jpg vs ISIC_7501669.jpg: 0.8360
# ISIC_1510820.jpg vs ISIC_7502317.jpg: 0.5211
# ISIC_1510820.jpg vs ISIC_7502935.jpg: 0.7257
# ISIC_1510820.jpg vs ISIC_7503199.jpg: 0.6284
# ISIC_1510820.jpg vs ISIC_7504384.jpg: 0.6964
# ISIC_1510820.jpg vs ISIC_7504825.jpg: 0.5518
# ISIC_1510820.jpg vs ISIC_7505887.jpg: 0.7428
# ISIC_1510820.jpg vs ISIC_7506745.jpg: 0.6560
# ISIC_1510820.jpg vs ISIC_7507905.jpg: 0.7942
# ISIC_1510820.jpg vs ISIC_7511353.jpg: 0.2605
# ISIC_1510820.jpg vs ISIC_7511434.jpg: 0.8491
# ISIC_1510820.jpg vs ISIC_7511758.jpg: 0.8166
# ISIC_1510820.jpg vs ISIC_7513617.jpg: 0.7237
# ISIC_1510820.jpg vs ISIC_7513737.jpg: 0.8306
# ISIC_1510820.jpg vs ISIC_7514527.jpg: 0.3216
# ISIC_1510820.jpg vs ISIC_7517764.jpg: 0.7163
# ISIC_1510820.jpg vs ISIC_7518616.jpg: 0.4564
# ISIC_1510820.jpg vs ISIC_7520133.jpg: 0.6109
# ISIC_1510820.jpg vs ISIC_7522358.jpg: 0.7160
# ISIC_1510820.jpg vs ISIC_7522632.jpg: 0.3926
# ISIC_1510820.jpg vs ISIC_7522882.jpg: 0.8795
# ISIC_1510820.jpg vs ISIC_7525904.jpg: 0.8416
# ISIC_1510820.jpg vs ISIC_7526898.jpg: 0.6694
# ISIC_1510820.jpg vs ISIC_7526999.jpg: 0.2763
# ISIC_1510820.jpg vs ISIC_7527610.jpg: 0.4110
# ISIC_1510820.jpg vs ISIC_7530370.jpg: 0.3593
# ISIC_1510820.jpg vs ISIC_7533893.jpg: 0.4672
# ISIC_1510820.jpg vs ISIC_7533998.jpg: 0.7227
# ISIC_1510820.jpg vs ISIC_7536031.jpg: 0.8881
# ISIC_1510820.jpg vs ISIC_7540410.jpg: 0.6651
# ISIC_1510820.jpg vs ISIC_7540663.jpg: 0.4611
# ISIC_1510820.jpg vs ISIC_7540893.jpg: 0.3002
# ISIC_1510820.jpg vs ISIC_7541281.jpg: 0.4935
# ISIC_1510820.jpg vs ISIC_7541564.jpg: 0.8828
# ISIC_1510820.jpg vs ISIC_7549311.jpg: 0.6936
# ISIC_1510820.jpg vs ISIC_7551036.jpg: 0.3216
# ISIC_1510820.jpg vs ISIC_7551811.jpg: 0.5009
# ISIC_1510820.jpg vs ISIC_7553223.jpg: 0.6906
# ISIC_1510820.jpg vs ISIC_7553317.jpg: 0.7034
# ISIC_1510820.jpg vs ISIC_7553987.jpg: 0.3028
# ISIC_1510820.jpg vs ISIC_7554237.jpg: 0.4798
# ISIC_1510820.jpg vs ISIC_7560441.jpg: 0.3698
# ISIC_1510820.jpg vs ISIC_7562455.jpg: 0.7541
# ISIC_1510820.jpg vs ISIC_7562805.jpg: 0.6346
# ISIC_1510820.jpg vs ISIC_7563708.jpg: 0.7056
# ISIC_1510820.jpg vs ISIC_7563835.jpg: 0.4291
# ISIC_1510820.jpg vs ISIC_7566175.jpg: 0.5347
# ISIC_1510820.jpg vs ISIC_7567821.jpg: 0.4857
# ISIC_1510820.jpg vs ISIC_7570492.jpg: 0.0853
# ISIC_1510820.jpg vs ISIC_7573213.jpg: 0.7201
# ISIC_1510820.jpg vs ISIC_7573463.jpg: 0.8209
# ISIC_1510820.jpg vs ISIC_7574085.jpg: 0.4119
# ISIC_1510820.jpg vs ISIC_7575027.jpg: 0.2429
# ISIC_1510820.jpg vs ISIC_7576311.jpg: 0.6374
# ISIC_1510820.jpg vs ISIC_7577155.jpg: 0.8768
# ISIC_1510820.jpg vs ISIC_7577303.jpg: 0.6974
# ISIC_1510820.jpg vs ISIC_7578283.jpg: 0.6734
# ISIC_1510820.jpg vs ISIC_7579848.jpg: 0.5195
# ISIC_1510820.jpg vs ISIC_7581451.jpg: 0.6391
# ISIC_1510820.jpg vs ISIC_7581494.jpg: 0.8859
# ISIC_1510820.jpg vs ISIC_7582923.jpg: 0.7563
# ISIC_1510820.jpg vs ISIC_7584134.jpg: 0.6099
# ISIC_1510820.jpg vs ISIC_7584926.jpg: 0.3720
# ISIC_1510820.jpg vs ISIC_7585202.jpg: 0.5944
# ISIC_1510820.jpg vs ISIC_7585351.jpg: 0.5771
# ISIC_1510820.jpg vs ISIC_7586290.jpg: 0.6366
# ISIC_1510820.jpg vs ISIC_7587095.jpg: 0.8636
# ISIC_1510820.jpg vs ISIC_7587191.jpg: 0.2346
# ISIC_1510820.jpg vs ISIC_7587476.jpg: 0.8211
# ISIC_1510820.jpg vs ISIC_7588607.jpg: 0.6337
# ISIC_1510820.jpg vs ISIC_7589420.jpg: 0.8174
# ISIC_1510820.jpg vs ISIC_7589828.jpg: 0.4302
# ISIC_1510820.jpg vs ISIC_7590061.jpg: 0.4741
# ISIC_1510820.jpg vs ISIC_7590800.jpg: 0.8007
# ISIC_1510820.jpg vs ISIC_7591125.jpg: 0.7262
# ISIC_1510820.jpg vs ISIC_7591683.jpg: 0.2965
# ISIC_1510820.jpg vs ISIC_7592323.jpg: 0.8083
# ISIC_1510820.jpg vs ISIC_7593089.jpg: 0.7230
# ISIC_1510820.jpg vs ISIC_7593339.jpg: 0.3778
# ISIC_1510820.jpg vs ISIC_7595646.jpg: 0.5137
# ISIC_1510820.jpg vs ISIC_7595718.jpg: 0.1001
# ISIC_1510820.jpg vs ISIC_7598211.jpg: 0.7706
# ISIC_1510820.jpg vs ISIC_7599726.jpg: 0.5014
# ISIC_1510820.jpg vs ISIC_7599730.jpg: 0.8285
# ISIC_1510820.jpg vs ISIC_7599734.jpg: 0.7440
# ISIC_1510820.jpg vs ISIC_7599882.jpg: 0.7881
# ISIC_1510820.jpg vs ISIC_7600077.jpg: 0.2823
# ISIC_1510820.jpg vs ISIC_7600777.jpg: 0.7897
# ISIC_1510820.jpg vs ISIC_7603453.jpg: 0.4778
# ISIC_1510820.jpg vs ISIC_7604683.jpg: 0.2480
# ISIC_1510820.jpg vs ISIC_7610042.jpg: 0.4766
# ISIC_1510820.jpg vs ISIC_7611037.jpg: 0.6908
# ISIC_1510820.jpg vs ISIC_7611734.jpg: 0.5452
# ISIC_1510820.jpg vs ISIC_7613717.jpg: 0.8479
# ISIC_1510820.jpg vs ISIC_7616219.jpg: 0.6781
# ISIC_1510820.jpg vs ISIC_7617371.jpg: 0.8511
# ISIC_1510820.jpg vs ISIC_7617438.jpg: 0.1883
# ISIC_1510820.jpg vs ISIC_7620523.jpg: 0.7192
# ISIC_1510820.jpg vs ISIC_7622418.jpg: 0.8172
# ISIC_1510820.jpg vs ISIC_7623074.jpg: 0.5891
# ISIC_1510820.jpg vs ISIC_7623848.jpg: 0.8126
# ISIC_1510820.jpg vs ISIC_7626131.jpg: 0.4357
# ISIC_1510820.jpg vs ISIC_7627502.jpg: 0.2597
# ISIC_1510820.jpg vs ISIC_7627576.jpg: 0.2110
# ISIC_1510820.jpg vs ISIC_7627778.jpg: 0.7974
# ISIC_1510820.jpg vs ISIC_7630288.jpg: 0.4671
# ISIC_1510820.jpg vs ISIC_7630611.jpg: 0.6697
# ISIC_1510820.jpg vs ISIC_7635759.jpg: 0.7847
# ISIC_1510820.jpg vs ISIC_7636162.jpg: 0.2356
# ISIC_1510820.jpg vs ISIC_7636412.jpg: 0.4774
# ISIC_1510820.jpg vs ISIC_7636842.jpg: 0.5743
# ISIC_1510820.jpg vs ISIC_7636930.jpg: 0.6140
# ISIC_1510820.jpg vs ISIC_7637046.jpg: 0.8408
# ISIC_1510820.jpg vs ISIC_7638072.jpg: 0.7623
# ISIC_1510820.jpg vs ISIC_7638885.jpg: 0.6305
# ISIC_1510820.jpg vs ISIC_7640239.jpg: 0.7900
# ISIC_1510820.jpg vs ISIC_7644360.jpg: 0.5121
# ISIC_1510820.jpg vs ISIC_7644866.jpg: 0.5850
# ISIC_1510820.jpg vs ISIC_7649184.jpg: 0.7987
# ISIC_1510820.jpg vs ISIC_7651538.jpg: 0.9170
# ISIC_1510820.jpg vs ISIC_7651760.jpg: 0.5232
# ISIC_1510820.jpg vs ISIC_7653328.jpg: 0.7806
# ISIC_1510820.jpg vs ISIC_7654912.jpg: 0.7444
# ISIC_1510820.jpg vs ISIC_7656837.jpg: 0.2093
# ISIC_1510820.jpg vs ISIC_7656871.jpg: 0.5806
# ISIC_1510820.jpg vs ISIC_7657493.jpg: 0.2039
# ISIC_1510820.jpg vs ISIC_7658312.jpg: 0.3201
# ISIC_1510820.jpg vs ISIC_7660676.jpg: 0.7014
# ISIC_1510820.jpg vs ISIC_7662282.jpg: 0.4055
# ISIC_1510820.jpg vs ISIC_7663385.jpg: 0.3503
# ISIC_1510820.jpg vs ISIC_7665759.jpg: 0.4075
# ISIC_1510820.jpg vs ISIC_7667543.jpg: 0.8628
# ISIC_1510820.jpg vs ISIC_7667553.jpg: 0.6847
# ISIC_1510820.jpg vs ISIC_7667909.jpg: 0.8331
# ISIC_1510820.jpg vs ISIC_7668118.jpg: 0.3411
# ISIC_1510820.jpg vs ISIC_7669867.jpg: 0.6620
# ISIC_1510820.jpg vs ISIC_7671624.jpg: 0.8382
# ISIC_1510820.jpg vs ISIC_7672975.jpg: 0.6678
# ISIC_1510820.jpg vs ISIC_7673649.jpg: 0.7496
# ISIC_1510820.jpg vs ISIC_7674913.jpg: 0.2692
# ISIC_1510820.jpg vs ISIC_7679244.jpg: 0.7526
# ISIC_1510820.jpg vs ISIC_7680807.jpg: 0.8979
# ISIC_1510820.jpg vs ISIC_7682110.jpg: 0.6541
# ISIC_1510820.jpg vs ISIC_7683015.jpg: 0.6827
# ISIC_1510820.jpg vs ISIC_7684185.jpg: 0.1901
# ISIC_1510820.jpg vs ISIC_7684478.jpg: 0.3987
# ISIC_1510820.jpg vs ISIC_7685786.jpg: 0.1129
# ISIC_1510820.jpg vs ISIC_7689193.jpg: 0.5542
# ISIC_1510820.jpg vs ISIC_7692562.jpg: 0.9015
# ISIC_1510820.jpg vs ISIC_7695262.jpg: 0.7396
# ISIC_1510820.jpg vs ISIC_7697239.jpg: 0.7272
# ISIC_1510820.jpg vs ISIC_7697976.jpg: 0.4374
# ISIC_1510820.jpg vs ISIC_7701922.jpg: 0.7829
# ISIC_1510820.jpg vs ISIC_7704226.jpg: 0.7652
# ISIC_1510820.jpg vs ISIC_7708303.jpg: 0.8145
# ISIC_1510820.jpg vs ISIC_7709533.jpg: 0.8512
# ISIC_1510820.jpg vs ISIC_7710261.jpg: 0.5975
# ISIC_1510820.jpg vs ISIC_7711587.jpg: 0.3078
# ISIC_1510820.jpg vs ISIC_7713266.jpg: 0.5535
# ISIC_1510820.jpg vs ISIC_7713989.jpg: 0.8172
# ISIC_1510820.jpg vs ISIC_7716813.jpg: 0.7412
# ISIC_1510820.jpg vs ISIC_7717659.jpg: 0.8285
# ISIC_1510820.jpg vs ISIC_7718013.jpg: 0.5577
# ISIC_1510820.jpg vs ISIC_7722840.jpg: 0.7886
# ISIC_1510820.jpg vs ISIC_7723435.jpg: 0.8731
# ISIC_1510820.jpg vs ISIC_7724330.jpg: 0.3758
# ISIC_1510820.jpg vs ISIC_7725016.jpg: 0.4183
# ISIC_1510820.jpg vs ISIC_7725975.jpg: 0.7123
# ISIC_1510820.jpg vs ISIC_7727226.jpg: 0.7233
# ISIC_1510820.jpg vs ISIC_7729587.jpg: 0.8617
# ISIC_1510820.jpg vs ISIC_7731414.jpg: 0.7581
# ISIC_1510820.jpg vs ISIC_7733324.jpg: 0.8221
# ISIC_1510820.jpg vs ISIC_7733430.jpg: 0.7064
# ISIC_1510820.jpg vs ISIC_7733738.jpg: 0.5642
# ISIC_1510820.jpg vs ISIC_7733862.jpg: 0.3294
# ISIC_1510820.jpg vs ISIC_7736823.jpg: 0.8445
# ISIC_1510820.jpg vs ISIC_7737938.jpg: 0.7386
# ISIC_1510820.jpg vs ISIC_7740348.jpg: 0.4431
# ISIC_1510820.jpg vs ISIC_7743612.jpg: 0.6274
# ISIC_1510820.jpg vs ISIC_7744928.jpg: 0.5959
# ISIC_1510820.jpg vs ISIC_7750466.jpg: 0.8850
# ISIC_1510820.jpg vs ISIC_7750754.jpg: 0.7985
# ISIC_1510820.jpg vs ISIC_7753184.jpg: 0.7910
# ISIC_1510820.jpg vs ISIC_7753214.jpg: 0.8990
# ISIC_1510820.jpg vs ISIC_7753758.jpg: 0.5565
# ISIC_1510820.jpg vs ISIC_7754315.jpg: 0.6510
# ISIC_1510820.jpg vs ISIC_7754398.jpg: 0.7839
# ISIC_1510820.jpg vs ISIC_7756539.jpg: 0.9021
# ISIC_1510820.jpg vs ISIC_7756930.jpg: 0.7386
# ISIC_1510820.jpg vs ISIC_7761552.jpg: 0.2453
# ISIC_1510820.jpg vs ISIC_7764199.jpg: 0.3922
# ISIC_1510820.jpg vs ISIC_7764660.jpg: 0.2427
# ISIC_1510820.jpg vs ISIC_7766238.jpg: 0.7540
# ISIC_1510820.jpg vs ISIC_7766708.jpg: 0.8786
# ISIC_1510820.jpg vs ISIC_7769967.jpg: 0.5275
# ISIC_1510820.jpg vs ISIC_7770700.jpg: 0.6715
# ISIC_1510820.jpg vs ISIC_7773170.jpg: 0.7210
# ISIC_1510820.jpg vs ISIC_7774245.jpg: 0.6597
# ISIC_1510820.jpg vs ISIC_7775286.jpg: 0.6764
# ISIC_1510820.jpg vs ISIC_7778585.jpg: 0.4291
# ISIC_1510820.jpg vs ISIC_7779204.jpg: 0.5136
# ISIC_1510820.jpg vs ISIC_7779819.jpg: 0.3367
# ISIC_1510820.jpg vs ISIC_7779978.jpg: 0.4128
# ISIC_1510820.jpg vs ISIC_7781376.jpg: 0.2678
# ISIC_1510820.jpg vs ISIC_7781873.jpg: 0.4942
# ISIC_1510820.jpg vs ISIC_7781890.jpg: 0.7964
# ISIC_1510820.jpg vs ISIC_7784350.jpg: 0.7525
# ISIC_1510820.jpg vs ISIC_7785749.jpg: 0.7297
# ISIC_1510820.jpg vs ISIC_7788266.jpg: 0.3427
# ISIC_1510820.jpg vs ISIC_7789951.jpg: 0.9047
# ISIC_1510820.jpg vs ISIC_7791160.jpg: 0.6187
# ISIC_1510820.jpg vs ISIC_7791220.jpg: 0.7929
# ISIC_1510820.jpg vs ISIC_7794460.jpg: 0.8076
# ISIC_1510820.jpg vs ISIC_7799170.jpg: 0.2598
# ISIC_1510820.jpg vs ISIC_7800457.jpg: 0.5782
# ISIC_1510820.jpg vs ISIC_7800797.jpg: 0.7400
# ISIC_1510820.jpg vs ISIC_7801187.jpg: 0.5262
# ISIC_1510820.jpg vs ISIC_7803451.jpg: 0.8726
# ISIC_1510820.jpg vs ISIC_7803555.jpg: 0.7418
# ISIC_1510820.jpg vs ISIC_7804428.jpg: 0.5278
# ISIC_1510820.jpg vs ISIC_7804487.jpg: 0.6565
# ISIC_1510820.jpg vs ISIC_7804865.jpg: 0.5131
# ISIC_1510820.jpg vs ISIC_7805224.jpg: 0.7144
# ISIC_1510820.jpg vs ISIC_7805245.jpg: 0.6580
# ISIC_1510820.jpg vs ISIC_7805593.jpg: 0.4563
# ISIC_1510820.jpg vs ISIC_7806397.jpg: 0.5711
# ISIC_1510820.jpg vs ISIC_7807204.jpg: 0.6728
# ISIC_1510820.jpg vs ISIC_7811647.jpg: 0.8488
# ISIC_1510820.jpg vs ISIC_7811700.jpg: 0.7393
# ISIC_1510820.jpg vs ISIC_7813650.jpg: 0.8654
# ISIC_1510820.jpg vs ISIC_7813701.jpg: 0.5880
# ISIC_1510820.jpg vs ISIC_7814337.jpg: 0.5751
# ISIC_1510820.jpg vs ISIC_7816515.jpg: 0.7365
# ISIC_1510820.jpg vs ISIC_7816675.jpg: 0.2833
# ISIC_1510820.jpg vs ISIC_7817967.jpg: 0.7596
# ISIC_1510820.jpg vs ISIC_7818146.jpg: 0.3141
# ISIC_1510820.jpg vs ISIC_7824107.jpg: 0.7080
# ISIC_1510820.jpg vs ISIC_7825185.jpg: 0.7765
# ISIC_1510820.jpg vs ISIC_7827284.jpg: 0.7544
# ISIC_1510820.jpg vs ISIC_7832143.jpg: 0.2802
# ISIC_1510820.jpg vs ISIC_7833167.jpg: 0.6765
# ISIC_1510820.jpg vs ISIC_7833479.jpg: 0.2589
# ISIC_1510820.jpg vs ISIC_7834148.jpg: 0.8281
# ISIC_1510820.jpg vs ISIC_7835022.jpg: 0.8425
# ISIC_1510820.jpg vs ISIC_7835852.jpg: 0.8037
# ISIC_1510820.jpg vs ISIC_7836684.jpg: 0.1561
# ISIC_1510820.jpg vs ISIC_7837910.jpg: 0.4332
# ISIC_1510820.jpg vs ISIC_7838894.jpg: 0.6266
# ISIC_1510820.jpg vs ISIC_7841575.jpg: 0.2384
# ISIC_1510820.jpg vs ISIC_7842633.jpg: 0.4415
# ISIC_1510820.jpg vs ISIC_7842792.jpg: 0.5072
# ISIC_1510820.jpg vs ISIC_7844469.jpg: 0.4352
# ISIC_1510820.jpg vs ISIC_7844923.jpg: 0.4570
# ISIC_1510820.jpg vs ISIC_7844958.jpg: 0.8228
# ISIC_1510820.jpg vs ISIC_7845902.jpg: 0.8736
# ISIC_1510820.jpg vs ISIC_7846243.jpg: 0.7122
# ISIC_1510820.jpg vs ISIC_7847120.jpg: 0.7811
# ISIC_1510820.jpg vs ISIC_7847196.jpg: 0.6729
# ISIC_1510820.jpg vs ISIC_7848111.jpg: 0.8765
# ISIC_1510820.jpg vs ISIC_7849540.jpg: 0.6124
# ISIC_1510820.jpg vs ISIC_7849547.jpg: 0.1942
# ISIC_1510820.jpg vs ISIC_7851287.jpg: 0.6102
# ISIC_1510820.jpg vs ISIC_7853900.jpg: 0.3572
# ISIC_1510820.jpg vs ISIC_7857142.jpg: 0.3482
# ISIC_1510820.jpg vs ISIC_7858885.jpg: 0.7615
# ISIC_1510820.jpg vs ISIC_7860512.jpg: 0.5536
# ISIC_1510820.jpg vs ISIC_7861832.jpg: 0.8581
# ISIC_1510820.jpg vs ISIC_7862347.jpg: 0.2962
# ISIC_1510820.jpg vs ISIC_7863452.jpg: 0.7423
# ISIC_1510820.jpg vs ISIC_7863462.jpg: 0.8733
# ISIC_1510820.jpg vs ISIC_7863624.jpg: 0.2610
# ISIC_1510820.jpg vs ISIC_7865262.jpg: 0.8077
# ISIC_1510820.jpg vs ISIC_7866404.jpg: 0.7202
# ISIC_1510820.jpg vs ISIC_7868581.jpg: 0.8455
# ISIC_1510820.jpg vs ISIC_7872575.jpg: 0.7719
# ISIC_1510820.jpg vs ISIC_7873500.jpg: 0.7564
# ISIC_1510820.jpg vs ISIC_7876012.jpg: 0.5858
# ISIC_1510820.jpg vs ISIC_7876842.jpg: 0.8010
# ISIC_1510820.jpg vs ISIC_7877822.jpg: 0.2413
# ISIC_1510820.jpg vs ISIC_7880400.jpg: 0.1444
# ISIC_1510820.jpg vs ISIC_7881142.jpg: 0.7727
# ISIC_1510820.jpg vs ISIC_7881449.jpg: 0.5681
# ISIC_1510820.jpg vs ISIC_7881583.jpg: 0.7497
# ISIC_1510820.jpg vs ISIC_7883705.jpg: 0.8218
# ISIC_1510820.jpg vs ISIC_7883929.jpg: 0.7317
# ISIC_1510820.jpg vs ISIC_7884316.jpg: 0.7309
# ISIC_1510820.jpg vs ISIC_7885342.jpg: 0.3686
# ISIC_1510820.jpg vs ISIC_7888244.jpg: 0.6400
# ISIC_1510820.jpg vs ISIC_7888355.jpg: 0.1698
# ISIC_1510820.jpg vs ISIC_7889097.jpg: 0.2262
# ISIC_1510820.jpg vs ISIC_7889482.jpg: 0.7598
# ISIC_1510820.jpg vs ISIC_7889564.jpg: 0.4268
# ISIC_1510820.jpg vs ISIC_7892693.jpg: 0.8048
# ISIC_1510820.jpg vs ISIC_7896855.jpg: 0.8508
# ISIC_1510820.jpg vs ISIC_7898290.jpg: 0.8300
# ISIC_1510820.jpg vs ISIC_7901644.jpg: 0.3580
# ISIC_1510820.jpg vs ISIC_7902499.jpg: 0.8842
# ISIC_1510820.jpg vs ISIC_7903252.jpg: 0.7545
# ISIC_1510820.jpg vs ISIC_7904453.jpg: 0.7464
# ISIC_1510820.jpg vs ISIC_7904829.jpg: 0.4144
# ISIC_1510820.jpg vs ISIC_7906425.jpg: 0.2990
# ISIC_1510820.jpg vs ISIC_7907818.jpg: 0.6036
# ISIC_1510820.jpg vs ISIC_7910179.jpg: 0.7397
# ISIC_1510820.jpg vs ISIC_7913461.jpg: 0.6470
# ISIC_1510820.jpg vs ISIC_7916227.jpg: 0.6829
# ISIC_1510820.jpg vs ISIC_7916312.jpg: 0.7833
# ISIC_1510820.jpg vs ISIC_7916620.jpg: 0.7792
# ISIC_1510820.jpg vs ISIC_7917687.jpg: 0.2318
# ISIC_1510820.jpg vs ISIC_7918881.jpg: 0.8332
# ISIC_1510820.jpg vs ISIC_7920594.jpg: 0.7678
# ISIC_1510820.jpg vs ISIC_7921623.jpg: 0.6199
# ISIC_1510820.jpg vs ISIC_7922587.jpg: 0.5531
# ISIC_1510820.jpg vs ISIC_7923162.jpg: 0.3082
# ISIC_1510820.jpg vs ISIC_7925249.jpg: 0.7449
# ISIC_1510820.jpg vs ISIC_7926183.jpg: 0.8404
# ISIC_1510820.jpg vs ISIC_7927194.jpg: 0.8475
# ISIC_1510820.jpg vs ISIC_7927766.jpg: 0.8578
# ISIC_1510820.jpg vs ISIC_7927872.jpg: 0.2810
# ISIC_1510820.jpg vs ISIC_7933766.jpg: 0.3977
# ISIC_1510820.jpg vs ISIC_7934933.jpg: 0.8694
# ISIC_1510820.jpg vs ISIC_7937038.jpg: 0.5331
# ISIC_1510820.jpg vs ISIC_7939244.jpg: 0.4266
# ISIC_1510820.jpg vs ISIC_7940557.jpg: 0.1493
# ISIC_1510820.jpg vs ISIC_7941445.jpg: 0.7834
# ISIC_1510820.jpg vs ISIC_7941553.jpg: 0.1568
# ISIC_1510820.jpg vs ISIC_7941883.jpg: 0.8545
# ISIC_1510820.jpg vs ISIC_7942433.jpg: 0.4938
# ISIC_1510820.jpg vs ISIC_7945119.jpg: 0.6945
# ISIC_1510820.jpg vs ISIC_7945724.jpg: 0.6994
# ISIC_1510820.jpg vs ISIC_7946199.jpg: 0.5509
# ISIC_1510820.jpg vs ISIC_7949118.jpg: 0.4275
# ISIC_1510820.jpg vs ISIC_7949783.jpg: 0.4503
# ISIC_1510820.jpg vs ISIC_7951953.jpg: 0.8553
# ISIC_1510820.jpg vs ISIC_7953193.jpg: 0.8227
# ISIC_1510820.jpg vs ISIC_7953713.jpg: 0.8236
# ISIC_1510820.jpg vs ISIC_7956211.jpg: 0.2207
# ISIC_1510820.jpg vs ISIC_7956738.jpg: 0.0855
# ISIC_1510820.jpg vs ISIC_7958352.jpg: 0.5969
# ISIC_1510820.jpg vs ISIC_7958450.jpg: 0.8628
# ISIC_1510820.jpg vs ISIC_7960593.jpg: 0.2642
# ISIC_1510820.jpg vs ISIC_7960890.jpg: 0.7876
# ISIC_1510820.jpg vs ISIC_7965714.jpg: 0.1067
# ISIC_1510820.jpg vs ISIC_7968234.jpg: 0.4090
# ISIC_1510820.jpg vs ISIC_7969416.jpg: 0.7654
# ISIC_1510820.jpg vs ISIC_7969975.jpg: 0.6172
# ISIC_1510820.jpg vs ISIC_7974344.jpg: 0.7221
# ISIC_1510820.jpg vs ISIC_7974685.jpg: 0.2129
# ISIC_1510820.jpg vs ISIC_7974758.jpg: 0.8573
# ISIC_1510820.jpg vs ISIC_7975018.jpg: 0.6560
# ISIC_1510820.jpg vs ISIC_7991392.jpg: 0.8078
# ISIC_1510820.jpg vs ISIC_7991645.jpg: 0.8470
# ISIC_1510820.jpg vs ISIC_8024498.jpg: 0.5497
# ISIC_1510820.jpg vs ISIC_8031116.jpg: 0.6476
# ISIC_1510820.jpg vs ISIC_8038522.jpg: 0.1063
# ISIC_1510820.jpg vs ISIC_8039634.jpg: 0.6848
# ISIC_1510820.jpg vs ISIC_8039807.jpg: 0.6137
# ISIC_1510820.jpg vs ISIC_8045425.jpg: 0.2354
# ISIC_1510820.jpg vs ISIC_8074146.jpg: 0.4145
# ISIC_1510820.jpg vs ISIC_8083476.jpg: 0.8561
# ISIC_1510820.jpg vs ISIC_8105127.jpg: 0.6596
# ISIC_1510820.jpg vs ISIC_8106490.jpg: 0.4672
# ISIC_1510820.jpg vs ISIC_8108065.jpg: 0.5076
# ISIC_1510820.jpg vs ISIC_8108132.jpg: 0.7962
# ISIC_1510820.jpg vs ISIC_8109696.jpg: 0.6905
# ISIC_1510820.jpg vs ISIC_8111571.jpg: 0.6321
# ISIC_1510820.jpg vs ISIC_8111800.jpg: 0.8296
# ISIC_1510820.jpg vs ISIC_8113829.jpg: 0.5000
# ISIC_1510820.jpg vs ISIC_8115282.jpg: 0.7248
# ISIC_1510820.jpg vs ISIC_8119887.jpg: 0.8502
# ISIC_1510820.jpg vs ISIC_8119925.jpg: 0.7450
# ISIC_1510820.jpg vs ISIC_8120614.jpg: 0.6102
# ISIC_1510820.jpg vs ISIC_8122278.jpg: 0.7144
# ISIC_1510820.jpg vs ISIC_8122560.jpg: 0.7812
# ISIC_1510820.jpg vs ISIC_8125150.jpg: 0.7270
# ISIC_1510820.jpg vs ISIC_8128662.jpg: 0.6357
# ISIC_1510820.jpg vs ISIC_8128698.jpg: 0.7512
# ISIC_1510820.jpg vs ISIC_8130168.jpg: 0.6932
# ISIC_1510820.jpg vs ISIC_8130468.jpg: 0.7155
# ISIC_1510820.jpg vs ISIC_8131069.jpg: 0.6280
# ISIC_1510820.jpg vs ISIC_8131501.jpg: 0.5678
# ISIC_1510820.jpg vs ISIC_8131838.jpg: 0.8238
# ISIC_1510820.jpg vs ISIC_8134340.jpg: 0.1422
# ISIC_1510820.jpg vs ISIC_8134560.jpg: 0.7188
# ISIC_1510820.jpg vs ISIC_8135508.jpg: 0.6226
# ISIC_1510820.jpg vs ISIC_8135585.jpg: 0.6388
# ISIC_1510820.jpg vs ISIC_8136476.jpg: 0.7031
# ISIC_1510820.jpg vs ISIC_8136608.jpg: 0.5963
# ISIC_1510820.jpg vs ISIC_8138606.jpg: 0.8403
# ISIC_1510820.jpg vs ISIC_8139780.jpg: 0.5638
# ISIC_1510820.jpg vs ISIC_8143813.jpg: 0.6650
# ISIC_1510820.jpg vs ISIC_8143966.jpg: 0.6773
# ISIC_1510820.jpg vs ISIC_8144137.jpg: 0.8129
# ISIC_1510820.jpg vs ISIC_8145729.jpg: 0.1945
# ISIC_1510820.jpg vs ISIC_8147475.jpg: 0.4928
# ISIC_1510820.jpg vs ISIC_8149637.jpg: 0.5623
# ISIC_1510820.jpg vs ISIC_8149766.jpg: 0.8294
# ISIC_1510820.jpg vs ISIC_8151681.jpg: 0.8194
# ISIC_1510820.jpg vs ISIC_8154548.jpg: 0.4114
# ISIC_1510820.jpg vs ISIC_8155289.jpg: 0.2142
# ISIC_1510820.jpg vs ISIC_8157391.jpg: 0.7473
# ISIC_1510820.jpg vs ISIC_8159613.jpg: 0.2706
# ISIC_1510820.jpg vs ISIC_8160347.jpg: 0.8593
# ISIC_1510820.jpg vs ISIC_8161072.jpg: 0.6342
# ISIC_1510820.jpg vs ISIC_8161707.jpg: 0.8394
# ISIC_1510820.jpg vs ISIC_8163612.jpg: 0.3857
# ISIC_1510820.jpg vs ISIC_8165385.jpg: 0.8544
# ISIC_1510820.jpg vs ISIC_8166112.jpg: 0.7059
# ISIC_1510820.jpg vs ISIC_8167408.jpg: 0.2723
# ISIC_1510820.jpg vs ISIC_8168431.jpg: 0.8559
# ISIC_1510820.jpg vs ISIC_8171556.jpg: 0.1974
# ISIC_1510820.jpg vs ISIC_8172884.jpg: 0.7076
# ISIC_1510820.jpg vs ISIC_8173513.jpg: 0.4217
# ISIC_1510820.jpg vs ISIC_8173707.jpg: 0.7738
# ISIC_1510820.jpg vs ISIC_8176711.jpg: 0.7628
# ISIC_1510820.jpg vs ISIC_8178489.jpg: 0.4580
# ISIC_1510820.jpg vs ISIC_8178720.jpg: 0.6374
# ISIC_1510820.jpg vs ISIC_8179466.jpg: 0.7152
# ISIC_1510820.jpg vs ISIC_8179931.jpg: 0.3474
# ISIC_1510820.jpg vs ISIC_8179996.jpg: 0.8539
# ISIC_1510820.jpg vs ISIC_8182216.jpg: 0.0998
# ISIC_1510820.jpg vs ISIC_8185342.jpg: 0.3803
# ISIC_1510820.jpg vs ISIC_8187725.jpg: 0.1638
# ISIC_1510820.jpg vs ISIC_8187908.jpg: 0.7653
# ISIC_1510820.jpg vs ISIC_8188409.jpg: 0.3170
# ISIC_1510820.jpg vs ISIC_8189421.jpg: 0.7041
# ISIC_1510820.jpg vs ISIC_8190844.jpg: 0.8010
# ISIC_1510820.jpg vs ISIC_8191278.jpg: 0.7259
# ISIC_1510820.jpg vs ISIC_8194816.jpg: 0.3411
# ISIC_1510820.jpg vs ISIC_8195106.jpg: 0.5779
# ISIC_1510820.jpg vs ISIC_8196059.jpg: 0.2367
# ISIC_1510820.jpg vs ISIC_8196688.jpg: 0.8244
# ISIC_1510820.jpg vs ISIC_8197106.jpg: 0.6898
# ISIC_1510820.jpg vs ISIC_8199232.jpg: 0.1973
# ISIC_1510820.jpg vs ISIC_8200009.jpg: 0.7026
# ISIC_1510820.jpg vs ISIC_8200852.jpg: 0.7625
# ISIC_1510820.jpg vs ISIC_8200860.jpg: 0.7690
# ISIC_1510820.jpg vs ISIC_8202664.jpg: 0.3486
# ISIC_1510820.jpg vs ISIC_8203599.jpg: 0.7224
# ISIC_1510820.jpg vs ISIC_8205569.jpg: 0.7955
# ISIC_1510820.jpg vs ISIC_8206728.jpg: 0.3246
# ISIC_1510820.jpg vs ISIC_8207167.jpg: 0.5909
# ISIC_1510820.jpg vs ISIC_8208627.jpg: 0.4947
# ISIC_1510820.jpg vs ISIC_8209167.jpg: 0.5566
# ISIC_1510820.jpg vs ISIC_8209208.jpg: 0.7446
# ISIC_1510820.jpg vs ISIC_8211041.jpg: 0.6602
# ISIC_1510820.jpg vs ISIC_8212176.jpg: 0.3077
# ISIC_1510820.jpg vs ISIC_8216046.jpg: 0.4310
# ISIC_1510820.jpg vs ISIC_8217534.jpg: 0.7813
# ISIC_1510820.jpg vs ISIC_8218124.jpg: 0.3008
# ISIC_1510820.jpg vs ISIC_8223262.jpg: 0.8597
# ISIC_1510820.jpg vs ISIC_8224453.jpg: 0.4511
# ISIC_1510820.jpg vs ISIC_8225764.jpg: 0.8188
# ISIC_1510820.jpg vs ISIC_8232490.jpg: 0.6646
# ISIC_1510820.jpg vs ISIC_8232642.jpg: 0.4840
# ISIC_1510820.jpg vs ISIC_8234611.jpg: 0.2387
# ISIC_1510820.jpg vs ISIC_8238434.jpg: 0.3110
# ISIC_1510820.jpg vs ISIC_8238755.jpg: 0.6277
# ISIC_1510820.jpg vs ISIC_8238831.jpg: 0.2398
# ISIC_1510820.jpg vs ISIC_8239619.jpg: 0.5543
# ISIC_1510820.jpg vs ISIC_8240408.jpg: 0.5328
# ISIC_1510820.jpg vs ISIC_8242793.jpg: 0.5847
# ISIC_1510820.jpg vs ISIC_8243766.jpg: 0.8748
# ISIC_1510820.jpg vs ISIC_8243811.jpg: 0.3569
# ISIC_1510820.jpg vs ISIC_8244515.jpg: 0.1795
# ISIC_1510820.jpg vs ISIC_8246087.jpg: 0.5859
# ISIC_1510820.jpg vs ISIC_8247144.jpg: 0.6529
# ISIC_1510820.jpg vs ISIC_8247638.jpg: 0.6274
# ISIC_1510820.jpg vs ISIC_8250062.jpg: 0.5775
# ISIC_1510820.jpg vs ISIC_8251076.jpg: 0.1919
# ISIC_1510820.jpg vs ISIC_8252760.jpg: 0.4220
# ISIC_1510820.jpg vs ISIC_8253208.jpg: 0.7997
# ISIC_1510820.jpg vs ISIC_8253521.jpg: 0.3845
# ISIC_1510820.jpg vs ISIC_8256702.jpg: 0.2457
# ISIC_1510820.jpg vs ISIC_8258916.jpg: 0.6797
# ISIC_1510820.jpg vs ISIC_8259540.jpg: 0.6466
# ISIC_1510820.jpg vs ISIC_8263010.jpg: 0.7192
# ISIC_1510820.jpg vs ISIC_8263315.jpg: 0.5403
# ISIC_1510820.jpg vs ISIC_8267340.jpg: 0.1153
# ISIC_1510820.jpg vs ISIC_8267789.jpg: 0.7439
# ISIC_1510820.jpg vs ISIC_8270879.jpg: 0.5639
# ISIC_1510820.jpg vs ISIC_8274465.jpg: 0.4217
# ISIC_1510820.jpg vs ISIC_8274649.jpg: 0.5643
# ISIC_1510820.jpg vs ISIC_8276011.jpg: 0.8115
# ISIC_1510820.jpg vs ISIC_8277903.jpg: 0.8863
# ISIC_1510820.jpg vs ISIC_8278493.jpg: 0.6197
# ISIC_1510820.jpg vs ISIC_8279605.jpg: 0.8754
# ISIC_1510820.jpg vs ISIC_8280147.jpg: 0.7411
# ISIC_1510820.jpg vs ISIC_8280969.jpg: 0.6922
# ISIC_1510820.jpg vs ISIC_8282150.jpg: 0.7842
# ISIC_1510820.jpg vs ISIC_8283552.jpg: 0.7679
# ISIC_1510820.jpg vs ISIC_8284060.jpg: 0.4615
# ISIC_1510820.jpg vs ISIC_8285611.jpg: 0.6549
# ISIC_1510820.jpg vs ISIC_8286820.jpg: 0.2870
# ISIC_1510820.jpg vs ISIC_8290327.jpg: 0.8440
# ISIC_1510820.jpg vs ISIC_8290815.jpg: 0.5448
# ISIC_1510820.jpg vs ISIC_8293323.jpg: 0.5724
# ISIC_1510820.jpg vs ISIC_8294494.jpg: 0.6326
# ISIC_1510820.jpg vs ISIC_8294587.jpg: 0.7831
# ISIC_1510820.jpg vs ISIC_8296738.jpg: 0.6521
# ISIC_1510820.jpg vs ISIC_8298051.jpg: 0.7589
# ISIC_1510820.jpg vs ISIC_8299116.jpg: 0.8493
# ISIC_1510820.jpg vs ISIC_8299739.jpg: 0.7582
# ISIC_1510820.jpg vs ISIC_8301281.jpg: 0.5817
# ISIC_1510820.jpg vs ISIC_8302863.jpg: 0.7684
# ISIC_1510820.jpg vs ISIC_8305995.jpg: 0.8716
# ISIC_1510820.jpg vs ISIC_8307265.jpg: 0.6951
# ISIC_1510820.jpg vs ISIC_8310539.jpg: 0.1966
# ISIC_1510820.jpg vs ISIC_8310711.jpg: 0.8739
# ISIC_1510820.jpg vs ISIC_8312411.jpg: 0.8002
# ISIC_1510820.jpg vs ISIC_8313489.jpg: 0.5226
# ISIC_1510820.jpg vs ISIC_8313831.jpg: 0.7707
# ISIC_1510820.jpg vs ISIC_8314801.jpg: 0.6718
# ISIC_1510820.jpg vs ISIC_8318169.jpg: 0.4211
# ISIC_1510820.jpg vs ISIC_8319946.jpg: 0.7745
# ISIC_1510820.jpg vs ISIC_8321575.jpg: 0.5802
# ISIC_1510820.jpg vs ISIC_8323806.jpg: 0.2494
# ISIC_1510820.jpg vs ISIC_8325595.jpg: 0.8124
# ISIC_1510820.jpg vs ISIC_8327784.jpg: 0.8357
# ISIC_1510820.jpg vs ISIC_8328232.jpg: 0.6933
# ISIC_1510820.jpg vs ISIC_8331130.jpg: 0.5690
# ISIC_1510820.jpg vs ISIC_8332176.jpg: 0.7204
# ISIC_1510820.jpg vs ISIC_8332544.jpg: 0.4981
# ISIC_1510820.jpg vs ISIC_8334170.jpg: 0.7471
# ISIC_1510820.jpg vs ISIC_8334342.jpg: 0.8229
# ISIC_1510820.jpg vs ISIC_8335421.jpg: 0.8638
# ISIC_1510820.jpg vs ISIC_8336115.jpg: 0.4411
# ISIC_1510820.jpg vs ISIC_8336838.jpg: 0.8002
# ISIC_1510820.jpg vs ISIC_8338699.jpg: 0.7446
# ISIC_1510820.jpg vs ISIC_8341368.jpg: 0.8761
# ISIC_1510820.jpg vs ISIC_8341620.jpg: 0.6423
# ISIC_1510820.jpg vs ISIC_8341922.jpg: 0.8651
# ISIC_1510820.jpg vs ISIC_8344285.jpg: 0.2692
# ISIC_1510820.jpg vs ISIC_8344755.jpg: 0.2159
# ISIC_1510820.jpg vs ISIC_8345288.jpg: 0.1450
# ISIC_1510820.jpg vs ISIC_8346281.jpg: 0.6906
# ISIC_1510820.jpg vs ISIC_8350329.jpg: 0.5029
# ISIC_1510820.jpg vs ISIC_8352015.jpg: 0.8929
# ISIC_1510820.jpg vs ISIC_8352016.jpg: 0.2549
# ISIC_1510820.jpg vs ISIC_8353253.jpg: 0.8200
# ISIC_1510820.jpg vs ISIC_8354561.jpg: 0.5001
# ISIC_1510820.jpg vs ISIC_8354874.jpg: 0.8055
# ISIC_1510820.jpg vs ISIC_8357411.jpg: 0.1782
# ISIC_1510820.jpg vs ISIC_8357923.jpg: 0.5238
# ISIC_1510820.jpg vs ISIC_8363542.jpg: 0.6915
# ISIC_1510820.jpg vs ISIC_8366946.jpg: 0.6614
# ISIC_1510820.jpg vs ISIC_8368715.jpg: 0.2357
# ISIC_1510820.jpg vs ISIC_8369268.jpg: 0.8891
# ISIC_1510820.jpg vs ISIC_8370269.jpg: 0.8179
# ISIC_1510820.jpg vs ISIC_8370736.jpg: 0.8593
# ISIC_1510820.jpg vs ISIC_8371546.jpg: 0.6820
# ISIC_1510820.jpg vs ISIC_8372206.jpg: 0.3790
# ISIC_1510820.jpg vs ISIC_8372452.jpg: 0.7816
# ISIC_1510820.jpg vs ISIC_8373471.jpg: 0.6965
# ISIC_1510820.jpg vs ISIC_8373691.jpg: 0.8774
# ISIC_1510820.jpg vs ISIC_8374182.jpg: 0.7576
# ISIC_1510820.jpg vs ISIC_8375281.jpg: 0.4672
# ISIC_1510820.jpg vs ISIC_8375292.jpg: 0.8489
# ISIC_1510820.jpg vs ISIC_8379889.jpg: 0.8214
# ISIC_1510820.jpg vs ISIC_8380448.jpg: 0.7058
# ISIC_1510820.jpg vs ISIC_8380991.jpg: 0.8306
# ISIC_1510820.jpg vs ISIC_8384497.jpg: 0.8321
# ISIC_1510820.jpg vs ISIC_8386329.jpg: 0.5571
# ISIC_1510820.jpg vs ISIC_8386691.jpg: 0.8126
# ISIC_1510820.jpg vs ISIC_8389896.jpg: 0.8010
# ISIC_1510820.jpg vs ISIC_8390230.jpg: 0.8729
# ISIC_1510820.jpg vs ISIC_8392134.jpg: 0.7584
# ISIC_1510820.jpg vs ISIC_8393618.jpg: 0.4623
# ISIC_1510820.jpg vs ISIC_8397422.jpg: 0.3085
# ISIC_1510820.jpg vs ISIC_8397575.jpg: 0.7724
# ISIC_1510820.jpg vs ISIC_8402252.jpg: 0.7168
# ISIC_1510820.jpg vs ISIC_8402882.jpg: 0.6903
# ISIC_1510820.jpg vs ISIC_8402960.jpg: 0.7711
# ISIC_1510820.jpg vs ISIC_8404725.jpg: 0.8115
# ISIC_1510820.jpg vs ISIC_8405160.jpg: 0.2877
# ISIC_1510820.jpg vs ISIC_8405212.jpg: 0.4893
# ISIC_1510820.jpg vs ISIC_8407148.jpg: 0.2174
# ISIC_1510820.jpg vs ISIC_8408444.jpg: 0.6327
# ISIC_1510820.jpg vs ISIC_8409612.jpg: 0.7610
# ISIC_1510820.jpg vs ISIC_8409838.jpg: 0.8796
# ISIC_1510820.jpg vs ISIC_8409913.jpg: 0.4635
# ISIC_1510820.jpg vs ISIC_8412460.jpg: 0.8036
# ISIC_1510820.jpg vs ISIC_8414455.jpg: 0.5361
# ISIC_1510820.jpg vs ISIC_8415134.jpg: 0.8834
# ISIC_1510820.jpg vs ISIC_8415934.jpg: 0.8780
# ISIC_1510820.jpg vs ISIC_8416956.jpg: 0.7118
# ISIC_1510820.jpg vs ISIC_8417195.jpg: 0.2233
# ISIC_1510820.jpg vs ISIC_8419052.jpg: 0.8976
# ISIC_1510820.jpg vs ISIC_8419210.jpg: 0.6226
# ISIC_1510820.jpg vs ISIC_8423177.jpg: 0.7344
# ISIC_1510820.jpg vs ISIC_8424579.jpg: 0.3755
# ISIC_1510820.jpg vs ISIC_8424990.jpg: 0.3160
# ISIC_1510820.jpg vs ISIC_8425779.jpg: 0.2022
# ISIC_1510820.jpg vs ISIC_8429000.jpg: 0.6940
# ISIC_1510820.jpg vs ISIC_8430333.jpg: 0.6608
# ISIC_1510820.jpg vs ISIC_8430358.jpg: 0.7619
# ISIC_1510820.jpg vs ISIC_8430720.jpg: 0.3008
# ISIC_1510820.jpg vs ISIC_8435488.jpg: 0.7384
# ISIC_1510820.jpg vs ISIC_8437178.jpg: 0.3011
# ISIC_1510820.jpg vs ISIC_8439105.jpg: 0.6514
# ISIC_1510820.jpg vs ISIC_8439124.jpg: 0.8316
# ISIC_1510820.jpg vs ISIC_8439859.jpg: 0.8543
# ISIC_1510820.jpg vs ISIC_8440465.jpg: 0.6192
# ISIC_1510820.jpg vs ISIC_8441463.jpg: 0.9341
# ISIC_1510820.jpg vs ISIC_8441864.jpg: 0.4077
# ISIC_1510820.jpg vs ISIC_8443566.jpg: 0.5236
# ISIC_1510820.jpg vs ISIC_8443707.jpg: 0.7120
# ISIC_1510820.jpg vs ISIC_8443990.jpg: 0.4557
# ISIC_1510820.jpg vs ISIC_8444333.jpg: 0.6439
# ISIC_1510820.jpg vs ISIC_8444587.jpg: 0.3506
# ISIC_1510820.jpg vs ISIC_8445938.jpg: 0.5603
# ISIC_1510820.jpg vs ISIC_8446191.jpg: 0.6928
# ISIC_1510820.jpg vs ISIC_8450504.jpg: 0.4479
# ISIC_1510820.jpg vs ISIC_8451459.jpg: 0.8987
# ISIC_1510820.jpg vs ISIC_8451738.jpg: 0.7825
# ISIC_1510820.jpg vs ISIC_8452182.jpg: 0.7306
# ISIC_1510820.jpg vs ISIC_8453693.jpg: 0.7274
# ISIC_1510820.jpg vs ISIC_8455665.jpg: 0.3248
# ISIC_1510820.jpg vs ISIC_8456118.jpg: 0.4741
# ISIC_1510820.jpg vs ISIC_8457788.jpg: 0.8847
# ISIC_1510820.jpg vs ISIC_8459597.jpg: 0.7468
# ISIC_1510820.jpg vs ISIC_8460973.jpg: 0.5688
# ISIC_1510820.jpg vs ISIC_8461939.jpg: 0.8589
# ISIC_1510820.jpg vs ISIC_8466497.jpg: 0.4837
# ISIC_1510820.jpg vs ISIC_8468029.jpg: 0.8322
# ISIC_1510820.jpg vs ISIC_8469607.jpg: 0.5515
# ISIC_1510820.jpg vs ISIC_8470330.jpg: 0.6495
# ISIC_1510820.jpg vs ISIC_8471690.jpg: 0.8270
# ISIC_1510820.jpg vs ISIC_8472353.jpg: 0.5881
# ISIC_1510820.jpg vs ISIC_8472528.jpg: 0.8479
# ISIC_1510820.jpg vs ISIC_8473523.jpg: 0.8632
# ISIC_1510820.jpg vs ISIC_8474368.jpg: 0.5867
# ISIC_1510820.jpg vs ISIC_8474376.jpg: 0.4747
# ISIC_1510820.jpg vs ISIC_8474980.jpg: 0.8833
# ISIC_1510820.jpg vs ISIC_8476628.jpg: 0.8075
# ISIC_1510820.jpg vs ISIC_8477203.jpg: 0.2789
# ISIC_1510820.jpg vs ISIC_8477641.jpg: 0.3326
# ISIC_1510820.jpg vs ISIC_8477949.jpg: 0.8333
# ISIC_1510820.jpg vs ISIC_8478992.jpg: 0.7081
# ISIC_1510820.jpg vs ISIC_8481618.jpg: 0.6348
# ISIC_1510820.jpg vs ISIC_8482652.jpg: 0.8304
# ISIC_1510820.jpg vs ISIC_8482659.jpg: 0.8378
# ISIC_1510820.jpg vs ISIC_8483677.jpg: 0.7625
# ISIC_1510820.jpg vs ISIC_8488103.jpg: 0.6475
# ISIC_1510820.jpg vs ISIC_8488212.jpg: 0.5058
# ISIC_1510820.jpg vs ISIC_8488326.jpg: 0.6796
# ISIC_1510820.jpg vs ISIC_8493711.jpg: 0.3809
# ISIC_1510820.jpg vs ISIC_8497203.jpg: 0.5362
# ISIC_1510820.jpg vs ISIC_8497431.jpg: 0.9171
# ISIC_1510820.jpg vs ISIC_8499885.jpg: 0.5287
# ISIC_1510820.jpg vs ISIC_8501183.jpg: 0.5824
# ISIC_1510820.jpg vs ISIC_8502049.jpg: 0.8692
# ISIC_1510820.jpg vs ISIC_8503349.jpg: 0.8178
# ISIC_1510820.jpg vs ISIC_8503681.jpg: 0.1808
# ISIC_1510820.jpg vs ISIC_8503714.jpg: 0.6929
# ISIC_1510820.jpg vs ISIC_8506354.jpg: 0.8577
# ISIC_1510820.jpg vs ISIC_8509539.jpg: 0.6954
# ISIC_1510820.jpg vs ISIC_8509703.jpg: 0.8820
# ISIC_1510820.jpg vs ISIC_8510555.jpg: 0.8133
# ISIC_1510820.jpg vs ISIC_8511501.jpg: 0.6691
# ISIC_1510820.jpg vs ISIC_8511669.jpg: 0.8034
# ISIC_1510820.jpg vs ISIC_8515579.jpg: 0.8253
# ISIC_1510820.jpg vs ISIC_8515796.jpg: 0.4459
# ISIC_1510820.jpg vs ISIC_8517129.jpg: 0.7891
# ISIC_1510820.jpg vs ISIC_8517928.jpg: 0.8240
# ISIC_1510820.jpg vs ISIC_8518143.jpg: 0.8669
# ISIC_1510820.jpg vs ISIC_8520181.jpg: 0.1422
# ISIC_1510820.jpg vs ISIC_8520278.jpg: 0.8110
# ISIC_1510820.jpg vs ISIC_8521108.jpg: 0.6670
# ISIC_1510820.jpg vs ISIC_8525942.jpg: 0.8766
# ISIC_1510820.jpg vs ISIC_8526421.jpg: 0.1002
# ISIC_1510820.jpg vs ISIC_8527595.jpg: 0.6848
# ISIC_1510820.jpg vs ISIC_8529178.jpg: 0.7881
# ISIC_1510820.jpg vs ISIC_8531444.jpg: 0.9200
# ISIC_1510820.jpg vs ISIC_8532318.jpg: 0.8371
# ISIC_1510820.jpg vs ISIC_8532460.jpg: 0.5667
# ISIC_1510820.jpg vs ISIC_8532615.jpg: 0.6444
# ISIC_1510820.jpg vs ISIC_8533492.jpg: 0.6617
# ISIC_1510820.jpg vs ISIC_8535552.jpg: 0.7974
# ISIC_1510820.jpg vs ISIC_8537801.jpg: 0.5480
# ISIC_1510820.jpg vs ISIC_8538346.jpg: 0.6623
# ISIC_1510820.jpg vs ISIC_8538657.jpg: 0.7555
# ISIC_1510820.jpg vs ISIC_8538920.jpg: 0.8788
# ISIC_1510820.jpg vs ISIC_8539371.jpg: 0.7087
# ISIC_1510820.jpg vs ISIC_8541427.jpg: 0.7918
# ISIC_1510820.jpg vs ISIC_8542563.jpg: 0.6675
# ISIC_1510820.jpg vs ISIC_8544906.jpg: 0.5402
# ISIC_1510820.jpg vs ISIC_8546598.jpg: 0.4114
# ISIC_1510820.jpg vs ISIC_8546808.jpg: 0.3552
# ISIC_1510820.jpg vs ISIC_8553381.jpg: 0.4240
# ISIC_1510820.jpg vs ISIC_8553859.jpg: 0.8287
# ISIC_1510820.jpg vs ISIC_8554769.jpg: 0.1410
# ISIC_1510820.jpg vs ISIC_8556045.jpg: 0.7335
# ISIC_1510820.jpg vs ISIC_8556548.jpg: 0.4986
# ISIC_1510820.jpg vs ISIC_8557216.jpg: 0.8177
# ISIC_1510820.jpg vs ISIC_8559362.jpg: 0.4882
# ISIC_1510820.jpg vs ISIC_8562955.jpg: 0.7011
# ISIC_1510820.jpg vs ISIC_8569603.jpg: 0.1556
# ISIC_1510820.jpg vs ISIC_8570212.jpg: 0.2717
# ISIC_1510820.jpg vs ISIC_8570369.jpg: 0.7771
# ISIC_1510820.jpg vs ISIC_8570643.jpg: 0.3855
# ISIC_1510820.jpg vs ISIC_8570719.jpg: 0.8449
# ISIC_1510820.jpg vs ISIC_8570923.jpg: 0.8644
# ISIC_1510820.jpg vs ISIC_8572342.jpg: 0.8468
# ISIC_1510820.jpg vs ISIC_8572576.jpg: 0.7947
# ISIC_1510820.jpg vs ISIC_8573752.jpg: 0.3279
# ISIC_1510820.jpg vs ISIC_8574381.jpg: 0.6438
# ISIC_1510820.jpg vs ISIC_8574400.jpg: 0.8559
# ISIC_1510820.jpg vs ISIC_8576019.jpg: 0.3966
# ISIC_1510820.jpg vs ISIC_8576575.jpg: 0.7584
# ISIC_1510820.jpg vs ISIC_8578130.jpg: 0.8536
# ISIC_1510820.jpg vs ISIC_8581330.jpg: 0.8171
# ISIC_1510820.jpg vs ISIC_8581411.jpg: 0.1793
# ISIC_1510820.jpg vs ISIC_8583322.jpg: 0.1449
# ISIC_1510820.jpg vs ISIC_8588789.jpg: 0.4578
# ISIC_1510820.jpg vs ISIC_8590131.jpg: 0.2760
# ISIC_1510820.jpg vs ISIC_8591728.jpg: 0.7279
# ISIC_1510820.jpg vs ISIC_8592885.jpg: 0.7874
# ISIC_1510820.jpg vs ISIC_8596085.jpg: 0.4936
# ISIC_1510820.jpg vs ISIC_8600691.jpg: 0.7413
# ISIC_1510820.jpg vs ISIC_8600924.jpg: 0.7835
# ISIC_1510820.jpg vs ISIC_8601288.jpg: 0.5271
# ISIC_1510820.jpg vs ISIC_8604723.jpg: 0.3602
# ISIC_1510820.jpg vs ISIC_8605641.jpg: 0.5160
# ISIC_1510820.jpg vs ISIC_8605723.jpg: 0.7573
# ISIC_1510820.jpg vs ISIC_8607491.jpg: 0.5124
# ISIC_1510820.jpg vs ISIC_8609145.jpg: 0.1829
# ISIC_1510820.jpg vs ISIC_8611354.jpg: 0.1067
# ISIC_1510820.jpg vs ISIC_8612112.jpg: 0.2299
# ISIC_1510820.jpg vs ISIC_8612981.jpg: 0.2859
# ISIC_1510820.jpg vs ISIC_8613602.jpg: 0.7531
# ISIC_1510820.jpg vs ISIC_8613848.jpg: 0.2284
# ISIC_1510820.jpg vs ISIC_8615435.jpg: 0.6985
# ISIC_1510820.jpg vs ISIC_8616497.jpg: 0.6674
# ISIC_1510820.jpg vs ISIC_8616859.jpg: 0.8259
# ISIC_1510820.jpg vs ISIC_8618676.jpg: 0.6087
# ISIC_1510820.jpg vs ISIC_8620474.jpg: 0.5634
# ISIC_1510820.jpg vs ISIC_8620516.jpg: 0.1574
# ISIC_1510820.jpg vs ISIC_8622019.jpg: 0.7487
# ISIC_1510820.jpg vs ISIC_8623375.jpg: 0.7402
# ISIC_1510820.jpg vs ISIC_8623387.jpg: 0.7327
# ISIC_1510820.jpg vs ISIC_8626011.jpg: 0.8685
# ISIC_1510820.jpg vs ISIC_8626442.jpg: 0.8818
# ISIC_1510820.jpg vs ISIC_8627702.jpg: 0.8044
# ISIC_1510820.jpg vs ISIC_8627791.jpg: 0.4956
# ISIC_1510820.jpg vs ISIC_8630965.jpg: 0.4084
# ISIC_1510820.jpg vs ISIC_8630968.jpg: 0.7969
# ISIC_1510820.jpg vs ISIC_8631345.jpg: 0.6417
# ISIC_1510820.jpg vs ISIC_8632205.jpg: 0.6366
# ISIC_1510820.jpg vs ISIC_8632904.jpg: 0.1784
# ISIC_1510820.jpg vs ISIC_8634773.jpg: 0.7713
# ISIC_1510820.jpg vs ISIC_8636180.jpg: 0.8532
# ISIC_1510820.jpg vs ISIC_8636351.jpg: 0.7833
# ISIC_1510820.jpg vs ISIC_8637657.jpg: 0.7309
# ISIC_1510820.jpg vs ISIC_8638834.jpg: 0.4798
# ISIC_1510820.jpg vs ISIC_8640384.jpg: 0.8179
# ISIC_1510820.jpg vs ISIC_8641097.jpg: 0.5941
# ISIC_1510820.jpg vs ISIC_8641927.jpg: 0.6751
# ISIC_1510820.jpg vs ISIC_8642588.jpg: 0.3664
# ISIC_1510820.jpg vs ISIC_8643238.jpg: 0.6768
# ISIC_1510820.jpg vs ISIC_8645931.jpg: 0.8018
# ISIC_1510820.jpg vs ISIC_8648668.jpg: 0.6333
# ISIC_1510820.jpg vs ISIC_8649904.jpg: 0.5901
# ISIC_1510820.jpg vs ISIC_8650022.jpg: 0.8173
# ISIC_1510820.jpg vs ISIC_8651294.jpg: 0.4411
# ISIC_1510820.jpg vs ISIC_8651681.jpg: 0.8050
# ISIC_1510820.jpg vs ISIC_8652577.jpg: 0.8834
# ISIC_1510820.jpg vs ISIC_8652658.jpg: 0.1650
# ISIC_1510820.jpg vs ISIC_8652723.jpg: 0.7374
# ISIC_1510820.jpg vs ISIC_8654232.jpg: 0.6201
# ISIC_1510820.jpg vs ISIC_8655348.jpg: 0.1438
# ISIC_1510820.jpg vs ISIC_8655397.jpg: 0.3401
# ISIC_1510820.jpg vs ISIC_8655515.jpg: 0.7904
# ISIC_1510820.jpg vs ISIC_8655808.jpg: 0.5535
# ISIC_1510820.jpg vs ISIC_8656544.jpg: 0.7976
# ISIC_1510820.jpg vs ISIC_8657008.jpg: 0.6902
# ISIC_1510820.jpg vs ISIC_8658727.jpg: 0.3372
# ISIC_1510820.jpg vs ISIC_8660499.jpg: 0.8747
# ISIC_1510820.jpg vs ISIC_8662209.jpg: 0.7515
# ISIC_1510820.jpg vs ISIC_8663772.jpg: 0.8597
# ISIC_1510820.jpg vs ISIC_8664024.jpg: 0.7698
# ISIC_1510820.jpg vs ISIC_8664078.jpg: 0.6455
# ISIC_1510820.jpg vs ISIC_8664210.jpg: 0.8284
# ISIC_1510820.jpg vs ISIC_8664687.jpg: 0.8592
# ISIC_1510820.jpg vs ISIC_8664911.jpg: 0.4438
# ISIC_1510820.jpg vs ISIC_8668157.jpg: 0.2019
# ISIC_1510820.jpg vs ISIC_8670486.jpg: 0.7745
# ISIC_1510820.jpg vs ISIC_8671883.jpg: 0.7934
# ISIC_1510820.jpg vs ISIC_8675845.jpg: 0.4224
# ISIC_1510820.jpg vs ISIC_8676919.jpg: 0.8459
# ISIC_1510820.jpg vs ISIC_8677397.jpg: 0.3666
# ISIC_1510820.jpg vs ISIC_8679128.jpg: 0.8094
# ISIC_1510820.jpg vs ISIC_8679619.jpg: 0.6045
# ISIC_1510820.jpg vs ISIC_8682592.jpg: 0.2475
# ISIC_1510820.jpg vs ISIC_8683202.jpg: 0.7446
# ISIC_1510820.jpg vs ISIC_8683473.jpg: 0.6435
# ISIC_1510820.jpg vs ISIC_8684658.jpg: 0.7636
# ISIC_1510820.jpg vs ISIC_8685059.jpg: 0.6162
# ISIC_1510820.jpg vs ISIC_8688128.jpg: 0.7270
# ISIC_1510820.jpg vs ISIC_8688992.jpg: 0.7099
# ISIC_1510820.jpg vs ISIC_8690265.jpg: 0.7942
# ISIC_1510820.jpg vs ISIC_8691320.jpg: 0.7658
# ISIC_1510820.jpg vs ISIC_8693147.jpg: 0.8138
# ISIC_1510820.jpg vs ISIC_8696087.jpg: 0.6100
# ISIC_1510820.jpg vs ISIC_8699871.jpg: 0.5040
# ISIC_1510820.jpg vs ISIC_8703136.jpg: 0.8020
# ISIC_1510820.jpg vs ISIC_8705042.jpg: 0.2954
# ISIC_1510820.jpg vs ISIC_8705723.jpg: 0.5914
# ISIC_1510820.jpg vs ISIC_8707911.jpg: 0.7787
# ISIC_1510820.jpg vs ISIC_8709476.jpg: 0.6959
# ISIC_1510820.jpg vs ISIC_8711172.jpg: 0.8274
# ISIC_1510820.jpg vs ISIC_8712678.jpg: 0.7515
# ISIC_1510820.jpg vs ISIC_8713613.jpg: 0.8490
# ISIC_1510820.jpg vs ISIC_8714050.jpg: 0.6695
# ISIC_1510820.jpg vs ISIC_8715037.jpg: 0.6319
# ISIC_1510820.jpg vs ISIC_8715813.jpg: 0.2114
# ISIC_1510820.jpg vs ISIC_8718545.jpg: 0.6076
# ISIC_1510820.jpg vs ISIC_8723928.jpg: 0.8571
# ISIC_1510820.jpg vs ISIC_8724607.jpg: 0.8687
# ISIC_1510820.jpg vs ISIC_8724813.jpg: 0.6774
# ISIC_1510820.jpg vs ISIC_8725020.jpg: 0.4899
# ISIC_1510820.jpg vs ISIC_8728322.jpg: 0.5311
# ISIC_1510820.jpg vs ISIC_8729639.jpg: 0.3585
# ISIC_1510820.jpg vs ISIC_8731126.jpg: 0.5766
# ISIC_1510820.jpg vs ISIC_8732496.jpg: 0.7717
# ISIC_1510820.jpg vs ISIC_8732904.jpg: 0.7594
# ISIC_1510820.jpg vs ISIC_8733176.jpg: 0.6087
# ISIC_1510820.jpg vs ISIC_8733507.jpg: 0.8775
# ISIC_1510820.jpg vs ISIC_8733834.jpg: 0.8076
# ISIC_1510820.jpg vs ISIC_8737280.jpg: 0.6356
# ISIC_1510820.jpg vs ISIC_8739480.jpg: 0.7443
# ISIC_1510820.jpg vs ISIC_8741719.jpg: 0.6693
# ISIC_1510820.jpg vs ISIC_8741900.jpg: 0.8699
# ISIC_1510820.jpg vs ISIC_8744510.jpg: 0.7858
# ISIC_1510820.jpg vs ISIC_8747198.jpg: 0.9082
# ISIC_1510820.jpg vs ISIC_8748275.jpg: 0.4488
# ISIC_1510820.jpg vs ISIC_8749499.jpg: 0.2788
# ISIC_1510820.jpg vs ISIC_8749647.jpg: 0.6080
# ISIC_1510820.jpg vs ISIC_8754299.jpg: 0.3757
# ISIC_1510820.jpg vs ISIC_8754331.jpg: 0.7867
# ISIC_1510820.jpg vs ISIC_8754753.jpg: 0.1763
# ISIC_1510820.jpg vs ISIC_8755413.jpg: 0.2110
# ISIC_1510820.jpg vs ISIC_8755702.jpg: 0.5425
# ISIC_1510820.jpg vs ISIC_8755717.jpg: 0.3808
# ISIC_1510820.jpg vs ISIC_8755718.jpg: 0.8165
# ISIC_1510820.jpg vs ISIC_8756106.jpg: 0.8833
# ISIC_1510820.jpg vs ISIC_8757232.jpg: 0.6567
# ISIC_1510820.jpg vs ISIC_8758219.jpg: 0.6004
# ISIC_1510820.jpg vs ISIC_8758404.jpg: 0.6604
# ISIC_1510820.jpg vs ISIC_8758498.jpg: 0.8282
# ISIC_1510820.jpg vs ISIC_8758652.jpg: 0.5188
# ISIC_1510820.jpg vs ISIC_8759013.jpg: 0.6954
# ISIC_1510820.jpg vs ISIC_8759397.jpg: 0.2304
# ISIC_1510820.jpg vs ISIC_8759537.jpg: 0.3597
# ISIC_1510820.jpg vs ISIC_8760258.jpg: 0.8018
# ISIC_1510820.jpg vs ISIC_8760915.jpg: 0.5576
# ISIC_1510820.jpg vs ISIC_8762162.jpg: 0.7941
# ISIC_1510820.jpg vs ISIC_8763593.jpg: 0.3434
# ISIC_1510820.jpg vs ISIC_8764178.jpg: 0.8476
# ISIC_1510820.jpg vs ISIC_8765861.jpg: 0.6066
# ISIC_1510820.jpg vs ISIC_8767904.jpg: 0.2292
# ISIC_1510820.jpg vs ISIC_8768167.jpg: 0.5712
# ISIC_1510820.jpg vs ISIC_8768857.jpg: 0.1931
# ISIC_1510820.jpg vs ISIC_8769458.jpg: 0.7692
# ISIC_1510820.jpg vs ISIC_8773197.jpg: 0.8142
# ISIC_1510820.jpg vs ISIC_8774910.jpg: 0.5655
# ISIC_1510820.jpg vs ISIC_8780817.jpg: 0.6172
# ISIC_1510820.jpg vs ISIC_8783748.jpg: 0.7090
# ISIC_1510820.jpg vs ISIC_8784137.jpg: 0.3715
# ISIC_1510820.jpg vs ISIC_8785587.jpg: 0.6160
# ISIC_1510820.jpg vs ISIC_8785816.jpg: 0.3358
# ISIC_1510820.jpg vs ISIC_8787227.jpg: 0.8007
# ISIC_1510820.jpg vs ISIC_8787842.jpg: 0.4341
# ISIC_1510820.jpg vs ISIC_8789502.jpg: 0.4581
# ISIC_1510820.jpg vs ISIC_8790782.jpg: 0.0817
# ISIC_1510820.jpg vs ISIC_8794737.jpg: 0.2777
# ISIC_1510820.jpg vs ISIC_8795995.jpg: 0.8391
# ISIC_1510820.jpg vs ISIC_8798037.jpg: 0.8456
# ISIC_1510820.jpg vs ISIC_8799403.jpg: 0.3150
# ISIC_1510820.jpg vs ISIC_8800182.jpg: 0.6434
# ISIC_1510820.jpg vs ISIC_8802849.jpg: 0.7137
# ISIC_1510820.jpg vs ISIC_8806466.jpg: 0.7627
# ISIC_1510820.jpg vs ISIC_8807139.jpg: 0.5426
# ISIC_1510820.jpg vs ISIC_8808618.jpg: 0.5039
# ISIC_1510820.jpg vs ISIC_8810056.jpg: 0.7778
# ISIC_1510820.jpg vs ISIC_8812635.jpg: 0.6068
# ISIC_1510820.jpg vs ISIC_8812840.jpg: 0.8256
# ISIC_1510820.jpg vs ISIC_8814160.jpg: 0.1187
# ISIC_1510820.jpg vs ISIC_8815184.jpg: 0.2322
# ISIC_1510820.jpg vs ISIC_8817234.jpg: 0.7532
# ISIC_1510820.jpg vs ISIC_8817936.jpg: 0.7611
# ISIC_1510820.jpg vs ISIC_8818310.jpg: 0.4583
# ISIC_1510820.jpg vs ISIC_8818534.jpg: 0.8274
# ISIC_1510820.jpg vs ISIC_8821039.jpg: 0.5003
# ISIC_1510820.jpg vs ISIC_8823822.jpg: 0.7046
# ISIC_1510820.jpg vs ISIC_8824409.jpg: 0.7711
# ISIC_1510820.jpg vs ISIC_8826064.jpg: 0.7363
# ISIC_1510820.jpg vs ISIC_8829327.jpg: 0.5048
# ISIC_1510820.jpg vs ISIC_8829628.jpg: 0.2248
# ISIC_1510820.jpg vs ISIC_8830409.jpg: 0.6476
# ISIC_1510820.jpg vs ISIC_8832128.jpg: 0.4054
# ISIC_1510820.jpg vs ISIC_8832183.jpg: 0.6633
# ISIC_1510820.jpg vs ISIC_8833575.jpg: 0.1272
# ISIC_1510820.jpg vs ISIC_8834954.jpg: 0.8482
# ISIC_1510820.jpg vs ISIC_8838823.jpg: 0.7182
# ISIC_1510820.jpg vs ISIC_8839128.jpg: 0.7331
# ISIC_1510820.jpg vs ISIC_8841828.jpg: 0.1523
# ISIC_1510820.jpg vs ISIC_8842675.jpg: 0.7777
# ISIC_1510820.jpg vs ISIC_8845382.jpg: 0.6833
# ISIC_1510820.jpg vs ISIC_8847751.jpg: 0.4201
# ISIC_1510820.jpg vs ISIC_8848544.jpg: 0.3295
# ISIC_1510820.jpg vs ISIC_8848924.jpg: 0.8094
# ISIC_1510820.jpg vs ISIC_8849561.jpg: 0.8413
# ISIC_1510820.jpg vs ISIC_8850011.jpg: 0.2949
# ISIC_1510820.jpg vs ISIC_8854304.jpg: 0.6166
# ISIC_1510820.jpg vs ISIC_8854658.jpg: 0.8888
# ISIC_1510820.jpg vs ISIC_8855370.jpg: 0.7626
# ISIC_1510820.jpg vs ISIC_8856035.jpg: 0.8111
# ISIC_1510820.jpg vs ISIC_8856127.jpg: 0.6561
# ISIC_1510820.jpg vs ISIC_8857430.jpg: 0.4232
# ISIC_1510820.jpg vs ISIC_8858828.jpg: 0.5461
# ISIC_1510820.jpg vs ISIC_8860494.jpg: 0.7297
# ISIC_1510820.jpg vs ISIC_8862994.jpg: 0.5004
# ISIC_1510820.jpg vs ISIC_8863347.jpg: 0.7784
# ISIC_1510820.jpg vs ISIC_8866881.jpg: 0.7566
# ISIC_1510820.jpg vs ISIC_8866925.jpg: 0.8143
# ISIC_1510820.jpg vs ISIC_8872191.jpg: 0.7531
# ISIC_1510820.jpg vs ISIC_8872520.jpg: 0.7078
# ISIC_1510820.jpg vs ISIC_8872567.jpg: 0.6910
# ISIC_1510820.jpg vs ISIC_8875139.jpg: 0.4161
# ISIC_1510820.jpg vs ISIC_8876322.jpg: 0.3092
# ISIC_1510820.jpg vs ISIC_8876387.jpg: 0.7104
# ISIC_1510820.jpg vs ISIC_8877773.jpg: 0.8237
# ISIC_1510820.jpg vs ISIC_8878727.jpg: 0.8449
# ISIC_1510820.jpg vs ISIC_8882478.jpg: 0.4990
# ISIC_1510820.jpg vs ISIC_8882896.jpg: 0.3020
# ISIC_1510820.jpg vs ISIC_8883090.jpg: 0.7427
# ISIC_1510820.jpg vs ISIC_8883894.jpg: 0.7640
# ISIC_1510820.jpg vs ISIC_8885123.jpg: 0.3742
# ISIC_1510820.jpg vs ISIC_8886139.jpg: 0.7545
# ISIC_1510820.jpg vs ISIC_8890526.jpg: 0.3427
# ISIC_1510820.jpg vs ISIC_8890530.jpg: 0.4653
# ISIC_1510820.jpg vs ISIC_8891438.jpg: 0.2432
# ISIC_1510820.jpg vs ISIC_8891943.jpg: 0.6734
# ISIC_1510820.jpg vs ISIC_8891951.jpg: 0.7811
# ISIC_1510820.jpg vs ISIC_8894662.jpg: 0.6652
# ISIC_1510820.jpg vs ISIC_8895569.jpg: 0.2065
# ISIC_1510820.jpg vs ISIC_8897880.jpg: 0.3143
# ISIC_1510820.jpg vs ISIC_8899172.jpg: 0.7584
# ISIC_1510820.jpg vs ISIC_8900885.jpg: 0.7640
# ISIC_1510820.jpg vs ISIC_8901241.jpg: 0.8729
# ISIC_1510820.jpg vs ISIC_8902792.jpg: 0.8869
# ISIC_1510820.jpg vs ISIC_8903911.jpg: 0.4673
# ISIC_1510820.jpg vs ISIC_8904119.jpg: 0.7844
# ISIC_1510820.jpg vs ISIC_8909153.jpg: 0.6013
# ISIC_1510820.jpg vs ISIC_8910224.jpg: 0.5703
# ISIC_1510820.jpg vs ISIC_8910785.jpg: 0.8735
# ISIC_1510820.jpg vs ISIC_8912651.jpg: 0.5166
# ISIC_1510820.jpg vs ISIC_8913315.jpg: 0.9296
# ISIC_1510820.jpg vs ISIC_8913327.jpg: 0.2039
# ISIC_1510820.jpg vs ISIC_8913346.jpg: 0.8203
# ISIC_1510820.jpg vs ISIC_8914743.jpg: 0.6564
# ISIC_1510820.jpg vs ISIC_8914803.jpg: 0.8078
# ISIC_1510820.jpg vs ISIC_8915548.jpg: 0.6815
# ISIC_1510820.jpg vs ISIC_8916523.jpg: 0.8135
# ISIC_1510820.jpg vs ISIC_8916845.jpg: 0.5870
# ISIC_1510820.jpg vs ISIC_8917277.jpg: 0.8719
# ISIC_1510820.jpg vs ISIC_8917520.jpg: 0.6050
# ISIC_1510820.jpg vs ISIC_8919030.jpg: 0.6832
# ISIC_1510820.jpg vs ISIC_8921222.jpg: 0.6065
# ISIC_1510820.jpg vs ISIC_8921308.jpg: 0.4370
# ISIC_1510820.jpg vs ISIC_8921417.jpg: 0.6418
# ISIC_1510820.jpg vs ISIC_8923222.jpg: 0.1423
# ISIC_1510820.jpg vs ISIC_8924679.jpg: 0.8693
# ISIC_1510820.jpg vs ISIC_8926785.jpg: 0.9171
# ISIC_1510820.jpg vs ISIC_8928109.jpg: 0.7673
# ISIC_1510820.jpg vs ISIC_8928203.jpg: 0.7811
# ISIC_1510820.jpg vs ISIC_8928254.jpg: 0.7723
# ISIC_1510820.jpg vs ISIC_8929017.jpg: 0.2417
# ISIC_1510820.jpg vs ISIC_8929362.jpg: 0.6443
# ISIC_1510820.jpg vs ISIC_8929405.jpg: 0.7450
# ISIC_1510820.jpg vs ISIC_8931482.jpg: 0.3418
# ISIC_1510820.jpg vs ISIC_8932951.jpg: 0.2386
# ISIC_1510820.jpg vs ISIC_8935177.jpg: 0.7408
# ISIC_1510820.jpg vs ISIC_8936821.jpg: 0.7651
# ISIC_1510820.jpg vs ISIC_8938888.jpg: 0.3629
# ISIC_1510820.jpg vs ISIC_8941830.jpg: 0.6781
# ISIC_1510820.jpg vs ISIC_8942163.jpg: 0.7231
# ISIC_1510820.jpg vs ISIC_8942427.jpg: 0.5643
# ISIC_1510820.jpg vs ISIC_8942790.jpg: 0.5486
# ISIC_1510820.jpg vs ISIC_8943817.jpg: 0.7145
# ISIC_1510820.jpg vs ISIC_8945286.jpg: 0.8296
# ISIC_1510820.jpg vs ISIC_8946926.jpg: 0.6666
# ISIC_1510820.jpg vs ISIC_8947444.jpg: 0.7433
# ISIC_1510820.jpg vs ISIC_8948314.jpg: 0.8114
# ISIC_1510820.jpg vs ISIC_8949833.jpg: 0.4230
# ISIC_1510820.jpg vs ISIC_8950567.jpg: 0.3282
# ISIC_1510820.jpg vs ISIC_8951672.jpg: 0.6283
# ISIC_1510820.jpg vs ISIC_8951943.jpg: 0.7566
# ISIC_1510820.jpg vs ISIC_8952876.jpg: 0.3056
# ISIC_1510820.jpg vs ISIC_8953808.jpg: 0.3908
# ISIC_1510820.jpg vs ISIC_8954491.jpg: 0.8052
# ISIC_1510820.jpg vs ISIC_8954775.jpg: 0.6177
# ISIC_1510820.jpg vs ISIC_8958259.jpg: 0.7536
# ISIC_1510820.jpg vs ISIC_8958344.jpg: 0.8551
# ISIC_1510820.jpg vs ISIC_8959686.jpg: 0.6658
# ISIC_1510820.jpg vs ISIC_8961647.jpg: 0.3164
# ISIC_1510820.jpg vs ISIC_8962905.jpg: 0.3550
# ISIC_1510820.jpg vs ISIC_8964189.jpg: 0.8200
# ISIC_1510820.jpg vs ISIC_8964270.jpg: 0.5745
# ISIC_1510820.jpg vs ISIC_8964604.jpg: 0.5943
# ISIC_1510820.jpg vs ISIC_8964676.jpg: 0.5194
# ISIC_1510820.jpg vs ISIC_8965024.jpg: 0.4331
# ISIC_1510820.jpg vs ISIC_8966656.jpg: 0.6263
# ISIC_1510820.jpg vs ISIC_8969962.jpg: 0.7543
# ISIC_1510820.jpg vs ISIC_8971006.jpg: 0.8310
# ISIC_1510820.jpg vs ISIC_8971245.jpg: 0.8590
# ISIC_1510820.jpg vs ISIC_8972367.jpg: 0.7501
# ISIC_1510820.jpg vs ISIC_8974538.jpg: 0.7794
# ISIC_1510820.jpg vs ISIC_8976229.jpg: 0.7407
# ISIC_1510820.jpg vs ISIC_8980146.jpg: 0.6123
# ISIC_1510820.jpg vs ISIC_8980733.jpg: 0.6799
# ISIC_1510820.jpg vs ISIC_8980995.jpg: 0.6465
# ISIC_1510820.jpg vs ISIC_8983743.jpg: 0.5865
# ISIC_1510820.jpg vs ISIC_8984115.jpg: 0.8583
# ISIC_1510820.jpg vs ISIC_8984864.jpg: 0.7998
# ISIC_1510820.jpg vs ISIC_8985582.jpg: 0.2566
# ISIC_1510820.jpg vs ISIC_8987790.jpg: 0.8027
# ISIC_1510820.jpg vs ISIC_8987816.jpg: 0.7395
# ISIC_1510820.jpg vs ISIC_8987944.jpg: 0.6336
# ISIC_1510820.jpg vs ISIC_8988074.jpg: 0.8204
# ISIC_1510820.jpg vs ISIC_8988125.jpg: 0.5887
# ISIC_1510820.jpg vs ISIC_8989246.jpg: 0.0985
# ISIC_1510820.jpg vs ISIC_8989674.jpg: 0.7484
# ISIC_1510820.jpg vs ISIC_8990534.jpg: 0.2006
# ISIC_1510820.jpg vs ISIC_8991162.jpg: 0.6171
# ISIC_1510820.jpg vs ISIC_8994705.jpg: 0.8457
# ISIC_1510820.jpg vs ISIC_8998656.jpg: 0.8280
# ISIC_1510820.jpg vs ISIC_8999365.jpg: 0.7584
# ISIC_1510820.jpg vs ISIC_9002130.jpg: 0.7641
# ISIC_1510820.jpg vs ISIC_9002755.jpg: 0.8074
# ISIC_1510820.jpg vs ISIC_9003624.jpg: 0.4655
# ISIC_1510820.jpg vs ISIC_9005237.jpg: 0.6394
# ISIC_1510820.jpg vs ISIC_9006649.jpg: 0.5646
# ISIC_1510820.jpg vs ISIC_9008398.jpg: 0.8558
# ISIC_1510820.jpg vs ISIC_9009245.jpg: 0.4077
# ISIC_1510820.jpg vs ISIC_9010905.jpg: 0.7901
# ISIC_1510820.jpg vs ISIC_9013687.jpg: 0.7649
# ISIC_1510820.jpg vs ISIC_9013692.jpg: 0.6247
# ISIC_1510820.jpg vs ISIC_9013834.jpg: 0.5535
# ISIC_1510820.jpg vs ISIC_9017571.jpg: 0.7116
# ISIC_1510820.jpg vs ISIC_9018217.jpg: 0.6572
# ISIC_1510820.jpg vs ISIC_9018830.jpg: 0.6093
# ISIC_1510820.jpg vs ISIC_9019607.jpg: 0.7712
# ISIC_1510820.jpg vs ISIC_9020555.jpg: 0.8132
# ISIC_1510820.jpg vs ISIC_9021401.jpg: 0.4722
# ISIC_1510820.jpg vs ISIC_9022771.jpg: 0.4638
# ISIC_1510820.jpg vs ISIC_9025314.jpg: 0.8433
# ISIC_1510820.jpg vs ISIC_9027197.jpg: 0.5491
# ISIC_1510820.jpg vs ISIC_9031778.jpg: 0.5174
# ISIC_1510820.jpg vs ISIC_9034421.jpg: 0.5025
# ISIC_1510820.jpg vs ISIC_9034553.jpg: 0.7719
# ISIC_1510820.jpg vs ISIC_9034965.jpg: 0.7022
# ISIC_1510820.jpg vs ISIC_9035628.jpg: 0.4526
# ISIC_1510820.jpg vs ISIC_9036933.jpg: 0.6428
# ISIC_1510820.jpg vs ISIC_9039512.jpg: 0.8617
# ISIC_1510820.jpg vs ISIC_9040585.jpg: 0.7305
# ISIC_1510820.jpg vs ISIC_9042020.jpg: 0.7545
# ISIC_1510820.jpg vs ISIC_9042055.jpg: 0.2325
# ISIC_1510820.jpg vs ISIC_9046624.jpg: 0.7235
# ISIC_1510820.jpg vs ISIC_9047445.jpg: 0.4022
# ISIC_1510820.jpg vs ISIC_9048634.jpg: 0.4613
# ISIC_1510820.jpg vs ISIC_9049820.jpg: 0.8249
# ISIC_1510820.jpg vs ISIC_9050193.jpg: 0.7348
# ISIC_1510820.jpg vs ISIC_9055320.jpg: 0.2197
# ISIC_1510820.jpg vs ISIC_9057036.jpg: 0.8054
# ISIC_1510820.jpg vs ISIC_9058971.jpg: 0.8294
# ISIC_1510820.jpg vs ISIC_9059207.jpg: 0.7574
# ISIC_1510820.jpg vs ISIC_9060254.jpg: 0.7734
# ISIC_1510820.jpg vs ISIC_9061798.jpg: 0.4769
# ISIC_1510820.jpg vs ISIC_9063375.jpg: 0.7222
# ISIC_1510820.jpg vs ISIC_9063657.jpg: 0.2729
# ISIC_1510820.jpg vs ISIC_9065092.jpg: 0.7396
# ISIC_1510820.jpg vs ISIC_9066369.jpg: 0.4779
# ISIC_1510820.jpg vs ISIC_9067229.jpg: 0.3700
# ISIC_1510820.jpg vs ISIC_9068075.jpg: 0.7647
# ISIC_1510820.jpg vs ISIC_9073521.jpg: 0.7709
# ISIC_1510820.jpg vs ISIC_9073812.jpg: 0.3518
# ISIC_1510820.jpg vs ISIC_9074443.jpg: 0.7250
# ISIC_1510820.jpg vs ISIC_9075192.jpg: 0.6853
# ISIC_1510820.jpg vs ISIC_9076973.jpg: 0.2041
# ISIC_1510820.jpg vs ISIC_9078802.jpg: 0.3721
# ISIC_1510820.jpg vs ISIC_9079297.jpg: 0.4359
# ISIC_1510820.jpg vs ISIC_9080885.jpg: 0.6240
# ISIC_1510820.jpg vs ISIC_9083661.jpg: 0.6869
# ISIC_1510820.jpg vs ISIC_9084283.jpg: 0.6435
# ISIC_1510820.jpg vs ISIC_9086006.jpg: 0.5849
# ISIC_1510820.jpg vs ISIC_9087762.jpg: 0.7204
# ISIC_1510820.jpg vs ISIC_9091107.jpg: 0.7630
# ISIC_1510820.jpg vs ISIC_9091217.jpg: 0.6816
# ISIC_1510820.jpg vs ISIC_9092837.jpg: 0.6241
# ISIC_1510820.jpg vs ISIC_9093052.jpg: 0.6005
# ISIC_1510820.jpg vs ISIC_9093252.jpg: 0.2468
# ISIC_1510820.jpg vs ISIC_9093510.jpg: 0.2745
# ISIC_1510820.jpg vs ISIC_9097422.jpg: 0.7969
# ISIC_1510820.jpg vs ISIC_9097649.jpg: 0.6633
# ISIC_1510820.jpg vs ISIC_9100035.jpg: 0.6908
# ISIC_1510820.jpg vs ISIC_9100455.jpg: 0.1427
# ISIC_1510820.jpg vs ISIC_9101004.jpg: 0.4336
# ISIC_1510820.jpg vs ISIC_9101048.jpg: 0.7517
# ISIC_1510820.jpg vs ISIC_9101762.jpg: 0.4657
# ISIC_1510820.jpg vs ISIC_9101868.jpg: 0.3339
# ISIC_1510820.jpg vs ISIC_9102112.jpg: 0.7775
# ISIC_1510820.jpg vs ISIC_9105385.jpg: 0.1695
# ISIC_1510820.jpg vs ISIC_9105386.jpg: 0.3007
# ISIC_1510820.jpg vs ISIC_9105772.jpg: 0.2280
# ISIC_1510820.jpg vs ISIC_9107177.jpg: 0.7996
# ISIC_1510820.jpg vs ISIC_9107651.jpg: 0.7472
# ISIC_1510820.jpg vs ISIC_9107678.jpg: 0.3439
# ISIC_1510820.jpg vs ISIC_9108482.jpg: 0.8871
# ISIC_1510820.jpg vs ISIC_9108838.jpg: 0.5473
# ISIC_1510820.jpg vs ISIC_9109293.jpg: 0.7884
# ISIC_1510820.jpg vs ISIC_9112240.jpg: 0.7136
# ISIC_1510820.jpg vs ISIC_9114248.jpg: 0.1828
# ISIC_1510820.jpg vs ISIC_9122069.jpg: 0.6914
# ISIC_1510820.jpg vs ISIC_9124451.jpg: 0.7325
# ISIC_1510820.jpg vs ISIC_9124728.jpg: 0.7275
# ISIC_1510820.jpg vs ISIC_9125989.jpg: 0.7625
# ISIC_1510820.jpg vs ISIC_9126660.jpg: 0.3387
# ISIC_1510820.jpg vs ISIC_9131529.jpg: 0.7489
# ISIC_1510820.jpg vs ISIC_9131769.jpg: 0.7985
# ISIC_1510820.jpg vs ISIC_9132494.jpg: 0.7850
# ISIC_1510820.jpg vs ISIC_9135550.jpg: 0.3246
# ISIC_1510820.jpg vs ISIC_9135940.jpg: 0.2386
# ISIC_1510820.jpg vs ISIC_9136878.jpg: 0.6219
# ISIC_1510820.jpg vs ISIC_9140624.jpg: 0.5715
# ISIC_1510820.jpg vs ISIC_9140839.jpg: 0.6565
# ISIC_1510820.jpg vs ISIC_9140847.jpg: 0.8188
# ISIC_1510820.jpg vs ISIC_9141993.jpg: 0.3695
# ISIC_1510820.jpg vs ISIC_9142163.jpg: 0.6820
# ISIC_1510820.jpg vs ISIC_9142737.jpg: 0.6752
# ISIC_1510820.jpg vs ISIC_9142988.jpg: 0.4570
# ISIC_1510820.jpg vs ISIC_9145307.jpg: 0.7547
# ISIC_1510820.jpg vs ISIC_9148590.jpg: 0.4553
# ISIC_1510820.jpg vs ISIC_9149390.jpg: 0.6369
# ISIC_1510820.jpg vs ISIC_9149497.jpg: 0.8552
# ISIC_1510820.jpg vs ISIC_9151043.jpg: 0.7680
# ISIC_1510820.jpg vs ISIC_9151798.jpg: 0.6970
# ISIC_1510820.jpg vs ISIC_9151820.jpg: 0.6409
# ISIC_1510820.jpg vs ISIC_9156895.jpg: 0.7869
# ISIC_1510820.jpg vs ISIC_9156944.jpg: 0.6433
# ISIC_1510820.jpg vs ISIC_9157640.jpg: 0.8185
# ISIC_1510820.jpg vs ISIC_9158342.jpg: 0.4330
# ISIC_1510820.jpg vs ISIC_9161496.jpg: 0.5483
# ISIC_1510820.jpg vs ISIC_9162488.jpg: 0.8359
# ISIC_1510820.jpg vs ISIC_9163528.jpg: 0.8432
# ISIC_1510820.jpg vs ISIC_9164605.jpg: 0.4082
# ISIC_1510820.jpg vs ISIC_9167543.jpg: 0.8572
# ISIC_1510820.jpg vs ISIC_9167607.jpg: 0.5487
# ISIC_1510820.jpg vs ISIC_9168537.jpg: 0.7676
# ISIC_1510820.jpg vs ISIC_9168740.jpg: 0.6271
# ISIC_1510820.jpg vs ISIC_9169910.jpg: 0.2412
# ISIC_1510820.jpg vs ISIC_9170035.jpg: 0.6694
# ISIC_1510820.jpg vs ISIC_9171223.jpg: 0.8172
# ISIC_1510820.jpg vs ISIC_9172945.jpg: 0.7949
# ISIC_1510820.jpg vs ISIC_9173786.jpg: 0.6170
# ISIC_1510820.jpg vs ISIC_9174152.jpg: 0.6804
# ISIC_1510820.jpg vs ISIC_9174552.jpg: 0.7469
# ISIC_1510820.jpg vs ISIC_9175873.jpg: 0.8606
# ISIC_1510820.jpg vs ISIC_9180287.jpg: 0.8991
# ISIC_1510820.jpg vs ISIC_9181340.jpg: 0.3743
# ISIC_1510820.jpg vs ISIC_9181743.jpg: 0.6618
# ISIC_1510820.jpg vs ISIC_9182942.jpg: 0.8729
# ISIC_1510820.jpg vs ISIC_9183084.jpg: 0.4331
# ISIC_1510820.jpg vs ISIC_9184615.jpg: 0.4983
# ISIC_1510820.jpg vs ISIC_9185938.jpg: 0.7378
# ISIC_1510820.jpg vs ISIC_9186721.jpg: 0.5183
# ISIC_1510820.jpg vs ISIC_9187566.jpg: 0.1855
# ISIC_1510820.jpg vs ISIC_9187921.jpg: 0.5083
# ISIC_1510820.jpg vs ISIC_9188862.jpg: 0.7796
# ISIC_1510820.jpg vs ISIC_9192257.jpg: 0.7344
# ISIC_1510820.jpg vs ISIC_9192659.jpg: 0.7644
# ISIC_1510820.jpg vs ISIC_9194284.jpg: 0.3497
# ISIC_1510820.jpg vs ISIC_9194658.jpg: 0.9099
# ISIC_1510820.jpg vs ISIC_9195184.jpg: 0.7083
# ISIC_1510820.jpg vs ISIC_9197861.jpg: 0.6864
# ISIC_1510820.jpg vs ISIC_9198651.jpg: 0.7025
# ISIC_1510820.jpg vs ISIC_9199792.jpg: 0.7193
# ISIC_1510820.jpg vs ISIC_9203162.jpg: 0.7253
# ISIC_1510820.jpg vs ISIC_9203929.jpg: 0.3481
# ISIC_1510820.jpg vs ISIC_9207339.jpg: 0.3683
# ISIC_1510820.jpg vs ISIC_9207601.jpg: 0.8472
# ISIC_1510820.jpg vs ISIC_9211310.jpg: 0.3043
# ISIC_1510820.jpg vs ISIC_9211681.jpg: 0.1469
# ISIC_1510820.jpg vs ISIC_9214715.jpg: 0.4022
# ISIC_1510820.jpg vs ISIC_9215680.jpg: 0.2915
# ISIC_1510820.jpg vs ISIC_9216693.jpg: 0.4417
# ISIC_1510820.jpg vs ISIC_9218660.jpg: 0.6964
# ISIC_1510820.jpg vs ISIC_9218956.jpg: 0.4884
# ISIC_1510820.jpg vs ISIC_9220014.jpg: 0.5064
# ISIC_1510820.jpg vs ISIC_9221947.jpg: 0.8723
# ISIC_1510820.jpg vs ISIC_9224692.jpg: 0.8955
# ISIC_1510820.jpg vs ISIC_9224823.jpg: 0.8704
# ISIC_1510820.jpg vs ISIC_9225102.jpg: 0.6944
# ISIC_1510820.jpg vs ISIC_9228022.jpg: 0.6087
# ISIC_1510820.jpg vs ISIC_9233285.jpg: 0.5474
# ISIC_1510820.jpg vs ISIC_9234032.jpg: 0.8504
# ISIC_1510820.jpg vs ISIC_9234969.jpg: 0.6101
# ISIC_1510820.jpg vs ISIC_9235283.jpg: 0.4867
# ISIC_1510820.jpg vs ISIC_9235659.jpg: 0.7403
# ISIC_1510820.jpg vs ISIC_9237549.jpg: 0.8022
# ISIC_1510820.jpg vs ISIC_9237660.jpg: 0.7956
# ISIC_1510820.jpg vs ISIC_9239212.jpg: 0.8073
# ISIC_1510820.jpg vs ISIC_9240897.jpg: 0.7812
# ISIC_1510820.jpg vs ISIC_9241147.jpg: 0.7944
# ISIC_1510820.jpg vs ISIC_9242278.jpg: 0.5997
# ISIC_1510820.jpg vs ISIC_9242758.jpg: 0.6687
# ISIC_1510820.jpg vs ISIC_9243223.jpg: 0.8809
# ISIC_1510820.jpg vs ISIC_9243861.jpg: 0.8742
# ISIC_1510820.jpg vs ISIC_9246598.jpg: 0.8499
# ISIC_1510820.jpg vs ISIC_9249558.jpg: 0.8044
# ISIC_1510820.jpg vs ISIC_9250246.jpg: 0.8515
# ISIC_1510820.jpg vs ISIC_9251197.jpg: 0.5958
# ISIC_1510820.jpg vs ISIC_9252742.jpg: 0.8293
# ISIC_1510820.jpg vs ISIC_9254796.jpg: 0.4180
# ISIC_1510820.jpg vs ISIC_9258565.jpg: 0.8158
# ISIC_1510820.jpg vs ISIC_9259972.jpg: 0.7087
# ISIC_1510820.jpg vs ISIC_9262161.jpg: 0.5085
# ISIC_1510820.jpg vs ISIC_9265002.jpg: 0.8489
# ISIC_1510820.jpg vs ISIC_9265385.jpg: 0.7380
# ISIC_1510820.jpg vs ISIC_9269637.jpg: 0.7011
# ISIC_1510820.jpg vs ISIC_9272530.jpg: 0.7326
# ISIC_1510820.jpg vs ISIC_9272888.jpg: 0.4574
# ISIC_1510820.jpg vs ISIC_9273234.jpg: 0.4135
# ISIC_1510820.jpg vs ISIC_9274269.jpg: 0.5609
# ISIC_1510820.jpg vs ISIC_9275700.jpg: 0.7355
# ISIC_1510820.jpg vs ISIC_9277130.jpg: 0.8090
# ISIC_1510820.jpg vs ISIC_9280090.jpg: 0.6890
# ISIC_1510820.jpg vs ISIC_9280462.jpg: 0.5646
# ISIC_1510820.jpg vs ISIC_9280822.jpg: 0.5197
# ISIC_1510820.jpg vs ISIC_9282097.jpg: 0.7693
# ISIC_1510820.jpg vs ISIC_9286333.jpg: 0.6210
# ISIC_1510820.jpg vs ISIC_9286553.jpg: 0.5460
# ISIC_1510820.jpg vs ISIC_9286642.jpg: 0.8112
# ISIC_1510820.jpg vs ISIC_9286788.jpg: 0.7180
# ISIC_1510820.jpg vs ISIC_9287804.jpg: 0.8709
# ISIC_1510820.jpg vs ISIC_9289075.jpg: 0.8430
# ISIC_1510820.jpg vs ISIC_9289090.jpg: 0.8080
# ISIC_1510820.jpg vs ISIC_9289197.jpg: 0.8582
# ISIC_1510820.jpg vs ISIC_9291596.jpg: 0.2879
# ISIC_1510820.jpg vs ISIC_9291828.jpg: 0.4280
# ISIC_1510820.jpg vs ISIC_9295328.jpg: 0.8541
# ISIC_1510820.jpg vs ISIC_9296170.jpg: 0.8262
# ISIC_1510820.jpg vs ISIC_9299316.jpg: 0.3026
# ISIC_1510820.jpg vs ISIC_9303631.jpg: 0.6895
# ISIC_1510820.jpg vs ISIC_9306623.jpg: 0.2236
# ISIC_1510820.jpg vs ISIC_9306935.jpg: 0.7399
# ISIC_1510820.jpg vs ISIC_9307335.jpg: 0.6026
# ISIC_1510820.jpg vs ISIC_9307630.jpg: 0.6705
# ISIC_1510820.jpg vs ISIC_9308141.jpg: 0.7323
# ISIC_1510820.jpg vs ISIC_9308692.jpg: 0.5254
# ISIC_1510820.jpg vs ISIC_9308958.jpg: 0.7651
# ISIC_1510820.jpg vs ISIC_9309879.jpg: 0.5415
# ISIC_1510820.jpg vs ISIC_9310576.jpg: 0.5970
# ISIC_1510820.jpg vs ISIC_9310591.jpg: 0.3532
# ISIC_1510820.jpg vs ISIC_9312065.jpg: 0.2059
# ISIC_1510820.jpg vs ISIC_9313535.jpg: 0.4893
# ISIC_1510820.jpg vs ISIC_9317637.jpg: 0.5285
# ISIC_1510820.jpg vs ISIC_9318490.jpg: 0.3159
# ISIC_1510820.jpg vs ISIC_9318565.jpg: 0.8607
# ISIC_1510820.jpg vs ISIC_9320994.jpg: 0.7300
# ISIC_1510820.jpg vs ISIC_9322605.jpg: 0.2413
# ISIC_1510820.jpg vs ISIC_9323866.jpg: 0.8666
# ISIC_1510820.jpg vs ISIC_9325387.jpg: 0.1927
# ISIC_1510820.jpg vs ISIC_9326230.jpg: 0.7865
# ISIC_1510820.jpg vs ISIC_9327290.jpg: 0.6881
# ISIC_1510820.jpg vs ISIC_9332396.jpg: 0.2272
# ISIC_1510820.jpg vs ISIC_9335018.jpg: 0.7973
# ISIC_1510820.jpg vs ISIC_9336332.jpg: 0.2714
# ISIC_1510820.jpg vs ISIC_9337556.jpg: 0.9226
# ISIC_1510820.jpg vs ISIC_9340537.jpg: 0.4137
# ISIC_1510820.jpg vs ISIC_9341159.jpg: 0.4530
# ISIC_1510820.jpg vs ISIC_9341631.jpg: 0.2339
# ISIC_1510820.jpg vs ISIC_9341718.jpg: 0.8387
# ISIC_1510820.jpg vs ISIC_9342419.jpg: 0.4684
# ISIC_1510820.jpg vs ISIC_9342809.jpg: 0.3544
# ISIC_1510820.jpg vs ISIC_9343272.jpg: 0.3822
# ISIC_1510820.jpg vs ISIC_9345094.jpg: 0.7630
# ISIC_1510820.jpg vs ISIC_9346529.jpg: 0.7136
# ISIC_1510820.jpg vs ISIC_9347323.jpg: 0.3948
# ISIC_1510820.jpg vs ISIC_9350250.jpg: 0.8907
# ISIC_1510820.jpg vs ISIC_9351693.jpg: 0.8430
# ISIC_1510820.jpg vs ISIC_9353360.jpg: 0.1512
# ISIC_1510820.jpg vs ISIC_9354346.jpg: 0.5346
# ISIC_1510820.jpg vs ISIC_9358108.jpg: 0.8606
# ISIC_1510820.jpg vs ISIC_9358483.jpg: 0.6228
# ISIC_1510820.jpg vs ISIC_9359141.jpg: 0.6976
# ISIC_1510820.jpg vs ISIC_9362634.jpg: 0.6352
# ISIC_1510820.jpg vs ISIC_9363884.jpg: 0.8651
# ISIC_1510820.jpg vs ISIC_9364759.jpg: 0.5962
# ISIC_1510820.jpg vs ISIC_9367956.jpg: 0.3342
# ISIC_1510820.jpg vs ISIC_9368385.jpg: 0.1215
# ISIC_1510820.jpg vs ISIC_9369230.jpg: 0.5203
# ISIC_1510820.jpg vs ISIC_9369302.jpg: 0.8406
# ISIC_1510820.jpg vs ISIC_9372129.jpg: 0.6967
# ISIC_1510820.jpg vs ISIC_9372173.jpg: 0.4985
# ISIC_1510820.jpg vs ISIC_9374581.jpg: 0.3785
# ISIC_1510820.jpg vs ISIC_9374689.jpg: 0.8996
# ISIC_1510820.jpg vs ISIC_9376658.jpg: 0.1325
# ISIC_1510820.jpg vs ISIC_9381475.jpg: 0.5380
# ISIC_1510820.jpg vs ISIC_9381492.jpg: 0.6950
# ISIC_1510820.jpg vs ISIC_9381773.jpg: 0.7832
# ISIC_1510820.jpg vs ISIC_9381977.jpg: 0.8308
# ISIC_1510820.jpg vs ISIC_9383016.jpg: 0.6699
# ISIC_1510820.jpg vs ISIC_9388406.jpg: 0.4875
# ISIC_1510820.jpg vs ISIC_9390336.jpg: 0.8847
# ISIC_1510820.jpg vs ISIC_9390440.jpg: 0.8631
# ISIC_1510820.jpg vs ISIC_9391523.jpg: 0.7837
# ISIC_1510820.jpg vs ISIC_9391983.jpg: 0.4563
# ISIC_1510820.jpg vs ISIC_9393653.jpg: 0.5088
# ISIC_1510820.jpg vs ISIC_9398826.jpg: 0.2279
# ISIC_1510820.jpg vs ISIC_9400341.jpg: 0.6935
# ISIC_1510820.jpg vs ISIC_9400786.jpg: 0.2961
# ISIC_1510820.jpg vs ISIC_9404212.jpg: 0.7367
# ISIC_1510820.jpg vs ISIC_9405195.jpg: 0.8558
# ISIC_1510820.jpg vs ISIC_9405386.jpg: 0.7266
# ISIC_1510820.jpg vs ISIC_9407352.jpg: 0.7847
# ISIC_1510820.jpg vs ISIC_9411047.jpg: 0.6761
# ISIC_1510820.jpg vs ISIC_9411778.jpg: 0.6077
# ISIC_1510820.jpg vs ISIC_9411850.jpg: 0.8836
# ISIC_1510820.jpg vs ISIC_9414296.jpg: 0.6428
# ISIC_1510820.jpg vs ISIC_9415087.jpg: 0.7916
# ISIC_1510820.jpg vs ISIC_9415347.jpg: 0.8539
# ISIC_1510820.jpg vs ISIC_9416647.jpg: 0.5290
# ISIC_1510820.jpg vs ISIC_9416839.jpg: 0.5477
# ISIC_1510820.jpg vs ISIC_9419168.jpg: 0.7659
# ISIC_1510820.jpg vs ISIC_9420623.jpg: 0.8194
# ISIC_1510820.jpg vs ISIC_9422657.jpg: 0.7533
# ISIC_1510820.jpg vs ISIC_9424112.jpg: 0.8568
# ISIC_1510820.jpg vs ISIC_9424470.jpg: 0.3510
# ISIC_1510820.jpg vs ISIC_9425285.jpg: 0.7832
# ISIC_1510820.jpg vs ISIC_9425501.jpg: 0.3555
# ISIC_1510820.jpg vs ISIC_9428732.jpg: 0.7507
# ISIC_1510820.jpg vs ISIC_9429943.jpg: 0.3145
# ISIC_1510820.jpg vs ISIC_9430171.jpg: 0.6636
# ISIC_1510820.jpg vs ISIC_9431076.jpg: 0.5723
# ISIC_1510820.jpg vs ISIC_9431437.jpg: 0.8115
# ISIC_1510820.jpg vs ISIC_9432288.jpg: 0.4386
# ISIC_1510820.jpg vs ISIC_9433769.jpg: 0.8445
# ISIC_1510820.jpg vs ISIC_9434018.jpg: 0.7937
# ISIC_1510820.jpg vs ISIC_9436771.jpg: 0.7820
# ISIC_1510820.jpg vs ISIC_9438854.jpg: 0.7623
# ISIC_1510820.jpg vs ISIC_9439231.jpg: 0.4995
# ISIC_1510820.jpg vs ISIC_9439244.jpg: 0.2032
# ISIC_1510820.jpg vs ISIC_9439908.jpg: 0.3859
# ISIC_1510820.jpg vs ISIC_9440851.jpg: 0.6175
# ISIC_1510820.jpg vs ISIC_9440902.jpg: 0.1452
# ISIC_1510820.jpg vs ISIC_9443862.jpg: 0.7890
# ISIC_1510820.jpg vs ISIC_9448856.jpg: 0.8261
# ISIC_1510820.jpg vs ISIC_9450189.jpg: 0.5094
# ISIC_1510820.jpg vs ISIC_9451161.jpg: 0.8559
# ISIC_1510820.jpg vs ISIC_9454520.jpg: 0.6525
# ISIC_1510820.jpg vs ISIC_9454633.jpg: 0.7824
# ISIC_1510820.jpg vs ISIC_9455202.jpg: 0.6317
# ISIC_1510820.jpg vs ISIC_9455280.jpg: 0.4048
# ISIC_1510820.jpg vs ISIC_9456779.jpg: 0.2389
# ISIC_1510820.jpg vs ISIC_9458416.jpg: 0.5903
# ISIC_1510820.jpg vs ISIC_9458964.jpg: 0.4597
# ISIC_1510820.jpg vs ISIC_9462823.jpg: 0.6463
# ISIC_1510820.jpg vs ISIC_9464286.jpg: 0.5244
# ISIC_1510820.jpg vs ISIC_9465938.jpg: 0.7850
# ISIC_1510820.jpg vs ISIC_9467877.jpg: 0.8500
# ISIC_1510820.jpg vs ISIC_9471010.jpg: 0.8545
# ISIC_1510820.jpg vs ISIC_9471866.jpg: 0.8187
# ISIC_1510820.jpg vs ISIC_9472514.jpg: 0.6949
# ISIC_1510820.jpg vs ISIC_9472708.jpg: 0.7648
# ISIC_1510820.jpg vs ISIC_9472856.jpg: 0.7445
# ISIC_1510820.jpg vs ISIC_9473192.jpg: 0.8137
# ISIC_1510820.jpg vs ISIC_9473581.jpg: 0.7782
# ISIC_1510820.jpg vs ISIC_9474385.jpg: 0.8980
# ISIC_1510820.jpg vs ISIC_9476269.jpg: 0.7626
# ISIC_1510820.jpg vs ISIC_9476938.jpg: 0.8232
# ISIC_1510820.jpg vs ISIC_9478088.jpg: 0.2257
# ISIC_1510820.jpg vs ISIC_9480050.jpg: 0.7158
# ISIC_1510820.jpg vs ISIC_9480897.jpg: 0.6653
# ISIC_1510820.jpg vs ISIC_9483222.jpg: 0.3098
# ISIC_1510820.jpg vs ISIC_9484827.jpg: 0.4482
# ISIC_1510820.jpg vs ISIC_9486077.jpg: 0.8528
# ISIC_1510820.jpg vs ISIC_9487545.jpg: 0.3967
# ISIC_1510820.jpg vs ISIC_9490517.jpg: 0.4070
# ISIC_1510820.jpg vs ISIC_9495534.jpg: 0.1282
# ISIC_1510820.jpg vs ISIC_9496437.jpg: 0.8155
# ISIC_1510820.jpg vs ISIC_9497933.jpg: 0.4712
# ISIC_1510820.jpg vs ISIC_9498081.jpg: 0.7090
# ISIC_1510820.jpg vs ISIC_9499578.jpg: 0.3824
# ISIC_1510820.jpg vs ISIC_9502233.jpg: 0.8131
# ISIC_1510820.jpg vs ISIC_9502846.jpg: 0.5212
# ISIC_1510820.jpg vs ISIC_9503665.jpg: 0.3014
# ISIC_1510820.jpg vs ISIC_9504668.jpg: 0.7849
# ISIC_1510820.jpg vs ISIC_9505721.jpg: 0.7865
# ISIC_1510820.jpg vs ISIC_9506089.jpg: 0.5736
# ISIC_1510820.jpg vs ISIC_9508507.jpg: 0.7360
# ISIC_1510820.jpg vs ISIC_9508816.jpg: 0.7587
# ISIC_1510820.jpg vs ISIC_9509950.jpg: 0.7759
# ISIC_1510820.jpg vs ISIC_9511982.jpg: 0.3430
# ISIC_1510820.jpg vs ISIC_9512059.jpg: 0.8273
# ISIC_1510820.jpg vs ISIC_9515324.jpg: 0.2500
# ISIC_1510820.jpg vs ISIC_9516799.jpg: 0.8309
# ISIC_1510820.jpg vs ISIC_9519092.jpg: 0.8286
# ISIC_1510820.jpg vs ISIC_9519270.jpg: 0.2443
# ISIC_1510820.jpg vs ISIC_9520761.jpg: 0.7494
# ISIC_1510820.jpg vs ISIC_9521732.jpg: 0.7284
# ISIC_1510820.jpg vs ISIC_9522771.jpg: 0.6510
# ISIC_1510820.jpg vs ISIC_9523755.jpg: 0.8605
# ISIC_1510820.jpg vs ISIC_9526357.jpg: 0.8858
# ISIC_1510820.jpg vs ISIC_9528336.jpg: 0.8337
# ISIC_1510820.jpg vs ISIC_9528425.jpg: 0.8251
# ISIC_1510820.jpg vs ISIC_9531727.jpg: 0.8088
# ISIC_1510820.jpg vs ISIC_9532428.jpg: 0.7126
# ISIC_1510820.jpg vs ISIC_9537778.jpg: 0.9028
# ISIC_1510820.jpg vs ISIC_9539153.jpg: 0.5317
# ISIC_1510820.jpg vs ISIC_9539564.jpg: 0.6247
# ISIC_1510820.jpg vs ISIC_9540894.jpg: 0.8978
# ISIC_1510820.jpg vs ISIC_9543891.jpg: 0.8048
# ISIC_1510820.jpg vs ISIC_9544045.jpg: 0.8591
# ISIC_1510820.jpg vs ISIC_9545152.jpg: 0.7388
# ISIC_1510820.jpg vs ISIC_9547337.jpg: 0.7548
# ISIC_1510820.jpg vs ISIC_9550350.jpg: 0.8052
# ISIC_1510820.jpg vs ISIC_9552129.jpg: 0.3919
# ISIC_1510820.jpg vs ISIC_9552986.jpg: 0.4621
# ISIC_1510820.jpg vs ISIC_9553857.jpg: 0.7876
# ISIC_1510820.jpg vs ISIC_9554478.jpg: 0.9124
# ISIC_1510820.jpg vs ISIC_9554637.jpg: 0.8348
# ISIC_1510820.jpg vs ISIC_9555049.jpg: 0.8760
# ISIC_1510820.jpg vs ISIC_9555226.jpg: 0.6498
# ISIC_1510820.jpg vs ISIC_9557584.jpg: 0.4594
# ISIC_1510820.jpg vs ISIC_9557753.jpg: 0.6499
# ISIC_1510820.jpg vs ISIC_9558797.jpg: 0.4096
# ISIC_1510820.jpg vs ISIC_9558823.jpg: 0.7774
# ISIC_1510820.jpg vs ISIC_9559734.jpg: 0.4663
# ISIC_1510820.jpg vs ISIC_9562557.jpg: 0.7623
# ISIC_1510820.jpg vs ISIC_9563612.jpg: 0.3597
# ISIC_1510820.jpg vs ISIC_9565009.jpg: 0.5890
# ISIC_1510820.jpg vs ISIC_9565522.jpg: 0.1140
# ISIC_1510820.jpg vs ISIC_9565834.jpg: 0.7999
# ISIC_1510820.jpg vs ISIC_9566153.jpg: 0.7199
# ISIC_1510820.jpg vs ISIC_9566781.jpg: 0.7281
# ISIC_1510820.jpg vs ISIC_9566869.jpg: 0.7883
# ISIC_1510820.jpg vs ISIC_9566995.jpg: 0.4098
# ISIC_1510820.jpg vs ISIC_9567086.jpg: 0.7533
# ISIC_1510820.jpg vs ISIC_9568052.jpg: 0.1841
# ISIC_1510820.jpg vs ISIC_9570510.jpg: 0.3985
# ISIC_1510820.jpg vs ISIC_9572182.jpg: 0.8166
# ISIC_1510820.jpg vs ISIC_9572206.jpg: 0.8047
# ISIC_1510820.jpg vs ISIC_9575100.jpg: 0.3510
# ISIC_1510820.jpg vs ISIC_9576017.jpg: 0.6759
# ISIC_1510820.jpg vs ISIC_9576872.jpg: 0.1389
# ISIC_1510820.jpg vs ISIC_9576983.jpg: 0.7306
# ISIC_1510820.jpg vs ISIC_9578201.jpg: 0.7708
# ISIC_1510820.jpg vs ISIC_9581438.jpg: 0.7226
# ISIC_1510820.jpg vs ISIC_9582219.jpg: 0.8127
# ISIC_1510820.jpg vs ISIC_9582272.jpg: 0.3045
# ISIC_1510820.jpg vs ISIC_9585872.jpg: 0.7477
# ISIC_1510820.jpg vs ISIC_9586848.jpg: 0.8798
# ISIC_1510820.jpg vs ISIC_9586945.jpg: 0.8710
# ISIC_1510820.jpg vs ISIC_9593129.jpg: 0.5826
# ISIC_1510820.jpg vs ISIC_9594594.jpg: 0.2688
# ISIC_1510820.jpg vs ISIC_9595936.jpg: 0.7593
# ISIC_1510820.jpg vs ISIC_9597559.jpg: 0.6635
# ISIC_1510820.jpg vs ISIC_9597570.jpg: 0.5506
# ISIC_1510820.jpg vs ISIC_9598408.jpg: 0.0945
# ISIC_1510820.jpg vs ISIC_9598745.jpg: 0.5154
# ISIC_1510820.jpg vs ISIC_9601410.jpg: 0.8389
# ISIC_1510820.jpg vs ISIC_9603620.jpg: 0.5916
# ISIC_1510820.jpg vs ISIC_9604611.jpg: 0.6517
# ISIC_1510820.jpg vs ISIC_9605223.jpg: 0.7260
# ISIC_1510820.jpg vs ISIC_9610143.jpg: 0.5928
# ISIC_1510820.jpg vs ISIC_9610738.jpg: 0.5983
# ISIC_1510820.jpg vs ISIC_9610783.jpg: 0.8253
# ISIC_1510820.jpg vs ISIC_9610931.jpg: 0.3804
# ISIC_1510820.jpg vs ISIC_9610986.jpg: 0.6065
# ISIC_1510820.jpg vs ISIC_9611096.jpg: 0.4727
# ISIC_1510820.jpg vs ISIC_9612009.jpg: 0.3818
# ISIC_1510820.jpg vs ISIC_9612128.jpg: 0.8192
# ISIC_1510820.jpg vs ISIC_9613406.jpg: 0.7304
# ISIC_1510820.jpg vs ISIC_9614732.jpg: 0.7942
# ISIC_1510820.jpg vs ISIC_9615309.jpg: 0.7623
# ISIC_1510820.jpg vs ISIC_9622423.jpg: 0.6692
# ISIC_1510820.jpg vs ISIC_9622814.jpg: 0.6805
# ISIC_1510820.jpg vs ISIC_9624902.jpg: 0.6550
# ISIC_1510820.jpg vs ISIC_9626060.jpg: 0.7730
# ISIC_1510820.jpg vs ISIC_9626654.jpg: 0.7124
# ISIC_1510820.jpg vs ISIC_9630631.jpg: 0.7046
# ISIC_1510820.jpg vs ISIC_9632466.jpg: 0.7114
# ISIC_1510820.jpg vs ISIC_9633578.jpg: 0.7321
# ISIC_1510820.jpg vs ISIC_9634301.jpg: 0.2948
# ISIC_1510820.jpg vs ISIC_9634338.jpg: 0.7117
# ISIC_1510820.jpg vs ISIC_9634605.jpg: 0.4015
# ISIC_1510820.jpg vs ISIC_9635714.jpg: 0.3158
# ISIC_1510820.jpg vs ISIC_9636985.jpg: 0.6119
# ISIC_1510820.jpg vs ISIC_9638086.jpg: 0.8188
# ISIC_1510820.jpg vs ISIC_9638464.jpg: 0.4798
# ISIC_1510820.jpg vs ISIC_9640394.jpg: 0.3046
# ISIC_1510820.jpg vs ISIC_9641213.jpg: 0.7526
# ISIC_1510820.jpg vs ISIC_9643133.jpg: 0.5920
# ISIC_1510820.jpg vs ISIC_9643232.jpg: 0.8213
# ISIC_1510820.jpg vs ISIC_9647724.jpg: 0.7096
# ISIC_1510820.jpg vs ISIC_9648810.jpg: 0.8493
# ISIC_1510820.jpg vs ISIC_9649315.jpg: 0.8603
# ISIC_1510820.jpg vs ISIC_9649614.jpg: 0.8431
# ISIC_1510820.jpg vs ISIC_9650416.jpg: 0.8635
# ISIC_1510820.jpg vs ISIC_9650954.jpg: 0.3989
# ISIC_1510820.jpg vs ISIC_9652852.jpg: 0.6361
# ISIC_1510820.jpg vs ISIC_9654373.jpg: 0.7512
# ISIC_1510820.jpg vs ISIC_9656902.jpg: 0.7325
# ISIC_1510820.jpg vs ISIC_9657093.jpg: 0.3828
# ISIC_1510820.jpg vs ISIC_9661048.jpg: 0.7596
# ISIC_1510820.jpg vs ISIC_9661408.jpg: 0.6588
# ISIC_1510820.jpg vs ISIC_9661960.jpg: 0.6491
# ISIC_1510820.jpg vs ISIC_9662365.jpg: 0.5341
# ISIC_1510820.jpg vs ISIC_9662621.jpg: 0.4840
# ISIC_1510820.jpg vs ISIC_9664555.jpg: 0.8919
# ISIC_1510820.jpg vs ISIC_9668427.jpg: 0.5651
# ISIC_1510820.jpg vs ISIC_9668890.jpg: 0.6829
# ISIC_1510820.jpg vs ISIC_9669609.jpg: 0.7523
# ISIC_1510820.jpg vs ISIC_9669857.jpg: 0.8668
# ISIC_1510820.jpg vs ISIC_9671653.jpg: 0.4400
# ISIC_1510820.jpg vs ISIC_9672112.jpg: 0.8897
# ISIC_1510820.jpg vs ISIC_9676047.jpg: 0.5166
# ISIC_1510820.jpg vs ISIC_9676644.jpg: 0.5322
# ISIC_1510820.jpg vs ISIC_9677559.jpg: 0.7079
# ISIC_1510820.jpg vs ISIC_9677935.jpg: 0.4826
# ISIC_1510820.jpg vs ISIC_9679360.jpg: 0.8532
# ISIC_1510820.jpg vs ISIC_9681414.jpg: 0.6140
# ISIC_1510820.jpg vs ISIC_9681634.jpg: 0.7161
# ISIC_1510820.jpg vs ISIC_9684090.jpg: 0.5978
# ISIC_1510820.jpg vs ISIC_9684470.jpg: 0.6663
# ISIC_1510820.jpg vs ISIC_9685490.jpg: 0.6435
# ISIC_1510820.jpg vs ISIC_9686337.jpg: 0.7042
# ISIC_1510820.jpg vs ISIC_9688400.jpg: 0.6141
# ISIC_1510820.jpg vs ISIC_9690651.jpg: 0.5564
# ISIC_1510820.jpg vs ISIC_9691681.jpg: 0.5759
# ISIC_1510820.jpg vs ISIC_9695660.jpg: 0.4955
# ISIC_1510820.jpg vs ISIC_9697692.jpg: 0.8368
# ISIC_1510820.jpg vs ISIC_9700357.jpg: 0.6175
# ISIC_1510820.jpg vs ISIC_9700398.jpg: 0.8292
# ISIC_1510820.jpg vs ISIC_9702840.jpg: 0.6803
# ISIC_1510820.jpg vs ISIC_9703604.jpg: 0.5451
# ISIC_1510820.jpg vs ISIC_9705000.jpg: 0.4096
# ISIC_1510820.jpg vs ISIC_9705679.jpg: 0.8596
# ISIC_1510820.jpg vs ISIC_9705839.jpg: 0.7819
# ISIC_1510820.jpg vs ISIC_9708816.jpg: 0.7575
# ISIC_1510820.jpg vs ISIC_9708917.jpg: 0.6933
# ISIC_1510820.jpg vs ISIC_9710645.jpg: 0.8984
# ISIC_1510820.jpg vs ISIC_9711522.jpg: 0.7508
# ISIC_1510820.jpg vs ISIC_9715108.jpg: 0.7706
# ISIC_1510820.jpg vs ISIC_9715397.jpg: 0.7599
# ISIC_1510820.jpg vs ISIC_9716246.jpg: 0.8687
# ISIC_1510820.jpg vs ISIC_9718279.jpg: 0.8765
# ISIC_1510820.jpg vs ISIC_9718744.jpg: 0.7759
# ISIC_1510820.jpg vs ISIC_9719597.jpg: 0.5488
# ISIC_1510820.jpg vs ISIC_9721106.jpg: 0.8960
# ISIC_1510820.jpg vs ISIC_9722578.jpg: 0.4192
# ISIC_1510820.jpg vs ISIC_9722875.jpg: 0.8417
# ISIC_1510820.jpg vs ISIC_9723994.jpg: 0.8753
# ISIC_1510820.jpg vs ISIC_9729042.jpg: 0.1405
# ISIC_1510820.jpg vs ISIC_9729470.jpg: 0.7584
# ISIC_1510820.jpg vs ISIC_9729657.jpg: 0.3714
# ISIC_1510820.jpg vs ISIC_9730582.jpg: 0.7736
# ISIC_1510820.jpg vs ISIC_9730892.jpg: 0.7689
# ISIC_1510820.jpg vs ISIC_9731733.jpg: 0.7332
# ISIC_1510820.jpg vs ISIC_9732518.jpg: 0.7949
# ISIC_1510820.jpg vs ISIC_9733455.jpg: 0.6415
# ISIC_1510820.jpg vs ISIC_9734753.jpg: 0.3819
# ISIC_1510820.jpg vs ISIC_9735478.jpg: 0.1337
# ISIC_1510820.jpg vs ISIC_9737755.jpg: 0.4141
# ISIC_1510820.jpg vs ISIC_9737799.jpg: 0.3223
# ISIC_1510820.jpg vs ISIC_9737914.jpg: 0.7001
# ISIC_1510820.jpg vs ISIC_9738859.jpg: 0.5071
# ISIC_1510820.jpg vs ISIC_9739400.jpg: 0.8784
# ISIC_1510820.jpg vs ISIC_9740139.jpg: 0.8043
# ISIC_1510820.jpg vs ISIC_9740435.jpg: 0.5564
# ISIC_1510820.jpg vs ISIC_9741760.jpg: 0.8202
# ISIC_1510820.jpg vs ISIC_9742031.jpg: 0.8046
# ISIC_1510820.jpg vs ISIC_9742572.jpg: 0.5013
# ISIC_1510820.jpg vs ISIC_9743234.jpg: 0.7782
# ISIC_1510820.jpg vs ISIC_9744808.jpg: 0.5558
# ISIC_1510820.jpg vs ISIC_9746585.jpg: 0.4158
# ISIC_1510820.jpg vs ISIC_9747254.jpg: 0.1868
# ISIC_1510820.jpg vs ISIC_9750982.jpg: 0.7412
# ISIC_1510820.jpg vs ISIC_9751225.jpg: 0.2217
# ISIC_1510820.jpg vs ISIC_9751559.jpg: 0.3613
# ISIC_1510820.jpg vs ISIC_9751666.jpg: 0.2630
# ISIC_1510820.jpg vs ISIC_9752088.jpg: 0.4084
# ISIC_1510820.jpg vs ISIC_9752704.jpg: 0.6414
# ISIC_1510820.jpg vs ISIC_9752812.jpg: 0.8627
# ISIC_1510820.jpg vs ISIC_9754437.jpg: 0.7037
# ISIC_1510820.jpg vs ISIC_9756144.jpg: 0.7681
# ISIC_1510820.jpg vs ISIC_9757672.jpg: 0.7247
# ISIC_1510820.jpg vs ISIC_9758068.jpg: 0.5685
# ISIC_1510820.jpg vs ISIC_9760396.jpg: 0.5772
# ISIC_1510820.jpg vs ISIC_9760502.jpg: 0.7801
# ISIC_1510820.jpg vs ISIC_9760777.jpg: 0.7316
# ISIC_1510820.jpg vs ISIC_9762076.jpg: 0.7122
# ISIC_1510820.jpg vs ISIC_9762786.jpg: 0.6010
# ISIC_1510820.jpg vs ISIC_9764424.jpg: 0.7022
# ISIC_1510820.jpg vs ISIC_9765963.jpg: 0.5571
# ISIC_1510820.jpg vs ISIC_9767158.jpg: 0.8198
# ISIC_1510820.jpg vs ISIC_9769594.jpg: 0.4387
# ISIC_1510820.jpg vs ISIC_9769723.jpg: 0.7229
# ISIC_1510820.jpg vs ISIC_9771418.jpg: 0.4967
# ISIC_1510820.jpg vs ISIC_9771565.jpg: 0.3066
# ISIC_1510820.jpg vs ISIC_9774915.jpg: 0.4238
# ISIC_1510820.jpg vs ISIC_9775373.jpg: 0.5321
# ISIC_1510820.jpg vs ISIC_9776988.jpg: 0.7316
# ISIC_1510820.jpg vs ISIC_9777743.jpg: 0.8396
# ISIC_1510820.jpg vs ISIC_9780160.jpg: 0.6220
# ISIC_1510820.jpg vs ISIC_9780161.jpg: 0.8465
# ISIC_1510820.jpg vs ISIC_9780991.jpg: 0.8016
# ISIC_1510820.jpg vs ISIC_9781447.jpg: 0.8909
# ISIC_1510820.jpg vs ISIC_9781886.jpg: 0.8031
# ISIC_1510820.jpg vs ISIC_9782592.jpg: 0.7273
# ISIC_1510820.jpg vs ISIC_9783961.jpg: 0.8595
# ISIC_1510820.jpg vs ISIC_9785592.jpg: 0.8587
# ISIC_1510820.jpg vs ISIC_9786035.jpg: 0.5933
# ISIC_1510820.jpg vs ISIC_9786628.jpg: 0.2737
# ISIC_1510820.jpg vs ISIC_9787314.jpg: 0.1486
# ISIC_1510820.jpg vs ISIC_9787513.jpg: 0.3870
# ISIC_1510820.jpg vs ISIC_9788810.jpg: 0.6319
# ISIC_1510820.jpg vs ISIC_9789324.jpg: 0.3847
# ISIC_1510820.jpg vs ISIC_9790934.jpg: 0.7899
# ISIC_1510820.jpg vs ISIC_9792406.jpg: 0.3833
# ISIC_1510820.jpg vs ISIC_9793946.jpg: 0.7481
# ISIC_1510820.jpg vs ISIC_9797245.jpg: 0.7260
# ISIC_1510820.jpg vs ISIC_9798583.jpg: 0.7860
# ISIC_1510820.jpg vs ISIC_9802201.jpg: 0.4385
# ISIC_1510820.jpg vs ISIC_9805149.jpg: 0.7178
# ISIC_1510820.jpg vs ISIC_9806516.jpg: 0.8417
# ISIC_1510820.jpg vs ISIC_9806811.jpg: 0.6180
# ISIC_1510820.jpg vs ISIC_9808390.jpg: 0.8576
# ISIC_1510820.jpg vs ISIC_9808824(1).jpg: 0.4195
# ISIC_1510820.jpg vs ISIC_9808824.jpg: 0.4195
# ISIC_1510820.jpg vs ISIC_9812833(1).jpg: 0.8216
# ISIC_1510820.jpg vs ISIC_9812833.jpg: 0.8216
# ISIC_1510820.jpg vs ISIC_9813807(1).jpg: 0.8186
# ISIC_1510820.jpg vs ISIC_9813807.jpg: 0.8186
# ISIC_1510820.jpg vs ISIC_9814124.jpg: 0.2382
# ISIC_1510820.jpg vs ISIC_9814134.jpg: 0.1803
# ISIC_1510820.jpg vs ISIC_9816232.jpg: 0.7745
# ISIC_1510820.jpg vs ISIC_9819477(1).jpg: 0.7910
# ISIC_1510820.jpg vs ISIC_9819477.jpg: 0.7910
# ISIC_1510820.jpg vs ISIC_9821590.jpg: 0.6738
# ISIC_1510820.jpg vs ISIC_9821912.jpg: 0.6710
# ISIC_1510820.jpg vs ISIC_9828652.jpg: 0.4519
# ISIC_1510820.jpg vs ISIC_9829355(1).jpg: 0.6422
# ISIC_1510820.jpg vs ISIC_9829355.jpg: 0.6422
# ISIC_1510820.jpg vs ISIC_9830086.jpg: 0.6305
# ISIC_1510820.jpg vs ISIC_9835307.jpg: 0.8543
# ISIC_1510820.jpg vs ISIC_9838627.jpg: 0.7100
# ISIC_1510820.jpg vs ISIC_9839024.jpg: 0.2988
# ISIC_1510820.jpg vs ISIC_9839094.jpg: 0.3107
# ISIC_1510820.jpg vs ISIC_9839916.jpg: 0.7158
# ISIC_1510820.jpg vs ISIC_9842982.jpg: 0.2014
# ISIC_1510820.jpg vs ISIC_9843071.jpg: 0.7148
# ISIC_1510820.jpg vs ISIC_9843964.jpg: 0.7217
# ISIC_1510820.jpg vs ISIC_9843995.jpg: 0.5818
# ISIC_1510820.jpg vs ISIC_9844548.jpg: 0.8794
# ISIC_1510820.jpg vs ISIC_9845561.jpg: 0.8997
# ISIC_1510820.jpg vs ISIC_9845867.jpg: 0.1418
# ISIC_1510820.jpg vs ISIC_9846010.jpg: 0.3886
# ISIC_1510820.jpg vs ISIC_9846074.jpg: 0.8101
# ISIC_1510820.jpg vs ISIC_9846473.jpg: 0.3046
# ISIC_1510820.jpg vs ISIC_9848140.jpg: 0.6864
# ISIC_1510820.jpg vs ISIC_9849474.jpg: 0.8789
# ISIC_1510820.jpg vs ISIC_9849915.jpg: 0.8841
# ISIC_1510820.jpg vs ISIC_9850411.jpg: 0.2796
# ISIC_1510820.jpg vs ISIC_9850663.jpg: 0.5650
# ISIC_1510820.jpg vs ISIC_9851570.jpg: 0.8022
# ISIC_1510820.jpg vs ISIC_9852671.jpg: 0.6962
# ISIC_1510820.jpg vs ISIC_9853298.jpg: 0.8224
# ISIC_1510820.jpg vs ISIC_9854060.jpg: 0.8759
# ISIC_1510820.jpg vs ISIC_9855157.jpg: 0.7631
# ISIC_1510820.jpg vs ISIC_9858719.jpg: 0.7365
# ISIC_1510820.jpg vs ISIC_9862104.jpg: 0.6439
# ISIC_1510820.jpg vs ISIC_9866896.jpg: 0.4255
# ISIC_1510820.jpg vs ISIC_9868655.jpg: 0.8491
# ISIC_1510820.jpg vs ISIC_9870231.jpg: 0.4815
# ISIC_1510820.jpg vs ISIC_9871669.jpg: 0.5447
# ISIC_1510820.jpg vs ISIC_9873208.jpg: 0.8304
# ISIC_1510820.jpg vs ISIC_9875614.jpg: 0.5421
# ISIC_1510820.jpg vs ISIC_9877452.jpg: 0.4825
# ISIC_1510820.jpg vs ISIC_9882513.jpg: 0.6861
# ISIC_1510820.jpg vs ISIC_9882710.jpg: 0.4033
# ISIC_1510820.jpg vs ISIC_9887854.jpg: 0.5616
# ISIC_1510820.jpg vs ISIC_9890091.jpg: 0.8727
# ISIC_1510820.jpg vs ISIC_9895171.jpg: 0.5138
# ISIC_1510820.jpg vs ISIC_9897022.jpg: 0.5923
# ISIC_1510820.jpg vs ISIC_9899809.jpg: 0.6197
# ISIC_1510820.jpg vs ISIC_9903236.jpg: 0.7394
# ISIC_1510820.jpg vs ISIC_9910831.jpg: 0.8478
# ISIC_1510820.jpg vs ISIC_9911183.jpg: 0.7822
# ISIC_1510820.jpg vs ISIC_9911568.jpg: 0.8250
# ISIC_1510820.jpg vs ISIC_9911882.jpg: 0.6914
# ISIC_1510820.jpg vs ISIC_9912997.jpg: 0.7470
# ISIC_1510820.jpg vs ISIC_9916252.jpg: 0.4773
# ISIC_1510820.jpg vs ISIC_9916306.jpg: 0.7732
# ISIC_1510820.jpg vs ISIC_9918495.jpg: 0.2633
# ISIC_1510820.jpg vs ISIC_9921046.jpg: 0.7115
# ISIC_1510820.jpg vs ISIC_9922988.jpg: 0.7663
# ISIC_1510820.jpg vs ISIC_9925177.jpg: 0.4655
# ISIC_1510820.jpg vs ISIC_9926395.jpg: 0.8013
# ISIC_1510820.jpg vs ISIC_9927447.jpg: 0.7278
# ISIC_1510820.jpg vs ISIC_9927724.jpg: 0.5144
# ISIC_1510820.jpg vs ISIC_9927852.jpg: 0.6237
# ISIC_1510820.jpg vs ISIC_9928214.jpg: 0.5891
# ISIC_1510820.jpg vs ISIC_9928802.jpg: 0.6246
# ISIC_1510820.jpg vs ISIC_9929302.jpg: 0.7505
# ISIC_1510820.jpg vs ISIC_9930189.jpg: 0.6669
# ISIC_1510820.jpg vs ISIC_9930607.jpg: 0.7210
# ISIC_1510820.jpg vs ISIC_9931481.jpg: 0.4684
# ISIC_1510820.jpg vs ISIC_9932650.jpg: 0.7995
# ISIC_1510820.jpg vs ISIC_9932714.jpg: 0.3493
# ISIC_1510820.jpg vs ISIC_9935338.jpg: 0.8341
# ISIC_1510820.jpg vs ISIC_9936277.jpg: 0.7746
# ISIC_1510820.jpg vs ISIC_9937461.jpg: 0.1947
# ISIC_1510820.jpg vs ISIC_9937768.jpg: 0.4372
# ISIC_1510820.jpg vs ISIC_9938017.jpg: 0.1025
# ISIC_1510820.jpg vs ISIC_9940478.jpg: 0.5788
# ISIC_1510820.jpg vs ISIC_9941030.jpg: 0.7951
# ISIC_1510820.jpg vs ISIC_9941424.jpg: 0.3959
# ISIC_1510820.jpg vs ISIC_9941666.jpg: 0.8399
# ISIC_1510820.jpg vs ISIC_9944611.jpg: 0.7987
# ISIC_1510820.jpg vs ISIC_9951368.jpg: 0.8524
# ISIC_1510820.jpg vs ISIC_9952355.jpg: 0.1705
# ISIC_1510820.jpg vs ISIC_9952818.jpg: 0.6410
# ISIC_1510820.jpg vs ISIC_9955410.jpg: 0.1544
# ISIC_1510820.jpg vs ISIC_9956468.jpg: 0.8049
# ISIC_1510820.jpg vs ISIC_9957341.jpg: 0.7513
# ISIC_1510820.jpg vs ISIC_9957368.jpg: 0.2322
# ISIC_1510820.jpg vs ISIC_9960485.jpg: 0.6386
# ISIC_1510820.jpg vs ISIC_9960583.jpg: 0.1729
# ISIC_1510820.jpg vs ISIC_9960678.jpg: 0.8771
# ISIC_1510820.jpg vs ISIC_9960693.jpg: 0.7249
# ISIC_1510820.jpg vs ISIC_9961370.jpg: 0.6059
# ISIC_1510820.jpg vs ISIC_9962990.jpg: 0.4707
# ISIC_1510820.jpg vs ISIC_9963289.jpg: 0.5869
# ISIC_1510820.jpg vs ISIC_9967412.jpg: 0.8749
# ISIC_1510820.jpg vs ISIC_9968023.jpg: 0.7698
# ISIC_1510820.jpg vs ISIC_9968928.jpg: 0.4422
# ISIC_1510820.jpg vs ISIC_9969308.jpg: 0.6424
# ISIC_1510820.jpg vs ISIC_9969800.jpg: 0.8247
# ISIC_1510820.jpg vs ISIC_9972828.jpg: 0.2024
# ISIC_1510820.jpg vs ISIC_9975957.jpg: 0.7513
# ISIC_1510820.jpg vs ISIC_9976054.jpg: 0.7507
# ISIC_1510820.jpg vs ISIC_9976538.jpg: 0.6439
# ISIC_1510820.jpg vs ISIC_9978346.jpg: 0.2588
# ISIC_1510820.jpg vs ISIC_9979369.jpg: 0.3631
# ISIC_1510820.jpg vs ISIC_9979673.jpg: 0.7848
# ISIC_1510820.jpg vs ISIC_9980216.jpg: 0.4124
# ISIC_1510820.jpg vs ISIC_9980453.jpg: 0.7491
# ISIC_1510820.jpg vs ISIC_9982814.jpg: 0.7430
# ISIC_1510820.jpg vs ISIC_9983015.jpg: 0.3750
# ISIC_1510820.jpg vs ISIC_9983147.jpg: 0.7561
# ISIC_1510820.jpg vs ISIC_9983668.jpg: 0.8396
# ISIC_1510820.jpg vs ISIC_9986023.jpg: 0.7258
# ISIC_1510820.jpg vs ISIC_9987035.jpg: 0.3664
# ISIC_1510820.jpg vs ISIC_9987545.jpg: 0.5831
# ISIC_1510820.jpg vs ISIC_9989537.jpg: 0.4363
# ISIC_1510820.jpg vs ISIC_9990202.jpg: 0.6422
# ISIC_1510820.jpg vs ISIC_9990629.jpg: 0.7843
# ISIC_1510820.jpg vs ISIC_9991958.jpg: 0.8143
# ISIC_1510820.jpg vs ISIC_9992226.jpg: 0.4674
# ISIC_1510820.jpg vs ISIC_9992485.jpg: 0.8537
# ISIC_1510820.jpg vs ISIC_9996992.jpg: 0.5666
# ISIC_1510820.jpg vs ISIC_9997917.jpg: 0.2108
# ISIC_1510820.jpg vs ISIC_9999302.jpg: 0.5945
# ✅ Saved to similarity_results_reference.csv

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/siam2/similarity_results_reference.csv")

plt.figure(figsize=(8, 5))
sns.histplot(df["Similarity"], bins=30, kde=True)
plt.title("Distribution of Similarity Scores")
plt.xlabel("Similarity Score")
plt.ylabel("Frequency")
plt.show()


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

top_n = 10
top_matches = df.sort_values("Similarity", ascending=False).head(top_n)

fig, axes = plt.subplots(1, top_n+1, figsize=(20, 4))

# Show reference image
axes[0].imshow(Image.open(ref_path))
axes[0].set_title("Reference")
axes[0].axis("off")

# Show similar images
for i, row in enumerate(top_matches.itertuples(), start=1):
    axes[i].imshow(Image.open(os.path.join(img_dir, row.Image2)))
    axes[i].set_title(f"{row.Similarity:.2f}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
!python3 train_siam.py | tee training_log.txt


In [ ]:
# /usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
#   warnings.warn(
# /usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
#   warnings.warn(msg)
# Using device: cuda
# Epoch 2: Train Loss=0.1424, Train Acc=0.706, Val Loss=0.1542, Val Acc=0.687
# 💾 Model saved at epoch 1
# ⏱ 49.2s
#
# Epoch 3: Train Loss=0.0644, Train Acc=0.898, Val Loss=0.1967, Val Acc=0.709
# 💾 Model saved at epoch 2
# ⏱ 49.2s
#
# Epoch 4: Train Loss=0.0394, Train Acc=0.948, Val Loss=0.2216, Val Acc=0.644
# ⏱ 49.4s
#
# Epoch 5: Train Loss=0.0382, Train Acc=0.952, Val Loss=0.2326, Val Acc=0.639
# ⏱ 48.7s
#
# Epoch 6: Train Loss=0.0235, Train Acc=0.971, Val Loss=0.2816, Val Acc=0.624
# ⏱ 49.0s
#
# Epoch 7: Train Loss=0.0179, Train Acc=0.977, Val Loss=0.3083, Val Acc=0.611
# ⏱ 49.2s
#
# Epoch 8: Train Loss=0.0196, Train Acc=0.973, Val Loss=0.2984, Val Acc=0.639
# ⏱ 48.5s
#
# Epoch 9: Train Loss=0.0241, Train Acc=0.971, Val Loss=0.4065, Val Acc=0.585
# ⏱ 49.1s
#
# Epoch 10: Train Loss=0.0190, Train Acc=0.978, Val Loss=0.2904, Val Acc=0.663
# ⏱ 49.0s
#
# Epoch 11: Train Loss=0.0152, Train Acc=0.983, Val Loss=0.3866, Val Acc=0.591
# ⏱ 49.5s
#
# Epoch 12: Train Loss=0.0104, Train Acc=0.990, Val Loss=0.3180, Val Acc=0.632
# ⏱ 48.9s
#
# Epoch 13: Train Loss=0.0205, Train Acc=0.976, Val Loss=0.2530, Val Acc=0.601
# ⏱ 49.0s
#
# Epoch 14: Train Loss=0.0128, Train Acc=0.986, Val Loss=0.3139, Val Acc=0.634
# ⏱ 49.1s
#
# Epoch 15: Train Loss=0.0156, Train Acc=0.982, Val Loss=0.3348, Val Acc=0.587
# ⏱ 49.2s
#
# Epoch 16: Train Loss=0.0176, Train Acc=0.979, Val Loss=0.3193, Val Acc=0.615
# ⏱ 48.6s
#
# ✅ Training complete! Best val accuracy: 0.709

In [ ]:
%cd /content/drive/MyDrive/siam2
%run train_siam.py

In [ ]:
# /content/drive/MyDrive/siam2
# /usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
#   warnings.warn(
# /usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
#   warnings.warn(msg)
# Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
# 100%|██████████| 44.7M/44.7M [00:00<00:00, 252MB/s]
# Epoch 1: train_acc=0.523, val_acc=0.607
# ⏱ 84.4s
# Epoch 2: train_acc=0.699, val_acc=0.879
# ⏱ 80.1s
# Epoch 3: train_acc=0.778, val_acc=0.914
# ⏱ 80.4s
# Epoch 4: train_acc=0.824, val_acc=0.993
# ⏱ 80.3s
# Epoch 5: train_acc=0.875, val_acc=1.000
# ⏱ 80.9s
# Epoch 6: train_acc=0.915, val_acc=1.000
# ⏱ 81.6s
# Epoch 7: train_acc=0.932, val_acc=0.993
# ⏱ 80.3s
# Epoch 8: train_acc=0.966, val_acc=1.000
# ⏱ 82.4s
# Epoch 9: train_acc=0.949, val_acc=1.000
# ⏱ 81.4s
# Epoch 10: train_acc=1.000, val_acc=1.000
# ⏱ 79.9s
# Epoch 11: train_acc=0.966, val_acc=1.000
# ⏱ 79.6s
# Epoch 12: train_acc=0.972, val_acc=1.000
# ⏱ 79.6s
# Epoch 13: train_acc=0.994, val_acc=1.000
# ⏱ 79.5s
# Epoch 14: train_acc=0.994, val_acc=1.000
# ⏱ 79.3s
# Epoch 15: train_acc=1.000, val_acc=1.000
# ⏱ 79.5s
# ✅ Best val accuracy: 1.000

In [ ]:
!python3 train_siam.py


In [ ]:
# Using device: cuda
# /usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
#   warnings.warn(
# /usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
#   warnings.warn(msg)
# Epoch 2: Train Loss=0.1256, Train Acc=0.750, Val Loss=0.0838, Val Acc=0.866
# 💾 Model saved at epoch 1
# ⏱ 70.2s
#
# Epoch 3: Train Loss=0.0572, Train Acc=0.916, Val Loss=0.0419, Val Acc=0.943
# 💾 Model saved at epoch 2
# ⏱ 67.3s
#
# Epoch 4: Train Loss=0.0352, Train Acc=0.952, Val Loss=0.0258, Val Acc=0.964
# 💾 Model saved at epoch 3
# ⏱ 67.1s
#
# Epoch 5: Train Loss=0.0232, Train Acc=0.970, Val Loss=0.0185, Val Acc=0.976
# 💾 Model saved at epoch 4
# ⏱ 67.5s
#
# Epoch 6: Train Loss=0.0168, Train Acc=0.981, Val Loss=0.0195, Val Acc=0.979
# 💾 Model saved at epoch 5
# ⏱ 68.2s
#
# Epoch 7: Train Loss=0.0218, Train Acc=0.972, Val Loss=0.0410, Val Acc=0.950
# ⏱ 67.7s
#
# Epoch 8: Train Loss=0.0174, Train Acc=0.979, Val Loss=0.0129, Val Acc=0.988
# 💾 Model saved at epoch 7
# ⏱ 67.3s
#
# Epoch 9: Train Loss=0.0088, Train Acc=0.990, Val Loss=0.0161, Val Acc=0.985
# ⏱ 67.4s
#
# Epoch 10: Train Loss=0.0206, Train Acc=0.971, Val Loss=0.0241, Val Acc=0.974
# ⏱ 66.9s
#
# Epoch 11: Train Loss=0.0121, Train Acc=0.984, Val Loss=0.0189, Val Acc=0.988
# ⏱ 66.8s
#
# Epoch 12: Train Loss=0.0031, Train Acc=0.997, Val Loss=0.0129, Val Acc=0.989
# 💾 Model saved at epoch 11
# ⏱ 67.5s
#
# Epoch 13: Train Loss=0.0028, Train Acc=0.998, Val Loss=0.0181, Val Acc=0.984
# ⏱ 68.1s
#
# Epoch 14: Train Loss=0.0124, Train Acc=0.985, Val Loss=0.0419, Val Acc=0.947
# ⏱ 67.1s
#
# Epoch 15: Train Loss=0.0138, Train Acc=0.981, Val Loss=0.0402, Val Acc=0.946
# ⏱ 67.2s
#
# Epoch 16: Train Loss=0.0112, Train Acc=0.988, Val Loss=0.0149, Val Acc=0.989
# 💾 Model saved at epoch 15
# ⏱ 67.3s
#
# ✅ Training complete! Best val accuracy: 0.989

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from train_siam import PairDataset, validate, FocalLoss
from model_siam import SiameseNetwork
import random

# -----------------------
# Benchmarking function
# -----------------------
def benchmark_model(model_path, test_pkl, batch_size=16, device="cpu", similarity_threshold=0.5):
    """
    Loads a trained Siamese model and evaluates it against the test set,
    compares with a random baseline, and reports improvement.
    """
    # Load model
    model = SiameseNetwork()
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    # Load test dataset
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])
    test_ds = PairDataset(test_pkl, transform)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    # Loss function (not strictly needed for benchmarking)
    loss_fn = FocalLoss()

    # Compute accuracy
    _, test_acc = validate(model, test_loader, loss_fn, device)
    print(f"✅ Model Accuracy: {test_acc*100:.2f}%")

    # -----------------------
    # Random baseline
    # -----------------------
    all_labels = [label.item() for _, _, label in test_ds]
    shuffled_labels = all_labels.copy()
    random.shuffle(shuffled_labels)
    correct_random = sum([a==b for a,b in zip(all_labels, shuffled_labels)])
    baseline_acc = correct_random / len(all_labels)
    print(f"🎯 Random baseline: {baseline_acc*100:.2f}%")

    # Improvement
    improvement = test_acc - baseline_acc
    print(f"📈 Improvement over random baseline: {improvement*100:.2f}%")

    return test_acc, baseline_acc, improvement


# -----------------------
# Example usage
# -----------------------
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_path = "/content/drive/MyDrive/siam2/siamese_best.pth"
    test_pkl = "/content/drive/MyDrive/siam2/siamese_isic_val.pkl"

    benchmark_model(model_path, test_pkl, batch_size=16, device=device)


✅ Model Accuracy: 70.90%
🎯 Random baseline: 51.00%
📈 Improvement over random baseline: 19.90%